# 《Stochastic Interpolants: A Unifying Framework for Flows and Diffusions》论文笔记

本笔记围绕随机插值框架，介绍概率路径、速度场、Score、ODE 与 SDE 的关系，并讨论训练采样算法、实验结果及其与 Flow Matching、Diffusion 等方法的联系。正文按章节展开，数学基础和详细公式推导见附录。

**正文顺序：** [第 0 章](#chapter-0) · [第 1 章](#chapter-1) · [第 2 章](#chapter-2) · [第 3 章](#chapter-3) · [第 4 章](#chapter-4) · [第 5 章](#chapter-5) · [第 6 章](#chapter-6) · [第 7 章](#chapter-7) · [第 8 章](#chapter-8) · [第 9 章](#chapter-9)

**数学推导导航：**

| 数学主题 | 推导位置 |
|---|---|
| MSE 最优性与条件期望 | [A.2.2](#app-a-2-2) |
| 独立高斯噪声合并 | [A.3.3](#app-a-3-3) |
| 连续性方程 | [B.2.2](#app-b-2-2) |
| Score 修正与 ODE/SDE 同边缘分布 | [B.3.3](#app-b-3-3) |
| ODE 的 likelihood | [B.2.4](#app-b-2-4) |
| 布朗桥方差与跨时间协方差 | [C.1.4](#app-c-1-4) |
| SDE 时间反转与符号约定 | [C.3](#app-c-3-2) |
| Girsanov 与终点 KL | [C.4](#app-c-4)、[A.4.5](#app-a-4-5) |
| 高斯噪声预测与 Score | [D.1.3](#app-d-1-3) |
| Tweedie 公式与去噪 | [D.2](#app-d-2) |
| 从 SDE 恢复 Probability Flow ODE | [D.3](#app-d-3) |
| 单侧线性插值的速度与三种预测目标转换 | [F.1](#app-f-1) |
| SDE 误差拆分与最优扩散强度 | [F.2](#app-f-2) |
| OU / VP / VE 的解析计算 | [F.3](#app-f-3) |
| CondOT 与 VP 的系数代入 | [F.4](#app-f-4) |
| Mirror 的速度与噪声日程 | [F.5](#app-f-5) |

---

<a id="chapter-0"></a>

# 第 0 章：论文概述与核心结论

**论文名称：** *Stochastic Interpolants: A Unifying Framework for Flows and Diffusions*

**论文链接：** https://arxiv.org/abs/2303.08797

本文研究一个生成模型中的基础问题：

> 能否建立一个统一的数学框架，使确定性的 Flow 模型与随机性的 Diffusion 模型描述同一条概率分布演化路径，并且能够在两种生成方式之间自由选择？

论文提出的答案是 **Stochastic Interpolants（随机插值）**。

它的核心思想是：**先人为构造连接两个概率分布的随机插值路径，再从这条概率路径推导出对应的 ODE 和 SDE，而不是先规定动力学，再研究它产生的概率分布。**


## 0.1 研究背景：为什么需要统一 Flow 与 Diffusion


### 1. 生成模型的本质是什么？

生成模型希望学习一个目标概率分布。

假设真实数据满足：

$$
X_1\sim \rho_1(x)
\tag{0.1}
$$

其中：

- $X_1\in\mathbb R^d$：真实数据，例如图像、轨迹等。
- $\rho_1(x)$：真实数据的概率密度。
- $d$：数据的维度。

我们希望从一个容易采样的初始分布出发，例如标准高斯：

$$
X_0\sim \rho_0=\mathcal N(0,I_d)
\tag{0.2}
$$

通过某种生成过程，将初始分布逐渐变换为真实数据分布：

$$
\boxed{\rho_0\longrightarrow\rho_1}
\tag{0.3}
$$

这里需要特别区分两个概念：

**样本轨迹**描述一个具体样本如何随时间变化：

$$
X_0\longrightarrow X_t\longrightarrow X_1
$$

**概率路径**描述所有样本在每个时刻形成的整体分布：

$$
\rho_0\longrightarrow\rho_t\longrightarrow\rho_1
$$

生成模型最终要匹配的是目标概率分布，而不是要求每条生成轨迹都等于某条预先指定的真实轨迹。

因此，**不同的样本运动方式，可以对应相同的概率路径。**

这正是本文能够统一 Flow 与 Diffusion 的关键。


### 2. Flow：通过确定性 ODE 生成数据

Flow 类生成模型可以使用常微分方程（ODE）描述样本运动：

$$
\boxed{
\frac{dY_t}{dt}=b(t,Y_t)
}
\tag{0.4}
$$

其中：

- $Y_t$：生成过程在时刻 $t$ 的状态。
- $b(t,x)$：时间相关的速度场。
- $t\in[0,1]$：生成时间。

给定初始状态 $Y_0$，在适当的正则性条件下，ODE 决定一条确定性的轨迹。

例如：

$$
Y_1=Y_0+\int_0^1 b(t,Y_t)\,dt
\tag{0.5}
$$

这里的速度场可以由神经网络预测：

$$
b_\theta(t,x)\approx b(t,x)
$$

但我们真正关心的不是单条轨迹，而是 ODE 对概率分布的作用。

如果 $Y_t$ 的概率密度为 $\rho_t$，那么它满足**连续性方程**：

$$
\boxed{
\partial_t\rho_t(x)
+
\nabla\cdot\left(\rho_t(x)b(t,x)\right)=0
}
\tag{0.6}
$$

这个方程表示概率质量守恒。

它告诉我们：速度场 $b$ 不仅决定单个样本如何运动，还决定整体概率分布如何演化。

连续性方程将在第 2 章解释，完整推导见[附录 B.2.2](#app-b-2-2)。


### 3. Diffusion：通过随机 SDE 生成数据

与 Flow 不同，Diffusion 使用随机微分方程（SDE）描述样本运动。

考虑一个一般形式：

$$
\boxed{
dY_t=f(t,Y_t)\,dt+g(t)\,dW_t
}
\tag{0.7}
$$

其中：

- $f(t,x)$：漂移项（Drift），决定样本的平均运动方向。
- $g(t)$：扩散系数，控制随机扰动强度。
- $W_t$：标准布朗运动。
- $dW_t$：布朗运动的随机增量。

与 ODE 不同，即使给定相同的初始状态，SDE 也可能产生不同的随机轨迹。

SDE 对应的概率密度演化由 **Fokker–Planck 方程**描述：

$$
\boxed{
\partial_t\rho_t
=
-\nabla\cdot(f\rho_t)
+
\frac12g^2(t)\Delta\rho_t
}
\tag{0.8}
$$

与连续性方程相比，多出了：

$$
\frac12g^2(t)\Delta\rho_t
$$

这一项描述随机噪声对概率密度的扩散作用。

这里暂时不需要掌握 Fokker–Planck 方程的证明。第 2 章将使用这一方程；从 Itô 公式出发的完整证明见附录 C.2.6。


### 4. Flow 与 Diffusion 的共同点

虽然 ODE 与 SDE 的样本运动方式不同，但它们都在描述概率分布随时间的变化。

| 比较维度 | Flow / ODE | Diffusion / SDE |
|---|---|---|
| 样本运动 | 确定性 | 随机性 |
| 动力学 | 速度场 | 漂移场 + 扩散项 |
| 密度演化 | 连续性方程 | Fokker–Planck 方程 |
| 生成过程 | 数值积分 | 随机数值积分 |
| 核心目标 | 从初始分布生成目标分布 | 从初始分布生成目标分布 |

因此，一个自然的问题是：

**如果我们事先知道想要的概率路径 $\rho_t$，是否可以同时构造出产生这条概率路径的 ODE 和 SDE？**

这就是随机插值框架研究的核心问题之一。

---


## 0.2 随机插值的核心思想


### 1. 从动力学出发，转变为从概率路径出发

传统的建模思路通常是：

$$
\text{定义 ODE/SDE}
\longrightarrow
\text{得到概率路径}
$$

而随机插值采用另一种思路：

$$
\boxed{
\text{构造随机插值}
\longrightarrow
\text{得到概率路径}
\longrightarrow
\text{推导 ODE/SDE}
}
\tag{0.9}
$$

这个区别非常重要。

它意味着，我们可以先设计数据在中间时刻的分布，再研究应该使用什么动力学来生成这些分布。


### 2. 随机插值的数学定义

考虑两个随机变量：

$$
X_0\sim\rho_0,\qquad X_1\sim\rho_1
$$

定义：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{0.10}
$$

其中：

- $I(t,X_0,X_1)$：连接两个端点的插值函数。
- $\gamma(t)$：中间高斯噪声的幅度。
- $Z\sim\mathcal N(0,I_d)$：独立于端点的标准高斯噪声。

要求：

$$
I(0,X_0,X_1)=X_0
$$

$$
I(1,X_0,X_1)=X_1
$$

以及：

$$
\gamma(0)=\gamma(1)=0
\tag{0.11}
$$

于是：

$$
X_{t=0}=X_0
$$

$$
X_{t=1}=X_1
$$

因此：

$$
\boxed{
\rho_{t=0}=\rho_0,\qquad
\rho_{t=1}=\rho_1
}
\tag{0.12}
$$

这说明随机插值可以在有限时间内连接两个给定的概率分布。

**需要注意：** 两个端点可以具有任意给定的联合分布，只要其边缘分别是 $\rho_0,\rho_1$。这个联合分布称为 Coupling（耦合）。


### 3. 一个具体例子

选择最简单的线性插值：

$$
I(t,X_0,X_1)=(1-t)X_0+tX_1
\tag{0.13}
$$

再选择：

$$
\gamma(t)=\sqrt{a\,t(1-t)},\qquad a>0
\tag{0.14}
$$

得到：

$$
\boxed{
X_t=(1-t)X_0+tX_1+\sqrt{a\,t(1-t)}Z
}
\tag{0.15}
$$

观察三个时刻。

当 $t=0$：

$$
X_0=X_0
$$

当 $t=\frac12$：

$$
X_{1/2}
=
\frac12X_0+\frac12X_1+\frac{\sqrt a}{2}Z
$$

当 $t=1$：

$$
X_1=X_1
$$

也就是说，插值过程在中间时刻引入高斯扰动，但不会改变端点分布。

通过改变 $I$ 和 $\gamma$，我们可以构造不同的中间概率路径。


### 4. 从随机插值恢复生成动力学

论文接下来证明：

随机插值诱导的概率路径 $\rho_t$ 满足连续性方程：

$$
\partial_t\rho_t+\nabla\cdot(b\rho_t)=0
\tag{0.16}
$$

其中速度场为：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{0.17}
$$

这里的 $\dot X_t$ 是原始随机插值样本对时间的导数。

再定义：

$$
\boxed{
s(t,x)=\nabla_x\log\rho_t(x)
}
\tag{0.18}
$$

即概率密度的 Score。

那么，可以构造：

**Probability Flow ODE：**

$$
\boxed{
dY_t=b(t,Y_t)\,dt
}
\tag{0.19}
$$

**前向 SDE：**

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}\,dW_t
}
\tag{0.20}
$$

**后向 SDE：**

沿原时间变量 $t$ 从 $1$ 向 $0$ 积分，采用后向 Itô 约定：

$$
\boxed{
dY_t=
[b(t,Y_t)-\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}\,d\overleftarrow W_t
}
\tag{0.21}
$$

其中 $\varepsilon(t)\ge0$ 是可以选择的扩散强度。

在相应正则性和方程适定性条件下，这些过程具有相同的单时刻边缘分布 $\rho_t$。

因此：

$$
\boxed{
\text{同一条概率路径}
\quad\Longrightarrow\quad
\text{ODE 与一族 SDE}
}
\tag{0.22}
$$

这就是论文最核心的统一思想。

---


## 0.3 论文的主要理论结论

本节先总结论文的主要结果，具体证明留到对应章节。


### 结论一：随机插值可以连接任意给定的端点分布

通过：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

只要满足端点条件，就能保证：

$$
\rho_{t=0}=\rho_0,\qquad \rho_{t=1}=\rho_1
$$

因此不必先求解某个动力学方程，才能构造连接两个分布的概率路径。


### 结论二：随机插值自然诱导速度场和连续性方程

定义：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
\tag{0.23}
$$

就有：

$$
\partial_t\rho_t+\nabla\cdot(b\rho_t)=0
$$

这是从随机插值连接到 ODE 的数学基础。

需要注意，$b$ 是给定当前状态后的条件平均速度，不是某条随机插值轨迹本身的速度。


### 结论三：速度场与 Score 可以通过回归学习

由于：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

根据条件期望的最小均方误差性质：

$$
\boxed{
b=
\arg\min_f
\mathbb E\left[\|f(t,X_t)-\dot X_t\|^2\right]
}
\tag{0.24}
$$

这里的最小化对可测函数 $f$ 进行，并对时间和样本取期望。

类似地，利用高斯噪声的条件期望可以得到：

$$
\boxed{
s(t,x)
=
-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
}
\tag{0.25}
$$

其中要求 $\gamma(t)>0$。

因此速度场和 Score 都可以转化为神经网络的监督学习问题。


### 结论四：ODE 与 SDE 可以描述相同的概率路径

这是论文最重要的理论结果之一。

给定 $b$ 和 $s$，选择任意满足条件的 $\varepsilon(t)$，可以构造：

$$
dY_t=
[b+\varepsilon s]dt+\sqrt{2\varepsilon}\,dW_t
$$

它与 ODE：

$$
dY_t=b\,dt
$$

可以具有相同的单时刻边缘分布。

**但两者的样本轨迹一般不同。**

这意味着：

- 概率路径由随机插值决定。
- 生成动力学可以选择 ODE 或 SDE。
- 扩散强度提供了额外的采样自由度。


### 结论五：SDE 的生成误差可以通过 KL 进行控制

论文使用 Girsanov 定理研究 SDE 的误差。

当两个 SDE 具有相同扩散系数，并满足所需条件时，其路径分布之间的 KL 可以由漂移误差的平方积分表示。

再利用数据处理不等式，可以控制最终生成分布的 KL。

这使得速度场和 Score 的学习误差能够与生成质量建立理论联系。


### 结论六：单侧线性插值可以只使用一个预测器

对于：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{0.26}
$$

定义噪声条件均值：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

那么：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
}
\tag{0.27}
$$

同时：

$$
\boxed{
b(t,x)=
\frac{\dot\beta(t)}{\beta(t)}x+
\left(
\dot\alpha(t)-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right)\eta_z(t,x)
}
\tag{0.28}
$$

其中要求相关分母非零。

这说明：

**只需要一个噪声预测器，就可以同时恢复边缘速度场和 Score。**

因此，单侧线性随机插值可以用一个神经网络支持 ODE 与 SDE 生成。

这也是我们后面统一 CondOT、VP-SDE 等方法的重要基础。


### 结论七：随机插值为多种生成方法提供统一视角

论文进一步研究了随机插值与以下方法的关系：

- Score-Based Diffusion
- Denoising
- Rectified Flow / Reflow
- Schrödinger Bridge

它们并非完全相同的算法，而是可以在概率路径、速度场、Score 或随机控制等层面建立数学联系。

特别需要区分：**统一的数学框架不意味着这些方法的训练目标、数值算法和实际性能完全等价。**

---


## 0.4 全文数学框架与逻辑关系

现在把前面的结论串起来。

首先，选择两个端点分布：

$$
\rho_0,\rho_1
$$

构造随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

由此得到一条边缘概率路径：

$$
\{\rho_t\}_{t\in[0,1]}
$$

接下来，计算两个核心数学对象：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

$$
\boxed{
s(t,x)=\nabla_x\log\rho_t(x)
}
$$

其中：

- $b$ 描述概率质量的整体运输。
- $s$ 描述当前概率密度的对数梯度。

然后，利用这两个场构造生成动力学：

$$
\boxed{
\begin{aligned}
\text{ODE:}\quad
&dY_t=b\,dt\\[4pt]
\text{Forward SDE:}\quad
&dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}\,dW_t\\[4pt]
\text{Backward SDE:}\quad
&dY_t=(b-\varepsilon s)dt+\sqrt{2\varepsilon}\,d\overleftarrow W_t
\end{aligned}
}
\tag{0.29}
$$

最后，通过监督学习得到 $b_\theta,s_\theta$，使用数值积分完成生成。

全文的逻辑可以概括为：

$$
\boxed{
\begin{gathered}
\text{端点分布与耦合}\\
\Downarrow\\
\text{随机插值 }X_t\\
\Downarrow\\
\text{边缘概率路径 }\rho_t\\
\Downarrow\\
\text{速度场 }b\text{ 与 Score }s\\
\Downarrow\\
\text{ODE / SDE 生成动力学}\\
\Downarrow\\
\text{神经网络学习与数值采样}
\end{gathered}
}
\tag{0.30}
$$

**理解这条逻辑链，是理解整篇论文的关键。**

第 2 章将严格证明这条链中的每一步；第 3～5 章讨论特殊插值及其与其他方法的联系；第 6～7 章讨论训练、采样和实验；第 9 章则利用单侧线性插值，把不同生成模型代入统一公式。

---


<a id="chapter-1"></a>

# 第 1 章：Introduction

第 0 章已经总结了论文的核心结论。本章回到作者的研究思路，重点回答三个问题：

1. 为什么已有的 Flow 和 Diffusion 方法还需要一个统一框架？
2. 随机插值相比已有方法解决了哪些问题？
3. 本文的理论贡献究竟是什么？

这一章主要解释研究动机，数学定理的严格证明从第 2 章开始。


## 1.1 ODE 与 SDE 生成模型的研究背景


### 1. 基于确定性运输的生成模型

早期 Normalizing Flow 主要研究如何构造一个可逆映射：

$$
Y_1=T(Y_0)
\tag{1.1}
$$

使得：

$$
Y_0\sim\rho_0
\quad\Longrightarrow\quad
Y_1\sim\rho_1
$$

其中 $T$ 是从初始分布到目标分布的运输映射。

连续时间 Normalizing Flow 则使用 ODE 定义这一映射：

$$
\frac{dY_t}{dt}=b_\theta(t,Y_t)
\tag{1.2}
$$

它具有两个重要特点。

首先，给定初始状态，生成轨迹是确定性的。

其次，在满足适当条件时，可以通过连续变量替换公式计算概率密度：

$$
\frac{d}{dt}\log\rho_t(Y_t)
=
-\nabla\cdot b_\theta(t,Y_t)
\tag{1.3}
$$

这使得连续 Flow 适合进行似然估计。

但传统基于最大似然的连续 Flow 训练通常需要求解 ODE，并对求解过程进行梯度计算，因此训练成本较高。

作者希望保留 ODE 的优势，同时避免训练阶段必须反复求解生成动力学。


### 2. 基于随机扩散的生成模型

Score-Based Diffusion 则采用不同思路。

先通过正向扩散过程，将真实数据逐渐转换为简单的噪声分布。

论文重点讨论的一个例子是 Ornstein–Uhlenbeck（OU）过程：

$$
\boxed{
dY_\tau=-Y_\tau\,d\tau+\sqrt{2}\,dW_\tau
}
\tag{1.4}
$$

其中 $\tau$ 表示正向加噪时间。

如果初始状态满足：

$$
Y_0\sim\rho_{\mathrm{data}}
$$

则 OU 过程的解析解为：

$$
\boxed{
Y_\tau
\overset d=
e^{-\tau}Y_0+
\sqrt{1-e^{-2\tau}}Z
}
\tag{1.5}
$$

其中 $Z\sim\mathcal N(0,I_d)$ 独立于 $Y_0$。

当 $\tau\to\infty$ 时：

$$
e^{-\tau}\to0
$$

$$
\sqrt{1-e^{-2\tau}}\to1
$$

因此：

$$
Y_\tau\overset d\longrightarrow\mathcal N(0,I_d)
\tag{1.6}
$$

也就是说，OU 过程能够逐渐把任意初始数据分布转变为标准高斯分布。

接下来，通过学习时间相关的 Score：

$$
s(\tau,x)=\nabla_x\log\rho_\tau(x)
\tag{1.7}
$$

就可以构造反向 SDE，从高斯噪声恢复数据分布。

**这一思路的重要性在于：正向加噪过程容易构造，反向生成过程则可以通过 Score 学习获得。**

不过，论文指出，OU 过程精确达到高斯平衡分布需要无限时间，实际生成模型只能截断这一过程。

这里的“无限时间”是针对 OU 的精确高斯极限，并不是说所有扩散模型都无法在有限时间连接两个分布。


### 3. ODE 与 SDE 并非完全独立

Score-Based Diffusion 的一个重要发现是：

**同一条扩散概率路径，也可以由确定性的 Probability Flow ODE 描述。**

也就是说，扩散模型不一定只能使用随机 SDE 进行生成。

反过来，本文希望研究更一般的问题：

> 如果不从特定的扩散 SDE 出发，而是直接指定一条连接两个分布的概率路径，能否系统地构造对应的 ODE 和 SDE？

这使得研究对象从某一种具体生成算法，转变为更一般的**概率分布运输问题**。

---


## 1.2 现有方法的问题与论文的研究动机

作者主要关注以下几个问题。


### 1. 对初始分布和扩散过程的依赖

传统 Score-Based Diffusion 通常选择高斯作为噪声分布，并采用预先设计的加噪过程。

例如 OU 过程：

$$
dY_\tau=-Y_\tau d\tau+\sqrt2\,dW_\tau
$$

虽然这种选择非常有效，但从数学上看，生成模型并不一定需要高斯起点，也不一定需要 OU 动力学。

更一般地，我们可能希望连接：

$$
\rho_0\longrightarrow\rho_1
$$

其中两个分布都是任意给定的概率分布。

因此，论文希望建立一个不依赖特定高斯加噪过程的生成框架。


### 2. 有限时间精确连接两个分布

对于 OU 扩散，精确的高斯平衡分布在无限时间极限下获得。

随机插值则直接构造：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

并通过端点约束保证：

$$
X_{t=0}=X_0,\qquad X_{t=1}=X_1
$$

因此可以在有限时间区间 $[0,1]$ 内精确连接两个给定的端点分布。

这里的**精确**指理论随机插值的端点分布完全正确。

实际使用神经网络近似速度场或 Score，再进行有限步数值积分时，仍然会存在训练误差和离散化误差。


### 3. 概率路径与生成动力学没有充分解耦

这是作者最重要的研究动机。

在传统扩散模型中，通常先指定正向 SDE：

$$
dY_t=f(t,Y_t)dt+g(t)dW_t
$$

然后由它决定概率路径：

$$
\{\rho_t\}_{t\in[0,1]}
$$

因此，概率路径的设计与随机动力学的选择紧密联系。

但从概率分布演化的角度看，这两个问题实际上可以分开：

**第一个问题：希望数据经过什么样的中间分布？**

即选择：

$$
\rho_0\longrightarrow\rho_t\longrightarrow\rho_1
$$

**第二个问题：采用什么动力学实现这条概率路径？**

可以选择：

$$
\text{ODE}
$$

也可以选择：

$$
\text{SDE}
$$

甚至可以选择不同扩散强度的 SDE。

随机插值框架将这两个问题分离。

这一点也是论文 Figure 1 所表达的核心思想：ODE 与 SDE 的轨迹不同，但可以共享同一条边缘概率路径。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 4. ODE 与 SDE 的生成误差缺乏统一分析

即使理论上 ODE 和 SDE 可以产生相同的概率路径，实际模型仍然需要通过神经网络学习对应的速度场和 Score。

假设：

$$
b_\theta(t,x)\approx b(t,x)
$$

$$
s_\theta(t,x)\approx s(t,x)
$$

那么会产生一个问题：

**网络预测误差究竟会如何影响最终生成分布？**

特别是：

- 速度场 MSE 很小，是否意味着最终生成分布的 KL 很小？
- ODE 与 SDE 对模型误差的敏感性是否相同？
- 扩散强度是否能够改善理论误差上界？

论文的重要贡献之一，就是利用连续性方程、Fokker–Planck 方程和 Girsanov 定理研究这些问题。

其中一个关键结果是：在相应假设下，SDE 的生成分布 KL 可以由漂移回归误差控制；而 ODE 的相应分析还需要考虑生成分布与目标分布之间的 Fisher Divergence。

这说明两种动力学虽然可以共享理想概率路径，但在模型近似误差下具有不同的理论性质。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

---


## 1.3 本文的主要贡献

根据原论文第 1.2 节 *Main contributions and organization*，可以将本文的主要贡献概括为六个方面。


### 贡献一：建立随机插值的统一数学框架

论文证明，满足相应假设的随机插值可以连接两个任意给定的端点概率密度。

其边缘概率密度同时满足：

- 一阶连续性方程；
- 一族具有可调扩散系数的前向 Fokker–Planck 方程；
- 对应的后向 Fokker–Planck 方程。

因此，同一条概率路径可以通过 ODE 或 SDE 实现。


### 贡献二：将生成动力学的学习转化为二次回归问题

论文证明，生成动力学需要的速度场和 Score 可以表示为二次目标函数的唯一最小化解（在相应函数空间和几乎处处意义下）。

这意味着训练阶段可以直接采样随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

然后构造监督目标，而不需要先求解生成 ODE 或 SDE。

论文还提出了针对插值密度 Score 的二次目标，并讨论了去噪器与 Score 的关系。


### 贡献三：建立 ODE 与 SDE 的似然误差理论

论文研究神经网络近似误差如何影响生成分布。

对于 SDE，利用 Girsanov 定理和 KL 的数据处理不等式，可以得到生成误差的理论控制。

对于 ODE，则需要额外考虑 Fisher Divergence。

论文还分析了如何通过选择扩散强度来优化相应的理论误差上界。


### 贡献四：推导生成模型的密度与交叉熵估计方法

对于 ODE，可以使用连续变量替换公式计算密度。

对于 SDE，论文构造了基于辅助随机过程的密度表示，并给出了交叉熵估计方法。

因此，随机插值框架不仅用于采样，也能够用于概率密度评估。


### 贡献五：建立与其他生成方法的联系

论文进一步研究：

- 单侧随机插值与高斯起点生成模型；
- Mirror Interpolants；
- Score-Based Diffusion 与随机局部化；
- Denoising；
- Rectified Flow；
- Schrödinger Bridge。

特别是，论文证明通过进一步优化随机插值，可以与 Schrödinger Bridge 的最优随机运输问题建立联系。

这里需要强调：**任意随机插值并不自动是最优 Schrödinger Bridge，必须进行额外优化。**


### 贡献六：给出实际训练和采样算法，并进行数值验证

论文讨论了：

- 如何利用经验数据估计训练目标；
- 如何选择速度场与 Score 的参数化；
- 如何利用去噪器进行采样；
- 如何处理数值稳定性；
- 如何选择 ODE 或 SDE 生成过程。

随后通过二维分布、高维高斯混合分布和图像生成实验验证相关方法。

以上贡献与原论文第 1.2 节及摘要的表述相对应。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

---


### 第 0～1 章总结

通过前两章，我们需要建立的核心认识是：

**生成模型的本质是概率分布之间的运输，而 ODE 与 SDE 只是实现这种运输的不同动力学方式。**

随机插值框架将问题分为三个层次：

$$
\boxed{
\underbrace{\text{随机插值}}_{\text{设计概率路径}}
\quad\longrightarrow\quad
\underbrace{\text{速度场与 Score}}_{\text{描述密度演化}}
\quad\longrightarrow\quad
\underbrace{\text{ODE / SDE}}_{\text{实现生成采样}}
}
$$

这三个层次分别回答：

1. **生成过程中，概率分布应该怎样变化？**
2. **怎样用数学场描述这种变化？**
3. **怎样通过确定性或随机动力学生成样本？**


<a id="chapter-2"></a>

# 第 2 章：Stochastic Interpolant Framework

本章是整篇论文最重要的理论基础。

第 0～1 章已经介绍了随机插值的核心思想：**先构造一条连接两个概率分布的路径，再从这条路径推导生成模型的 ODE 和 SDE。**

但是，这里存在几个需要严格回答的问题：

1. 随机插值只是人为构造的随机变量，为什么能够定义一条连续的概率路径？
2. 已知随机插值的表达式，为什么能够得到速度场和 Score？
3. 为什么这些场可以通过简单的均方误差进行学习？
4. 为什么同一条概率路径既可以由 ODE 描述，也可以由 SDE 描述？
5. 当速度场和 Score 存在预测误差时，能否控制最终生成分布的误差？
6. 除了生成样本，能否计算生成模型的概率密度？

本章将按照这六个问题展开。

为避免符号混淆，先约定：

| 符号 | 含义 |
|---|---|
| $X_t$ | 人为构造的随机插值 |
| $Y_t$ | 通过 ODE 或 SDE 生成的随机变量 |
| $\rho_t(x)$ | 随机插值在时刻 $t$ 的概率密度 |
| $b(t,x)$ | 边缘概率流的速度场 |
| $v(t,x)$ | 插值函数时间导数的条件期望 |
| $s(t,x)$ | 概率密度的 Score |
| $\gamma(t)$ | 随机插值中的高斯噪声系数 |
| $\varepsilon(t)$ | SDE 中可选择的扩散强度 |
| $d$ | 数据空间维度 |

特别注意：

$$
\boxed{b(t,x)\neq v(t,x)}
$$

一般情况下，两者并不相等。后面会推导它们之间的关系。

---


## 2.1 随机插值的定义、端点耦合与概率密度


### 2.1.1 核心结论

本节希望证明：

**给定任意两个端点概率分布，我们可以直接构造一个随机过程，使它在有限时间内连接这两个分布，并且能够明确写出任意中间时刻的概率密度。**

这个构造就是 Stochastic Interpolant。


### 2.1.2 随机插值的数学定义

假设有两个随机变量：

$$
X_0\sim\rho_0,\qquad X_1\sim\rho_1
\tag{2.1}
$$

其中：

- $\rho_0$：初始概率密度。
- $\rho_1$：目标概率密度。

首先，需要指定端点的联合分布：

$$
(X_0,X_1)\sim\nu
\tag{2.2}
$$

这里的 $\nu$ 称为 **Coupling（耦合）**。

它满足：

$$
\int\nu(x_0,x_1)\,dx_1=\rho_0(x_0)
\tag{2.3}
$$

$$
\int\nu(x_0,x_1)\,dx_0=\rho_1(x_1)
\tag{2.4}
$$

也就是说，无论两个端点之间如何配对，它们各自的边缘分布必须正确。

例如，最简单的独立耦合：

$$
\nu(x_0,x_1)=\rho_0(x_0)\rho_1(x_1)
\tag{2.5}
$$

但也可以使用其他耦合，例如最优传输耦合。

**为什么需要耦合？**

因为我们不仅需要从两个分布中分别采样，还需要确定一条插值路径连接哪两个端点样本。

不同耦合会产生不同的样本配对，从而影响中间概率路径。

接下来，定义随机插值：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{2.6}
$$

其中：

$$
Z\sim\mathcal N(0,I_d)
$$

且 $Z$ 与 $(X_0,X_1)$ 独立。

插值函数满足：

$$
I(0,x_0,x_1)=x_0
$$

$$
I(1,x_0,x_1)=x_1
\tag{2.7}
$$

噪声系数满足：

$$
\gamma(0)=\gamma(1)=0
\tag{2.8}
$$

因此：

$$
X_{t=0}=X_0,\qquad X_{t=1}=X_1
$$

自然得到：

$$
\boxed{
\rho_{t=0}=\rho_0,\qquad \rho_{t=1}=\rho_1
}
\tag{2.9}
$$

这一步已经说明，随机插值能够在有限时间内精确连接两个端点分布。

但这里的精确性只针对理论构造的随机插值，还没有涉及神经网络近似和数值积分。


### 2.1.3 如何得到中间时刻的概率密度？

现在考虑：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

我们的目标是求：

$$
\rho_t(x)
$$

也就是随机变量 $X_t$ 的边缘概率密度。

这里需要使用两个数学知识：

- **条件高斯分布**：见附录 A。
- **全概率公式**：见附录 A。

首先，固定两个端点：

$$
X_0=x_0,\qquad X_1=x_1
$$

此时：

$$
X_t=I(t,x_0,x_1)+\gamma(t)Z
$$

由于 $Z$ 是标准高斯，因此：

$$
X_t\mid X_0=x_0,X_1=x_1
\sim
\mathcal N\left(
I(t,x_0,x_1),
\gamma^2(t)I_d
\right)
\tag{2.10}
$$

对于 $\gamma(t)>0$，条件概率密度为：

$$
\begin{aligned}
&p_t(x\mid x_0,x_1)\\
&=
\frac{1}{(2\pi\gamma^2(t))^{d/2}}
\exp\left[
-\frac{\|x-I(t,x_0,x_1)\|^2}{2\gamma^2(t)}
\right]
\end{aligned}
\tag{2.11}
$$

但这只是固定两个端点时的条件密度。

实际生成过程中，端点本身也是随机采样的。

因此，根据全概率公式：

$$
\boxed{
\rho_t(x)=
\int
p_t(x\mid x_0,x_1)
\,\nu(dx_0,dx_1)
}
\tag{2.12}
$$

如果耦合存在联合密度，也可以写成：

$$
\rho_t(x)=
\int
p_t(x\mid x_0,x_1)
\nu(x_0,x_1)\,dx_0dx_1
\tag{2.13}
$$

这就是随机插值的边缘概率密度。

注意式 (2.12) 的含义：

**每一组端点对应一个高斯条件分布，对所有端点组合进行加权平均，就得到当前时刻的整体概率分布。**

因此，中间密度本质上是一个连续的高斯混合分布。


### 2.1.4 本节结论

到这里，我们完成了第一步：

$$
\boxed{
(X_0,X_1)\sim\nu
\quad\Longrightarrow\quad
X_t
\quad\Longrightarrow\quad
\rho_t
}
$$

现在已经知道如何构造概率路径，但还不知道如何让 ODE 或 SDE 产生这条路径。

下一节将从随机插值的时间导数出发，推导对应的速度场和 Score。

---


## 2.2 速度场、Score 与连续性方程的推导


### 2.2.1 核心结论

本节要证明三个重要结果。

**第一个结论：随机插值的边缘速度场是样本速度的条件期望。**

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{2.14}
$$

**第二个结论：随机插值的概率密度满足连续性方程。**

$$
\boxed{
\partial_t\rho_t+\nabla\cdot(\rho_tb)=0
}
\tag{2.15}
$$

**第三个结论：随机插值的 Score 可以通过高斯噪声的条件期望计算。**

$$
\boxed{
s(t,x)=
-\frac{1}{\gamma(t)}
\mathbb E[Z\mid X_t=x]
}
\tag{2.16}
$$

下面分别推导。


### 2.2.2 从样本速度推导边缘速度场

随机插值为：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

对时间求导：

$$
\boxed{
\dot X_t=
\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
}
\tag{2.17}
$$

其中：

- $\partial_tI$：插值函数对时间的偏导数。
- $\dot\gamma$：噪声系数对时间的导数。

这里存在一个关键问题：

$$
\dot X_t
$$

依赖于：

$$
X_0,\quad X_1,\quad Z
$$

但是我们希望构造的 ODE 速度场只能依赖：

$$
t,\quad x
$$

也就是说，给定当前状态 $X_t=x$，需要确定当前位置的速度。

自然的做法是对所有满足 $X_t=x$ 的潜在样本组合取条件平均：

$$
\boxed{
b(t,x)=
\mathbb E[
\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
\mid X_t=x
]
}
\tag{2.18}
$$

这个 $b(t,x)$ 就是边缘速度场。

但目前它只是一个定义。

为什么这个定义一定能够正确描述概率密度的变化？

需要证明连续性方程。


### 2.2.3 连续性方程的详细推导


对任意光滑紧支撑测试函数应用链式法则、条件期望与分部积分，可得到下式。

完整推导见[附录 B.2.2：随机插值的连续性方程](#app-b-2-2)。

$$
\boxed{
\partial_t\rho_t+
\nabla\cdot(\rho_tb)=0
}
\tag{2.28}
$$

**这就完成了证明。**

需要注意，以上推导要求具有足够的可积性和正则性，以交换求导与期望。


#### 为什么这个结果重要？

我们最初构造的 $X_t$ 并不是通过 ODE 得到的。

但现在证明，它的边缘密度满足：

$$
\partial_t\rho_t+\nabla\cdot(\rho_tb)=0
$$

这恰好是 ODE 概率流的连续性方程。

因此，只要满足相应的存在唯一性条件，就可以构造：

$$
\frac{dY_t}{dt}=b(t,Y_t)
$$

使其边缘分布与随机插值一致。

注意：

$$
\boxed{
Y_t\overset d=X_t
}
$$

表示相同时间的边缘分布一致，而不是每条样本轨迹相同。


### 2.2.4 Score 的详细推导


对条件高斯分布求对数梯度，再按潜变量的后验取条件期望。对一般插值 $I+\gamma Z$，这里要求 $\gamma(t)>0$。

完整推导见[附录 D.1.2～D.1.3：Fisher Identity 与高斯噪声 Score](#app-d-1-3)。

$$
\boxed{
s(t,x)=
-\frac{1}{\gamma(t)}
\mathbb E[Z\mid X_t=x]
}
\tag{2.33}
$$

这就是 Score 与噪声预测之间的关系。

**核心理解：**

虽然每个训练样本的高斯噪声 $Z$ 是随机的，但在给定 $X_t=x$ 后，其条件期望是一个确定性的函数。

因此可以训练一个神经网络预测：

$$
\eta_\theta(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

再恢复：

$$
s_\theta(t,x)=-\frac{\eta_\theta(t,x)}{\gamma(t)}
$$


### 2.2.5 为什么论文还定义了另一个速度场 $v$？

论文还定义：

$$
\boxed{
v(t,x)=
\mathbb E[
\partial_tI(t,X_0,X_1)
\mid X_t=x
]
}
\tag{2.34}
$$

注意，$v$ 只包含插值函数的时间导数，不包含高斯噪声系数变化带来的速度。

而：

$$
b(t,x)=
\mathbb E[
\partial_tI+\dot\gamma Z
\mid X_t=x
]
$$

展开条件期望：

$$
b=v+\dot\gamma\mathbb E[Z\mid X_t=x]
\tag{2.35}
$$

根据刚才推导的 Score 公式：

$$
\mathbb E[Z\mid X_t=x]=-\gamma s
$$

代入：

$$
b=v-\gamma\dot\gamma s
$$

最终得到：

$$
\boxed{
b(t,x)=v(t,x)-\gamma(t)\dot\gamma(t)s(t,x)
}
\tag{2.36}
$$

这个公式非常重要。

它说明随机插值的边缘速度可以拆成两部分：

$$
\underbrace{v}_{\text{插值函数引起的运输}}
-
\underbrace{\gamma\dot\gamma s}_{\text{噪声方差变化引起的修正}}
$$

当：

$$
\gamma(t)=0
$$

且不存在相关端点奇异性时，通常有：

$$
b=v
$$

但当 $\gamma(t)\neq0$ 时，一般不能直接把 $v$ 当作概率流 ODE 的速度。


### 2.2.6 本节总结

我们已经从随机插值推导出：

$$
\boxed{
\begin{aligned}
b(t,x)
&=\mathbb E[\dot X_t\mid X_t=x]\\
s(t,x)
&=-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}\\
b(t,x)
&=v(t,x)-\gamma(t)\dot\gamma(t)s(t,x)
\end{aligned}
}
\tag{2.37}
$$

同时证明：

$$
\boxed{
\partial_t\rho_t+\nabla\cdot(\rho_tb)=0
}
\tag{2.38}
$$

到这里，我们已经完成了从**随机插值到概率流速度场**的数学推导。

下一步需要回答：既然 $b$、$v$ 和 $s$ 都涉及条件期望，如何通过神经网络训练得到它们？

---


## 2.3 速度场与 Score 的训练目标


### 2.3.1 核心结论

上一节推导出了：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

以及：

$$
s(t,x)=-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
$$

但在实际训练中，我们不知道这些条件期望的解析表达式。

例如，对于一个高维图像分布，几乎不可能直接计算：

$$
\mathbb E[Z\mid X_t=x]
$$

因此，我们希望通过神经网络学习这些函数。

论文证明，这些条件期望都可以表示为**二次优化目标的最小化解**。

这意味着，只需要采样随机插值及其已知的监督量，就可以训练模型，而不需要显式计算中间概率密度 $\rho_t$。


### 2.3.2 为什么 MSE 能够学习条件期望？


设 $m(X)=\mathbb E[Y\mid X]$，且 $Y$ 平方可积。均方误差可作如下正交分解：

完整推导见[附录 A.2.2：条件期望的 MSE 最优性](#app-a-2-2)。

$$
\boxed{
\mathbb E\|f-Y\|^2
=
\mathbb E\|f-m\|^2+
\mathbb E\|m-Y\|^2
}
\tag{2.43}
$$

第二项不依赖 $f$，第一项非负。

因此最优解为：

$$
\boxed{
f^*(x)=\mathbb E[Y\mid X=x]
}
\tag{2.44}
$$

这个结论解释了一个看似矛盾的现象：

**即使训练标签 $Y$ 是随机的，使用 MSE 训练得到的最优预测器仍然是一个确定性的条件平均函数。**


### 2.3.3 速度场的训练目标

随机插值的速度为：

$$
\dot X_t=\partial_tI+\dot\gamma Z
$$

定义监督目标：

$$
A_b=\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
\tag{2.45}
$$

根据上一节：

$$
b(t,x)=\mathbb E[A_b\mid X_t=x]
$$

因此可以训练：

$$
\boxed{
\mathcal L_b(\theta)=
\mathbb E_{t,X_0,X_1,Z}
\left[
\|b_\theta(t,X_t)-A_b\|^2
\right]
}
\tag{2.46}
$$

其中：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

由条件期望的 MSE 最优性：

$$
\boxed{
b_\theta^*(t,x)=b(t,x)
}
\tag{2.47}
$$

这里的等号指无限函数表达能力、理想优化条件下，在训练分布几乎处处成立。

论文还使用等价的二次泛函：

$$
\mathcal J_b[\widehat b]
=
\mathbb E\left[
\frac12\|\widehat b(t,X_t)\|^2
-\widehat b(t,X_t)^\top A_b
\right]
\tag{2.48}
$$

为什么它与 MSE 等价？

因为：

$$
\frac12\|\widehat b-A_b\|^2
=
\frac12\|\widehat b\|^2
-\widehat b^\top A_b
+\frac12\|A_b\|^2
$$

所以：

$$
\mathcal J_b[\widehat b]
=
\frac12\mathbb E\|\widehat b-A_b\|^2
-\frac12\mathbb E\|A_b\|^2
\tag{2.49}
$$

最后一项与模型无关。

因此，最小化这两个目标得到相同的最优解。


### 2.3.4 Score 的训练目标

根据第 2.2 节：

$$
s(t,x)=-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
$$

最直接的方式是训练 Score 网络：

$$
\mathcal L_s(\phi)=
\mathbb E\left[
\left\|
s_\phi(t,X_t)+\frac Z{\gamma(t)}
\right\|^2
\right]
\tag{2.50}
$$

其最优解为：

$$
s_\phi^*(t,x)=s(t,x)
$$

但这种方式存在一个问题：

当：

$$
\gamma(t)\to0
$$

监督目标：

$$
-\frac Z{\gamma(t)}
$$

可能具有非常大的数值尺度。

因此，可以换一种参数化。

定义噪声预测器：

$$
\eta_\phi(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

直接训练：

$$
\boxed{
\mathcal L_\eta(\phi)=
\mathbb E\left[
\|\eta_\phi(t,X_t)-Z\|^2
\right]
}
\tag{2.51}
$$

然后恢复：

$$
\boxed{
s_\phi(t,x)=
-\frac{\eta_\phi(t,x)}{\gamma(t)}
}
\tag{2.52}
$$

这种做法使监督目标 $Z$ 的尺度不再随 $\gamma(t)$ 发散。

但要注意：**训练目标稳定，并不代表恢复 Score 时的除法一定数值稳定。**

尤其在端点附近，仍然需要特别处理。


### 2.3.5 直接预测 $b$ 与分别预测 $v,s$

论文还定义：

$$
v(t,x)=\mathbb E[\partial_tI\mid X_t=x]
$$

因此可以单独训练：

$$
\boxed{
\mathcal L_v(\theta)=
\mathbb E\left[
\|v_\theta(t,X_t)-\partial_tI\|^2
\right]
}
\tag{2.53}
$$

同时训练 Score：

$$
s_\phi(t,x)
$$

最后通过：

$$
\boxed{
b_{\theta,\phi}
=
v_\theta-\gamma\dot\gamma s_\phi
}
\tag{2.54}
$$

恢复边缘速度。

所以存在两种主要方式：

**方式一：直接预测边缘速度**

$$
b_\theta\longrightarrow\text{ODE}
$$

如果还需要 SDE，则额外需要 Score。

**方式二：分别预测插值速度与 Score**

$$
(v_\theta,s_\phi)
\longrightarrow b_{\theta,\phi}
\longrightarrow\text{ODE/SDE}
$$

两种方法在理想函数空间中都可以恢复正确的场，但在有限模型容量、有限数据和数值优化条件下，表现不一定相同。

后面第 4 章还会看到：

**对于单侧线性随机插值，速度场与 Score 之间存在更强的代数关系，因此可以只训练一个预测器。**


### 2.3.6 本节总结

通过条件期望的 MSE 最优性，我们得到了：

$$
\boxed{
\begin{aligned}
b(t,x)
&=\arg\min_f
\mathbb E[\|f(t,X_t)-\dot X_t\|^2]\\
v(t,x)
&=\arg\min_f
\mathbb E[\|f(t,X_t)-\partial_tI\|^2]\\
\eta(t,x)
&=\arg\min_f
\mathbb E[\|f(t,X_t)-Z\|^2]
\end{aligned}
}
\tag{2.55}
$$

这里的最小化针对整个时间—状态函数，等式在相应分布几乎处处意义下成立。

这一结果的重要意义是：

**训练生成模型不需要提前知道 $\rho_t$ 的解析表达式，也不需要在训练阶段求解生成 ODE/SDE。**

只需要能够从随机插值中采样，就能构造训练数据。

---


## 2.4 ODE、前向 SDE 与反向 SDE 的统一推导


### 2.4.1 核心结论

这是全文最重要的统一结果。

上一节已经得到了随机插值的概率路径：

$$
\rho_t
$$

以及它的两个场：

$$
b(t,x),\qquad s(t,x)
$$

本节要证明：

**对于同一条概率路径，可以构造确定性的 ODE，以及扩散强度可调的前向和反向 SDE。**

最终结果为：

$$
\boxed{
\frac{dY_t}{dt}=b(t,Y_t)
}
\tag{2.56}
$$

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+\sqrt{2\varepsilon(t)}\,dW_t
}
\tag{2.57}
$$

以及沿原时间从 $1$ 向 $0$ 积分的后向 SDE：

$$
\boxed{
dY_t=
[b(t,Y_t)-\varepsilon(t)s(t,Y_t)]dt
+\sqrt{2\varepsilon(t)}\,d\overleftarrow W_t
}
\tag{2.58}
$$

下面解释它们为什么成立。


### 2.4.2 为什么 ODE 能够产生相同的概率路径？

第 2.2 节已经证明：

$$
\partial_t\rho_t+\nabla\cdot(\rho_tb)=0
\tag{2.59}
$$

另一方面，考虑 ODE：

$$
dY_t=b(t,Y_t)dt
$$

假设 $Y_t$ 的概率密度为 $q_t$。

根据 ODE 的概率密度演化规律：

$$
\partial_tq_t+\nabla\cdot(q_tb)=0
\tag{2.60}
$$

如果：

$$
q_0=\rho_0
$$

那么 $q_t$ 和 $\rho_t$ 满足相同的连续性方程、相同的初始条件。

在适当的存在唯一性条件下：

$$
\boxed{
q_t=\rho_t
}
\tag{2.61}
$$

这说明 ODE 能够生成与随机插值相同的边缘概率路径。

这里的证明依赖于连续性方程解的唯一性，而不仅仅是两个方程形式相同。


### 2.4.3 为什么 SDE 不能直接使用漂移 $b$？


若直接以 $b$ 为 SDE 漂移，布朗噪声会额外引入 $\varepsilon\Delta\rho_t$，改变目标密度演化。对于仅依赖时间的 $\varepsilon(t)\ge0$，可选择：

完整推导见[附录 B.3.3：Score 修正与扩散项抵消](#app-b-3-3)。

$$
\boxed{
f=b+\varepsilon s
}
\tag{2.69}
$$

代入原 SDE：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{2.70}
$$

这就是前向 SDE。


#### 为什么要增加 $\varepsilon s$？

因为布朗噪声会在 Fokker–Planck 方程中引入：

$$
+\varepsilon\Delta\rho_t
$$

为了不改变目标概率路径，需要在漂移中增加：

$$
+\varepsilon s
$$

Score 修正产生的密度变化恰好抵消随机扩散项，因而保留原来的连续性方程；计算见[附录 B.3.3](#app-b-3-3)。

**这就是 ODE 与 SDE 可以共享同一概率路径的根本原因。**

需要注意：这里构造的是一个满足要求的漂移场，并不意味着满足相同密度方程的漂移场只有这一个。


### 2.4.4 为什么可以自由选择扩散强度？

观察：

$$
f=b+\varepsilon s
$$

无论选择什么非负的 $\varepsilon(t)$，只要相应 SDE 适定，Fokker–Planck 中的扩散项都能被 Score 修正项抵消。

因此：

$$
\boxed{
\varepsilon(t)
\text{ 可以作为独立的动力学设计参数}
}
$$

例如：

当：

$$
\varepsilon(t)=0
$$

得到：

$$
dY_t=b(t,Y_t)dt
$$

也就是 ODE。

当：

$$
\varepsilon(t)>0
$$

得到随机 SDE。

不同 $\varepsilon$ 对应不同的随机动力学，但理想连续时间下可以保持相同的边缘概率路径。

这里还要区分：

$$
\gamma(t)\neq\varepsilon(t)
$$

其中：

- $\gamma(t)$ 决定随机插值本身。
- $\varepsilon(t)$ 决定生成 SDE 的随机性。

改变 $\gamma$ 通常会改变目标概率路径。

改变 $\varepsilon$ 则是在同一目标概率路径上选择不同的动力学。


### 2.4.5 反向 SDE 为什么出现负号？


令递增反向时间 $\tau=1-t$，并令 $q_\tau=\rho_{1-\tau}$。由时间反演公式，对 $f=b+\varepsilon s$、$g^2=2\varepsilon$，反向漂移为 $-f+g^2s=-b+\varepsilon s$。下式右侧各场在原时间 $1-\tau$ 处取值：

完整推导见[附录 C.3：SDE 时间反转](#app-c-3-2)。

$$
\boxed{
dR_\tau=
[-b(1-\tau,R_\tau)+\varepsilon(1-\tau)s(1-\tau,R_\tau)]d\tau
+
\sqrt{2\varepsilon(1-\tau)}dW_\tau
}
\tag{2.79}
$$

如果改回原论文使用的递减时间 $t:1\to0$，就得到后向 Itô 表示：

$$
\boxed{
dY_t=
[b(t,Y_t)-\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}d\overleftarrow W_t
}
\tag{2.80}
$$

**注意：**

这里不能把 $d\overleftarrow W_t$ 简单理解为普通正向布朗增量。

它使用后向随机积分约定，且时间沿 $1\to0$ 演化。


### 2.4.6 本节总结

我们已经证明：

$$
\boxed{
\begin{aligned}
\text{ODE:}\quad
&dY_t=b\,dt\\[4pt]
\text{Forward SDE:}\quad
&dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}\,dW_t\\[4pt]
\text{Backward SDE:}\quad
&dY_t=(b-\varepsilon s)dt+\sqrt{2\varepsilon}\,d\overleftarrow W_t
\end{aligned}
}
\tag{2.81}
$$

其中后向 SDE 使用原时间递减的约定。

这三个方程在理想条件下共享同一条边缘概率路径。

但是，实际训练中我们只能得到：

$$
b_\theta\approx b,\qquad s_\phi\approx s
$$

那么模型误差会如何影响最终生成分布？

这就是下一节要解决的问题。

---


## 2.5 ODE/SDE 的 KL 误差分析与理论保证


### 2.5.1 核心结论

前面证明了：如果使用真实的速度场 $b$ 和 Score $s$，ODE 与 SDE 可以产生目标概率路径。

但实际生成模型使用的是神经网络：

$$
\widehat b(t,x)\approx b(t,x)
$$

$$
\widehat s(t,x)\approx s(t,x)
$$

这意味着模型实际生成的分布：

$$
q_t(x)
$$

一般不再等于目标分布：

$$
\rho_t(x)
$$

因此，我们真正需要研究的是：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
}
\tag{2.82}
$$

能否通过训练时的速度场和 Score 误差来控制？

论文给出了两个重要结果：

**对于 ODE：** KL 的时间变化不仅取决于速度场误差，还涉及目标分布与模型分布的 Score 差异。

**对于 SDE：** 在适当条件下，可以使用 Girsanov 定理，将最终分布的 KL 上界与两个 SDE 的漂移平方误差联系起来。

下面分别推导。


### 2.5.2 KL 散度的定义

对于两个概率密度 $\rho(x)$ 和 $q(x)$，定义：

$$
\boxed{
D_{\mathrm{KL}}(\rho\|q)
=
\int \rho(x)\log\frac{\rho(x)}{q(x)}\,dx
}
\tag{2.83}
$$

KL 散度用于衡量两个概率分布之间的差异。

它满足：

$$
D_{\mathrm{KL}}(\rho\|q)\ge0
$$

当两个分布几乎处处相等时，KL 为零。

需要注意，KL 不是对称的：

$$
D_{\mathrm{KL}}(\rho\|q)
\neq
D_{\mathrm{KL}}(q\|\rho)
$$

本节主要研究：

$$
D_{\mathrm{KL}}(\rho_t\|q_t)
$$

其中：

- $\rho_t$：理论随机插值的目标概率密度。
- $q_t$：神经网络生成动力学实际产生的概率密度。

相关数学基础见附录 A：KL 散度。


### 2.5.3 ODE 的 KL 误差推导

考虑真实 ODE：

$$
dY_t=b(t,Y_t)dt
$$

对应密度：

$$
\rho_t
$$

以及近似 ODE：

$$
d\widehat Y_t=\widehat b(t,\widehat Y_t)dt
$$

对应密度：

$$
q_t
$$

它们分别满足：

$$
\partial_t\rho_t=-\nabla\cdot(\rho_tb)
\tag{2.84}
$$

$$
\partial_tq_t=-\nabla\cdot(q_t\widehat b)
\tag{2.85}
$$

现在定义时间相关的 KL：

$$
D(t)=
\int\rho_t\log\frac{\rho_t}{q_t}\,dx
\tag{2.86}
$$

对时间求导：

$$
\begin{aligned}
\frac{dD}{dt}
={}&
\int\partial_t\rho_t
\log\frac{\rho_t}{q_t}\,dx\\
&+
\int\rho_t
\left(
\frac{\partial_t\rho_t}{\rho_t}
-
\frac{\partial_tq_t}{q_t}
\right)dx
\end{aligned}
\tag{2.87}
$$

由于概率密度归一化：

$$
\int\rho_t\,dx=1
$$

所以：

$$
\int\partial_t\rho_t\,dx=0
$$

因此：

$$
\frac{dD}{dt}
=
\int\partial_t\rho_t
\log\frac{\rho_t}{q_t}\,dx
-
\int\rho_t\frac{\partial_tq_t}{q_t}\,dx
\tag{2.88}
$$

接下来分别处理这两项。

**第一项**

代入：

$$
\partial_t\rho_t=-\nabla\cdot(\rho_tb)
$$

得到：

$$
-\int\nabla\cdot(\rho_tb)
\log\frac{\rho_t}{q_t}\,dx
$$

通过分部积分：

$$
=
\int\rho_tb\cdot
\nabla\log\frac{\rho_t}{q_t}\,dx
$$

定义：

$$
s_\rho=\nabla\log\rho_t
$$

$$
s_q=\nabla\log q_t
$$

于是：

$$
\nabla\log\frac{\rho_t}{q_t}
=s_\rho-s_q
$$

第一项等于：

$$
\int\rho_tb\cdot(s_\rho-s_q)\,dx
\tag{2.89}
$$

**第二项**

代入：

$$
\partial_tq_t=-\nabla\cdot(q_t\widehat b)
$$

得到：

$$
-\int\rho_t\frac{\partial_tq_t}{q_t}\,dx
=
\int\rho_t
\frac{\nabla\cdot(q_t\widehat b)}{q_t}\,dx
\tag{2.90}
$$

根据散度乘积法则：

$$
\nabla\cdot(q_t\widehat b)
=
q_t\nabla\cdot\widehat b
+
\widehat b\cdot\nabla q_t
$$

两边除以 $q_t$：

$$
\frac{\nabla\cdot(q_t\widehat b)}{q_t}
=
\nabla\cdot\widehat b
+
\widehat b\cdot s_q
$$

因此：

$$
\begin{aligned}
-\int\rho_t\frac{\partial_tq_t}{q_t}\,dx
={}&
\int\rho_t\nabla\cdot\widehat b\,dx\\
&+
\int\rho_t\widehat b\cdot s_q\,dx
\end{aligned}
\tag{2.91}
$$

对第一项再次分部积分：

$$
\int\rho_t\nabla\cdot\widehat b\,dx
=
-\int\widehat b\cdot\nabla\rho_t\,dx
$$

利用：

$$
\nabla\rho_t=\rho_ts_\rho
$$

得到：

$$
\int\rho_t\nabla\cdot\widehat b\,dx
=
-\int\rho_t\widehat b\cdot s_\rho\,dx
$$

所以式 (2.91) 变成：

$$
-\int\rho_t\widehat b\cdot(s_\rho-s_q)\,dx
\tag{2.92}
$$

最后，将式 (2.89) 与式 (2.92) 相加：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(\rho_t\|q_t)
=
\int\rho_t
(b-\widehat b)\cdot(s_\rho-s_q)\,dx
}
\tag{2.93}
$$

这就是 ODE 的 KL 演化公式。


#### 这个公式说明了什么？

观察右侧：

$$
(b-\widehat b)\cdot(s_\rho-s_q)
$$

它包含两个误差：

**速度场误差：**

$$
b-\widehat b
$$

**两个分布之间的 Score 差异：**

$$
s_\rho-s_q
$$

因此，ODE 的 KL 变化不能仅由速度场 MSE 直接确定。

利用 Cauchy–Schwarz 不等式（见附录 A）：

$$
\begin{aligned}
\left|\frac{dD}{dt}\right|
\le{}&
\sqrt{\mathbb E_{\rho_t}\|b-\widehat b\|^2}\\
&\cdot
\sqrt{\mathbb E_{\rho_t}\|s_\rho-s_q\|^2}
\end{aligned}
\tag{2.94}
$$

其中：

$$
\boxed{
\mathbb E_{\rho_t}\|s_\rho-s_q\|^2
}
\tag{2.95}
$$

是两个分布之间的一种 Fisher Divergence。

**因此，即使速度场误差较小，也不能仅凭这个事实就保证 ODE 最终生成分布的 KL 较小。**

还需要考虑两个分布的 Score 差异，或者其他能够控制密度演化的正则性条件。


### 2.5.4 SDE 的 KL 误差推导

现在考虑两个 SDE。

真实 SDE：

$$
dY_t=f(t,Y_t)dt+\sqrt{2\varepsilon}\,dW_t
\tag{2.96}
$$

近似 SDE：

$$
d\widehat Y_t=
\widehat f(t,\widehat Y_t)dt
+\sqrt{2\varepsilon}\,d\widehat W_t
\tag{2.97}
$$

这里暂时令：

$$
\varepsilon>0
$$

为常数，并假设两个过程具有相同的初始分布。

定义漂移误差：

$$
\delta(t,x)=f(t,x)-\widehat f(t,x)
\tag{2.98}
$$

我们希望分析：

$$
D_{\mathrm{KL}}(\rho_1\|q_1)
$$

但直接计算两个终点分布的 KL 比较困难。

论文使用了一个重要思路：

**先研究整个随机过程的路径分布，再利用路径 KL 控制终点 KL。**

设：

- $P$：真实 SDE 在路径空间上的概率测度。
- $Q$：近似 SDE 在路径空间上的概率测度。

这里的路径空间可以理解为：

$$
\omega=\{Y_t:0\le t\le1\}
$$

即一整条随机轨迹。


#### 第一步：Girsanov 定理


在相同初始分布、相同扩散系数 $\sqrt{2\varepsilon}I_d$、$\varepsilon>0$，以及所需绝对连续性和可积性条件下：

完整推导见[附录 C.4：Girsanov 路径 KL](#app-c-4)。

$$
\boxed{
D_{\mathrm{KL}}(P\|Q)
=
\frac{1}{4\varepsilon}
\int_0^1
\mathbb E_{\rho_t}
\|f(t,Y_t)-\widehat f(t,Y_t)\|^2dt
}
\tag{2.99}
$$

系数来自 Girsanov 中的 $1/2$ 与扩散方差 $2\varepsilon$；归一化漂移差的计算统一见[附录 C.4.2～C.4.4](#app-c-4)。


#### 第二步：从路径 KL 得到终点 KL


从整条随机路径中取出终点是确定性映射，因此数据处理不等式给出：

完整推导见[附录 A.4.4～A.4.5：从路径 KL 到终点 KL](#app-a-4-5)。

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
\le
D_{\mathrm{KL}}(P\|Q)
}
\tag{2.100}
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
\le
\frac{1}{4\varepsilon}
\int_0^1
\mathbb E_{\rho_t}
\|f-\widehat f\|^2dt
}
\tag{2.101}
$$

这就是关键结论。

**最终生成分布的 KL，可以通过两个 SDE 的漂移平方误差控制。**


### 2.5.5 将 SDE 漂移误差拆成速度场与 Score 误差


记 $A=\int_0^1\mathbb E_{\rho_t}\|b-\widehat b\|^2dt$、$B=\int_0^1\mathbb E_{\rho_t}\|s-\widehat s\|^2dt$。对相同初始分布、相同扩散系数 $\sqrt{2\varepsilon}$ 的 SDE，在 Girsanov 条件下有：

完整推导见[附录 F.2：SDE 误差上界与最优扩散强度](#app-f-2)。

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
\le
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
\tag{2.109}
$$

这个结果非常值得理解。

第一项：

$$
\frac{A}{2\varepsilon}
$$

随着扩散强度增大而减小。

第二项：

$$
\frac{\varepsilon B}{2}
$$

随着扩散强度增大而增大。

因此，扩散强度存在一个理论上的权衡。


当 $A,B>0$ 且固定时，使该上界最小的扩散强度为（求导步骤见[附录 F.2](#app-f-2)）：

$$
\boxed{
\varepsilon^*=\sqrt{\frac AB}
}
\tag{2.110}
$$

这表示：

**在该误差上界下，存在一个使上界最小的扩散强度。**

但要特别注意：

- 这里优化的是 KL 上界，不是精确 KL。
- 推导使用了特定的不等式，因此上界可能不紧。
- 实际模型的数值积分误差没有包含在这个连续时间结果中。
- 不能据此断言 SDE 一定优于 ODE。


### 2.5.6 本节总结

对于 ODE：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(\rho_t\|q_t)
=
\mathbb E_{\rho_t}
[(b-\widehat b)\cdot(s_\rho-s_q)]
}
\tag{2.111}
$$

对于具有相同扩散系数的 SDE，在相应假设下：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
\le
\frac{1}{4\varepsilon}
\int_0^1\mathbb E_{\rho_t}
\|f-\widehat f\|^2dt
}
\tag{2.112}
$$

因此，论文建立了从**神经网络回归误差到最终生成分布误差**的理论联系。

这也是随机插值框架不仅是一种生成模型构造方法，而且具有误差分析价值的重要原因。

---


## 2.6 概率密度估计与 Likelihood


### 2.6.1 核心结论

前面讨论的主要是：

$$
Y_0\sim\rho_0
\quad\longrightarrow\quad
Y_1\sim q_1
$$

也就是如何生成样本。

但概率生成模型还有一个重要问题：

**给定某个样本 $x$，能否计算模型认为它出现的概率密度 $q_1(x)$？**

这就是 Likelihood Estimation。

论文分别研究了 ODE 和 SDE 的概率密度计算。

其中：

- ODE 可以通过连续变量替换公式计算密度。
- SDE 需要借助辅助随机过程，得到密度的随机路径期望表示。


### 2.6.2 ODE 的概率密度估计


令 $q_t$ 为 ODE $\dot Y_t=b(t,Y_t)$ 的密度。连续性方程与链式法则给出：

完整推导见[附录 B.2.4：沿 ODE 轨迹的密度变化](#app-b-2-4)。

$$
\boxed{
\frac{d}{dt}\log q_t(Y_t)
=
-\nabla\cdot b(t,Y_t)
}
\tag{2.119}
$$

对时间积分：

$$
\log q_1(Y_1)-\log q_0(Y_0)
=
-\int_0^1\nabla\cdot b(t,Y_t)\,dt
$$

因此：

$$
\boxed{
\log q_1(Y_1)
=
\log q_0(Y_0)
-
\int_0^1\nabla\cdot b(t,Y_t)\,dt
}
\tag{2.120}
$$

这就是连续 Normalizing Flow 的概率密度计算公式。


#### 如何理解这个公式？

考虑：

$$
\nabla\cdot b
$$

它表示速度场的局部体积膨胀率。

当：

$$
\nabla\cdot b>0
$$

表示局部体积膨胀。

概率质量守恒，因此概率密度会降低。

对应：

$$
\frac d{dt}\log q_t(Y_t)<0
$$

反过来，当：

$$
\nabla\cdot b<0
$$

局部体积收缩，概率密度会升高。

因此，ODE 的密度变化可以通过速度场的散度计算。

这也解释了为什么 Continuous Normalizing Flow 可以计算精确形式的连续时间 Likelihood。

这里的“精确形式”不意味着数值计算没有误差：实际求解 ODE 和散度积分时仍然需要数值近似。


### 2.6.3 为什么 SDE 的 Likelihood 更复杂？

对于 SDE：

$$
dY_t=f(t,Y_t)dt+\sqrt{2\varepsilon}\,dW_t
\tag{2.121}
$$

它的密度满足：

$$
\partial_tq_t
=
-\nabla\cdot(fq_t)+\varepsilon\Delta q_t
\tag{2.122}
$$

与 ODE 不同，右侧多出了二阶扩散项：

$$
\varepsilon\Delta q_t
$$

因此，不能直接沿单条随机轨迹使用：

$$
\frac d{dt}\log q_t(Y_t)=-\nabla\cdot f
$$

因为随机轨迹的函数变化需要使用 Itô 公式，其中包含二阶导数项。

这意味着：

**SDE 的概率密度不能直接使用 ODE 的散度积分公式计算。**

论文采用另一种方法：构造辅助随机过程，通过随机路径期望表示模型密度。


### 2.6.4 SDE 的辅助随机过程

考虑模型的前向 SDE：

$$
dY_t=f(t,Y_t)dt+\sqrt{2\varepsilon}\,dW_t
\tag{2.123}
$$

其中：

$$
f=\widehat b+\varepsilon\widehat s
$$

其密度 $q_t$ 满足：

$$
\partial_tq_t
=
-\nabla\cdot(fq_t)+\varepsilon\Delta q_t
\tag{2.124}
$$

展开散度：

$$
\nabla\cdot(fq_t)
=
f\cdot\nabla q_t+q_t\nabla\cdot f
$$

所以：

$$
\partial_tq_t
=
-f\cdot\nabla q_t
-q_t\nabla\cdot f
+\varepsilon\Delta q_t
\tag{2.125}
$$

整理：

$$
\boxed{
\partial_tq_t
+
f\cdot\nabla q_t
-\varepsilon\Delta q_t
=
-(\nabla\cdot f)q_t
}
\tag{2.126}
$$

这个方程提示我们，可以通过适当的辅助后向扩散来构造随机表示。

定义辅助过程 $R_t$，沿原时间从 $1$ 向 $0$ 积分：

$$
\boxed{
dR_t=f(t,R_t)dt+
\sqrt{2\varepsilon}\,d\overleftarrow W_t
}
\tag{2.127}
$$

并指定：

$$
R_1=x
$$

注意这里辅助过程使用的是：

$$
f=\widehat b+\varepsilon\widehat s
$$

它不是第 2.4 节用于保持目标边缘分布的后向 SDE 漂移：

$$
\widehat b-\varepsilon\widehat s
$$

这两个过程的用途不同：

- 第 2.4 节的后向 SDE 用于生成相反时间方向的概率路径。
- 这里的辅助后向 SDE 用于表示前向生成模型的概率密度。

不能混为一谈。


### 2.6.5 从辅助过程得到密度表示

这里使用一个随机分析工具：

**Feynman–Kac 型随机表示。**

相关数学基础见附录 C：Feynman–Kac 公式与后向 Itô 公式。

我们希望把 PDE：

$$
\partial_tq_t
+
f\cdot\nabla q_t
-\varepsilon\Delta q_t
=
-(\nabla\cdot f)q_t
$$

转换为沿辅助随机过程的期望。

沿着从 $t=1$ 向 $t=0$ 演化的辅助过程 $R_t$，考虑：

$$
q_t(R_t)
$$

使用后向 Itô 公式，其漂移项为：

$$
\left[
\partial_tq_t
+
f\cdot\nabla q_t
-
\varepsilon\Delta q_t
\right]dt
$$

根据式 (2.126)，这个漂移项等于：

$$
-(\nabla\cdot f)q_t\,dt
$$

因此：

$$
dq_t(R_t)
=
-(\nabla\cdot f)(t,R_t)q_t(R_t)\,dt
+
\text{随机积分项}
\tag{2.128}
$$

为了消去漂移，定义积分因子：

$$
M_t=
\exp\left[
\int_0^t\nabla\cdot f(r,R_r)\,dr
\right]
$$

形式上对：

$$
M_tq_t(R_t)
$$

使用随机乘积法则，可以抵消上述漂移项。

在适当可积性条件下，对辅助随机路径取期望，得到：

$$
\boxed{
q_1(x)
=
\mathbb E_{\mathrm{aux}}^{\,R_1=x}
\left[
q_0(R_0)
\exp\left(
-\int_0^1\nabla\cdot f(t,R_t)\,dt
\right)
\right]
}
\tag{2.129}
$$

这就是 SDE 的密度随机表示。

它与 ODE 公式的形式有相似之处。

ODE：

$$
q_1(Y_1)
=
q_0(Y_0)
\exp\left[
-\int_0^1\nabla\cdot b(t,Y_t)\,dt
\right]
\tag{2.130}
$$

SDE：

$$
q_1(x)
=
\mathbb E_{\mathrm{aux}}^{\,R_1=x}
\left[
q_0(R_0)
\exp\left(
-\int_0^1\nabla\cdot f(t,R_t)\,dt
\right)
\right]
\tag{2.131}
$$

区别在于：

**ODE 沿一条确定性轨迹计算密度，而 SDE 需要对辅助随机轨迹取期望。**

当扩散强度为零，并且相关极限满足适当条件时，随机路径退化为确定性路径，二者可以建立联系。


### 2.6.6 Cross-Entropy 的计算

如果真实数据分布为：

$$
\rho_1(x)
$$

模型生成密度为：

$$
q_1(x)
$$

则交叉熵定义为：

$$
\boxed{
H(\rho_1,q_1)
=
-\mathbb E_{x\sim\rho_1}
[\log q_1(x)]
}
\tag{2.132}
$$

如果能够计算或估计：

$$
q_1(x)
$$

就可以进一步估计模型在真实数据上的交叉熵。

但对于 SDE，需要注意：

$$
q_1(x)=\mathbb E[W]
$$

并不意味着：

$$
\log q_1(x)=\mathbb E[\log W]
$$

因为对数函数是非线性的。

根据 Jensen 不等式：

$$
\mathbb E[\log W]\le\log\mathbb E[W]
$$

因此，使用有限条辅助随机路径进行 Monte Carlo 估计，再取对数，通常会引入估计偏差。

这也是 SDE Likelihood 估计需要额外考虑的问题。


### 2.6.7 本节总结

对于 ODE：

$$
\boxed{
\log q_1(Y_1)
=
\log q_0(Y_0)
-
\int_0^1\nabla\cdot b(t,Y_t)\,dt
}
\tag{2.133}
$$

对于 SDE：

$$
\boxed{
q_1(x)
=
\mathbb E_{\mathrm{aux}}^{\,R_1=x}
\left[
q_0(R_0)
\exp\left(
-\int_0^1\nabla\cdot f(t,R_t)\,dt
\right)
\right]
}
\tag{2.134}
$$

因此，随机插值框架不仅能够构造生成动力学，也能够建立概率密度估计方法。

---


# 第 2 章总结

本章从随机插值的定义出发，完成了整篇论文最核心的理论推导。

首先，通过端点耦合、插值函数和高斯噪声构造：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
$$

由此得到概率路径：

$$
\{\rho_t\}_{t\in[0,1]}
$$

然后证明：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

以及：

$$
\boxed{
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
}
$$

并得到：

$$
\boxed{
b=v-\gamma\dot\gamma s
}
$$

利用条件期望的 MSE 最优性，可以通过监督学习估计这些场。

接下来，使用连续性方程和 Fokker–Planck 方程，推导出：

$$
\boxed{
\begin{aligned}
\text{ODE:}\quad
&dY_t=b\,dt\\
\text{Forward SDE:}\quad
&dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}\,dW_t\\
\text{Backward SDE:}\quad
&dY_t=(b-\varepsilon s)dt+\sqrt{2\varepsilon}\,d\overleftarrow W_t
\end{aligned}
}
$$

最后，论文利用 KL、Girsanov 和随机密度表示，研究了模型近似误差及概率密度估计。


## 本章最需要掌握的三个结论

**第一，随机插值决定的是概率路径，而不是某个特定的生成动力学。**

**第二，速度场 $b$ 和 Score $s$ 是连接概率路径与 ODE/SDE 的两个核心数学对象。**

**第三，ODE 与 SDE 可以共享相同的理想边缘分布，但它们的样本轨迹、误差控制方式和数值采样性质并不相同。**

第 3 章将进一步讨论随机插值的特殊形式，包括 Diffusive Interpolants、One-Sided Interpolants、Mirror Interpolants，以及 Schrödinger Bridge。


<a id="chapter-3"></a>

# 第 3 章：Instantiations and Extensions

第 2 章建立了随机插值的基本理论：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
\tag{3.1}
$$

其中：

- $X_0\sim\rho_0$、$X_1\sim\rho_1$：两个端点随机变量。
- $I(t,X_0,X_1)$：插值函数。
- $Z\sim\mathcal N(0,I_d)$：独立标准高斯噪声。
- $\gamma(t)$：控制中间高斯扰动的系数。

我们已经证明，随机插值诱导的概率路径 $\rho_t$ 可以由 ODE 或 SDE 实现。

但前面的结论仍然比较抽象。

**第 3 章的目标，就是研究随机插值框架的几种特殊形式，并进一步说明它如何与布朗桥、扩散过程以及 Schrödinger Bridge 联系起来。**

本章包含四个部分：

- **Diffusive Interpolants**：研究随机插值与布朗桥的联系。
- **One-Sided Interpolants**：研究当一个端点为高斯分布时，随机插值如何简化。
- **Mirror Interpolants**：研究两个端点分布相同时的特殊构造。
- **Schrödinger Bridge**：研究如何在所有可行的随机路径中选择具有最小随机控制代价的路径。

其中，3.1～3.3 主要讨论**随机插值的不同构造方式**，3.4 则进一步研究**概率路径的最优性**。

---


## 3.1 Diffusive Interpolants 与 Brownian Bridge


### 3.1.1 核心结论

本节希望说明：

**随机插值可以构造与布朗桥具有相同单时刻边缘分布的随机过程，但两者的完整路径分布通常不同。**

这是一个很重要的区别。

随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

使用一个高斯随机变量 $Z$，就可以定义所有时刻的随机状态。

而布朗桥是由整个布朗运动过程构造的，它在不同时刻具有特定的时间相关性。

因此，我们需要区分：

$$
\boxed{
\text{单时刻边缘分布相同}
\neq
\text{整个随机过程相同}
}
$$

下面从 Brownian Motion 开始推导。


### 3.1.2 Brownian Motion 的基本性质

设：

$$
W_t,\qquad t\in[0,1]
$$

是一个 $d$ 维标准布朗运动。

它满足：

$$
W_0=0
$$

$$
W_t-W_s\sim\mathcal N(0,(t-s)I_d),
\qquad t>s
$$

并且不相交时间区间上的增量相互独立。

因此：

$$
\mathbb E[W_t]=0
$$

$$
\operatorname{Cov}(W_s,W_t)=\min(s,t)I_d
\tag{3.2}
$$

其中 $\operatorname{Cov}$ 表示协方差矩阵。

特别地：

$$
W_t\sim\mathcal N(0,tI_d)
$$

也就是说，布朗运动的方差随着时间线性增加。

但我们希望构造的随机过程需要同时满足两个端点约束。

例如，希望随机扰动在：

$$
t=0,\qquad t=1
$$

都为零。

普通布朗运动只满足：

$$
W_0=0
$$

一般不满足：

$$
W_1=0
$$

因此需要引入 Brownian Bridge。


### 3.1.3 Brownian Bridge 的定义

定义：

$$
\boxed{
B_t=W_t-tW_1
}
\tag{3.3}
$$

这就是从 $0$ 到 $0$ 的标准布朗桥。

首先验证端点。

当 $t=0$：

$$
B_0=W_0=0
$$

当 $t=1$：

$$
B_1=W_1-W_1=0
$$

因此：

$$
\boxed{
B_0=B_1=0
}
\tag{3.4}
$$

这意味着布朗桥在两个端点处没有随机扰动。

但在中间时刻，它仍然具有随机性。

接下来计算它的概率分布。


### 3.1.4 Brownian Bridge 的方差推导


对 $B_t=W_t-tW_1$，其均值为零，协方差为 $t(1-t)I_d$，故：

完整推导见[附录 C.1.4：布朗桥协方差与方差](#app-c-1-4)。

$$
\boxed{
B_t\sim\mathcal N(0,t(1-t)I_d)
}
\tag{3.7}
$$

这说明布朗桥在中间时刻的方差为：

$$
t(1-t)
$$

观察这个函数：

$$
t(1-t)
$$

当 $t=0$ 或 $t=1$ 时：

$$
t(1-t)=0
$$

当 $t=\frac12$ 时：

$$
t(1-t)=\frac14
$$

因此，布朗桥的随机扰动在中间时刻最大，在两个端点消失。

这与随机插值对 $\gamma(t)$ 的要求非常一致。


### 3.1.5 如何用随机插值构造布朗桥的边缘分布？

考虑随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

选择：

$$
\boxed{
\gamma(t)=\sqrt{t(1-t)}
}
\tag{3.8}
$$

于是：

$$
X_t=
I(t,X_0,X_1)+\sqrt{t(1-t)}Z
\tag{3.9}
$$

另一方面，考虑：

$$
\widetilde X_t=
I(t,X_0,X_1)+B_t
\tag{3.10}
$$

其中布朗桥 $B_t$ 与端点随机变量独立。

根据式 (3.7)：

$$
B_t\sim\mathcal N(0,t(1-t)I_d)
$$

而：

$$
\sqrt{t(1-t)}Z
\sim
\mathcal N(0,t(1-t)I_d)
$$

因此，在任意固定时刻 $t$：

$$
\boxed{
X_t\overset d=\widetilde X_t
}
\tag{3.11}
$$

也就是说，两者具有相同的单时刻边缘分布。

但是否意味着：

$$
\{X_t\}_{t\in[0,1]}
\overset d=
\{\widetilde X_t\}_{t\in[0,1]}
$$

答案一般是否定的。

原因在于两者的跨时间协方差不同。


### 3.1.6 为什么边缘相同不代表随机过程相同？

先考虑随机插值中的高斯扰动：

$$
G_t=\sqrt{t(1-t)}Z
$$

对于两个时刻 $s,t$：

$$
\begin{aligned}
\operatorname{Cov}(G_s,G_t)
&=
\sqrt{s(1-s)}
\sqrt{t(1-t)}
\operatorname{Cov}(Z)\\
&=
\sqrt{s(1-s)t(1-t)}I_d
\end{aligned}
\tag{3.12}
$$

而布朗桥的跨时间协方差为：

$$
\boxed{
\operatorname{Cov}(B_s,B_t)
=
[\min(s,t)-st]I_d
}
\tag{3.13}
$$


完整推导见[附录 C.1.4：布朗桥跨时间协方差](#app-c-1-4)。

比较式 (3.12) 和式 (3.13)，一般有：

$$
\sqrt{s(1-s)t(1-t)}
\neq
\min(s,t)-st
$$

因此：

$$
\boxed{
\text{随机插值高斯扰动与布朗桥具有相同边缘，但不同时间相关结构}
}
$$

这说明随机插值并不要求样本路径本身是布朗运动或布朗桥。

**随机插值首先构造的是概率路径，而不是随机微分方程意义下的扩散轨迹。**


### 3.1.7 Brownian Bridge 的 SDE 表示

为了进一步理解布朗桥，我们还可以推导它的 SDE。

标准布朗桥满足：

$$
\boxed{
dB_t=-\frac{B_t}{1-t}dt+dW_t,
\qquad t<1
}
\tag{3.14}
$$

为什么漂移项是：

$$
-\frac{B_t}{1-t}
$$

可以从线性 SDE 的解验证。

考虑：

$$
dB_t=-\frac{B_t}{1-t}dt+dW_t
$$

将它写成：

$$
dB_t+\frac{B_t}{1-t}dt=dW_t
$$

定义：

$$
U_t=\frac{B_t}{1-t}
$$

使用 Itô 乘积法则：

$$
dU_t=
\frac{1}{1-t}dB_t+
\frac{B_t}{(1-t)^2}dt
$$

代入 SDE：

$$
\begin{aligned}
dU_t
&=
\frac{1}{1-t}
\left(
-\frac{B_t}{1-t}dt+dW_t
\right)
+
\frac{B_t}{(1-t)^2}dt\\
&=
\frac{1}{1-t}dW_t
\end{aligned}
$$

因此：

$$
U_t=\int_0^t\frac{1}{1-s}dW_s
$$

得到：

$$
\boxed{
B_t=(1-t)\int_0^t\frac{1}{1-s}dW_s
}
\tag{3.15}
$$

根据 Itô 等距：

$$
\begin{aligned}
\operatorname{Var}(B_t)
&=
(1-t)^2\int_0^t\frac{1}{(1-s)^2}ds\\
&=
(1-t)^2
\left(
\frac{1}{1-t}-1
\right)\\
&=t(1-t)
\end{aligned}
\tag{3.16}
$$

这与前面的布朗桥方差完全一致。

同时，这个过程是中心高斯过程，且具有布朗桥的跨时间协方差，因此它与标准布朗桥具有相同的有限维分布。


#### 为什么漂移会在 $t\to1$ 时变大？

因为：

$$
-\frac{B_t}{1-t}
$$

会随着剩余时间 $1-t$ 减小而产生越来越强的回拉作用。

这使得过程能够在终点满足：

$$
B_1=0
$$

因此，Brownian Bridge 可以理解为：

**受到终点约束的布朗运动，其漂移会随着终点临近而增强。**

相关数学知识见附录 C：Brownian Bridge 与 Doob $h$-Transform。


### 3.1.8 本节总结

本节最重要的结论是：

$$
\boxed{
B_t\sim\mathcal N(0,t(1-t)I_d)
}
$$

因此可以使用：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

构造具有相同单时刻高斯扰动分布的随机插值。

但：

$$
\boxed{
\text{边缘分布相同}
\neq
\text{随机过程相同}
}
$$

随机插值并不要求真实样本路径是布朗桥。

这也再次说明：**随机插值框架首先关注概率分布演化，而不是某种固定的随机轨迹结构。**

---


## 3.2 One-Sided Interpolants：单侧随机插值


### 3.2.1 核心结论

第 2 章的随机插值是：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

它同时包含：

- 起点随机变量 $X_0$
- 终点随机变量 $X_1$
- 额外高斯噪声 $Z$

但是，在生成模型中，我们经常选择：

$$
X_0\sim\mathcal N(0,I_d)
$$

也就是说，**起点本身就是高斯随机变量。**

这时，起点 $X_0$ 与额外噪声 $Z$ 都是高斯。

那么能否把这两个高斯随机变量合并，从而简化随机插值？

答案是：在适当条件下可以。

这就是 One-Sided Interpolants 的主要动机。


### 3.2.2 从双侧线性随机插值出发


设 $X_0,Z$ 为独立标准高斯，且均与 $X_1$ 独立。对 $X_t=a(t)X_0+\beta(t)X_1+\gamma(t)Z$，令 $\alpha(t)=\sqrt{a^2(t)+\gamma^2(t)}$，再取独立于 $X_1$ 的标准高斯 $Z'$，则：

完整推导见[附录 A.3.3：独立高斯随机变量合并](#app-a-3-3)。

$$
\boxed{
X_t\overset d=
\alpha(t)Z'+\beta(t)X_1
}
\tag{3.21}
$$

这就是单侧线性随机插值的基本形式。

需要特别注意：

这个推导要求高斯部分与 $X_1$ 独立。

如果 $X_0$ 和 $X_1$ 采用相关耦合，那么即使 $aX_0+\gamma Z$ 的边缘仍然是高斯，也不能直接将它替换为一个独立于 $X_1$ 的高斯变量。


### 3.2.3 为什么叫 One-Sided？

这里的 One-Sided 并不是说只有一个端点。

仍然有：

$$
\rho_0\longrightarrow\rho_1
$$

两个端点分布。

所谓单侧，是指随机插值只需要显式采样一个非高斯数据端点，而另一端由高斯变量和时间系数构造。

其形式为：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{3.22}
$$

其中：

$$
Z\sim\mathcal N(0,I_d)
$$

$$
X_1\sim\rho_1
$$

并且：

$$
Z\perp X_1
$$

为了保证端点正确，需要：

$$
\alpha(0)=1,\qquad\beta(0)=0
$$

$$
\alpha(1)=0,\qquad\beta(1)=1
\tag{3.23}
$$

于是：

$$
X_{t=0}=Z\sim\mathcal N(0,I_d)
$$

$$
X_{t=1}=X_1\sim\rho_1
$$

这就得到从标准高斯到目标数据的随机插值。


### 3.2.4 单侧随机插值的概率密度

给定：

$$
X_1=x_1
$$

有：

$$
X_t=\alpha(t)Z+\beta(t)x_1
$$

因此：

$$
X_t\mid X_1=x_1
\sim
\mathcal N(\beta(t)x_1,\alpha^2(t)I_d)
\tag{3.24}
$$

所以条件密度为：

$$
p_t(x\mid x_1)
=
\frac{1}{(2\pi\alpha^2)^{d/2}}
\exp\left[
-\frac{\|x-\beta x_1\|^2}{2\alpha^2}
\right]
\tag{3.25}
$$

对数据分布积分：

$$
\boxed{
\rho_t(x)=
\int p_t(x\mid x_1)\rho_1(x_1)\,dx_1
}
\tag{3.26}
$$

因此，单侧随机插值的中间密度是数据分布经过缩放与高斯平滑后的结果。

这与传统扩散模型中的高斯加噪分布具有相似结构。


### 3.2.5 单侧随机插值的 Score


对 $X_t=\alpha(t)Z+\beta(t)X_1$，在 $\alpha(t)>0$ 时：

完整推导见[附录 D.1.3：高斯加噪的 Score](#app-d-1-3)。

$$
\boxed{
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{\alpha(t)}
}
\tag{3.28}
$$

定义噪声预测器：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

则：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
}
\tag{3.29}
$$

这与传统扩散模型中噪声预测和 Score 之间的关系一致。


### 3.2.6 单侧随机插值的速度场


记 $\eta_z=\mathbb E[Z\mid X_t=x]$、$\eta_1=\mathbb E[X_1\mid X_t=x]$。利用 $b=\dot\alpha\eta_z+\dot\beta\eta_1$ 和 $x=\alpha\eta_z+\beta\eta_1$，在 $\beta(t)\ne0$ 时得到：

完整推导见[附录 F.1.1：单侧线性速度场](#app-f-1-1)。

$$
\boxed{
b(t,x)=
\frac{\dot\beta(t)}{\beta(t)}x+
\left(
\dot\alpha(t)-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right)\eta_z(t,x)
}
\tag{3.35}
$$

结合：

$$
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
$$

得到：

$$
\boxed{
\begin{aligned}
b(t,x)
&=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z\\[4pt]
s(t,x)
&=-\frac{\eta_z}{\alpha}
\end{aligned}
}
\tag{3.36}
$$

**这说明单侧线性插值只需要一个噪声预测器，就可以同时恢复速度场和 Score。**

这个结果将在第 4 章进一步讨论，并用于统一 Flow Matching 与扩散模型。


### 3.2.7 单侧随机插值的局限

需要注意三个条件。

首先，起点通常需要是标准高斯。

其次，噪声变量与数据端点需要独立。

最后，使用上述显式消元公式时，需要注意：

$$
\alpha(t)=0
$$

或：

$$
\beta(t)=0
$$

的端点。

例如：

$$
\beta(0)=0
$$

因此式 (3.35) 在 $t=0$ 不能直接代入。

有些情况下，整个表达式的极限仍然存在，但必须通过极限分析或使用其他等价参数化来计算。


### 3.2.8 本节总结

单侧线性随机插值为：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
$$

其核心性质是：

$$
\boxed{
x=\alpha\eta_z+\beta\eta_1
}
$$

因此噪声预测器与数据预测器不是独立的。

最终：

$$
\boxed{
\text{一个预测器}
\Longrightarrow
\text{速度场与 Score}
\Longrightarrow
\text{ODE 与 SDE}
}
$$

这也是随机插值框架与现代扩散生成模型联系最紧密的部分之一。

---


## 3.3 Mirror Interpolants：镜像随机插值


### 3.3.1 核心结论

前面研究的随机插值主要用于连接两个不同分布：

$$
\rho_0\longrightarrow\rho_1
$$

但如果两个端点分布相同：

$$
\rho_0=\rho_1
$$

是否还有必要构造随机插值？

答案是有。

**端点分布相同，并不意味着中间概率路径必须保持不变。**

Mirror Interpolants 正是研究这种特殊情况。


### 3.3.2 Mirror Interpolants 的基本思想

假设：

$$
X_0\sim\rho
$$

希望构造一个过程：

$$
\rho\longrightarrow\rho
$$

其中中间时刻可以经过不同的概率分布。

一种简单构造是：

$$
\boxed{
X_t=K(t,X_0)+\gamma(t)Z
}
\tag{3.37}
$$

其中：

$$
K(0,x)=K(1,x)=x
$$

$$
\gamma(0)=\gamma(1)=0
$$

于是：

$$
X_{t=0}=X_0
$$

$$
X_{t=1}=X_0
$$

因此：

$$
\boxed{
\rho_{t=0}=\rho_{t=1}=\rho
}
\tag{3.38}
$$

这里不仅端点分布相同，在这个构造下，端点随机变量本身也相同。

但对于中间时刻：

$$
X_t=K(t,X_0)+\gamma(t)Z
$$

其分布一般不同于 $\rho$。


### 3.3.3 一个最简单的例子

选择：

$$
K(t,x)=x
$$

以及：

$$
\gamma(t)=\sqrt{a\,t(1-t)}
$$

则：

$$
\boxed{
X_t=X_0+\sqrt{a\,t(1-t)}Z
}
\tag{3.39}
$$

在两个端点：

$$
X_{t=0}=X_{t=1}=X_0
$$

但在中间时刻：

$$
X_{1/2}=X_0+\frac{\sqrt a}{2}Z
$$

因此中间分布是原始分布经过高斯平滑后的结果。

这条路径可以理解为：

$$
\boxed{
\text{原始分布}
\longrightarrow
\text{高斯平滑分布}
\longrightarrow
\text{原始分布}
}
$$

这说明即使端点分布完全相同，仍然可以构造非平凡的中间概率路径。


### 3.3.4 Mirror Interpolants 的速度场与 Score

考虑一般形式：

$$
X_t=K(t,X_0)+\gamma(t)Z
$$

对时间求导：

$$
\dot X_t=
\partial_tK(t,X_0)+\dot\gamma(t)Z
\tag{3.40}
$$

因此边缘速度场为：

$$
\boxed{
b(t,x)=
\mathbb E[
\partial_tK(t,X_0)+\dot\gamma(t)Z
\mid X_t=x
]
}
\tag{3.41}
$$

同样，由高斯条件分布可得：

$$
\boxed{
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
}
\tag{3.42}
$$

定义：

$$
v(t,x)=
\mathbb E[\partial_tK(t,X_0)\mid X_t=x]
$$

则：

$$
\boxed{
b=v-\gamma\dot\gamma s
}
\tag{3.43}
$$

这与第 2 章的统一公式完全一致。


### 3.3.5 特殊情况：只添加高斯噪声


若 $K(t,x)=x$，则 $\partial_tK=0$、$v=0$，从而：

完整推导见[附录 F.5：Mirror 的速度场](#app-f-5)。

$$
\boxed{
b(t,x)=-\gamma(t)\dot\gamma(t)s(t,x)
}
\tag{3.44}
$$

这说明在这种特殊情况下，**边缘速度场完全由 Score 决定**。

例如：

$$
\gamma(t)=\sqrt{a\,t(1-t)}
$$


此时 $\gamma\dot\gamma=\tfrac a2(1-2t)$，因此（计算见[附录 F.5](#app-f-5)）：

$$
\boxed{
b(t,x)=
-\frac a2(1-2t)s(t,x)
}
\tag{3.45}
$$

观察时间变化。

当：

$$
t<\frac12
$$

有：

$$
1-2t>0
$$

因此：

$$
b=-c(t)s,\qquad c(t)>0
$$

速度方向与 Score 相反。

这对应分布逐渐被高斯平滑的阶段。

当：

$$
t>\frac12
$$

有：

$$
1-2t<0
$$

因此：

$$
b=+c(t)s
$$

速度方向与 Score 相同。

这对应分布逐渐恢复的阶段。

当：

$$
t=\frac12
$$

有：

$$
b=0
$$

这里的 $b=0$ 表示中间时刻的瞬时边缘速度为零，并不表示整个随机过程停止变化。


### 3.3.6 Mirror Interpolants 的意义

Mirror Interpolants 说明：

**概率路径的设计空间比简单的“噪声到数据”更广。**

随机插值既可以用于：

$$
\rho_0\longrightarrow\rho_1
$$

也可以用于：

$$
\rho\longrightarrow\rho
$$

甚至可以通过不同的中间路径实现相同的端点分布。

这再次说明：

$$
\boxed{
\text{端点分布并不能唯一决定中间概率路径}
}
$$

而不同概率路径又可以进一步对应不同的 ODE/SDE 动力学。

---


## 3.4 Schrödinger Bridge：最优随机路径与变分推导


### 3.4.1 核心结论

前面已经证明：

给定两个端点分布：

$$
\rho_0,\qquad\rho_1
$$

我们可以构造无数条随机插值路径。

例如：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

不同的插值函数、耦合和噪声日程，都可以产生不同的中间概率路径。

那么自然产生一个问题：

**在所有连接两个端点分布的随机路径中，是否存在某种意义下最优的路径？**

Schrödinger Bridge（SB）研究的就是这个问题。

它的核心思想是：

> 在所有满足指定端点分布约束的随机过程之中，寻找与某个参考随机扩散过程最接近的过程。

这里的“最接近”通常通过路径概率测度之间的 KL 散度衡量。

因此：

$$
\boxed{
\text{Schrödinger Bridge}
=
\text{带端点约束的路径 KL 最小化}
}
$$

本节将从路径 KL 出发，推导控制能量形式，再利用变分法得到最优控制场，最后解释随机插值如何用于求解 SB。


### 3.4.2 Schrödinger Bridge 的基本定义

首先定义一个参考扩散过程：

$$
\boxed{
dY_t=\sqrt{2\varepsilon}\,dW_t
}
\tag{3.46}
$$

其中：

$$
\varepsilon>0
$$

表示扩散强度。

这里先选择最简单的无漂移布朗扩散作为参考过程。

设其路径概率测度为：

$$
P_{\mathrm{ref}}
$$

现在考虑一个带控制漂移的扩散过程：

$$
\boxed{
dY_t=u(t,Y_t)dt+\sqrt{2\varepsilon}\,dW_t
}
\tag{3.47}
$$

其中：

$$
u(t,x)
$$

是需要优化的控制场。

它的路径概率测度记为：

$$
P_u
$$

我们要求这个受控过程满足：

$$
Y_0\sim\rho_0
$$

$$
Y_1\sim\rho_1
\tag{3.48}
$$

然后寻找：

$$
\boxed{
\min_u D_{\mathrm{KL}}(P_u\|P_{\mathrm{ref}})
}
\tag{3.49}
$$

其中两个过程采用相同的初始分布 $\rho_0$。

这就是 Schrödinger Bridge 的基本随机控制形式。


### 3.4.3 为什么路径 KL 等价于控制能量？


参考过程漂移为零，受控过程漂移为 $u$，二者采用相同初始分布和扩散系数 $\sqrt{2\varepsilon}I_d$。由 Girsanov 定理，在适当绝对连续性及可积性条件下：

完整推导见[附录 C.4：Girsanov 与控制能量](#app-c-4)。

$$
\boxed{
D_{\mathrm{KL}}(P_u\|P_{\mathrm{ref}})
=
\frac{1}{4\varepsilon}
\mathbb E_{P_u}
\int_0^1\|u(t,Y_t)\|^2dt
}
\tag{3.51}
$$

因为：

$$
\mathbb E_{P_u}[\|u(t,Y_t)\|^2]
=
\int\rho_t(x)\|u(t,x)\|^2dx
$$

所以：

$$
\boxed{
D_{\mathrm{KL}}(P_u\|P_{\mathrm{ref}})
=
\frac{1}{4\varepsilon}
\int_0^1\int
\rho_t(x)\|u(t,x)\|^2dxdt
}
\tag{3.52}
$$

因此，最小化路径 KL 等价于最小化：

$$
\boxed{
\int_0^1\int
\rho_t(x)\|u(t,x)\|^2dxdt
}
\tag{3.53}
$$

这就是控制能量。


#### 为什么这个结果重要？

原始问题是在路径空间上最小化 KL：

$$
\min D_{\mathrm{KL}}(P_u\|P_{\mathrm{ref}})
$$

路径空间是无限维的，直接优化比较困难。

但通过 Girsanov 定理，我们将它转化成了一个时间—空间积分：

$$
\min_u
\int_0^1\int\rho_t\|u\|^2dxdt
$$

这使得问题可以进一步使用偏微分方程和变分法研究。


### 3.4.4 Schrödinger Bridge 的约束优化形式

受控 SDE 为：

$$
dY_t=u(t,Y_t)dt+\sqrt{2\varepsilon}dW_t
$$

根据 Fokker–Planck 方程：

$$
\boxed{
\partial_t\rho_t
=
-\nabla\cdot(\rho_tu)
+\varepsilon\Delta\rho_t
}
\tag{3.54}
$$

因此，SB 可以写成：

$$
\boxed{
\begin{aligned}
\min_{\rho,u}\quad&
\frac12
\int_0^1\int
\rho_t(x)\|u(t,x)\|^2dxdt\\
\text{s.t.}\quad&
\partial_t\rho_t+
\nabla\cdot(\rho_tu)
-\varepsilon\Delta\rho_t=0\\
&\rho_{t=0}=\rho_0\\
&\rho_{t=1}=\rho_1
\end{aligned}
}
\tag{3.55}
$$

这里将目标函数乘以常数：

$$
2\varepsilon
$$

不会改变最优解，因此可以使用：

$$
\frac12\int\rho\|u\|^2
$$

作为目标。

这个优化问题包含两个未知量：

- $\rho_t(x)$：中间概率密度。
- $u(t,x)$：控制漂移场。

并且它们必须满足 Fokker–Planck 约束。


### 3.4.5 使用拉格朗日乘子处理 PDE 约束

接下来使用变分法。

数学基础见附录 D：变分法与 Euler–Lagrange 方程。

对于约束：

$$
\partial_t\rho+
\nabla\cdot(\rho u)
-\varepsilon\Delta\rho=0
$$

引入拉格朗日乘子：

$$
\lambda(t,x)
$$

定义拉格朗日泛函：

$$
\begin{aligned}
\mathcal L(\rho,u,\lambda)
={}&
\int_0^1\int
\frac12\rho\|u\|^2dxdt\\
&+
\int_0^1\int
\lambda
\left[
\partial_t\rho+
\nabla\cdot(\rho u)
-\varepsilon\Delta\rho
\right]dxdt
\end{aligned}
\tag{3.56}
$$

其中 $\lambda$ 的作用是将 PDE 约束加入优化目标。

现在分别对：

$$
u,\qquad\rho
$$

进行变分。


### 3.4.6 对控制场 $u$ 进行变分

首先处理：

$$
\int\lambda\nabla\cdot(\rho u)\,dx
$$

利用分部积分：

$$
\int\lambda\nabla\cdot(\rho u)\,dx
=
-\int\rho u\cdot\nabla\lambda\,dx
\tag{3.57}
$$

忽略满足相应边界条件时消失的边界项。

因此，与 $u$ 有关的部分为：

$$
\int
\rho
\left[
\frac12\|u\|^2-u\cdot\nabla\lambda
\right]dx
\tag{3.58}
$$

对 $u$ 求变分：

$$
\frac{\delta\mathcal L}{\delta u}
=
\rho(u-\nabla\lambda)
\tag{3.59}
$$

令其为零：

$$
\rho(u-\nabla\lambda)=0
$$

在 $\rho>0$ 的区域：

$$
\boxed{
u^*=\nabla\lambda
}
\tag{3.60}
$$

这说明：

**Schrödinger Bridge 的最优控制漂移可以表示为某个标量势函数的梯度。**

这是一个非常重要的结构性结论。


### 3.4.7 对概率密度 $\rho$ 进行变分

现在处理拉格朗日泛函中与 $\rho$ 有关的部分。

原式为：

$$
\begin{aligned}
\mathcal L
={}&
\int_0^1\int
\frac12\rho\|u\|^2dxdt\\
&+
\int_0^1\int
\lambda\partial_t\rho\,dxdt\\
&+
\int_0^1\int
\lambda\nabla\cdot(\rho u)\,dxdt\\
&-
\varepsilon\int_0^1\int
\lambda\Delta\rho\,dxdt
\end{aligned}
\tag{3.61}
$$

分别处理后三项。

**第一项：时间分部积分**

$$
\int_0^1\lambda\partial_t\rho\,dt
=
[\lambda\rho]_0^1
-
\int_0^1\rho\partial_t\lambda\,dt
\tag{3.62}
$$

由于端点密度固定，变分时端点项不产生自由变化。

**第二项：空间散度分部积分**

$$
\int\lambda\nabla\cdot(\rho u)\,dx
=
-\int\rho u\cdot\nabla\lambda\,dx
\tag{3.63}
$$

**第三项：拉普拉斯分部积分**

在适当边界条件下：

$$
\int\lambda\Delta\rho\,dx
=
\int\rho\Delta\lambda\,dx
\tag{3.64}
$$

因此：

$$
-\varepsilon\int\lambda\Delta\rho\,dx
=
-\varepsilon\int\rho\Delta\lambda\,dx
$$

将这些结果代回，忽略固定端点项：

$$
\begin{aligned}
\mathcal L
=
\int_0^1\int\rho
\Big[
&\frac12\|u\|^2
-\partial_t\lambda\\
&-u\cdot\nabla\lambda
-\varepsilon\Delta\lambda
\Big]dxdt
\end{aligned}
\tag{3.65}
$$

对 $\rho$ 求变分：

$$
\frac{\delta\mathcal L}{\delta\rho}
=
\frac12\|u\|^2
-\partial_t\lambda
-u\cdot\nabla\lambda
-\varepsilon\Delta\lambda
\tag{3.66}
$$

令其为零：

$$
\frac12\|u\|^2
-\partial_t\lambda
-u\cdot\nabla\lambda
-\varepsilon\Delta\lambda=0
\tag{3.67}
$$

代入：

$$
u=\nabla\lambda
$$

得到：

$$
\frac12\|\nabla\lambda\|^2
-\partial_t\lambda
-\|\nabla\lambda\|^2
-\varepsilon\Delta\lambda=0
$$

整理：

$$
\boxed{
\partial_t\lambda
+
\frac12\|\nabla\lambda\|^2
+
\varepsilon\Delta\lambda
=0
}
\tag{3.68}
$$

这就是一个带黏性项的 Hamilton–Jacobi 方程。

因此，SB 的最优控制满足：

$$
\boxed{
\begin{aligned}
u^*&=\nabla\lambda\\
\partial_t\lambda
+\frac12\|\nabla\lambda\|^2
+\varepsilon\Delta\lambda&=0
\end{aligned}
}
\tag{3.69}
$$

同时，概率密度还必须满足 Fokker–Planck 方程与两个端点约束。

这构成一个耦合的最优性系统。


### 3.4.8 随机插值如何与 Schrödinger Bridge 联系？

前面第 2 章已经证明：

对于任意满足条件的随机插值概率路径 $\rho_t$，都可以构造：

$$
\boxed{
u=b+\varepsilon s
}
\tag{3.70}
$$

使得：

$$
\partial_t\rho_t
=
-\nabla\cdot(\rho_tu)
+\varepsilon\Delta\rho_t
$$

因此，随机插值能够提供一个满足端点约束的可行扩散过程。

这意味着：

$$
\boxed{
\text{随机插值}
\Longrightarrow
\text{SB 优化问题的可行候选路径}
}
$$

但是，需要特别强调：

**可行路径不等于最优路径。**

对于不同随机插值：

$$
I,\quad\gamma,\quad\nu
$$

我们会得到不同的：

$$
\rho_t,\quad b,\quad s
$$

从而得到不同的控制漂移：

$$
u=b+\varepsilon s
$$

对应的控制能量也不同：

$$
\mathcal E=
\int_0^1\int\rho_t\|u\|^2dxdt
\tag{3.71}
$$

因此，Schrödinger Bridge 需要进一步在这些可行路径中寻找控制能量最小的解。


### 3.4.9 为什么还需要优化概率路径？

考虑两个不同的随机插值：

$$
X_t^{(1)}
=
I_1(t,X_0,X_1)+\gamma_1(t)Z
$$

$$
X_t^{(2)}
=
I_2(t,X_0,X_1)+\gamma_2(t)Z
$$

假设它们都满足：

$$
\rho_0^{(1)}=\rho_0^{(2)}=\rho_0
$$

$$
\rho_1^{(1)}=\rho_1^{(2)}=\rho_1
$$

但是中间概率路径可能不同：

$$
\rho_t^{(1)}\neq\rho_t^{(2)}
$$

于是它们对应的控制场：

$$
u_1=b_1+\varepsilon s_1
$$

$$
u_2=b_2+\varepsilon s_2
$$

也一般不同。

因此：

$$
\mathcal E_1
=
\int\rho_t^{(1)}\|u_1\|^2
$$

$$
\mathcal E_2
=
\int\rho_t^{(2)}\|u_2\|^2
$$

可能满足：

$$
\mathcal E_1>\mathcal E_2
$$

那么第二条路径在 SB 的控制能量意义下更优。

这就是为什么：

**仅仅训练出随机插值的速度场和 Score，并不能保证得到 Schrödinger Bridge。**

还必须进一步优化插值所定义的概率路径。


### 3.4.10 随机插值与 SB 优化的关系总结

现在可以把整个逻辑整理为：

首先，构造随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

得到概率路径：

$$
\rho_t
$$

然后推导：

$$
b(t,x)
$$

以及：

$$
s(t,x)
$$

再构造扩散漂移：

$$
u=b+\varepsilon s
$$

于是得到一个满足端点约束的扩散过程：

$$
dY_t=u(t,Y_t)dt+\sqrt{2\varepsilon}dW_t
$$

最后，研究：

$$
\boxed{
\min
\int_0^1\int\rho_t(x)\|u(t,x)\|^2dxdt
}
\tag{3.72}
$$

其中最小化需要在满足端点和 Fokker–Planck 约束的可行概率路径与控制场上进行。

如果能够找到全局最优解，就得到相应参考扩散下的 Schrödinger Bridge。

因此：

$$
\boxed{
\text{随机插值提供可行路径}
\quad+\quad
\text{控制能量优化}
\quad\Longrightarrow\quad
\text{Schrödinger Bridge}
}
$$

这里还有一个重要区别：

**随机插值框架提供了构造可行解的方法，但是否能够通过所选插值参数化覆盖真正的 SB 最优解，需要额外分析。**

如果限制插值函数的形式过于简单，优化可能只能得到某个受限函数族内的最优路径，而不是整个 SB 问题的全局最优解。

---


# 第 3 章总结

本章研究了随机插值的三个特殊形式，以及它与最优随机运输的联系。


### 1. Diffusive Interpolants

通过 Brownian Bridge，我们看到：

$$
B_t\sim\mathcal N(0,t(1-t)I_d)
$$

因此可以使用：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

构造具有相同单时刻高斯扰动分布的随机插值。

但要注意：

$$
\boxed{
\text{边缘分布相同}
\neq
\text{路径分布相同}
}
$$


### 2. One-Sided Interpolants

当初始分布为标准高斯时，可以构造：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
$$

并利用：

$$
x=\alpha\eta_z+\beta\eta_1
$$

得到：

$$
\boxed{
\begin{aligned}
b&=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z\\
s&=-\frac{\eta_z}{\alpha}
\end{aligned}
}
$$

因此可以只使用一个预测器恢复速度场和 Score。


### 3. Mirror Interpolants

即使两个端点分布相同：

$$
\rho_0=\rho_1
$$

也可以构造非平凡的中间概率路径。

对于简单的高斯平滑镜像插值：

$$
X_t=X_0+\gamma(t)Z
$$

有：

$$
\boxed{
b=-\gamma\dot\gamma s
}
$$

这说明速度场可以完全由 Score 决定。


### 4. Schrödinger Bridge

Schrödinger Bridge 研究的是：

$$
\boxed{
\min_{P:\,P_0=\rho_0,\;P_1=\rho_1}
D_{\mathrm{KL}}(P\|P_{\mathrm{ref}})
}
$$

通过 Girsanov 定理，它可以转化为最小随机控制能量问题。

对于布朗扩散参考过程，其最优控制具有势函数梯度结构：

$$
\boxed{
u^*=\nabla\lambda
}
$$

并满足相应的 Hamilton–Jacobi 最优性条件。

**随机插值能够构造满足端点约束的可行扩散，但要得到 Schrödinger Bridge，还需要进一步优化概率路径。**

---

**本章最重要的认识是：随机插值不仅是一种连接噪声与数据的方式，而是一个能够构造不同概率路径、不同生成动力学，并进一步研究最优随机运输的统一数学框架。**

下一章将研究 **Spatially Linear Interpolants（空间线性随机插值）**，重点推导不同插值日程的数学性质，以及为什么单侧线性插值能够只使用一个预测器来同时恢复速度场和 Score。


<a id="chapter-4"></a>

# 第 4 章：Spatially Linear Interpolants

第 2 章建立了随机插值的统一理论，第 3 章研究了布朗桥、单侧插值以及 Schrödinger Bridge 等特殊情况。

本章进一步研究一种具有重要实际价值的随机插值：

**Spatially Linear Interpolants（空间线性随机插值）。**

这里的 Linear 指的是**插值函数对空间端点变量是线性的**，而不是要求时间系数必须是线性函数。

例如：

$$
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
$$

虽然 $\alpha(t)$、$\beta(t)$、$\gamma(t)$ 可以是非线性时间函数，但该插值仍然属于空间线性随机插值。

这种结构的重要性在于：我们能够利用线性关系，推导速度场、Score 以及不同条件预测器之间的代数关系。

特别是当初始分布为标准高斯时，**单侧线性插值可以仅使用一个预测器，同时恢复速度场和 Score，从而构造 ODE 与 SDE。**

本章围绕四个问题展开：

1. 线性随机插值具有什么特殊的数学结构？
2. 不同插值日程会怎样改变概率路径？
3. 插值噪声 $\gamma$ 与采样扩散强度 $\varepsilon$ 有什么区别？
4. 为什么单侧线性插值可以只使用一个预测器？

---


## 4.1 线性随机插值的定义与基本性质


### 4.1.1 核心结论

对于空间线性随机插值：

$$
\boxed{
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
}
\tag{4.1}
$$

我们可以推导出：

$$
\boxed{
b(t,x)=
\dot\alpha(t)m_0(t,x)
+\dot\beta(t)m_1(t,x)
+\dot\gamma(t)m_z(t,x)
}
\tag{4.2}
$$

其中：

$$
\begin{aligned}
m_0(t,x)&=\mathbb E[X_0\mid X_t=x]\\
m_1(t,x)&=\mathbb E[X_1\mid X_t=x]\\
m_z(t,x)&=\mathbb E[Z\mid X_t=x]
\end{aligned}
$$

同时：

$$
\boxed{
s(t,x)=-\frac{m_z(t,x)}{\gamma(t)}
}
\tag{4.3}
$$

这些条件均值还满足：

$$
\boxed{
x=\alpha(t)m_0(t,x)+\beta(t)m_1(t,x)+\gamma(t)m_z(t,x)
}
\tag{4.4}
$$

这三个公式构成空间线性随机插值的主要代数基础。


### 4.1.2 线性随机插值的定义

考虑两个端点随机变量：

$$
X_0\sim\rho_0,\qquad X_1\sim\rho_1
$$

它们具有某个联合分布：

$$
(X_0,X_1)\sim\nu
$$

这里的 $\nu$ 是端点耦合，不要求两个端点相互独立。

另外引入：

$$
Z\sim\mathcal N(0,I_d)
$$

并要求 $Z$ 独立于端点对 $(X_0,X_1)$。

定义：

$$
\boxed{
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
}
\tag{4.5}
$$

其中：

- $\alpha(t)$：初始端点的权重。
- $\beta(t)$：目标端点的权重。
- $\gamma(t)$：额外高斯扰动的幅度。

为了保证端点正确，要求：

$$
\begin{aligned}
\alpha(0)&=1,&\beta(0)&=0,&\gamma(0)&=0\\
\alpha(1)&=0,&\beta(1)&=1,&\gamma(1)&=0
\end{aligned}
\tag{4.6}
$$

于是：

$$
X_{t=0}=X_0
$$

$$
X_{t=1}=X_1
$$

因此：

$$
\rho_{t=0}=\rho_0,\qquad\rho_{t=1}=\rho_1
$$

这里需要强调：

**空间线性不意味着中间概率密度是两个端点密度的线性组合。**

一般情况下：

$$
\boxed{
\rho_t(x)\neq
\alpha(t)\rho_0(x)+\beta(t)\rho_1(x)
}
$$

因为随机变量的线性组合与概率密度的线性混合是两种不同操作。

例如：

$$
X_t=(1-t)X_0+tX_1
$$

表示两个随机变量的加权和，而不是以概率 $1-t$ 选择 $X_0$、以概率 $t$ 选择 $X_1$。

两者诱导的概率分布通常不同。


### 4.1.3 速度场的推导

首先对随机插值求时间导数：

$$
\dot X_t=
\dot\alpha(t)X_0+
\dot\beta(t)X_1+
\dot\gamma(t)Z
\tag{4.7}
$$

根据第 2.2 节的结论：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

代入：

$$
\begin{aligned}
b(t,x)
={}&
\mathbb E[
\dot\alpha X_0+\dot\beta X_1+\dot\gamma Z
\mid X_t=x]
\end{aligned}
$$

由于时间系数是确定性的，可以提出条件期望：

$$
\begin{aligned}
b(t,x)
={}&
\dot\alpha\mathbb E[X_0\mid X_t=x]\\
&+\dot\beta\mathbb E[X_1\mid X_t=x]\\
&+\dot\gamma\mathbb E[Z\mid X_t=x]
\end{aligned}
\tag{4.8}
$$

定义：

$$
m_0=\mathbb E[X_0\mid X_t=x]
$$

$$
m_1=\mathbb E[X_1\mid X_t=x]
$$

$$
m_z=\mathbb E[Z\mid X_t=x]
$$

于是：

$$
\boxed{
b=\dot\alpha m_0+\dot\beta m_1+\dot\gamma m_z
}
\tag{4.9}
$$

这就是空间线性随机插值的边缘速度场。

注意：

$$
\dot X_t
$$

是具体样本的速度，而：

$$
b(t,x)
$$

是给定当前位置后的条件平均速度。

即使 $\dot X_t$ 对时间是常数，$b(t,x)$ 也不一定是常数，因为条件分布会随时间和位置变化。


### 4.1.4 Score 的推导


将一般插值函数取为 $I=\alpha X_0+\beta X_1$，并记 $m_z(t,x)=\mathbb E[Z\mid X_t=x]$，直接得到：

完整推导见[附录 D.1.3：一般随机插值的 Score](#app-d-1-3)。

$$
\boxed{
s(t,x)=-\frac{m_z(t,x)}{\gamma(t)}
}
\tag{4.12}
$$

该公式要求 $\gamma(t)>0$。当 $\gamma=0$ 时，需要使用其他方法分析 Score。


### 4.1.5 三个条件均值之间的关系

从原始随机插值：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

出发。

给定：

$$
X_t=x
$$

两边取条件期望：

$$
\mathbb E[X_t\mid X_t=x]
=
\mathbb E[\alpha X_0+\beta X_1+\gamma Z\mid X_t=x]
$$

左侧为：

$$
\mathbb E[X_t\mid X_t=x]=x
$$

右侧展开：

$$
\alpha m_0+\beta m_1+\gamma m_z
$$

因此：

$$
\boxed{
x=\alpha m_0+\beta m_1+\gamma m_z
}
\tag{4.13}
$$

这是空间线性插值最重要的代数约束之一。

它说明：

$$
m_0,\quad m_1,\quad m_z
$$

并不是完全独立的。

但这里有一个需要注意的细节：

**一个向量方程只能提供 $d$ 个约束，而三个条件均值总共有 $3d$ 个分量。**

因此，一般情况下，仅凭式 (4.13) 不能从一个条件均值恢复另外两个。

这也是双侧线性随机插值与单侧线性随机插值的重要区别。


### 4.1.6 双侧线性插值为什么通常需要多个预测器？

对于双侧线性随机插值：

$$
b=\dot\alpha m_0+\dot\beta m_1+\dot\gamma m_z
$$

$$
s=-\frac{m_z}{\gamma}
$$

我们希望得到：

$$
b,\quad s
$$

如果已经知道 $m_z$，就能得到 Score。

但是速度场还依赖：

$$
m_0,\quad m_1
$$

利用：

$$
x=\alpha m_0+\beta m_1+\gamma m_z
$$

只能消去其中一个条件均值。

因此，通常还需要另一个独立的预测信息。

当然，这并不意味着一定要使用两个独立神经网络。

可以使用一个网络输出两个向量，也可以直接预测：

$$
b,\quad s
$$

这里讨论的是所需的**独立预测信息**，而不是网络数量。


### 4.1.7 本节总结

空间线性随机插值为：

$$
\boxed{
X_t=\alpha X_0+\beta X_1+\gamma Z
}
$$

其三个核心关系为：

$$
\boxed{
\begin{aligned}
b&=\dot\alpha m_0+\dot\beta m_1+\dot\gamma m_z\\
s&=-\frac{m_z}{\gamma}\\
x&=\alpha m_0+\beta m_1+\gamma m_z
\end{aligned}
}
\tag{4.14}
$$

后面所有线性插值的特殊结果，基本都可以从这三个公式出发推导。

---


## 4.2 不同插值日程及其数学性质


### 4.2.1 核心结论

空间线性随机插值中的三个时间系数：

$$
\alpha(t),\quad\beta(t),\quad\gamma(t)
$$

决定了中间概率路径的形状。

不同日程会影响：

- 样本在不同时间的插值位置。
- 中间分布的方差和协方差。
- 速度场的时间变化。
- 训练监督目标的数值尺度。
- 生成 ODE 的数值积分难度。

但需要注意：

**日程函数越平滑，并不必然意味着边缘 ODE 轨迹越直，也不必然意味着数值采样更容易。**

因为边缘速度场还取决于端点的条件分布。

下面讨论几种典型日程。


### 4.2.2 Linear Interpolant：线性时间日程

最简单的选择是：

$$
\boxed{
\alpha(t)=1-t,\qquad\beta(t)=t
}
\tag{4.15}
$$

如果不添加额外高斯噪声：

$$
\gamma(t)=0
$$

得到：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{4.16}
$$

对时间求导：

$$
\boxed{
\dot X_t=X_1-X_0
}
\tag{4.17}
$$

因此，对于固定的端点样本：

$$
X_0,\quad X_1
$$

样本沿直线运动，而且速度恒定。

但边缘速度场为：

$$
\boxed{
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
}
\tag{4.18}
$$

这个条件期望通常依赖：

$$
t,\quad x
$$

所以：

$$
\boxed{
\dot X_t=\text{常数}
\not\Rightarrow
b(t,x)=\text{常数}
}
$$

这也是为什么 Flow Matching 使用直线插值训练，推理时仍然需要多步 ODE 积分。

**关键区别：**

- 条件样本轨迹是直线。
- 边缘 ODE 轨迹不一定是直线。

二者只有在特殊条件下才会一致。


#### 添加高斯扰动

如果选择：

$$
\gamma(t)=\sqrt{a\,t(1-t)},\qquad a>0
$$

则：

$$
X_t=(1-t)X_0+tX_1+\sqrt{a\,t(1-t)}Z
\tag{4.19}
$$

对时间求导：

$$
\dot X_t=
X_1-X_0+
\frac{a(1-2t)}{2\sqrt{a\,t(1-t)}}Z
\tag{4.20}
$$

这时条件样本轨迹一般不再是直线。

同时可以看到，当：

$$
t\to0,\qquad t\to1
$$

时，噪声系数的导数可能发散。

这会给直接预测 $\dot X_t$ 的训练带来数值问题。

但不能直接据此断言边缘速度场 $b(t,x)$ 也必然发散，因为条件期望可能产生抵消。


### 4.2.3 Trigonometric Interpolant：三角函数日程

另一种常见选择是：

$$
\boxed{
\alpha(t)=\cos\left(\frac{\pi t}{2}\right)
}
$$

$$
\boxed{
\beta(t)=\sin\left(\frac{\pi t}{2}\right)
}
\tag{4.21}
$$

它满足：

$$
\alpha(0)=1,\quad\beta(0)=0
$$

$$
\alpha(1)=0,\quad\beta(1)=1
$$

因此：

$$
X_t=
\cos\left(\frac{\pi t}{2}\right)X_0+
\sin\left(\frac{\pi t}{2}\right)X_1
\tag{4.22}
$$

如果没有额外噪声，对时间求导：

$$
\boxed{
\dot X_t=
-\frac{\pi}{2}\sin\left(\frac{\pi t}{2}\right)X_0+
\frac{\pi}{2}\cos\left(\frac{\pi t}{2}\right)X_1
}
\tag{4.23}
$$

与线性时间日程不同，样本速度随时间变化。

这种日程有一个重要性质：

$$
\boxed{
\alpha^2(t)+\beta^2(t)=1
}
\tag{4.24}
$$

因为：

$$
\cos^2\theta+\sin^2\theta=1
$$

这个性质在分析方差时非常有用。


#### 为什么三角日程可能保持方差？

假设：

$$
X_0,\quad X_1
$$

相互独立，并且都具有零均值、单位协方差：

$$
\operatorname{Cov}(X_0)=\operatorname{Cov}(X_1)=I_d
$$

那么：

$$
X_t=\alpha X_0+\beta X_1
$$

其协方差为：

$$
\operatorname{Cov}(X_t)
=
\alpha^2I_d+\beta^2I_d
$$

因此：

$$
\operatorname{Cov}(X_t)
=
(\alpha^2+\beta^2)I_d
$$

利用式 (4.24)：

$$
\boxed{
\operatorname{Cov}(X_t)=I_d
}
\tag{4.25}
$$

这说明在上述假设下，三角日程可以保持总体协方差不变。

但如果：

$$
\operatorname{Cov}(X_1)\neq I_d
$$

或者：

$$
X_0,\quad X_1
$$

存在相关性，那么这个结论一般不成立。

因此，**三角日程具有方差保持结构，但不意味着对任意端点分布都严格保持方差。**


### 4.2.4 Gaussian Encoding-Decoding Interpolant

另一类构造是将概率路径理解为两个阶段：

$$
\boxed{
\text{初始分布}
\longrightarrow
\text{高斯中间分布}
\longrightarrow
\text{目标分布}
}
$$

可以通过分段时间系数实现。

例如，在前半段逐渐减小初始数据的权重，同时增加高斯噪声。

在后半段逐渐减少高斯噪声，同时增加目标数据的权重。

一种示意性的构造为：

$$
X_t=
\begin{cases}
(1-2t)X_0+2tZ,&0\le t\le\frac12\\[4pt]
(2-2t)Z+(2t-1)X_1,&\frac12<t\le1
\end{cases}
\tag{4.26}
$$

观察三个关键时刻：

当 $t=0$：

$$
X_0=X_0
$$

当 $t=\frac12$：

$$
X_{1/2}=Z
$$

当 $t=1$：

$$
X_1=X_1
$$

因此，中间时刻恰好是标准高斯分布。

这可以理解为：

**先将初始分布编码到高斯，再从高斯解码到目标分布。**

式 (4.26) 是说明这种构造思想的分段示例，并非要求所有 Gaussian Encoding-Decoding Interpolants 都采用该具体日程。

它与前面的直接线性插值不同。

直接线性插值：

$$
\rho_0\longrightarrow\rho_1
$$

Gaussian Encoding-Decoding：

$$
\rho_0\longrightarrow\mathcal N(0,I_d)\longrightarrow\rho_1
$$

两者具有相同端点，但中间概率路径不同。


### 4.2.5 不同日程的协方差分析

为了统一比较不同日程，考虑：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

定义：

$$
\mu_0=\mathbb E[X_0],\qquad
\mu_1=\mathbb E[X_1]
$$

以及：

$$
\Sigma_0=\operatorname{Cov}(X_0)
$$

$$
\Sigma_1=\operatorname{Cov}(X_1)
$$

$$
\Sigma_{01}=\operatorname{Cov}(X_0,X_1)
$$

由于 $Z$ 独立于端点：

$$
\mathbb E[X_t]=\alpha\mu_0+\beta\mu_1
\tag{4.27}
$$

其协方差为：

$$
\boxed{
\begin{aligned}
\operatorname{Cov}(X_t)
={}&
\alpha^2\Sigma_0+
\beta^2\Sigma_1+
\gamma^2I_d\\
&+\alpha\beta
\left(
\Sigma_{01}+\Sigma_{01}^{\mathsf T}
\right)
\end{aligned}
}
\tag{4.28}
$$

这个公式可以直接由协方差的双线性性质得到。

它揭示了四个影响因素：

1. 初始分布的协方差。
2. 目标分布的协方差。
3. 额外高斯噪声的方差。
4. 两个端点之间的相关性。

因此，概率路径不仅由时间日程决定，还受到端点耦合的影响。

这也是为什么不同 Coupling 可能产生不同的 Flow Matching 速度场。


### 4.2.6 本节总结

不同插值日程的主要特点可以概括为：

| 日程 | 典型形式 | 主要特点 |
|---|---|---|
| Linear | $\alpha=1-t,\ \beta=t$ | 无额外噪声时，条件样本轨迹为直线 |
| Trigonometric | $\alpha=\cos(\pi t/2),\ \beta=\sin(\pi t/2)$ | 满足 $\alpha^2+\beta^2=1$ |
| Gaussian Encoding-Decoding | 分段连接高斯中间分布 | 可以显式经过高斯中间分布 |

需要掌握的核心认识是：

$$
\boxed{
\text{时间日程}
+
\text{端点耦合}
+
\text{插值噪声}
\Longrightarrow
\text{中间概率路径}
}
$$

而不是仅由时间日程单独决定概率路径。

---


## 4.3 插值噪声与采样扩散强度的作用


### 4.3.1 核心结论

在随机插值框架中，有两个容易混淆的参数：

$$
\gamma(t),\qquad \varepsilon(t)
$$

它们都与随机性有关，但作用完全不同。

**插值噪声 $\gamma(t)$ 决定概率路径本身；采样扩散强度 $\varepsilon(t)$ 决定如何通过 SDE 实现这条概率路径。**

具体来说：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{4.29}
$$

中的 $\gamma$ 决定随机插值的边缘密度：

$$
\rho_t
$$

而：

$$
\boxed{
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}\,dW_t
}
\tag{4.30}
$$

中的 $\varepsilon$ 决定生成过程的随机扰动强度。

下面分别讨论。


### 4.3.2 改变 $\gamma(t)$ 为什么会改变概率路径？

考虑：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

固定插值函数和端点耦合。

对于任意中间时刻 $t$，给定两个端点时：

$$
X_t\mid X_0,X_1
\sim
\mathcal N(I(t,X_0,X_1),\gamma^2(t)I_d)
$$

因此：

$$
\gamma^2(t)
$$

直接决定条件分布的方差。

例如：

$$
\gamma_1(t)=0
$$

与：

$$
\gamma_2(t)=\sqrt{t(1-t)}
$$

会产生不同的中间分布。

第一种：

$$
X_t^{(1)}=I(t,X_0,X_1)
$$

第二种：

$$
X_t^{(2)}=I(t,X_0,X_1)+\sqrt{t(1-t)}Z
$$

即使端点分布相同，中间概率密度一般也不同：

$$
\rho_t^{(1)}\neq\rho_t^{(2)}
$$

进一步，由于：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

$$
s(t,x)=\nabla\log\rho_t(x)
$$

所以改变 $\gamma$ 通常也会改变：

$$
b,\qquad s
$$

因此：

$$
\boxed{
\gamma
\longrightarrow
\rho_t
\longrightarrow
(b,s)
}
\tag{4.31}
$$

这是插值噪声的作用链。


### 4.3.3 改变 $\varepsilon(t)$ 为什么不改变理想概率路径？


固定 $\rho_t,b,s$，采用漂移 $b+\varepsilon s$ 和扩散系数 $\sqrt{2\varepsilon}$。由 Score 修正项抵消扩散项，密度仍满足：

完整推导见[附录 B.3.3：保持概率路径的生成 SDE](#app-b-3-3)。

$$
\boxed{
\partial_t\rho_t=-\nabla\cdot(b\rho_t)
}
\tag{4.34}
$$

这说明，在相应正则性和适定性条件下，改变 $\varepsilon(t)$ 不会改变目标边缘概率路径。

但不同的 $\varepsilon$ 会产生不同的随机轨迹分布。

因此：

$$
\boxed{
\text{边缘概率路径相同}
\neq
\text{生成过程的路径分布相同}
}
$$


### 4.3.4 两个参数是否可以独立选择？

可以，但需要区分两个层面。

**理论层面：**

首先选择：

$$
I,\quad\gamma,\quad\nu
$$

确定：

$$
\rho_t,\quad b,\quad s
$$

然后可以选择合适的：

$$
\varepsilon(t)\ge0
$$

构造不同的 SDE。

因此，$\gamma$ 和 $\varepsilon$ 是两个不同的设计自由度。

**实际训练与采样层面：**

改变 $\gamma$ 通常意味着需要重新学习新的速度场和 Score。

而在固定 $\gamma$ 的情况下，如果已经准确学习了 $b,s$，可以尝试改变 $\varepsilon$，使用不同的 SDE 采样。

但是，在有限神经网络精度和有限数值积分步数下，不同 $\varepsilon$ 的实际生成分布可能不同。

原因包括：

- Score 预测误差。
- 速度场预测误差。
- SDE 离散化误差。
- 不同扩散强度对误差的放大或抑制作用。

因此，理论上的边缘一致性不等于实际有限步采样结果完全一致。


### 4.3.5 为什么不能简单令 $\varepsilon=\gamma^2$？

因为两者在数学上没有必然的对应关系。

$\gamma^2(t)$ 是随机插值中高斯扰动的方差。

而 SDE 的扩散项为：

$$
\sqrt{2\varepsilon(t)}dW_t
$$

在一个很小的时间间隔 $dt$ 内，随机增量的协方差约为：

$$
2\varepsilon(t)dt\,I_d
$$

因此：

- $\gamma^2(t)$：描述某个时刻随机插值的条件噪声方差。
- $2\varepsilon(t)dt$：描述 SDE 在一个无穷小时间间隔内注入的噪声方差。

二者不是同一种量。

在某些特殊模型中，可以通过特定选择使二者满足某种关系。

例如：

$$
\varepsilon(t)=\gamma(t)\dot\gamma(t)
$$

可能使某些 SDE 漂移项简化。

但这不是随机插值框架的普遍要求，而且右侧必须满足扩散强度非负等条件。


### 4.3.6 本节总结

本节最重要的区别是：

| 参数 | 所在位置 | 主要作用 |
|---|---|---|
| $\gamma(t)$ | 随机插值 $X_t=I+\gamma Z$ | 决定中间概率路径 |
| $\varepsilon(t)$ | SDE 的扩散系数 $\sqrt{2\varepsilon}$ | 决定生成动力学的随机强度 |

因此：

$$
\boxed{
\begin{aligned}
\gamma&:\text{设计概率路径}\\
\varepsilon&:\text{选择实现该路径的动力学}
\end{aligned}
}
\tag{4.35}
$$

这是随机插值框架中非常重要的两个自由度。

---


## 4.4 单侧线性插值：使用一个预测器恢复速度与 Score


### 4.4.1 核心结论

这是本章最重要的部分。

第 4.1 节已经证明，双侧线性随机插值：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

具有三个条件均值：

$$
m_0,\quad m_1,\quad m_z
$$

而它们只满足一个向量约束：

$$
x=\alpha m_0+\beta m_1+\gamma m_z
$$

因此，一般不能仅通过一个条件预测器恢复全部所需信息。

但当起点为独立标准高斯时，可以使用单侧形式：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{4.36}
$$

这时只剩下两个条件均值：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

$$
\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]
$$

它们满足：

$$
\boxed{
x=\alpha\eta_z+\beta\eta_1
}
\tag{4.37}
$$

因此，只需要知道其中一个，就可以恢复另一个。

进一步，速度场和 Score 都可以通过同一个预测器计算。


### 4.4.2 从噪声预测器恢复 Score


对 $X_t=\alpha Z+\beta X_1$，直接应用高斯噪声的 Score 恒等式：

完整推导见[附录 D.1.3：单侧线性插值的 Score](#app-d-1-3)。

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
}
\tag{4.38}
$$

因此，只需要训练：

$$
\eta_\theta(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

就能恢复 Score：

$$
s_\theta(t,x)=-\frac{\eta_\theta(t,x)}{\alpha(t)}
$$

这里要求：

$$
\alpha(t)>0
$$

在 $\alpha=0$ 的端点，必须通过极限或其他参数化处理。


### 4.4.3 从噪声预测器恢复速度场


由 $x=\alpha\eta_z+\beta\eta_1$ 消去数据条件均值，在 $\beta(t)\ne0$ 时有：

完整推导见[附录 F.1.1：从噪声预测恢复速度场](#app-f-1-1)。

$$
\boxed{
b(t,x)=
\frac{\dot\beta(t)}{\beta(t)}x+
\left(
\dot\alpha(t)
-\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right)\eta_z(t,x)
}
\tag{4.41}
$$

因此，单侧线性插值的两个核心公式为：

$$
\boxed{
\begin{aligned}
b(t,x)
&=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,x)\\[5pt]
s(t,x)
&=-\frac{\eta_z(t,x)}{\alpha}
\end{aligned}
}
\tag{4.42}
$$

**这就是为什么一个噪声预测器能够同时恢复速度场和 Score。**


### 4.4.4 是否也可以只使用数据预测器？


可以。记 $\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]$，在 $\alpha(t)\ne0$ 时，有：

完整推导见[附录 F.1.3：数据预测参数化](#app-f-1-3)。

$$
\boxed{
\begin{aligned}
b(t,x)
&=
\frac{\dot\alpha}{\alpha}x+
\left(
\dot\beta-\frac{\beta\dot\alpha}{\alpha}
\right)\eta_1\\[4pt]
s(t,x)
&=-\frac{x-\beta\eta_1}{\alpha^2}
\end{aligned}
}
\tag{4.46}
$$

这说明：

**噪声预测和数据预测在理想条件下可以互相转换。**

但它们的数值稳定性和训练损失权重并不完全相同。

例如，数据预测器的训练目标为：

$$
\mathcal L_{\mathrm{data}}
=
\mathbb E\|\eta_\theta(t,X_t)-X_1\|^2
$$

而噪声预测器的训练目标为：

$$
\mathcal L_{\mathrm{noise}}
=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
$$

虽然二者的最优条件期望存在代数关系，但在有限模型容量下，它们不必得到相同的实际结果。


### 4.4.5 进一步：直接预测速度也可以恢复 Score 吗？


可以，但要求 $D(t)=\alpha(t)\dot\beta(t)-\beta(t)\dot\alpha(t)\ne0$。将观测与速度写成关于两个条件均值的线性方程组，求逆后有：

完整推导见[附录 F.1.4：三种预测量的转换](#app-f-1-4)。

$$
\boxed{
\eta_z(t,x)=
\frac{\dot\beta(t)x-\beta(t)b(t,x)}{D(t)}
}
\tag{4.49}
$$

以及：

$$
\boxed{
\eta_1(t,x)=
\frac{\alpha(t)b(t,x)-\dot\alpha(t)x}{D(t)}
}
\tag{4.50}
$$

再根据：

$$
s=-\frac{\eta_z}{\alpha}
$$

得到：

$$
\boxed{
s(t,x)=
-\frac{\dot\beta(t)x-\beta(t)b(t,x)}
{\alpha(t)D(t)}
}
\tag{4.51}
$$

因此，当：

$$
\alpha(t)\neq0,\qquad D(t)\neq0
$$

时，直接预测边缘速度场 $b$，也能够恢复 Score。

这意味着单侧线性插值存在三种常见参数化：

$$
\boxed{
\text{Noise Prediction}
\quad\leftrightarrow\quad
\text{Data Prediction}
\quad\leftrightarrow\quad
\text{Velocity Prediction}
}
\tag{4.52}
$$

但这种等价是指**理想预测场之间的代数可转换性**，不是说三个训练目标在有限模型下完全等价。


### 4.4.6 一个具体例子：CondOT 线性日程

选择：

$$
\alpha(t)=1-t
$$

$$
\beta(t)=t
$$

则：

$$
\boxed{
X_t=(1-t)Z+tX_1
}
\tag{4.53}
$$

这是常见的 Conditional Optimal Transport 路径形式。

注意，这里使用 CondOT 是指其条件直线路径形式；如果端点采用独立耦合，并不意味着整体运输一定是全局最优传输。

首先计算时间导数：

$$
\dot\alpha=-1
$$

$$
\dot\beta=1
$$

因此：

$$
\dot X_t=X_1-Z
\tag{4.54}
$$

边缘速度场为：

$$
b(t,x)=\mathbb E[X_1-Z\mid X_t=x]
\tag{4.55}
$$


将该日程代入附录 F.1 的通式（计算见[附录 F.4.1](#app-f-4-1)），得到：

$$
\boxed{
b(t,x)=\frac{x-\eta_z(t,x)}{t}
}
\tag{4.56}
$$

而 Score 为：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{1-t}
}
\tag{4.57}
$$

所以：

$$
\boxed{
\begin{aligned}
b(t,x)&=\frac{x-\eta_z(t,x)}{t}\\[4pt]
s(t,x)&=-\frac{\eta_z(t,x)}{1-t}
\end{aligned}
}
\tag{4.58}
$$

这个例子说明，只需要训练一个噪声预测器，就能恢复 CondOT 的速度场与 Score。

不过，如果采用直接速度预测，训练目标更加简单：

$$
\boxed{
\mathcal L_v=
\mathbb E\left[
\|b_\theta(t,X_t)-(X_1-Z)\|^2
\right]
}
\tag{4.59}
$$

这就是常见的 Flow Matching 训练形式。


### 4.4.7 从一个预测器构造 ODE 与 SDE

现在假设我们已经训练好了：

$$
\eta_\theta(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

对于任意给定的单侧线性日程：

$$
X_t=\alpha Z+\beta X_1
$$

可以恢复：

$$
b_\theta(t,x)=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_\theta(t,x)
\tag{4.60}
$$

以及：

$$
s_\theta(t,x)=-\frac{\eta_\theta(t,x)}{\alpha}
\tag{4.61}
$$

于是：

**Probability Flow ODE：**

$$
\boxed{
dY_t=b_\theta(t,Y_t)dt
}
\tag{4.62}
$$

**前向 SDE：**

$$
\boxed{
dY_t=
[b_\theta(t,Y_t)+\varepsilon(t)s_\theta(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{4.63}
$$

将两个预测场代入：

$$
\boxed{
\begin{aligned}
dY_t
={}&
\left[
\frac{\dot\beta}{\beta}Y_t
+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
-\frac{\varepsilon}{\alpha}
\right)
\eta_\theta(t,Y_t)
\right]dt\\
&+\sqrt{2\varepsilon}\,dW_t
\end{aligned}
}
\tag{4.64}
$$

这说明：

**一个噪声预测器就可以支持不同扩散强度的 SDE 采样。**

对于后向 SDE，同样有：

$$
dY_t=
[b_\theta-\varepsilon s_\theta]dt+
\sqrt{2\varepsilon}d\overleftarrow W_t
$$

代入：

$$
\boxed{
\begin{aligned}
dY_t
={}&
\left[
\frac{\dot\beta}{\beta}Y_t
+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
+\frac{\varepsilon}{\alpha}
\right)
\eta_\theta(t,Y_t)
\right]dt\\
&+\sqrt{2\varepsilon}\,d\overleftarrow W_t
\end{aligned}
}
\tag{4.65}
$$

其中后向 SDE 使用原时间递减的约定。

需要注意，式 (4.64) 和式 (4.65) 都只在相关分母非零的内部时间区间直接适用。端点附近应使用极限分析、等价参数化或数值正则化。


### 4.4.8 为什么单侧线性插值特别适合生成模型？

原因主要有三个。

**第一，训练过程简单。**

可以直接采样：

$$
Z\sim\mathcal N(0,I_d)
$$

$$
X_1\sim\rho_1
$$

$$
t\sim p(t)
$$

然后构造：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

整个过程不需要求解 ODE 或 SDE。

**第二，监督目标容易计算。**

例如噪声预测：

$$
\mathcal L_\eta=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
$$

或者速度预测：

$$
\mathcal L_b=
\mathbb E\left[
\|b_\theta(t,X_t)-(\dot\alpha Z+\dot\beta X_1)\|^2
\right]
$$

**第三，同一个模型可以支持多种采样动力学。**

理想情况下，只要能够恢复：

$$
b,\quad s
$$

就可以构造：

$$
\text{ODE}
$$

以及：

$$
\text{不同 }\varepsilon(t)\text{ 对应的 SDE}
$$

因此：

$$
\boxed{
\text{单侧线性插值}
\longrightarrow
\text{一个预测器}
\longrightarrow
(b,s)
\longrightarrow
\text{ODE/SDE}
}
\tag{4.66}
$$

这是本章最重要的结论。

---


# 第 4 章总结

本章研究了空间线性随机插值的数学结构。

首先，对于双侧线性随机插值：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

有：

$$
\boxed{
\begin{aligned}
b&=\dot\alpha m_0+\dot\beta m_1+\dot\gamma m_z\\
s&=-\frac{m_z}{\gamma}\\
x&=\alpha m_0+\beta m_1+\gamma m_z
\end{aligned}
}
$$

不同的时间日程、端点耦合和插值噪声共同决定中间概率路径。

其次，需要明确区分：

$$
\boxed{
\gamma(t):\text{决定随机插值的概率路径}
}
$$

$$
\boxed{
\varepsilon(t):\text{决定生成 SDE 的扩散强度}
}
$$

最后，对于单侧线性插值：

$$
X_t=\alpha Z+\beta X_1
$$

有：

$$
x=\alpha\eta_z+\beta\eta_1
$$

因此，只需要一个预测器，就可以恢复：

$$
\boxed{
\begin{aligned}
b(t,x)
&=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,x)\\[4pt]
s(t,x)
&=-\frac{\eta_z(t,x)}{\alpha}
\end{aligned}
}
$$

**本章的核心认识是：单侧线性插值通过额外的代数约束，将速度预测、噪声预测和数据预测联系起来，从而为 Flow Matching 与 Diffusion 的统一提供了具体可计算的数学形式。**

下一章将进一步研究这些公式如何与 **OU、VP-SDE、VE-SDE、Denoising 和 Rectified Flow** 建立联系。

<a id="chapter-5"></a>

# 第 5 章：Connections with Other Methods

前面四章已经建立了随机插值的统一理论，并重点研究了空间线性随机插值：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{5.1}
$$

其中：

- $Z\sim\mathcal N(0,I_d)$：初始高斯噪声。
- $X_1\sim\rho_1$：目标数据。
- $Z$ 与 $X_1$ 相互独立。
- $t\in[0,1]$：时间从噪声端 $t=0$ 向数据端 $t=1$ 演化。

第 4 章证明，对于单侧线性插值，只需要一个预测器，就可以恢复速度场和 Score：

$$
\boxed{
\begin{aligned}
b(t,x)
&=\frac{\dot\beta}{\beta}x+
\left(\dot\alpha-\frac{\alpha\dot\beta}{\beta}\right)\eta_z(t,x)\\[4pt]
s(t,x)&=-\frac{\eta_z(t,x)}{\alpha}
\end{aligned}
}
\tag{5.2}
$$

其中：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

本章进一步研究：**这一统一框架与现有生成模型之间究竟是什么关系？**

重点讨论三个问题：

1. Score-Based Diffusion 中的 OU、VP、VE 扩散过程，如何写成随机插值形式？
2. 为什么预测噪声、预测数据和 Score Matching 在数学上密切相关？
3. Rectified Flow 为什么采用直线插值？Reflow 又为什么可能使生成轨迹更加平直？

需要先明确一个贯穿本章的区别：

**随机插值描述单时刻概率分布的演化；SDE 描述实现该概率演化的随机过程；神经网络的预测目标则是学习这些概率分布所对应的条件统计量。**

三者相互联系，但并不是同一个数学对象。

---


## 5.1 与 Score-Based Diffusion 的联系：OU、VP 与 VE


### 5.1.1 核心结论

Score-Based Diffusion 通常从一个前向加噪 SDE 出发：

$$
\boxed{
dY_\tau=f(\tau,Y_\tau)d\tau+g(\tau)dW_\tau
}
\tag{5.3}
$$

其中：

- $\tau$：扩散模型的前向加噪时间，从数据到噪声。
- $f(\tau,x)$：漂移函数。
- $g(\tau)$：扩散系数。
- $W_\tau$：标准布朗运动。

对于常见的线性扩散过程，其任意时刻的条件分布都可以写成：

$$
\boxed{
Y_\tau\mid Y_0=x_1
\sim
\mathcal N\left(
m(\tau)x_1,\sigma^2(\tau)I_d
\right)
}
\tag{5.4}
$$

因此，在单时刻分布意义下：

$$
\boxed{
Y_\tau\overset d=
m(\tau)X_1+\sigma(\tau)Z
}
\tag{5.5}
$$

这恰好是单侧线性随机插值的形式。

但是，扩散模型通常使用：

$$
\tau=0:\text{数据},\qquad \tau=T:\text{噪声}
$$

而前面随机插值使用：

$$
t=0:\text{噪声},\qquad t=1:\text{数据}
$$

因此，为了比较，需要引入时间反转：

$$
\boxed{
t=1-\frac{\tau}{T}
}
\tag{5.6}
$$

下面先推导最基本的 Ornstein–Uhlenbeck（OU）过程，再推广到 VP-SDE 和 VE-SDE。


### 5.1.2 Ornstein–Uhlenbeck 过程

考虑 OU 随机微分方程：

$$
\boxed{
dY_\tau=-\lambda Y_\tau d\tau+\sqrt{2\lambda}\,dW_\tau
}
\tag{5.7}
$$

其中：

$$
\lambda>0
$$

表示均值回复强度。

OU 过程的特点是：

- 漂移项 $-\lambda Y_\tau$ 将状态拉向零。
- 扩散项持续注入高斯噪声。
- 当时间足够长时，其分布趋向标准高斯分布。

我们先推导它的解析解。


#### 第一步：使用积分因子


对 $e^{\lambda\tau}Y_\tau$ 使用 Itô 乘积公式并积分，得到：

完整推导见[附录 F.3.1：OU 积分因子法](#app-f-3-1)。

$$
\boxed{
Y_\tau=
e^{-\lambda\tau}Y_0+
\sqrt{2\lambda}\int_0^\tau e^{-\lambda(\tau-r)}dW_r
}
\tag{5.8}
$$


#### 第二步：计算随机积分的分布


Itô 等距给出噪声协方差 $(1-e^{-2\lambda\tau})I_d$，因而：

完整推导见[附录 F.3.2：线性高斯随机积分的方差](#app-f-3-2)。

$$
\boxed{
Y_\tau\mid Y_0=x_1
\sim
\mathcal N\left(
e^{-\lambda\tau}x_1,
(1-e^{-2\lambda\tau})I_d
\right)
}
\tag{5.9}
$$

这就是 OU 过程的条件转移分布。


#### 第三步：写成随机插值形式

根据式 (5.9)，在任意固定时间 $\tau$：

$$
\boxed{
Y_\tau\overset d=
e^{-\lambda\tau}X_1+
\sqrt{1-e^{-2\lambda\tau}}Z
}
\tag{5.10}
$$

其中 $Z$ 独立于 $X_1$。

定义：

$$
m(\tau)=e^{-\lambda\tau}
$$

$$
\sigma(\tau)=\sqrt{1-e^{-2\lambda\tau}}
$$

则：

$$
Y_\tau\overset d=m(\tau)X_1+\sigma(\tau)Z
$$

这就是单侧线性插值的形式。

特别地：

$$
\boxed{
m^2(\tau)+\sigma^2(\tau)=1
}
\tag{5.11}
$$

当：

$$
\tau\to\infty
$$

有：

$$
m(\tau)\to0
$$

$$
\sigma(\tau)\to1
$$

因此：

$$
Y_\tau\Rightarrow\mathcal N(0,I_d)
$$

这里的 $\Rightarrow$ 表示依分布收敛。

**结论：OU 过程的每个单时刻条件分布，都可以由一个高斯线性插值表示。**

但 OU 过程使用的是随时间变化的布朗运动，而随机插值可以使用同一个 $Z$ 构造所有时刻，因此二者的完整路径分布一般不同。


### 5.1.3 VP-SDE：Variance Preserving SDE

OU 过程使用常数 $\lambda$。

扩散模型中更常见的是使用随时间变化的噪声日程：

$$
\beta_{\mathrm{VP}}(\tau)>0
$$

定义：

$$
\boxed{
dY_\tau=
-\frac12\beta_{\mathrm{VP}}(\tau)Y_\tau d\tau+
\sqrt{\beta_{\mathrm{VP}}(\tau)}dW_\tau
}
\tag{5.12}
$$

这就是 VP-SDE。

它可以看作时间非齐次 OU 过程。

下面推导它的解析条件分布。


#### 第一步：定义累计噪声强度


定义 $B(\tau)=\int_0^\tau\beta_{\mathrm{VP}}(r)dr$、$m(\tau)=e^{-B(\tau)/2}$。积分因子法得到：

完整推导见[附录 F.3.2：VP 的线性 SDE 解](#app-f-3-2)。

$$
Y_\tau=
m(\tau)Y_0+
m(\tau)
\int_0^\tau
\frac{\sqrt{\beta_{\mathrm{VP}}(r)}}{m(r)}dW_r
\tag{5.15}
$$

这个结果可以使用与 OU 过程相同的积分因子法得到。


#### 第二步：计算噪声方差


噪声积分是零均值高斯变量，协方差为 $[1-e^{-B(\tau)}]I_d$。因此：

完整推导见[附录 F.3.2：VP 的噪声方差](#app-f-3-2)。

$$
\boxed{
Y_\tau\mid Y_0=x_1
\sim
\mathcal N\left(
e^{-\frac12B(\tau)}x_1,
[1-e^{-B(\tau)}]I_d
\right)
}
\tag{5.17}
$$

定义：

$$
\boxed{
\begin{aligned}
m(\tau)&=e^{-\frac12B(\tau)}\\
\sigma(\tau)&=\sqrt{1-e^{-B(\tau)}}
\end{aligned}
}
\tag{5.18}
$$

于是：

$$
\boxed{
Y_\tau\overset d=m(\tau)X_1+\sigma(\tau)Z
}
\tag{5.19}
$$

并且：

$$
\boxed{
m^2(\tau)+\sigma^2(\tau)=1
}
\tag{5.20}
$$

这就是 VP-SDE 与单侧线性随机插值的直接联系。


#### 第三步：为什么叫 Variance Preserving？

假设：

$$
\mathbb E[X_1]=0
$$

并且：

$$
\operatorname{Cov}(X_1)=I_d
$$

由于 $X_1$ 与 $Z$ 独立：

$$
\operatorname{Cov}(Y_\tau)
=
m^2(\tau)I_d+\sigma^2(\tau)I_d
$$

利用：

$$
m^2+\sigma^2=1
$$

得到：

$$
\boxed{
\operatorname{Cov}(Y_\tau)=I_d
}
\tag{5.21}
$$

因此，它保持的是在特定数据协方差假设下的总体方差。

并不是说任意数据分布都满足：

$$
\operatorname{Cov}(Y_\tau)=\operatorname{Cov}(X_1)
$$

更准确地说，VP-SDE 具有：

$$
m^2+\sigma^2=1
$$

这一方差归一化结构。


### 5.1.4 VP-SDE 的 Score


将高斯加噪通式中的噪声系数取为 $\sigma(\tau)>0$、数据系数取为 $m(\tau)$，有：

完整推导见[附录 D.1.3：噪声预测恢复 Score](#app-d-1-3)。

$$
\boxed{
s_\tau(y)=
-\frac{\mathbb E[Z\mid Y_\tau=y]}{\sigma(\tau)}
}
\tag{5.23}
$$

因此，VP-SDE 中常见的噪声预测器：

$$
\epsilon_\theta(y,\tau)
$$

在 MSE 最优条件下满足：

$$
\epsilon_\theta(y,\tau)
=
\mathbb E[Z\mid Y_\tau=y]
$$

于是：

$$
\boxed{
s_\theta(y,\tau)=
-\frac{\epsilon_\theta(y,\tau)}{\sigma(\tau)}
}
\tag{5.24}
$$

这说明扩散模型中的噪声预测与随机插值中的 Score 回归，本质上来自同一个条件期望恒等式。


### 5.1.5 VP-SDE 的反向 SDE 与 Probability Flow ODE

对于一般前向 SDE：

$$
dY_\tau=f(\tau,Y_\tau)d\tau+g(\tau)dW_\tau
$$

当 $g$ 只依赖时间时，其反向时间 SDE 在原时间递减的约定下为：

$$
\boxed{
dY_\tau=
[f(\tau,Y_\tau)-g^2(\tau)s_\tau(Y_\tau)]d\tau
+
g(\tau)d\overleftarrow W_\tau
}
\tag{5.25}
$$

其中：

$$
s_\tau(y)=\nabla_y\log p_\tau(y)
$$

而对应的 Probability Flow ODE 为：

$$
\boxed{
\frac{dY_\tau}{d\tau}
=
f(\tau,Y_\tau)
-\frac12g^2(\tau)s_\tau(Y_\tau)
}
\tag{5.26}
$$

对于 VP-SDE：

$$
f(\tau,y)=-\frac12\beta_{\mathrm{VP}}(\tau)y
$$

$$
g^2(\tau)=\beta_{\mathrm{VP}}(\tau)
$$

因此：

**反向 SDE：**

$$
\boxed{
dY_\tau=
\left[
-\frac12\beta_{\mathrm{VP}}(\tau)Y_\tau
-\beta_{\mathrm{VP}}(\tau)s_\tau(Y_\tau)
\right]d\tau
+
\sqrt{\beta_{\mathrm{VP}}(\tau)}d\overleftarrow W_\tau
}
\tag{5.27}
$$

**Probability Flow ODE：**

$$
\boxed{
\frac{dY_\tau}{d\tau}
=
-\frac12\beta_{\mathrm{VP}}(\tau)Y_\tau
-\frac12\beta_{\mathrm{VP}}(\tau)s_\tau(Y_\tau)
}
\tag{5.28}
$$

注意，以上公式使用的是**数据到噪声的时间 $\tau$**。

生成时，从较大的 $\tau$ 向较小的 $\tau$ 积分。


#### 为什么两个公式的 Score 系数相差两倍？

因为反向 SDE 需要同时补偿随机扩散项：

$$
\frac12g^2\Delta p
$$

而 Probability Flow ODE 没有随机扩散项，只需要通过确定性漂移实现相同的密度演化。

完整推导见[附录 D.3：从 Fokker–Planck 方程恢复 Probability Flow ODE](#app-d-3)。

这与第 2 章建立 ODE/SDE 统一关系使用的是同一个恒等式。


### 5.1.6 VE-SDE：Variance Exploding SDE

VP-SDE 同时包含漂移和扩散：

$$
dY_\tau=
-\frac12\beta_{\mathrm{VP}}Y_\tau d\tau+
\sqrt{\beta_{\mathrm{VP}}}dW_\tau
$$

而 VE-SDE 采用另一种设计：

**不收缩原始数据，只不断增加高斯噪声。**

考虑：

$$
\boxed{
dY_\tau=g(\tau)dW_\tau
}
\tag{5.29}
$$

它没有漂移项。


令累计噪声方差 $v(\tau)=\int_0^\tau g^2(r)dr$，由[附录 F.3.3](#app-f-3-3) 的随机积分计算，得到：

$$
\boxed{
Y_\tau\mid Y_0=x_1
\sim
\mathcal N(x_1,v(\tau)I_d)
}
\tag{5.32}
$$

因此：

$$
\boxed{
Y_\tau\overset d=X_1+\sqrt{v(\tau)}Z
}
\tag{5.33}
$$

这就是 VE-SDE 的随机插值表示。

如果使用噪声标准差函数 $\sigma(\tau)$，并令：

$$
v(\tau)=\sigma^2(\tau)-\sigma^2(0)
$$

则：

$$
g^2(\tau)=\frac{d\sigma^2(\tau)}{d\tau}
$$

得到常见形式：

$$
\boxed{
dY_\tau=
\sqrt{\frac{d\sigma^2(\tau)}{d\tau}}\,dW_\tau
}
\tag{5.34}
$$

需要注意：

如果 $\sigma(0)>0$，那么通常需要将初始分布设为已经带有方差 $\sigma^2(0)$ 的扰动分布，才能得到总噪声方差为 $\sigma^2(\tau)$ 的表达式。


#### VE-SDE 的 Score

对于：

$$
Y_\tau=X_1+\sqrt{v(\tau)}Z
$$

有：

$$
\boxed{
s_\tau(y)=
-\frac{\mathbb E[Z\mid Y_\tau=y]}{\sqrt{v(\tau)}}
}
\tag{5.35}
$$

因此，VE-SDE 同样可以通过噪声预测恢复 Score。


#### VE-SDE 的反向 SDE

由于：

$$
f=0
$$

所以：

$$
\boxed{
dY_\tau=
-g^2(\tau)s_\tau(Y_\tau)d\tau
+
g(\tau)d\overleftarrow W_\tau
}
\tag{5.36}
$$

对应的 Probability Flow ODE 为：

$$
\boxed{
\frac{dY_\tau}{d\tau}
=
-\frac12g^2(\tau)s_\tau(Y_\tau)
}
\tag{5.37}
$$

可以看到，VE-SDE 的生成漂移完全由 Score 决定。


### 5.1.7 VE 与单侧插值的端点区别

这里有一个非常重要的细节。

前面使用的单侧生成插值为：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

并要求：

$$
\alpha(0)=1,\quad\beta(0)=0
$$

$$
\alpha(1)=0,\quad\beta(1)=1
$$

这样才能严格实现：

$$
\mathcal N(0,I_d)\longrightarrow\rho_1
$$

但 VE-SDE 的形式是：

$$
Y_\tau=X_1+\sigma(\tau)Z
$$

其中数据系数始终为：

$$
\beta=1
$$

即使噪声方差很大：

$$
Y_\tau=X_1+\sigma Z
$$

也不是严格的纯高斯分布。

只有在适当归一化或极限意义下，数据项的相对影响才可能趋于消失。

因此：

$$
\boxed{
\text{VE-SDE 与单侧线性插值具有相同的代数结构，但端点约束未必相同}
}
$$

这是理解随机插值统一框架时必须注意的地方。


### 5.1.8 OU、VP、VE 的统一比较

| 方法 | 前向 SDE | 单时刻条件分布 |
|---|---|---|
| OU | $dY=-\lambda Yd\tau+\sqrt{2\lambda}dW$ | $\mathcal N(e^{-\lambda\tau}x_1,(1-e^{-2\lambda\tau})I)$ |
| VP | $dY=-\frac12\beta_{\mathrm{VP}}Yd\tau+\sqrt{\beta_{\mathrm{VP}}}dW$ | $\mathcal N(mx_1,(1-m^2)I)$ |
| VE | $dY=g(\tau)dW$ | $\mathcal N(x_1,v(\tau)I)$ |

它们都可以写成：

$$
\boxed{
Y_\tau\overset d=m(\tau)X_1+\sigma(\tau)Z
}
\tag{5.38}
$$

但对应的时间系数不同。

**本节最重要的认识是：Score-Based Diffusion 的线性前向加噪过程，可以通过单侧线性随机插值描述其边缘概率路径；而其具体 SDE 则规定了跨时间的随机演化方式。**

---


## 5.2 与 Denoising、Score Matching 的联系


### 5.2.1 核心结论

这一节需要回答一个非常重要的问题：

**为什么扩散模型使用 MSE 预测噪声，就能够学习概率分布的 Score？**

进一步，为什么使用同一个模型，还可以得到数据去噪器：

$$
\widehat x_1=\mathbb E[X_1\mid X_t=x]
$$

核心原因是：

$$
\boxed{
\text{MSE 的最优预测器是条件期望}
}
$$

而对于高斯加噪过程，噪声的条件期望、数据的条件期望与边缘 Score 之间存在精确的代数关系。

原论文第 5.2 节进一步讨论了：如何使用去噪器预测其他时间的随机插值状态，以及如何通过迭代去噪构造生成过程。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 5.2.2 为什么 MSE 的最优解是条件期望？


对平方可积的 $U$，最小化 $\mathbb E\|f(V)-U\|^2$ 的理想预测器为：

完整推导见[附录 A.2.2：条件期望的 MSE 最优性](#app-a-2-2)。

$$
\boxed{
f^*(v)=\mathbb E[U\mid V=v]
}
\tag{5.40}
$$

这个结论是 Flow Matching、噪声预测和数据预测等回归训练目标的共同数学基础。


### 5.2.3 噪声预测为什么等价于学习 Score？

考虑单侧线性随机插值：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{5.41}
$$

训练一个噪声预测器：

$$
\eta_\theta(t,x)
$$

使用：

$$
\boxed{
\mathcal L_{\mathrm{noise}}
=
\mathbb E_{t,X_1,Z}
\left[
\|\eta_\theta(t,X_t)-Z\|^2
\right]
}
\tag{5.42}
$$

对于固定 $t$，根据前面的 MSE 最优性：

$$
\boxed{
\eta^*(t,x)=\mathbb E[Z\mid X_t=x]
}
\tag{5.43}
$$

第 4 章已经推导：

$$
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{\alpha(t)}
$$

因此：

$$
\boxed{
s(t,x)=-\frac{\eta^*(t,x)}{\alpha(t)}
}
\tag{5.44}
$$

也就是说：

**噪声预测器的最优输出经过一个确定性缩放，就是真实的边缘 Score。**

这就是 Denoising Score Matching 与噪声预测的联系。


#### 从损失函数角度再看一次

定义真实 Score：

$$
s(t,x)=\nabla_x\log\rho_t(x)
$$

以及由噪声预测器构造的：

$$
s_\theta(t,x)=-\frac{\eta_\theta(t,x)}{\alpha(t)}
$$

于是：

$$
\eta_\theta=-\alpha s_\theta
$$

代入噪声 MSE：

$$
\begin{aligned}
\mathcal L_{\mathrm{noise}}
&=
\mathbb E\|-\alpha s_\theta-Z\|^2\\
&=
\alpha^2
\mathbb E\left\|
s_\theta+\frac Z\alpha
\right\|^2
\end{aligned}
\tag{5.45}
$$

这里暂时固定时间 $t$。

根据条件期望的平方误差分解：

$$
\begin{aligned}
\mathbb E\left\|
s_\theta+\frac Z\alpha
\right\|^2
={}&
\mathbb E_{X_t}\|s_\theta-s\|^2\\
&+
C(t)
\end{aligned}
\tag{5.46}
$$

其中：

$$
C(t)=
\mathbb E\left\|
-\frac Z\alpha-s(t,X_t)
\right\|^2
$$

与模型参数无关。

因此：

$$
\boxed{
\mathcal L_{\mathrm{noise}}(t)
=
\alpha^2(t)
\mathbb E_{\rho_t}\|s_\theta-s\|^2
+
\alpha^2(t)C(t)
}
\tag{5.47}
$$

这说明噪声预测损失与 Score 回归损失只相差一个时间相关权重和一个与模型无关的常数。

需要注意：

如果对不同时间 $t$ 进行积分，那么：

$$
\alpha^2(t)
$$

会改变各时间点的训练权重。

因此，虽然在理想无限容量条件下两种训练方式具有相同的逐时刻最优解，但在有限网络容量下，它们的优化结果可能不同。


### 5.2.4 数据去噪器与 Tweedie 公式


记 Bayes 最优数据去噪器 $D(t,x)=\mathbb E[X_1\mid X_t=x]$。对 $X_t=\alpha Z+\beta X_1$，且 $\beta(t)\ne0$，有：

完整推导见[附录 D.2.1～D.2.2：Tweedie 公式](#app-d-2)。

$$
\boxed{
D(t,x)=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
}
\tag{5.49}
$$

利用：

$$
\eta_z=-\alpha s
$$

得到：

$$
\boxed{
D(t,x)=
\frac{x+\alpha^2(t)s(t,x)}{\beta(t)}
}
\tag{5.50}
$$

这是高斯加噪下 Tweedie 公式的一种形式。

特别地，当：

$$
\beta(t)=1
$$

即：

$$
X_t=X_1+\alpha(t)Z
$$

时：

$$
\boxed{
D(t,x)=x+\alpha^2(t)s(t,x)
}
\tag{5.51}
$$

这个公式具有非常直观的含义：

$$
\boxed{
\text{去噪后的估计}
=
\text{带噪观测}
+
\text{Score 修正}
}
$$

为什么是沿 Score 方向修正？

因为：

$$
s(t,x)=\nabla_x\log\rho_t(x)
$$

指向局部概率密度增加最快的方向。

因此，Score 提供了将带噪样本向高概率区域修正的信息。

但要注意：

$$
D(t,x)=\mathbb E[X_1\mid X_t=x]
$$

是后验均值，不一定是后验概率最大的点。

因此，它不必等于 MAP 估计，也不保证输出一定严格位于数据流形上。


### 5.2.5 从当前时刻预测任意其他时刻

原论文进一步研究：

给定：

$$
X_t=x
$$

能否预测：

$$
X_s
$$

其中 $s$ 可以大于或小于 $t$？

仍然考虑：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

以及：

$$
X_s=\alpha(s)Z+\beta(s)X_1
\tag{5.52}
$$

这里两个随机变量使用**同一个** $Z$ 和 $X_1$，即同一条随机插值路径上的两个时刻。

给定 $X_t=x$，有：

$$
\begin{aligned}
\mathbb E[X_s\mid X_t=x]
={}&
\alpha(s)\mathbb E[Z\mid X_t=x]\\
&+\beta(s)\mathbb E[X_1\mid X_t=x]
\end{aligned}
$$

因此：

$$
\mathbb E[X_s\mid X_t=x]
=
\alpha(s)\eta_z(t,x)+\beta(s)D(t,x)
\tag{5.53}
$$

代入式 (5.49)：

$$
D(t,x)=\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
$$

得到：

$$
\begin{aligned}
\mathbb E[X_s\mid X_t=x]
={}&
\alpha(s)\eta_z(t,x)\\
&+
\frac{\beta(s)}{\beta(t)}
\left[
x-\alpha(t)\eta_z(t,x)
\right]
\end{aligned}
$$

整理：

$$
\boxed{
\begin{aligned}
\mathbb E[X_s\mid X_t=x]
={}&
\frac{\beta(s)}{\beta(t)}x\\
&+
\left[
\alpha(s)-
\frac{\alpha(t)\beta(s)}{\beta(t)}
\right]\eta_z(t,x)
\end{aligned}
}
\tag{5.54}
$$

该公式在 $\beta(t)\neq0$ 时成立。

它说明：

**一个噪声预测器不仅能够恢复干净数据，还能够给出同一随机插值路径在任意其他时刻的条件均值。**

这也是原论文第 5.2 节的重要结果之一。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 5.2.6 为什么迭代去噪可以构造生成模型？

考虑从噪声向数据演化的时间序列：

$$
0=t_0<t_1<\cdots<t_N=1
$$

一种自然想法是：

$$
\boxed{
Y_{k+1}
=
\mathbb E[X_{t_{k+1}}\mid X_{t_k}=Y_k]
}
\tag{5.55}
$$

即每次使用去噪器预测下一个时刻的条件均值。

根据式 (5.54)，可以写成：

$$
\begin{aligned}
Y_{k+1}
={}&
\frac{\beta(t_{k+1})}{\beta(t_k)}Y_k\\
&+
\left[
\alpha(t_{k+1})-
\frac{\alpha(t_k)\beta(t_{k+1})}{\beta(t_k)}
\right]
\eta_z(t_k,Y_k)
\end{aligned}
\tag{5.56}
$$

在 $t_k=0$ 等分母为零的端点，需要使用相应的端点表达式或极限。

这个迭代过程看起来与 ODE 积分非常相似。

为了说明二者的联系，令：

$$
t_{k+1}=t_k+h
$$

其中：

$$
h\to0
$$

对时间系数进行 Taylor 展开：

$$
\alpha(t+h)=\alpha(t)+h\dot\alpha(t)+O(h^2)
$$

$$
\beta(t+h)=\beta(t)+h\dot\beta(t)+O(h^2)
$$

因此：

$$
\frac{\beta(t+h)}{\beta(t)}
=
1+h\frac{\dot\beta(t)}{\beta(t)}+O(h^2)
$$

另外：

$$
\begin{aligned}
\alpha(t+h)
-\frac{\alpha(t)\beta(t+h)}{\beta(t)}
={}&
h\left[
\dot\alpha(t)
-\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right]\\
&+O(h^2)
\end{aligned}
$$

代入式 (5.56)：

$$
\begin{aligned}
Y_{k+1}
={}&
Y_k+
h\frac{\dot\beta}{\beta}Y_k\\
&+
h\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,Y_k)
+O(h^2)
\end{aligned}
$$

而第 4 章已经证明：

$$
b(t,x)=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,x)
$$

所以：

$$
\boxed{
Y_{k+1}
=
Y_k+h\,b(t_k,Y_k)+O(h^2)
}
\tag{5.57}
$$

这正是 Probability Flow ODE 的 Euler 离散形式：

$$
\frac{dY_t}{dt}=b(t,Y_t)
$$

因此，在适当的正则性与收敛条件下：

$$
\boxed{
\text{无限细的迭代条件均值去噪}
\longrightarrow
\text{Probability Flow ODE}
}
\tag{5.58}
$$

这里要区分两个概念：

- 单次去噪：计算某个条件后验均值。
- 迭代去噪：通过连续的小时间步，构造完整的生成动力学。

单次去噪不一定能够正确采样目标分布；但适当构造的连续时间迭代，可以收敛到生成 ODE。


### 5.2.7 本节总结

本节建立了三个重要联系。

首先：

$$
\boxed{
\text{Noise MSE}
\Longrightarrow
\eta_z=\mathbb E[Z\mid X_t=x]
\Longrightarrow
s=-\frac{\eta_z}{\alpha}
}
$$

其次：

$$
\boxed{
D(t,x)=
\mathbb E[X_1\mid X_t=x]
=
\frac{x+\alpha^2s(t,x)}{\beta}
}
$$

最后：

$$
\boxed{
\text{迭代条件均值去噪}
\longrightarrow
\text{Probability Flow ODE}
}
$$

因此，Denoising、Score Matching 和 Flow Matching 并不是完全独立的思想。

在高斯线性插值下，它们可以通过条件期望和概率流建立直接的数学联系。

---


## 5.3 与 Rectified Flow、Reflow 的联系


### 5.3.1 核心结论

Rectified Flow 与随机插值框架的联系非常直接。

它使用线性插值：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{5.59}
$$

对应的条件速度为：

$$
\boxed{
\dot X_t=X_1-X_0
}
\tag{5.60}
$$

因此，可以通过速度回归学习：

$$
\boxed{
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
}
\tag{5.61}
$$

但 Rectified Flow 不仅关注如何学习速度场，还关注一个更进一步的问题：

**能否改变训练样本的端点耦合，使得生成 ODE 的轨迹越来越接近直线，从而减少采样所需的积分步数？**

这就引出了 Rectification 和 Reflow。

原论文第 5.3 节讨论了通过已有生成映射重新构造随机插值，并分析了在可逆性条件下，新 ODE 如何保持原来的终点映射，同时获得更简单的轨迹。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 5.3.2 为什么使用直线插值，ODE 轨迹仍然可能弯曲？

首先考虑：

$$
X_t=(1-t)X_0+tX_1
$$

对时间求导：

$$
\dot X_t=X_1-X_0
$$

对于固定的端点对 $(X_0,X_1)$，这确实是一条直线。

但模型学习的是：

$$
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
$$

假设有多组不同端点对，它们的直线插值在某个时刻经过相同或相近的位置。

例如：

$$
(X_0^{(1)},X_1^{(1)})
$$

和：

$$
(X_0^{(2)},X_1^{(2)})
$$

两组样本的条件速度分别为：

$$
v_1=X_1^{(1)}-X_0^{(1)}
$$

$$
v_2=X_1^{(2)}-X_0^{(2)}
$$

如果在某个 $(t,x)$ 处，两组端点对都具有非零的后验权重，那么边缘速度场会对这些条件速度进行平均：

$$
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
$$

因此，模型得到的速度可能既不等于 $v_1$，也不等于 $v_2$。

而随着时间和位置变化，条件权重也会改变。

因此：

$$
b(t,x)
$$

一般是一个非线性的时间—空间速度场。

生成 ODE：

$$
\boxed{
\frac{dY_t}{dt}=b(t,Y_t)
}
\tag{5.62}
$$

所产生的轨迹不一定是直线。

这就是 Rectified Flow 需要进一步进行 Rectification 的原因。


### 5.3.3 Rectification 的基本思想

假设我们已经训练好了速度场：

$$
b(t,x)
$$

并且能够求解：

$$
\frac{dY_t}{dt}=b(t,Y_t)
$$

从初始样本：

$$
Y_0=x_0
$$

出发，积分到：

$$
t=1
$$

得到：

$$
Y_1=T(x_0)
\tag{5.63}
$$

这里：

$$
T:\mathbb R^d\to\mathbb R^d
$$

是生成 ODE 的终点映射。

在理想情况下：

$$
X_0\sim\rho_0
$$

经过：

$$
X_1=T(X_0)
$$

得到：

$$
X_1\sim\rho_1
$$

也就是说：

$$
\boxed{
T_\#\rho_0=\rho_1
}
\tag{5.64}
$$

其中 $T_\#\rho_0$ 表示分布 $\rho_0$ 在映射 $T$ 下的推前分布。

现在我们已经有了一组新的端点对：

$$
\boxed{
(X_0,T(X_0))
}
\tag{5.65}
$$

与最初随机采样的端点对不同，这两个端点具有明确的确定性对应关系。

因此，我们可以用这组新端点重新训练一个 Flow Matching 模型。

这就是 Reflow 的基本思想。


### 5.3.4 重新构造直线插值

使用新的端点对：

$$
X_0,\qquad T(X_0)
$$

构造：

$$
\boxed{
X_t^{\mathrm{rec}}
=
(1-t)X_0+tT(X_0)
}
\tag{5.66}
$$

这里上标 $\mathrm{rec}$ 表示 rectified。

对时间求导：

$$
\boxed{
\dot X_t^{\mathrm{rec}}
=
T(X_0)-X_0
}
\tag{5.67}
$$

于是新的边缘速度场为：

$$
\boxed{
b^{\mathrm{rec}}(t,x)
=
\mathbb E[
T(X_0)-X_0
\mid X_t^{\mathrm{rec}}=x
]
}
\tag{5.68}
$$

对应的训练损失为：

$$
\boxed{
\mathcal L_{\mathrm{rec}}(\theta)
=
\mathbb E
\left[
\left\|
b_\theta^{\mathrm{rec}}
(t,X_t^{\mathrm{rec}})
-
[T(X_0)-X_0]
\right\|^2
\right]
}
\tag{5.69}
$$

这与普通 Flow Matching 的训练形式相同。

区别仅在于端点耦合：

**第一次训练：**

$$
(X_0,X_1)\sim\nu_0
$$

**Reflow 训练：**

$$
X_1=T(X_0)
$$

也就是说，Reflow 的主要改变不是损失函数，而是训练数据中两个端点的对应关系。


### 5.3.5 为什么 Reflow 有可能让轨迹变直？

关键在于：

$$
T(X_0)
$$

已经是原始生成 ODE 的终点。

因此，新的训练目标是直接连接：

$$
X_0\longrightarrow T(X_0)
$$

而不再使用原始端点耦合。

如果新的插值映射：

$$
M_t(x_0)=(1-t)x_0+tT(x_0)
\tag{5.70}
$$

对于每个 $t$ 都是可逆的，那么当前位置：

$$
x=M_t(x_0)
$$

就可以唯一确定起点：

$$
x_0=M_t^{-1}(x)
$$

因此，在给定 $(t,x)$ 后，不再需要对多个可能的起点求条件平均。

此时：

$$
\begin{aligned}
b^{\mathrm{rec}}(t,x)
&=
T(M_t^{-1}(x))-M_t^{-1}(x)
\end{aligned}
\tag{5.71}
$$

令：

$$
Y_t=M_t(x_0)
$$

则：

$$
\begin{aligned}
b^{\mathrm{rec}}(t,Y_t)
&=
T(M_t^{-1}(M_t(x_0)))
-
M_t^{-1}(M_t(x_0))\\
&=T(x_0)-x_0
\end{aligned}
$$

因此：

$$
\boxed{
\frac{dY_t}{dt}=T(x_0)-x_0
}
\tag{5.72}
$$

右侧与时间无关。

积分得到：

$$
\boxed{
Y_t=(1-t)x_0+tT(x_0)
}
\tag{5.73}
$$

这是一条严格的直线。

因此，在插值映射可逆、速度场精确等条件下，重新构造的 Probability Flow ODE 具有直线解。

这也是原论文 Theorem 47 所强调的结果。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 5.3.6 为什么需要可逆性条件？

前面的关键步骤是：

$$
x=M_t(x_0)
$$

能够唯一恢复：

$$
x_0=M_t^{-1}(x)
$$

如果 $M_t$ 不可逆，那么可能存在：

$$
x_0^{(1)}\neq x_0^{(2)}
$$

但：

$$
M_t(x_0^{(1)})=M_t(x_0^{(2)})
$$

这意味着两条插值轨迹在同一时刻到达同一个位置。

如果它们的速度不同：

$$
T(x_0^{(1)})-x_0^{(1)}
\neq
T(x_0^{(2)})-x_0^{(2)}
$$

那么边缘速度场仍然需要进行条件平均。

因此，不能直接断言生成 ODE 会严格沿着每条原始插值直线运动。

从微分角度看：

$$
M_t(x)=(1-t)x+tT(x)
$$

其 Jacobian 为：

$$
\boxed{
J_{M_t}(x)
=
(1-t)I_d+tJ_T(x)
}
\tag{5.74}
$$

其中：

$$
J_T(x)=\frac{\partial T(x)}{\partial x}
$$

是终点映射的 Jacobian。

若：

$$
\det J_{M_t}(x)\neq0
$$

则根据反函数定理，$M_t$ 在该点附近局部可逆。

但需要注意，**Jacobian 非奇异只直接保证局部可逆，不自动保证全局可逆**；全局结论还需要额外条件。

这也解释了为什么原论文将插值映射的可逆性单独列为假设。


### 5.3.7 Rectification 是否会改变最终生成分布？

这是本节非常重要的问题。

原始 ODE 的终点映射为：

$$
T(x_0)
$$

重新构造的插值为：

$$
X_t^{\mathrm{rec}}=(1-t)X_0+tT(X_0)
$$

因此：

$$
X_0^{\mathrm{rec}}=X_0
$$

$$
X_1^{\mathrm{rec}}=T(X_0)
$$

如果原始模型是精确的：

$$
T_\#\rho_0=\rho_1
$$

那么新的插值终点仍然满足：

$$
X_1^{\mathrm{rec}}\sim\rho_1
$$

所以：

$$
\boxed{
\rho_1^{\mathrm{rec}}=\rho_1
}
\tag{5.75}
$$

在第 5.3.5 节的可逆性条件下，还可以得到更强的结论：

$$
\boxed{
T^{\mathrm{rec}}=T
}
\tag{5.76}
$$

也就是说，新旧模型具有相同的终点映射。

但是中间概率路径一般不同：

$$
\rho_t^{\mathrm{rec}}\neq\rho_t
$$

这正是 Rectification 的目标：

**在不改变理想终点映射的情况下，重新设计中间路径，使生成动力学更加简单。**

不过，实际模型中需要考虑两个误差来源：

第一，原始模型的 $T$ 可能不是精确的目标分布映射。

第二，Reflow 模型的速度场也存在训练误差。

因此，在实际有限模型条件下，Reflow 可能改变最终生成分布，并不保证生成质量一定提高。


### 5.3.8 Reflow 与最优传输是什么关系？

前面已经说明：

Rectification 可以使生成轨迹变直。

那么是否意味着它找到了最优传输？

答案是：**不一定。**

考虑二次代价最优传输：

$$
\boxed{
\min_{T:\,T_\#\rho_0=\rho_1}
\mathbb E_{X_0\sim\rho_0}
\|T(X_0)-X_0\|^2
}
\tag{5.77}
$$

它不仅要求：

$$
T_\#\rho_0=\rho_1
$$

还要求在所有可行映射中，运输代价最小。

而 Rectification 主要研究的是：

$$
X_t=(1-t)X_0+tT(X_0)
$$

能否成为对应 ODE 的直线轨迹。

即使某个 $T$ 不是最优传输映射，只要满足适当可逆性条件，也可能构造出直线生成轨迹。

因此：

$$
\boxed{
\text{直线生成轨迹}
\not\Rightarrow
\text{最优传输映射}
}
\tag{5.78}
$$

这一点与原论文 Remark 48 一致。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 5.3.9 为什么多轮 Reflow 可能降低运输代价？

下面补充 Rectified Flow 文献中常见的一个数学解释。

假设当前端点耦合为：

$$
(X_0,X_1)
$$

构造：

$$
X_t=(1-t)X_0+tX_1
$$

其边缘速度场：

$$
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
$$

现在使用这个速度场生成新的 ODE 轨迹：

$$
\dot Y_t=b(t,Y_t)
$$

并令：

$$
Y_0\sim\rho_0
$$

在理想条件下：

$$
Y_1\sim\rho_1
$$

因此，新旧端点耦合具有相同的边缘分布。

接下来比较二次运输代价。

首先：

$$
Y_1-Y_0=\int_0^1b(t,Y_t)dt
$$

根据 Jensen 不等式：

$$
\left\|
\int_0^1b(t,Y_t)dt
\right\|^2
\le
\int_0^1\|b(t,Y_t)\|^2dt
$$

所以：

$$
\mathbb E\|Y_1-Y_0\|^2
\le
\int_0^1\mathbb E\|b(t,Y_t)\|^2dt
\tag{5.79}
$$

由于 ODE 的边缘分布与插值的边缘分布相同：

$$
Y_t\overset d=X_t
$$

因此：

$$
\int_0^1\mathbb E\|b(t,Y_t)\|^2dt
=
\int_0^1\mathbb E\|b(t,X_t)\|^2dt
\tag{5.80}
$$

又因为：

$$
b(t,X_t)=\mathbb E[X_1-X_0\mid X_t]
$$

根据条件期望的 Jensen 不等式：

$$
\mathbb E
\left\|
\mathbb E[X_1-X_0\mid X_t]
\right\|^2
\le
\mathbb E\|X_1-X_0\|^2
$$

因此：

$$
\int_0^1\mathbb E\|b(t,X_t)\|^2dt
\le
\mathbb E\|X_1-X_0\|^2
\tag{5.81}
$$

合并：

$$
\boxed{
\mathbb E\|Y_1-Y_0\|^2
\le
\mathbb E\|X_1-X_0\|^2
}
\tag{5.82}
$$

这说明：

**在理想连续时间条件下，Rectification 得到的新端点耦合，其二次运输代价不会超过旧耦合。**

需要特别注意，这个结论说明的是：

$$
C_{\mathrm{new}}\le C_{\mathrm{old}}
$$

而不是：

$$
C_{\mathrm{new}}<C_{\mathrm{old}}
$$

也就是说，运输代价是**非增**的，不保证每轮严格下降。

另外：

$$
C_{\mathrm{new}}\le C_{\mathrm{old}}
$$

也不意味着：

$$
C_{\mathrm{new}}=C_{\mathrm{OT}}
$$

因为代价非增并不自动保证收敛到全局最优传输代价。

这一点在理解多轮 Reflow 时尤其重要。


### 5.3.10 Reflow 是否一定能实现一步生成？

不能直接保证。

在理想情况下，如果 Rectification 后的速度场满足：

$$
b^{\mathrm{rec}}(t,Y_t)=T(X_0)-X_0
$$

那么生成轨迹为：

$$
Y_t=X_0+t[T(X_0)-X_0]
$$

于是：

$$
Y_1=X_0+[T(X_0)-X_0]
$$

因此：

$$
\boxed{
Y_1=X_0+b^{\mathrm{rec}}(0,X_0)
}
\tag{5.83}
$$

这意味着一步 Euler 积分就能够得到精确终点。

但这依赖：

- 理想速度场被精确学习。
- 重新构造的插值映射满足相应可逆性条件。
- 生成 ODE 确实具有直线解。
- 数值实现能够正确计算起点速度。

实际训练中，这些条件未必完全成立。

因此，多轮 Reflow 的主要目标是提高轨迹平直性、减少数值积分误差，而不是保证任意模型都能一步生成。

另外，**Reflow 与一步蒸馏不是同一个概念**。

Reflow 主要改变端点耦合并重新学习速度场。

一步蒸馏则可以直接训练一个映射：

$$
G_\theta(X_0)\approx T(X_0)
$$

不一定要求显式学习连续时间速度场。

两者可以结合，但数学目标不同。


### 5.3.11 本节总结

Rectified Flow 使用：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
$$

学习：

$$
\boxed{
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
}
$$

Reflow 则利用已有 ODE 的终点映射：

$$
T(X_0)
$$

构造新的端点耦合：

$$
\boxed{
(X_0,T(X_0))
}
$$

并重新训练速度场。

在适当可逆性条件下，新 ODE 可以沿直线：

$$
\boxed{
Y_t=(1-t)X_0+tT(X_0)
}
$$

运动，并保持原有的终点映射。

但必须区分：

$$
\boxed{
\begin{aligned}
\text{直线插值}&\neq\text{直线 ODE 轨迹}\\
\text{直线 ODE 轨迹}&\neq\text{最优传输}\\
\text{运输代价非增}&\neq\text{保证收敛到全局最优}\\
\text{Reflow}&\neq\text{必然一步生成}
\end{aligned}
}
$$

---


# 第 5 章总结

本章讨论了随机插值与三类生成方法之间的联系。

**第一，Score-Based Diffusion 可以从随机插值的角度理解。**

OU、VP、VE 等线性扩散过程的单时刻条件分布都具有：

$$
\boxed{
Y_\tau\overset d=m(\tau)X_1+\sigma(\tau)Z
}
$$

这一结构。

但随机插值与扩散 SDE 的完整路径分布一般不同。

**第二，Denoising 与 Score Matching 可以通过条件期望统一。**

对于：

$$
X_t=\alpha Z+\beta X_1
$$

有：

$$
\boxed{
\begin{aligned}
\eta_z(t,x)&=\mathbb E[Z\mid X_t=x]\\
s(t,x)&=-\frac{\eta_z(t,x)}{\alpha}\\
D(t,x)&=\frac{x+\alpha^2s(t,x)}{\beta}
\end{aligned}
}
$$

同时，迭代条件均值去噪在适当极限下可以得到 Probability Flow ODE。

**第三，Rectified Flow 与 Reflow 研究如何优化生成轨迹。**

Rectified Flow 通过线性插值学习边缘速度场。

Reflow 则利用已有生成映射重新构造端点耦合，使生成 ODE 有机会获得更平直的轨迹。

但轨迹平直性、运输代价最优性和一步生成能力是不同的数学性质。

因此，本章最终建立了如下联系：

$$
\boxed{
\begin{gathered}
\text{随机插值概率路径}\\
\Downarrow\\
\text{条件期望、速度场与 Score}\\
\Downarrow\\
\begin{array}{ccc}
\text{Score-Based Diffusion}
&
\text{Denoising}
&
\text{Rectified Flow}
\end{array}
\end{gathered}
}
$$

**这些方法的共同基础是概率分布的连续演化，而它们的主要区别在于概率路径的设计、预测目标的选择，以及生成动力学的构造方式。**

下一章将进入 **Algorithmic Aspects**，重点研究这些理论公式在实际神经网络训练和 ODE/SDE 采样中的实现方法。


<a id="chapter-6"></a>

# 第 6 章：Algorithmic Aspects

前面五章主要讨论了随机插值的数学理论，包括概率路径、速度场、Score、ODE/SDE 的统一关系，以及它们与 Diffusion、Flow Matching 和 Rectified Flow 的联系。

本章进一步回答两个实际问题：

1. **如何使用神经网络训练随机插值模型？** 训练数据如何构造？为什么可以直接使用样本速度和噪声作为监督信号，而不需要知道真实的边缘速度场与 Score？
2. **如何通过 ODE 或 SDE 生成样本？** 不同数值求解器有什么区别？如何理解时间步长、误差、数值稳定性以及计算成本之间的关系？

本章的核心结论是：

**随机插值模型可以通过直接采样端点和辅助噪声，构造无须模拟动力学的监督训练；训练完成后，再使用 ODE 或 SDE 数值积分实现生成。**

因此，整个算法可以分为：

$$
\boxed{
\text{构造随机插值}
\longrightarrow
\text{学习速度场与 Score}
\longrightarrow
\text{ODE/SDE 数值采样}
}
\tag{6.1}
$$


## 6.1 随机插值模型的训练方法


### 6.1.1 核心结论：用可观测样本监督不可直接计算的条件期望

考虑一般随机插值：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{6.2}
$$

其中：

$$
(X_0,X_1)\sim\nu,\qquad Z\sim\mathcal N(0,I_d)
$$

且 $Z$ 独立于端点对。

随机插值的时间导数为：

$$
\boxed{
\dot X_t=\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
}
\tag{6.3}
$$

根据第 2 章：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{6.4}
$$

以及：

$$
\boxed{
s(t,x)=-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
}
\tag{6.5}
$$

直接计算这两个条件期望通常非常困难。

因为对于给定的：

$$
X_t=x
$$

我们需要知道所有可能产生该位置的端点对与噪声，并计算它们的后验平均。

对于高维图像、轨迹或其他复杂数据分布，这通常不可行。

但是，我们可以直接采样：

$$
X_0,\quad X_1,\quad Z,\quad t
$$

并计算：

$$
X_t,\quad\dot X_t
$$

因此可以构造监督回归问题。

对于速度场：

$$
\boxed{
\mathcal L_b(\theta)
=
\mathbb E
\left[
\left\|
b_\theta(t,X_t)-\dot X_t
\right\|^2
\right]
}
\tag{6.6}
$$

对于 Score：

$$
\boxed{
\mathcal L_s(\phi)
=
\mathbb E
\left[
\left\|
s_\phi(t,X_t)+\frac{Z}{\gamma(t)}
\right\|^2
\right]
}
\tag{6.7}
$$

其中式 (6.7) 要求：

$$
\gamma(t)>0
$$

两者都是普通的监督学习目标，不需要在训练过程中求解生成 ODE 或 SDE。


### 6.1.2 为什么速度回归可以学习真实边缘速度场？

这一点是理解 Flow Matching 训练的关键。

令：

$$
U_t=\dot X_t
$$

真实速度场为：

$$
b(t,x)=\mathbb E[U_t\mid X_t=x]
$$

神经网络的损失为：

$$
\mathcal L_b(\theta)
=
\mathbb E\|b_\theta(t,X_t)-U_t\|^2
$$


将附录的通用结论用于输入 $(t,X_t)$、标签 $U_t=\dot X_t$，得到：

完整推导见[附录 A.2.2～A.2.3：回归损失的正交分解](#app-a-2-2)。

$$
\boxed{
\begin{aligned}
\mathcal L_b(\theta)
={}&
\mathbb E\|b_\theta(t,X_t)-b(t,X_t)\|^2\\
&+\mathbb E\|b(t,X_t)-\dot X_t\|^2
\end{aligned}
}
\tag{6.9}
$$

第二项与模型参数无关。

因此：

$$
\boxed{
\arg\min_\theta\mathcal L_b(\theta)
=
\arg\min_\theta
\mathbb E\|b_\theta(t,X_t)-b(t,X_t)\|^2
}
\tag{6.10}
$$

这里的等式是指两个目标具有相同的最优解集合。

所以，我们不需要直接计算：

$$
b(t,x)
$$

只需要使用容易计算的：

$$
\dot X_t
$$

作为监督信号。

**这就是随机插值训练能够避免显式求解条件期望的原因。**

需要注意，单个样本的：

$$
\dot X_t
$$

不一定等于：

$$
b(t,X_t)
$$

它只是一个带有条件随机性的监督目标。

训练通过大量样本，将这些条件速度进行统计平均。


### 6.1.3 Score 回归为什么同样成立？


由[附录 D.1.3](#app-d-1-3) 与[附录 A.2.2](#app-a-2-2)，在 $\gamma(t)>0$ 时，标签 $-Z/\gamma(t)$ 的条件均值是真实边缘 Score。因此 Score 回归与第 5.2 节的 Denoising Score Matching 使用同一原理。

实际训练中，直接使用：

$$
-\frac Z{\gamma(t)}
$$

可能存在数值问题。

例如，当：

$$
\gamma(t)\to0
$$

时，监督目标的尺度可能变得非常大。

因此，可以改为预测：

$$
\eta_\phi(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

使用：

$$
\boxed{
\mathcal L_\eta(\phi)
=
\mathbb E\|\eta_\phi(t,X_t)-Z\|^2
}
\tag{6.11}
$$

然后通过：

$$
\boxed{
s_\phi(t,x)=-\frac{\eta_\phi(t,x)}{\gamma(t)}
}
\tag{6.12}
$$

恢复 Score。

这两种参数化在理想情况下对应相同的条件统计量，但训练权重和数值性质不同。


### 6.1.4 一般随机插值的训练流程

对于一个 Batch，训练过程可以分为以下步骤。

**第一步：采样端点。**

$$
(X_0,X_1)\sim\nu
$$

这里的 $\nu$ 是端点耦合。

例如：

- 独立耦合：分别从 $\rho_0,\rho_1$ 独立采样。
- 配对耦合：使用已有对应关系的端点样本。
- OT 耦合：通过最优传输方法构造端点对应关系。

不同耦合会改变中间概率路径和速度场。

**第二步：采样时间。**

$$
t\sim p(t)
$$

常见选择是：

$$
t\sim\mathcal U(0,1)
$$

但也可以根据训练难度采用非均匀时间采样。

**第三步：采样辅助噪声。**

$$
Z\sim\mathcal N(0,I_d)
$$

**第四步：构造中间样本。**

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

**第五步：计算监督目标。**

速度监督：

$$
U_t=\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
$$

噪声监督：

$$
Z
$$

**第六步：执行网络前向传播。**

例如：

$$
\widehat b=b_\theta(t,X_t)
$$

$$
\widehat\eta=\eta_\phi(t,X_t)
$$

**第七步：计算损失并更新参数。**

$$
\boxed{
\mathcal L
=
\lambda_b\mathcal L_b+
\lambda_\eta\mathcal L_\eta
}
\tag{6.13}
$$

其中：

$$
\lambda_b,\lambda_\eta\ge0
$$

是损失权重。

这两个预测器可以使用独立网络，也可以共享主干并使用不同输出头。

需要注意：如果选择的参数化已经允许从一个预测器恢复另一个预测场，那么并不一定需要同时训练两个预测头。


### 6.1.5 单侧线性插值的训练

对于实际生成模型，最常用的形式之一是：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{6.14}
$$

此时：

$$
\dot X_t=\dot\alpha(t)Z+\dot\beta(t)X_1
\tag{6.15}
$$

如果直接训练速度场：

$$
\boxed{
\mathcal L_v
=
\mathbb E
\left[
\left\|
v_\theta(t,X_t)-
\left(\dot\alpha Z+\dot\beta X_1\right)
\right\|^2
\right]
}
\tag{6.16}
$$

如果训练噪声预测器：

$$
\boxed{
\mathcal L_\eta
=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
}
\tag{6.17}
$$

如果训练数据预测器：

$$
\boxed{
\mathcal L_{x_1}
=
\mathbb E\|D_\theta(t,X_t)-X_1\|^2
}
\tag{6.18}
$$

三种参数化的理想最优解分别为：

$$
\begin{aligned}
v^*(t,x)&=\mathbb E[\dot X_t\mid X_t=x]\\
\eta^*(t,x)&=\mathbb E[Z\mid X_t=x]\\
D^*(t,x)&=\mathbb E[X_1\mid X_t=x]
\end{aligned}
$$

根据第 4.4 节，它们在非退化时间区间内可以相互转换。

例如：

$$
\boxed{
v(t,x)=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta(t,x)
}
\tag{6.19}
$$

因此，预测目标的选择可以结合数值稳定性、任务维度和采样器设计进行考虑。

但不能简单认为三种损失在有限模型条件下完全等价。


### 6.1.6 条件生成模型如何训练？

前面默认目标分布为：

$$
X_1\sim\rho_1
$$

对于自动驾驶轨迹规划等任务，我们通常需要建模条件分布：

$$
\boxed{
X_1\sim\rho_1(\cdot\mid c)
}
\tag{6.20}
$$

其中 $c$ 可以是：

- 当前场景的感知编码。
- 地图与道路结构。
- 导航信息。
- 交通参与者的历史状态。

这时使用条件速度场：

$$
v_\theta(t,x,c)
$$

训练过程为：

$$
(X_1,c)\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I_d)
$$

$$
X_t=\alpha Z+\beta X_1
$$

损失为：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathbb E
\left[
\left\|
v_\theta(t,X_t,c)
-
(\dot\alpha Z+\dot\beta X_1)
\right\|^2
\right]
}
\tag{6.21}
$$

其最优预测器为：

$$
\boxed{
v^*(t,x,c)=
\mathbb E[
\dot\alpha Z+\dot\beta X_1
\mid X_t=x,c
]
}
\tag{6.22}
$$

特别地，当：

$$
\alpha(t)=1-t,\qquad\beta(t)=t
$$

有：

$$
X_t=(1-t)Z+tX_1
$$

$$
\dot X_t=X_1-Z
$$

因此：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathbb E
\left[
\|v_\theta(t,X_t,c)-(X_1-Z)\|^2
\right]
}
\tag{6.23}
$$

这就是常见的条件 Flow Matching 训练形式。

需要强调，条件 $c$ 并不要求在训练集中出现完全相同的场景多次。

网络通过跨场景共享参数，学习条件分布之间的统计规律。


### 6.1.7 训练时需要注意的数值问题

随机插值训练虽然不需要求解 ODE/SDE，但仍然存在一些数值问题。

**第一，时间端点可能出现奇异性。**

例如：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

则：

$$
\dot\gamma(t)=
\frac{1-2t}{2\sqrt{t(1-t)}}
$$

当：

$$
t\to0,\qquad t\to1
$$

时：

$$
|\dot\gamma(t)|\to\infty
$$

因此，速度监督：

$$
\dot X_t=\partial_tI+\dot\gamma Z
$$

可能具有较大的数值尺度。

一种处理方式是避免直接采样精确端点：

$$
t\sim\mathcal U(\delta,1-\delta)
$$

其中：

$$
0<\delta\ll1
$$

但这会改变训练覆盖的时间范围，端点附近的误差仍然需要单独考虑。

**第二，不同时间的监督目标尺度可能不同。**

例如：

$$
\|\dot X_t\|^2
$$

在不同时间可能存在较大差异。

因此，可以使用时间相关权重：

$$
\boxed{
\mathcal L_v
=
\mathbb E[
w(t)\|v_\theta(t,X_t)-\dot X_t\|^2
]
}
\tag{6.24}
$$

但需要注意，权重函数会改变不同时间点在整体训练目标中的相对重要性。

**第三，端点耦合影响训练目标的条件方差。**

根据式 (6.9)：

$$
\mathcal L_v
=
\mathbb E\|v_\theta-b\|^2+
\mathbb E\|\dot X_t-b\|^2
$$

第二项为不可约的条件方差：

$$
\mathbb E[
\operatorname{Var}(\dot X_t\mid X_t)
]
$$

不同端点耦合可能改变这一项。

因此，端点耦合不仅影响概率路径，也可能影响监督信号的统计难度。

不过，条件方差更小并不自动意味着最终生成效果一定更好，还需要结合模型容量、数据分布和数值采样进行判断。


### 6.1.8 本节总结

随机插值训练的核心是：

$$
\boxed{
\text{条件期望难以直接计算}
\quad\Longrightarrow\quad
\text{使用样本级监督进行 MSE 回归}
}
$$

对于速度场：

$$
\boxed{
v^*(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

对于噪声预测：

$$
\boxed{
\eta^*(t,x)=\mathbb E[Z\mid X_t=x]
}
$$

对于数据预测：

$$
\boxed{
D^*(t,x)=\mathbb E[X_1\mid X_t=x]
}
$$

训练阶段只需要构造随机插值样本，不需要执行生成过程的数值积分。

---


## 6.2 ODE 与 SDE 的采样算法及数值稳定性


### 6.2.1 核心结论

训练完成后，我们已经得到了近似速度场：

$$
b_\theta(t,x)
$$

以及可能的 Score：

$$
s_\phi(t,x)
$$

接下来需要从：

$$
Y_0\sim\rho_0
$$

出发，生成：

$$
Y_1\sim\rho_1
$$

理论上，可以使用：

$$
\boxed{
dY_t=b(t,Y_t)dt
}
\tag{6.25}
$$

或者：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{6.26}
$$

在精确预测场、适定性和正确初始分布等条件下，二者可以具有相同的边缘概率路径。

但实际采样需要离散时间：

$$
0=t_0<t_1<\cdots<t_N=1
$$

因此必须使用数值求解器。

不同求解器的主要区别在于：

- 每一步如何近似状态变化。
- 需要多少次神经网络推理。
- 局部误差与全局误差的阶数。
- 对速度场变化和数值刚性的敏感程度。


### 6.2.2 Euler 方法：最简单的 ODE 求解器

考虑：

$$
\frac{dY_t}{dt}=b_\theta(t,Y_t)
$$

在时间区间：

$$
[t_k,t_{k+1}]
$$

有：

$$
Y_{t_{k+1}}
=
Y_{t_k}
+
\int_{t_k}^{t_{k+1}}b_\theta(t,Y_t)dt
\tag{6.27}
$$

Euler 方法使用左端点速度近似整个积分：

$$
\int_{t_k}^{t_{k+1}}b_\theta(t,Y_t)dt
\approx
h_kb_\theta(t_k,Y_{t_k})
$$

其中：

$$
h_k=t_{k+1}-t_k
$$

因此：

$$
\boxed{
Y_{k+1}
=
Y_k+h_kb_\theta(t_k,Y_k)
}
\tag{6.28}
$$

每一步只需要一次网络推理。

因此：

$$
\boxed{
\mathrm{NFE}=N
}
\tag{6.29}
$$

其中 NFE 表示 Number of Function Evaluations，即速度场网络调用次数。

Euler 方法是一阶方法。

在速度场足够光滑、求解稳定等条件下：

- 局部截断误差为 $O(h^2)$。
- 全局离散化误差为 $O(h)$。

因此，如果将步长缩小一半，理想情况下全局误差约缩小一半。

但这只是渐近阶数，并不意味着所有问题都严格按这个比例变化。


### 6.2.3 Heun 方法：二阶预测—校正

Euler 方法只使用区间起点的速度。

如果速度场在区间内变化明显，那么：

$$
b(t_k,Y_k)
$$

可能无法准确代表整个区间的平均速度。

Heun 方法使用两次速度评估。

首先进行 Euler 预测：

$$
\boxed{
\widetilde Y_{k+1}
=
Y_k+h_kb_\theta(t_k,Y_k)
}
\tag{6.30}
$$

然后在预测终点计算速度：

$$
\widetilde b_{k+1}
=
b_\theta(t_{k+1},\widetilde Y_{k+1})
\tag{6.31}
$$

最后使用起点和终点速度的平均值：

$$
\boxed{
Y_{k+1}
=
Y_k+
\frac{h_k}{2}
\left[
b_\theta(t_k,Y_k)+
b_\theta(t_{k+1},\widetilde Y_{k+1})
\right]
}
\tag{6.32}
$$

因此，每一步通常需要两次网络推理：

$$
\boxed{
\mathrm{NFE}\approx2N
}
\tag{6.33}
$$

Heun 方法是二阶方法。

在适当光滑性条件下：

- 局部截断误差为 $O(h^3)$。
- 全局误差为 $O(h^2)$。

因此，在相同时间步数下，Heun 通常比 Euler 更准确。

但是：

**Heun 每一步需要更多网络推理，因此不能只比较积分步数，还需要比较总 NFE。**

例如：

- Euler：20 步，约 20 NFE。
- Heun：10 步，约 20 NFE。

两者计算成本接近，但实际误差取决于速度场的具体性质。


### 6.2.4 Midpoint 方法与 Heun 的区别

显式中点法也是二阶 Runge–Kutta 方法。

首先计算起点速度：

$$
k_1=b_\theta(t_k,Y_k)
$$

使用半步 Euler 预测中点：

$$
Y_{k+\frac12}
=
Y_k+\frac{h_k}{2}k_1
$$

然后计算中点速度：

$$
k_2=
b_\theta\left(
t_k+\frac{h_k}{2},
Y_{k+\frac12}
\right)
$$

最终：

$$
\boxed{
Y_{k+1}=Y_k+h_kk_2
}
\tag{6.34}
$$

Heun 和 Midpoint 都是二阶方法，但取样位置不同。

Heun 使用：

$$
\text{起点速度}+\text{预测终点速度}
$$

Midpoint 使用：

$$
\text{预测中点速度}
$$

两者每步通常都需要两次网络推理。

不能笼统地认为其中一种始终更好。


### 6.2.5 RK4：经典四阶 Runge–Kutta 方法

对于：

$$
\dot Y_t=b_\theta(t,Y_t)
$$

RK4 定义：

$$
k_1=b_\theta(t_k,Y_k)
$$

$$
k_2=b_\theta\left(
t_k+\frac{h_k}{2},
Y_k+\frac{h_k}{2}k_1
\right)
$$

$$
k_3=b_\theta\left(
t_k+\frac{h_k}{2},
Y_k+\frac{h_k}{2}k_2
\right)
$$

$$
k_4=b_\theta(t_k+h_k,Y_k+h_kk_3)
$$

最终：

$$
\boxed{
Y_{k+1}
=
Y_k+\frac{h_k}{6}
(k_1+2k_2+2k_3+k_4)
}
\tag{6.35}
$$

RK4 每步需要四次网络推理。

在适当条件下，其全局误差为：

$$
O(h^4)
$$

因此，RK4 可以使用较大的时间步长获得较高精度。

但对于高成本神经网络，四次推理可能使单步计算开销较大。

因此，实际生成任务需要综合考虑：

$$
\boxed{
\text{生成误差}
+
\text{NFE}
+
\text{推理延迟}
}
$$


### 6.2.6 自适应步长求解器

前面的 Euler、Heun、RK4 通常可以使用固定步长：

$$
h=\frac1N
$$

但速度场在不同时间可能具有不同的变化速度。

例如，在某些时间区间：

$$
\left\|
\frac{d}{dt}b(t,Y_t)
\right\|
$$

较大，需要更小的步长。

而在其他时间区间，速度场变化平缓，可以使用较大的步长。

因此，可以采用自适应步长求解器。

其基本思想是：

1. 使用两个不同精度的近似，估计当前步的局部误差。
2. 如果误差超过容差，则缩小步长。
3. 如果误差远小于容差，则适当增大步长。
4. 重复积分直到到达终点。

例如 Dormand–Prince 类方法可以使用嵌入式 Runge–Kutta 公式进行误差估计。

通常需要设置：

$$
\mathrm{rtol},\qquad\mathrm{atol}
$$

分别表示相对误差容差和绝对误差容差。

但需要注意：

**自适应求解器控制的是数值积分误差，不是神经网络预测误差。**

即使将积分容差设置得非常小，如果：

$$
b_\theta(t,x)\neq b(t,x)
$$

最终生成结果仍然可能存在明显偏差。


### 6.2.7 SDE 的 Euler–Maruyama 采样

现在考虑前向生成 SDE：

$$
dY_t=
[b_\theta(t,Y_t)+\varepsilon(t)s_\phi(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
\tag{6.36}
$$

定义漂移：

$$
F_\theta(t,x)
=
b_\theta(t,x)+\varepsilon(t)s_\phi(t,x)
$$

于是：

$$
dY_t=F_\theta(t,Y_t)dt+\sqrt{2\varepsilon(t)}dW_t
$$

在时间区间：

$$
[t_k,t_{k+1}]
$$

有：

$$
Y_{t_{k+1}}
=
Y_{t_k}
+
\int_{t_k}^{t_{k+1}}F_\theta(t,Y_t)dt
+
\int_{t_k}^{t_{k+1}}\sqrt{2\varepsilon(t)}dW_t
$$

Euler–Maruyama 方法将漂移项近似为：

$$
h_kF_\theta(t_k,Y_k)
$$

将扩散系数近似为当前时刻的值。

由于布朗运动增量满足：

$$
W_{t_{k+1}}-W_{t_k}
\sim
\mathcal N(0,h_kI_d)
$$

因此可以写成：

$$
W_{t_{k+1}}-W_{t_k}
=
\sqrt{h_k}\xi_k
$$

其中：

$$
\xi_k\sim\mathcal N(0,I_d)
$$

最终得到：

$$
\boxed{
\begin{aligned}
Y_{k+1}
={}&
Y_k+
h_k[b_\theta(t_k,Y_k)+\varepsilon(t_k)s_\phi(t_k,Y_k)]\\
&+\sqrt{2\varepsilon(t_k)h_k}\,\xi_k
\end{aligned}
}
\tag{6.37}
$$

这是随机插值框架下非常直接的 SDE 采样公式。

与 ODE 不同，SDE 每一步都需要采样新的随机噪声：

$$
\xi_k
$$

因此，即使初始状态相同，最终样本也可能不同。


### 6.2.8 SDE 数值误差与 ODE 有什么不同？

ODE 求解器通常讨论：

$$
\|Y_T-\widehat Y_T\|
$$

即数值解与真实解之间的误差。

SDE 的误差需要进一步区分。

**强误差（Strong Error）：**

比较同一布朗运动驱动下的真实路径与数值路径。

例如：

$$
\mathbb E\|Y_T-\widehat Y_T\|
$$

**弱误差（Weak Error）：**

比较某个测试函数在真实分布与数值分布下的期望。

例如：

$$
\left|
\mathbb E[\varphi(Y_T)]
-
\mathbb E[\varphi(\widehat Y_T)]
\right|
$$

对于生成模型，我们通常更关注生成样本的分布是否接近目标分布。

因此，弱误差与分布层面的误差具有直接关系。

Euler–Maruyama 在一般充分正则条件下：

- 强收敛阶通常为 $1/2$。
- 弱收敛阶通常为 $1$。

不过，本章讨论的 SDE 具有时间相关、状态无关的扩散系数，在更强的光滑性等条件下，某些数值方法可能获得更高的强收敛阶。

因此不能不加条件地将所有 SDE 都归为同一种误差阶数。


### 6.2.9 速度场误差如何影响最终生成结果？

这是数值采样中非常重要的问题。

假设真实速度场为：

$$
b(t,x)
$$

训练得到：

$$
b_\theta(t,x)
$$

定义预测误差：

$$
e_\theta(t,x)=b_\theta(t,x)-b(t,x)
$$

考虑两个 ODE：

$$
\dot Y_t=b(t,Y_t)
$$

$$
\dot{\widehat Y}_t=b_\theta(t,\widehat Y_t)
$$

令：

$$
\Delta_t=\widehat Y_t-Y_t
$$

则：

$$
\begin{aligned}
\dot\Delta_t
&=
b_\theta(t,\widehat Y_t)-b(t,Y_t)\\
&=
[b_\theta(t,\widehat Y_t)-b(t,\widehat Y_t)]\\
&\quad+
[b(t,\widehat Y_t)-b(t,Y_t)]
\end{aligned}
$$

因此：

$$
\|\dot\Delta_t\|
\le
\|e_\theta(t,\widehat Y_t)\|
+
\|b(t,\widehat Y_t)-b(t,Y_t)\|
$$

假设真实速度场对空间变量满足 Lipschitz 条件：

$$
\|b(t,x)-b(t,y)\|\le L\|x-y\|
$$

并且模型误差满足：

$$
\|e_\theta(t,x)\|\le\delta
$$

则：

$$
\frac{d}{dt}\|\Delta_t\|
\le
L\|\Delta_t\|+\delta
$$

如果初始状态相同：

$$
\Delta_0=0
$$

根据 Grönwall 不等式：

$$
\boxed{
\|\Delta_t\|
\le
\frac{\delta}{L}(e^{Lt}-1)
}
\tag{6.38}
$$

当 $L=0$ 时，取极限得到：

$$
\|\Delta_t\|\le\delta t
$$

这个公式说明：

**速度场预测误差可能沿着生成轨迹传播和放大。**

即使使用高阶数值求解器，也无法自动消除模型本身的误差。

因此，最终生成误差通常可以理解为两个主要来源：

$$
\boxed{
\text{生成误差}
\approx
\text{模型预测误差}
+
\text{数值离散化误差}
}
\tag{6.39}
$$

这里的加号表示概念上的误差来源分解，并非对任意分布距离都严格成立的等式。


### 6.2.10 时间日程与数值稳定性

不同插值日程可能导致不同的数值积分难度。

例如：

$$
X_t=(1-t)Z+tX_1
$$

对应的条件样本速度为：

$$
\dot X_t=X_1-Z
$$

这个监督目标不显式依赖时间。

但边缘速度场：

$$
b(t,x)=\mathbb E[X_1-Z\mid X_t=x]
$$

仍然可能随时间快速变化。

对于其他日程，例如：

$$
\alpha(t)=\cos\left(\frac{\pi t}{2}\right)
$$

$$
\beta(t)=\sin\left(\frac{\pi t}{2}\right)
$$

其条件速度为：

$$
\dot X_t=
-\frac{\pi}{2}\sin\left(\frac{\pi t}{2}\right)Z
+
\frac{\pi}{2}\cos\left(\frac{\pi t}{2}\right)X_1
$$

因此，监督目标的时间尺度也会发生变化。

在实际采样中，可能出现以下问题。

**第一，端点附近的系数奇异。**

例如：

$$
b(t,x)=\frac{x-\eta_z(t,x)}{t}
$$

当：

$$
t\to0
$$

时，公式中存在：

$$
\frac1t
$$

虽然真实表达式可能存在有限极限，但有限精度预测误差可能被放大。

**第二，速度场可能具有较大的空间梯度。**

如果：

$$
\left\|
\frac{\partial b}{\partial x}
\right\|
$$

较大，数值积分可能需要更小的步长。

**第三，时间步长不一定应该均匀分配。**

如果某些时间区间的速度场变化更快，可以适当增加这些区间的积分节点。

但这需要结合具体模型进行分析，而不是仅根据 $\alpha,\beta$ 的形式判断。


### 6.2.11 ODE 与 SDE 采样如何选择？

从理论上说，在精确预测场条件下，ODE 和相应 SDE 可以共享同一边缘概率路径。

但实际应用中，两者存在明显区别。

| 比较维度 | ODE | SDE |
|---|---|---|
| 动力学 | 确定性 | 随机性 |
| 随机性来源 | 初始噪声 | 初始噪声与过程噪声 |
| 所需预测场 | 速度场 $b$ | 速度场 $b$ 与 Score $s$ |
| 常见求解器 | Euler、Heun、RK4 | Euler–Maruyama 等 |
| 数值误差 | ODE 离散化误差 | 强误差、弱误差 |
| 同一初始状态 | 理想情况下输出确定 | 可以产生不同样本 |
| 主要特点 | 便于确定性采样与高阶积分 | 可以引入额外随机探索 |

对于自动驾驶轨迹规划，如果需要在固定场景下生成多条轨迹，可以通过：

$$
Z^{(1)},Z^{(2)},\ldots,Z^{(K)}
$$

采样不同初始噪声，然后使用 ODE：

$$
\dot Y_t^{(k)}=b_\theta(t,Y_t^{(k)},c)
$$

得到：

$$
Y_1^{(1)},Y_1^{(2)},\ldots,Y_1^{(K)}
$$

因此，**ODE 并不意味着生成模型没有多模态能力**。

多模态来自初始分布经过生成映射后的推前分布，而不要求采样过程本身必须是随机微分方程。

SDE 则可以在积分过程中进一步引入随机扰动，但不意味着它一定比 ODE 生成更多有效的行为模式。


### 6.2.12 本节总结

ODE 采样：

$$
\boxed{
\dot Y_t=b_\theta(t,Y_t)
}
$$

SDE 采样：

$$
\boxed{
dY_t=
(b_\theta+\varepsilon s_\phi)dt+
\sqrt{2\varepsilon}dW_t
}
$$

二者在理想条件下可以实现相同的边缘概率演化。

实际采样时，需要考虑：

$$
\boxed{
\text{模型误差}
+
\text{数值误差}
+
\text{计算成本}
}
$$

Euler、Heun、RK4 等求解器提供了不同的精度与计算成本权衡。

更高阶的求解器通常可以降低离散化误差，但不能自动修正模型预测误差。

---


# 第 6 章总结

本章将随机插值理论转化为具体算法。

训练阶段：

$$
\boxed{
(X_0,X_1,Z,t)
\longrightarrow
(X_t,\dot X_t)
\longrightarrow
\text{监督回归}
}
$$

核心数学基础是：

$$
\boxed{
\arg\min_f\mathbb E\|f(X)-Y\|^2
=
\mathbb E[Y\mid X]
}
$$

因此，神经网络可以通过样本级监督学习边缘速度场与 Score，而不需要显式计算复杂的条件分布。

采样阶段：

$$
\boxed{
Y_0\sim\rho_0
\longrightarrow
\text{ODE/SDE 数值积分}
\longrightarrow
Y_1\sim\rho_1
}
$$

理想连续时间理论保证的是概率路径之间的关系，而实际生成效果还受到模型预测误差与数值离散化误差影响。

---


<a id="chapter-7"></a>

# 第 7 章：Numerical Results

第 6 章已经说明，随机插值框架可以通过简单的回归目标学习速度场与 Score，并使用 ODE 或 SDE 进行采样。

但是，理论上能够生成相同的概率分布，并不意味着使用神经网络近似后，两种方法具有相同的生成质量。

因此，本章重点研究一个问题：

**当速度场和 Score 都存在预测误差时，确定性 ODE 与随机性 SDE，究竟哪一种生成方式更可靠？**

论文通过三个层次的实验回答这一问题：

- **二维分布实验**：直观观察 ODE/SDE 生成分布的差异，以及 $\gamma(t)$、$\varepsilon$ 的作用。
- **128 维 Gaussian Mixture 实验**：利用可以解析计算的速度场与 Score，定量研究神经网络近似误差。
- **图像生成实验**：检验随机插值框架能否扩展到高维真实数据，并展示不同插值设计的灵活性。

原论文的实际安排是：7.1 为二维实验，7.2 为 128 维高斯混合分布，7.3 为图像生成。这里按照你之前确定的目录，将前两者合并到 7.1，把图像实验放在 7.2，最后单独增加 7.3 进行理论分析。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


## 7.1 二维分布与高维 Gaussian Mixture 实验


### 7.1.1 核心结论

论文的低维和高维实验共同表明：

**在这些实验设置下，当学习得到的速度场和 Score 存在误差时，SDE 采样往往比 ODE 采样更加准确或稳健。**

同时，实验也发现：

1. 插值噪声 $\gamma(t)$ 会影响 ODE 的生成质量。
2. 采样扩散强度 $\varepsilon$ 可以改善某些模型的分布拟合，但过大的 $\varepsilon$ 需要更小的积分步长。
3. ODE 和 SDE 在精确预测场下具有相同边缘分布，但近似模型下不再保证一致。

这里的结论来自论文具体实验，不能推广为“SDE 在所有生成任务中都优于 ODE”。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.1.2 为什么先做二维实验？

高维生成模型通常很难直接观察生成概率密度。

例如，对于图像：

$$
x\in\mathbb R^{3\times128\times128}
$$

我们可以观察生成图像是否自然，但很难直观判断：

$$
\widehat\rho_1(x)
$$

与真实分布：

$$
\rho_1(x)
$$

之间究竟存在多大差异。

二维实验则不同。

因为：

$$
x\in\mathbb R^2
$$

我们可以直接绘制：

- 真实目标概率密度。
- 模型生成样本的分布。
- 核密度估计结果。
- 不同模型之间的概率密度误差。

因此，二维实验的主要目的不是证明方法能够处理复杂数据，而是**隔离并观察不同生成动力学的影响**。


### 7.1.3 实验中的两个重要变量

回顾随机插值：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{7.1}
$$

以及生成 SDE：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon s(t,Y_t)]dt+
\sqrt{2\varepsilon}\,dW_t
}
\tag{7.2}
$$

实验重点改变：

$$
\gamma(t),\qquad\varepsilon
$$

其中：

$$
\gamma(t)
$$

控制插值路径中的额外高斯噪声。

而：

$$
\varepsilon
$$

控制采样时的随机扩散强度。

当：

$$
\varepsilon=0
$$

式 (7.2) 退化为：

$$
\boxed{
dY_t=b(t,Y_t)dt
}
\tag{7.3}
$$

即 Probability Flow ODE。

当：

$$
\varepsilon>0
$$

则得到 SDE。

因此，通过改变：

$$
\varepsilon=0,\quad \varepsilon>0
$$

可以比较确定性与随机性生成。

通过改变：

$$
\gamma(t)
$$

则可以研究概率路径设计对最终生成效果的影响。


### 7.1.4 为什么 $\gamma(t)$ 会影响生成效果？

理论上，对于任意满足条件的 $\gamma(t)$，都可以构造从：

$$
\rho_0\longrightarrow\rho_1
$$

的精确概率路径。

但是，实际模型只能学习近似速度场：

$$
\widehat b(t,x)
$$

因此，不同概率路径会带来不同的学习难度。

例如：

$$
\gamma(t)=0
$$

对应不含额外潜变量的插值。

而：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

会在中间时刻增加高斯扰动，同时保持：

$$
\gamma(0)=\gamma(1)=0
$$

这样可以改变中间概率密度的平滑程度。

由于：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

不同的 $\gamma(t)$ 会改变条件分布，进而改变速度场。

因此，即使理论上最终都可以到达：

$$
\rho_1
$$

它们对应的神经网络学习问题也可能具有不同的难度。

论文二维实验发现，采用：

$$
\boxed{
\gamma(t)=\sqrt{t(1-t)}
}
\tag{7.4}
$$

的设置，在所比较的 ODE 模型中表现最好。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

这说明额外插值噪声并不必然损害生成质量。

相反，适当的插值噪声可能使学习问题更加有利。

但需要注意，最优的 $\gamma$ 依赖具体数据分布和模型设置，并不是一个普遍固定的函数。


### 7.1.5 如何定量比较二维生成分布？

设真实目标分布为：

$$
\rho_1(x)
$$

模型生成的分布为：

$$
\widehat\rho_1(x)
$$

如果两者具有密度，就可以比较：

$$
\log\rho_1(x)
$$

与：

$$
\log\widehat\rho_1(x)
$$

定义点态误差：

$$
\boxed{
e(x)=
\left|
\log\rho_1(x)-
\log\widehat\rho_1(x)
\right|
}
\tag{7.5}
$$

然后在选定的评估位置上统计：

$$
\mathbb E[e(x)]
$$

以及：

$$
\operatorname{Var}[e(x)]
$$

其中期望和方差的具体含义取决于评估点的采样方式。

论文使用了这类对数密度误差统计，并结合核密度估计图进行定量和定性比较。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

为什么比较对数密度？

因为概率密度可能跨越多个数量级。

例如：

$$
\rho_1(x)=10^{-2}
$$

与：

$$
\widehat\rho_1(x)=10^{-3}
$$

其绝对密度差为：

$$
9\times10^{-3}
$$

但两者相差十倍。

使用对数：

$$
|\log10^{-2}-\log10^{-3}|=\log10
$$

能够更直接地反映相对尺度差异。

不过，对数密度误差并不等同于 KL 散度，两者的数学定义不同。


### 7.1.6 二维实验的主要发现

论文二维实验的结果可以概括为三个方面。

**第一，SDE 在这些实验中通常优于 ODE。**

对于相同学习得到的速度场与 Score，使用：

$$
\varepsilon>0
$$

进行随机采样，通常能够得到更接近目标分布的结果。

**第二，插值噪声影响 ODE 性能。**

不同：

$$
\gamma(t)
$$

对应不同的速度场学习问题。

其中：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

在论文比较的 ODE 设置中表现最好。

**第三，较大的 $\varepsilon$ 会提高数值求解要求。**

当扩散强度增加：

$$
\sqrt{2\varepsilon}
$$

也会增加。

因此，单步随机增量的标准差为：

$$
\sqrt{2\varepsilon h}
$$

当：

$$
\varepsilon
$$

增大时，如果仍使用相同的步长：

$$
h
$$

每一步的随机扰动也会更大。

这可能要求更小的积分步长，才能准确逼近连续时间 SDE。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.1.7 为什么还需要 128 维 Gaussian Mixture 实验？

二维实验能够直观展示概率密度，但仍然存在一个局限：

**二维问题相对简单，不能充分反映高维生成模型的困难。**

因此，论文进一步研究：

$$
d=128
$$

的 Gaussian Mixture。

高斯混合分布具有两个重要优点。

第一，它能够表示多模态分布。

第二，在适当构造下，随机插值对应的概率密度、速度场和 Score 可以解析计算。

这使得我们不仅可以比较生成样本，还可以直接分析：

$$
\widehat b-b
$$

以及：

$$
\widehat s-s
$$

所造成的影响。


### 7.1.8 Gaussian Mixture 的数学定义

考虑高斯混合分布：

$$
\boxed{
\rho(x)=
\sum_{k=1}^{K}
\pi_k\mathcal N(x;\mu_k,\Sigma_k)
}
\tag{7.6}
$$

其中：

$$
\pi_k\ge0,\qquad
\sum_{k=1}^{K}\pi_k=1
$$

每个分量由：

$$
\mu_k,\quad\Sigma_k
$$

描述。

这里：

- $K$：高斯分量数量。
- $\pi_k$：第 $k$ 个分量的混合权重。
- $\mu_k$：均值。
- $\Sigma_k$：协方差矩阵。

当不同分量的均值相距较远时，整体分布可能呈现多个明显的概率峰。

这可以用来测试生成模型是否能够正确恢复多模态结构。


### 7.1.9 为什么 Gaussian Mixture 可以解析计算 Score？

对于：

$$
\rho(x)=\sum_{k=1}^{K}\pi_k\mathcal N(x;\mu_k,\Sigma_k)
$$

Score 定义为：

$$
s(x)=\nabla_x\log\rho(x)
$$

根据：

$$
\nabla_x\log\rho(x)=\frac{\nabla_x\rho(x)}{\rho(x)}
$$

首先求：

$$
\nabla_x\rho(x)
=
\sum_{k=1}^{K}
\pi_k\nabla_x\mathcal N(x;\mu_k,\Sigma_k)
$$

对于单个高斯：

$$
\nabla_x\log\mathcal N(x;\mu_k,\Sigma_k)
=
-\Sigma_k^{-1}(x-\mu_k)
$$

因此：

$$
\nabla_x\mathcal N(x;\mu_k,\Sigma_k)
=
-\mathcal N(x;\mu_k,\Sigma_k)
\Sigma_k^{-1}(x-\mu_k)
$$

代入：

$$
\nabla_x\rho(x)
=
-\sum_{k=1}^{K}
\pi_k\mathcal N(x;\mu_k,\Sigma_k)
\Sigma_k^{-1}(x-\mu_k)
$$

最终：

$$
\boxed{
s(x)=
-\frac{
\sum_{k=1}^{K}
\pi_k\mathcal N(x;\mu_k,\Sigma_k)
\Sigma_k^{-1}(x-\mu_k)
}{
\sum_{k=1}^{K}
\pi_k\mathcal N(x;\mu_k,\Sigma_k)
}
}
\tag{7.7}
$$

定义后验责任权重：

$$
r_k(x)=
\frac{
\pi_k\mathcal N(x;\mu_k,\Sigma_k)
}{
\sum_j\pi_j\mathcal N(x;\mu_j,\Sigma_j)
}
$$

则：

$$
\boxed{
s(x)=
-\sum_{k=1}^{K}
r_k(x)\Sigma_k^{-1}(x-\mu_k)
}
\tag{7.8}
$$

这个公式说明：

**Gaussian Mixture 的 Score 是各个高斯分量 Score 的后验加权平均。**

它与前面讨论的条件期望结构非常相似。


### 7.1.10 为什么 Gaussian Mixture 的速度场也可以解析计算？

考虑空间线性随机插值：

$$
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
\tag{7.9}
$$

假设端点来自独立高斯混合分布：

$$
X_0\sim\sum_i\pi_i^0\mathcal N(\mu_i^0,\Sigma_i^0)
$$

$$
X_1\sim\sum_j\pi_j^1\mathcal N(\mu_j^1,\Sigma_j^1)
$$

对于固定分量对：

$$
(i,j)
$$

由于高斯随机变量的线性组合仍然是高斯，因此：

$$
X_t\mid(i,j)
\sim
\mathcal N(m_{ij}(t),C_{ij}(t))
$$

其中：

$$
\boxed{
m_{ij}(t)=
\alpha(t)\mu_i^0+\beta(t)\mu_j^1
}
\tag{7.10}
$$

$$
\boxed{
C_{ij}(t)=
\alpha^2(t)\Sigma_i^0+
\beta^2(t)\Sigma_j^1+
\gamma^2(t)I_d
}
\tag{7.11}
$$

因此，整个中间分布仍然是 Gaussian Mixture：

$$
\boxed{
\rho_t(x)=
\sum_{i,j}
\pi_i^0\pi_j^1
\mathcal N(x;m_{ij}(t),C_{ij}(t))
}
\tag{7.12}
$$

对每个高斯分量，其概率流速度可以写为：

$$
\boxed{
b_{ij}(t,x)=
\dot m_{ij}(t)+
\frac12\dot C_{ij}(t)C_{ij}^{-1}(t)
[x-m_{ij}(t)]
}
\tag{7.13}
$$

这里采用论文附录 A 的公式；该形式依赖其协方差结构，不能不加条件地推广到任意非对易矩阵路径。

然后使用各分量的后验权重：

$$
r_{ij}(t,x)=
\frac{
\pi_i^0\pi_j^1\mathcal N(x;m_{ij},C_{ij})
}{
\rho_t(x)
}
$$

得到：

$$
\boxed{
b(t,x)=
\sum_{i,j}r_{ij}(t,x)b_{ij}(t,x)
}
\tag{7.14}
$$

这就是高斯混合随机插值的解析速度场。

因此，在这一实验中：

$$
b(t,x),\quad s(t,x)
$$

都可以通过数学公式计算，而不需要依赖神经网络。

这使得它成为研究模型近似误差的理想测试环境。论文附录 A 给出了对应解析公式。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

---


### 7.1.11 高维实验的具体设置与结果

论文采用的目标分布是一个 **128 维、5 个高斯分量的 Gaussian Mixture**：

$$
\rho_0=\mathcal N(0,I_{128})
$$

$$
\rho_1=\sum_{i=1}^{5}\pi_i\mathcal N(\mu_i,\Sigma_i)
$$

各分量均值通过随机采样生成：

$$
\mu_i\sim\mathcal N(0,7.5^2I_{128})
$$

协方差构造为：

$$
\boxed{
\Sigma_i=I_{128}+\frac1{128}W_i^\top W_i
}
\tag{7.15}
$$

其中 $W_i$ 的元素独立服从标准正态分布。

由于：

$$
W_i^\top W_i\succeq0
$$

所以：

$$
\Sigma_i\succ0
$$

从而保证协方差矩阵正定。

插值日程采用：

$$
\alpha(t)=1-t,\qquad
\beta(t)=t,\qquad
\gamma(t)=\sqrt{t(1-t)}
$$

论文比较了四种学习方案：

| 方案 | 第一个预测场 | 第二个预测场 |
|---|---|---|
| A | 直接学习 $b$ | 直接学习 $s$ |
| B | 直接学习 $b$ | 学习去噪器 $\eta_z$ |
| C | 学习 $v$，再恢复 $b$ | 直接学习 $s$ |
| D | 学习 $v$，再恢复 $b$ | 学习去噪器 $\eta_z$ |

这里：

$$
\boxed{
b=v-\gamma\dot\gamma s
}
\tag{7.16}
$$

因此，学习 $v$ 与直接学习 $b$ 是两种不同的参数化方案。

采样时，ODE 使用自适应 Dormand–Prince 求解器，SDE 使用 1000 步 Heun 型 SDE 求解器。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


#### 为什么需要比较四种方案？

因为即使两个预测目标在理论上可以相互转换，它们的神经网络训练误差也可能不同。

例如：

$$
\widehat b=\widehat v-\gamma\dot\gamma\widehat s
$$

定义：

$$
e_v=\widehat v-v
$$

$$
e_s=\widehat s-s
$$

则：

$$
\begin{aligned}
e_b
&=\widehat b-b\\
&=(\widehat v-v)-\gamma\dot\gamma(\widehat s-s)
\end{aligned}
$$

因此：

$$
\boxed{
e_b=e_v-\gamma\dot\gamma e_s
}
\tag{7.17}
$$

进一步：

$$
\begin{aligned}
\|e_b\|^2
={}&\|e_v\|^2
+(\gamma\dot\gamma)^2\|e_s\|^2\\
&-2\gamma\dot\gamma\,e_v^\top e_s
\end{aligned}
\tag{7.18}
$$

可以看到，间接恢复 $b$ 时，其误差受到两个预测器共同影响。

尤其当：

$$
|\gamma\dot\gamma|
$$

较大时，Score 预测误差可能对速度场造成明显影响。

因此，直接学习 $b$ 可能具有优势。

但式 (7.18) 也表明，误差之间可能发生抵消，所以不能仅凭这一公式断言直接学习 $b$ 必然更好。


#### 高维实验的主要发现

论文 Figure 10–12 得到了三个重要结果。

**第一，存在一个合适的非零扩散强度。**

当：

$$
\varepsilon
$$

较小时，生成分布倾向于在高斯模态内部过度集中，而尾部概率不足。

当：

$$
\varepsilon
$$

过大时，生成分布又可能过于分散。

因此，误差随 $\varepsilon$ 的变化可能呈现先下降、后上升的趋势。

论文比较的各组模型都在某个非零 $\varepsilon$ 处取得最佳结果。

**第二，直接学习 $b$ 通常优于分别学习 $v,s$ 后组合。**

这说明直接优化最终需要的速度场，在该实验中具有更好的数值表现。

**第三，学习去噪器 $\eta_z$ 通常优于直接学习 Score。**

原因与第 6.1 节讨论的数值问题有关：

$$
s=-\frac{\eta_z}{\gamma}
$$

当 $\gamma$ 很小时，直接 Score 回归可能出现较大的监督尺度和方差。

因此，先学习：

$$
\eta_z=\mathbb E[Z\mid X_t=x]
$$

再恢复 Score，可能更加稳定。

在论文的比较中，**直接学习 $b$ 与 $\eta_z$，再选择适当的非零 $\varepsilon$，取得了最佳表现**。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.1.12 KL 散度是如何计算的？

论文并没有直接计算完整 128 维分布的 KL 散度，而是将生成样本投影到前两个坐标，在二维边缘分布上进行 KDE 和 KL 估计。

这一点很重要，因为：

$$
\mathrm{KL}(\rho_1\|\widehat\rho_1)
$$

如果指完整的 128 维分布，与二维投影后的 KL 并不是同一个量。

下面为了简化符号，用：

$$
p(x)
$$

表示真实二维边缘密度，用：

$$
q(x)
$$

表示模型的二维边缘密度。

KL 散度定义为：

$$
\boxed{
D_{\mathrm{KL}}(p\|q)
=
\mathbb E_{x\sim p}
\left[
\log\frac{p(x)}{q(x)}
\right]
}
\tag{7.19}
$$

最直接的 Monte Carlo 估计是：

$$
\widehat D_{\mathrm{KL}}
=
\frac1N\sum_{i=1}^{N}
\left[
\log p(x_i)-\log q(x_i)
\right]
$$

其中：

$$
x_i\sim p
$$

论文进一步使用控制变量降低估计方差：

$$
\boxed{
\widehat D_{\mathrm{KL}}
=
\frac1N\sum_{i=1}^{N}
\left[
\log\frac{p(x_i)}{q(x_i)}
-
\left(
\frac{q(x_i)}{p(x_i)}-1
\right)
\right]
}
\tag{7.20}
$$

为什么可以添加第二项？

因为：

$$
\begin{aligned}
\mathbb E_{x\sim p}
\left[
\frac{q(x)}{p(x)}-1
\right]
&=
\int p(x)
\left[
\frac{q(x)}{p(x)}-1
\right]dx\\
&=
\int q(x)dx-\int p(x)dx\\
&=1-1\\
&=0
\end{aligned}
$$

所以，只要 $p,q$ 是正确归一化的密度，控制变量的期望就是零。

因此，添加该项不会改变总体期望：

$$
\mathbb E[\widehat D_{\mathrm{KL}}]
=
D_{\mathrm{KL}}(p\|q)
$$

此外，令：

$$
r=\frac{q(x)}{p(x)}>0
$$

则每个样本的贡献为：

$$
-\log r-r+1
$$

这里需要纠正一个容易混淆的符号：论文的控制变量形式应写成

$$
\boxed{
\log\frac{p(x)}{q(x)}
+
\frac{q(x)}{p(x)}-1
}
\tag{7.21}
$$

即：

$$
-\log r+r-1
$$

因为根据：

$$
\log r\le r-1
$$

有：

$$
-\log r+r-1\ge0
$$

这才具有逐样本非负的性质。

**因此，如果严格采用原文展示的式 (7.1) 中的负号形式，它与原文随后声称的“逐样本非负”并不一致。** 这里应当将它视为需要核对的公式符号问题，而不能直接接受两个结论同时成立。原文确实展示了该负号，并同时给出了非负性解释。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.1.13 本节总结

二维和 128 维 Gaussian Mixture 实验共同说明：

$$
\boxed{
\text{理论上边缘分布一致}
\not\Rightarrow
\text{有限模型下生成质量一致}
}
$$

实验支持以下认识：

- $\gamma(t)$ 会影响速度场与 Score 的学习难度。
- $\varepsilon$ 会影响近似模型的最终生成误差。
- 适当的非零扩散强度可能改善分布拟合。
- 直接学习 $b$ 与学习去噪器 $\eta_z$ 在论文实验中表现较好。

这些结果为第 2.5 节的 KL 误差分析提供了实验支持，但并不意味着实验已经证明所有模型或所有分布都满足相同规律。

---


## 7.2 图像生成实验


### 7.2.1 核心结论

图像生成实验的主要目的不是刷新生成质量指标，而是验证：

**随机插值框架能够从低维理论问题扩展到高维真实数据，并支持不同概率路径和采样动力学。**

论文使用：

$$
128\times128
$$

的 Oxford Flowers 图像数据集，分别研究：

1. 从高斯噪声生成图像的单侧随机插值。
2. 从真实图像出发、最终仍保持数据分布的 Mirror Interpolant。

作者也明确指出，图像生成是展示框架可扩展性和灵活性的实验，而不是论文的主要研究重点。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.2.2 单侧随机插值的图像生成

首先考虑：

$$
Z\sim\mathcal N(0,I_d)
$$

$$
X_1\sim\rho_{\mathrm{image}}
$$

其中：

$$
d=3\times128\times128
$$

表示 RGB 图像的像素维度。

构造：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{7.22}
$$

论文比较了两种时间日程。

**第一种：Linear Interpolant**

$$
\boxed{
X_t=(1-t)Z+tX_1
}
\tag{7.23}
$$

对应条件速度：

$$
\dot X_t=X_1-Z
$$

**第二种：Trigonometric Interpolant**

$$
\boxed{
X_t=
\cos\left(\frac{\pi t}{2}\right)Z+
\sin\left(\frac{\pi t}{2}\right)X_1
}
\tag{7.24}
$$

对应条件速度：

$$
\boxed{
\dot X_t=
-\frac{\pi}{2}\sin\left(\frac{\pi t}{2}\right)Z
+
\frac{\pi}{2}\cos\left(\frac{\pi t}{2}\right)X_1
}
\tag{7.25}
$$

两者具有相同的端点分布，但中间概率路径不同。


### 7.2.3 神经网络与训练方法

论文采用与 DDPM 相关的 U-Net 架构，并使用 Adam 优化器训练预测场。

主要学习对象包括：

$$
b_\theta(t,x)
$$

$$
s_\phi(t,x)
$$

以及：

$$
\eta_\psi(t,x)
$$

其中：

$$
\eta_\psi(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

对于单侧线性插值，三者之间具有第 4 章推导的代数关系。

例如：

$$
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
$$

以及：

$$
b(t,x)=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,x)
$$

因此，既可以直接训练所需的预测场，也可以训练去噪器后进行转换。

论文将这些不同的学习与采样方案应用于真实图像。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.2.4 ODE 与 SDE 的图像生成差异

对于 ODE：

$$
\boxed{
dY_t=b_\theta(t,Y_t)dt
}
\tag{7.26}
$$

给定同一个初始噪声：

$$
Y_0=z
$$

如果模型和求解器确定，那么输出也是确定的。

即：

$$
Y_1=T_\theta(z)
$$

对于 SDE：

$$
\boxed{
dY_t=
[b_\theta(t,Y_t)+\varepsilon s_\phi(t,Y_t)]dt+
\sqrt{2\varepsilon}dW_t
}
\tag{7.27}
$$

即使初始噪声相同：

$$
Y_0=z
$$

不同的布朗运动路径也可能得到不同的输出：

$$
Y_1^{(1)}\neq Y_1^{(2)}
$$

因此：

$$
\boxed{
\text{相同初始噪声}
+
\text{不同过程噪声}
\Longrightarrow
\text{不同生成图像}
}
\tag{7.28}
$$

论文通过固定初始样本，改变 $\varepsilon$，展示了不同的花卉生成结果。

实验使用自适应 ODE 求解器进行确定性采样，而对于：

$$
\varepsilon=1,\ 2,\ 4
$$

分别使用了 2000、2500、4000 步的 Heun 型 SDE 采样。

作者观察到，在这些示例中，增大 $\varepsilon$ 可以增加同一初始样本对应输出的多样性。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)

但这里需要区分：

**条件于同一个初始噪声的输出多样性**，不等于**整体生成分布的质量或多模态覆盖能力**。

因为即使使用 ODE，只要改变初始噪声，也可以生成不同图像。


### 7.2.5 Mirror Interpolant：从图像生成相近图像

论文还研究了一个非常有意思的设置：

$$
\boxed{
X_t=X_1+\gamma(t)Z
}
\tag{7.29}
$$

其中：

$$
\gamma(0)=\gamma(1)=0
$$

因此：

$$
X_{t=0}=X_1
$$

$$
X_{t=1}=X_1
$$

也就是说，随机插值的起点和终点是同一个随机变量。

从边缘分布来看：

$$
\boxed{
\rho_0=\rho_1
}
\tag{7.30}
$$

这就是 Mirror Interpolant。

为什么要研究这样一个过程？

因为它说明随机插值不仅能够用于：

$$
\text{高斯噪声}\longrightarrow\text{数据}
$$

也可以用于：

$$
\text{数据分布}\longrightarrow\text{同一个数据分布}
$$

但在这个过程中，中间时刻仍然可以引入噪声，并使用随机动力学产生新的样本。


### 7.2.6 Mirror Interpolant 的速度场推导


对 $X_t=X_1+\gamma(t)Z$，记 $\eta_z(t,x)=\mathbb E[Z\mid X_t=x]$，直接应用只加高斯噪声的结果：

完整推导见[附录 F.5：Mirror 速度与 Score 的关系](#app-f-5)。

$$
\boxed{
b(t,x)=\dot\gamma(t)\eta_z(t,x)
}
\tag{7.31}
$$

同时：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\gamma(t)}
}
\tag{7.32}
$$

因此，Mirror Interpolant 也可以仅通过一个去噪器恢复速度场和 Score。


### 7.2.7 为什么 Mirror ODE 与 Mirror SDE 的结果不同？

这是 Mirror Interpolant 最有意思的地方。

由于：

$$
X_t=X_1+\gamma(t)Z
$$

其边缘密度只取决于：

$$
\gamma^2(t)
$$

如果：

$$
\gamma(t)=\sqrt{t(1-t)}
$$

那么：

$$
\gamma(t)=\gamma(1-t)
$$

因此：

$$
\rho_t=\rho_{1-t}
$$

中间概率路径关于：

$$
t=\frac12
$$

对称。

对应的速度场为：

$$
b(t,x)=\dot\gamma(t)\eta_z(t,x)
$$

由于：

$$
\dot\gamma(1-t)=-\dot\gamma(t)
$$

而相同噪声方差对应相同的条件去噪器，所以：

$$
\boxed{
b(1-t,x)=-b(t,x)
}
\tag{7.33}
$$

因此，在理想精确 ODE 中，前半段的流动会被后半段反向抵消。

最终：

$$
\boxed{
Y_1=Y_0
}
\tag{7.34}
$$

但对于 SDE：

$$
dY_t=
[b(t,Y_t)+\varepsilon s(t,Y_t)]dt+
\sqrt{2\varepsilon}dW_t
$$

即使：

$$
\rho_0=\rho_1
$$

单条随机轨迹也不要求回到原始位置。

因此：

$$
Y_1\neq Y_0
$$

是可能的。

这并不违反概率分布一致性，因为：

$$
\boxed{
Y_0\sim\rho_1,\qquad Y_1\sim\rho_1
}
$$

只要求起点和终点的边缘分布相同，而不要求每个样本相同。

论文在 Oxford Flowers 数据集上展示了这一现象：使用 Mirror Interpolant 和非零扩散强度，可以从一张花卉图像生成另一张与之相近的花卉图像。[arXiv](https://arxiv.org/html/2303.08797v4?utm_source=chatgpt.com)


### 7.2.8 本节总结

图像实验验证了两个重要结论。

首先，随机插值可以用于高维真实数据生成：

$$
\boxed{
\mathcal N(0,I_d)
\longrightarrow
\rho_{\mathrm{image}}
}
$$

其次，随机插值也可以构造保持边缘分布不变的随机生成过程：

$$
\boxed{
\rho_{\mathrm{image}}
\longrightarrow
\rho_{\mathrm{image}}
}
$$

因此，随机插值的作用不仅是将噪声变成数据，更是为不同概率分布之间的连续演化提供统一设计方法。

---


## 7.3 实验结果分析与理论验证


### 7.3.1 核心结论

前面的实验结果可以总结为三个层次：

**第一，验证随机插值框架的灵活性。**

通过改变：

$$
I(t,X_0,X_1),\quad\gamma(t)
$$

可以构造不同概率路径。

**第二，验证采样动力学的灵活性。**

对于同一概率路径，可以选择：

$$
\varepsilon=0
$$

对应 ODE，或者：

$$
\varepsilon>0
$$

对应 SDE。

**第三，验证有限模型误差下 ODE/SDE 的性能差异。**

理论上二者可以共享相同边缘分布，但实际神经网络预测不精确时，生成质量可能不同。

其中第三点是论文实验的重要关注对象。


### 7.3.2 为什么 SDE 可能比 ODE 更稳健？

第 2.5 节已经讨论了随机生成模型的 KL 误差控制。

这里重新考虑：

$$
dY_t=
[b(t,Y_t)+\varepsilon s(t,Y_t)]dt+
\sqrt{2\varepsilon}dW_t
\tag{7.35}
$$

实际模型使用：

$$
\widehat b,\quad\widehat s
$$

因此：

$$
d\widehat Y_t=
[\widehat b(t,\widehat Y_t)+\varepsilon\widehat s(t,\widehat Y_t)]dt+
\sqrt{2\varepsilon}dW_t
\tag{7.36}
$$

定义：

$$
e_b=\widehat b-b
$$

$$
e_s=\widehat s-s
$$

则漂移误差为：

$$
\boxed{
e_F=e_b+\varepsilon e_s
}
\tag{7.37}
$$

对于具有相同扩散系数的两个 SDE，在 Girsanov 定理适用的条件下，其路径空间 KL 散度具有：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\frac1{4\varepsilon}
\mathbb E_{\mathbb P}
\int_0^1
\|e_b+\varepsilon e_s\|^2dt
}
\tag{7.38}
$$

这里为了突出主要关系，假设：

$$
\varepsilon>0
$$

为常数，并且两个过程具有相同初始分布；还需要满足相应绝对连续性与可积性条件。


应用[附录 F.2](#app-f-2) 的漂移误差拆分，可直接得到：

因此：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
\le
\frac1{2\varepsilon}
\mathbb E\int_0^1\|e_b\|^2dt
+
\frac{\varepsilon}{2}
\mathbb E\int_0^1\|e_s\|^2dt
}
\tag{7.39}
$$

这个公式非常重要。

它揭示了：

$$
\boxed{
\text{速度场误差项}\propto\frac1\varepsilon
}
$$

$$
\boxed{
\text{Score 误差项}\propto\varepsilon
}
$$

因此：

- 当 $\varepsilon$ 较小时，速度场误差的上界权重较大。
- 当 $\varepsilon$ 较大时，Score 误差的上界权重较大。

这解释了为什么存在一个适中的扩散强度可能更好。


### 7.3.3 如何理解最优扩散强度？

定义：

$$
A=
\mathbb E\int_0^1\|e_b\|^2dt
$$

$$
B=
\mathbb E\int_0^1\|e_s\|^2dt
$$

则式 (7.39) 为：

$$
\boxed{
D_{\mathrm{KL}}
\le
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
\tag{7.40}
$$


由[附录 F.2](#app-f-2) 的上界优化结果：

$$
\boxed{
\varepsilon^*=\sqrt{\frac AB}
}
\tag{7.41}
$$

这里假设：

$$
A>0,\qquad B>0
$$

并且 $A,B$ 在选择 $\varepsilon$ 时保持固定。

这个结果说明：

**最优扩散强度与速度场误差和 Score 误差的相对大小有关。**

如果：

$$
A\gg B
$$

即速度场误差较大，而 Score 相对准确，则：

$$
\varepsilon^*
$$

较大。

反过来，如果：

$$
B\gg A
$$

则：

$$
\varepsilon^*
$$

较小。

需要注意，这里得到的是**简化 KL 上界的最优值**，并不是所有生成质量指标的精确最优扩散强度。

实际还需要考虑数值离散化误差，以及误差之间的相关性。


### 7.3.4 为什么 ODE 不能直接令 $\varepsilon=0$ 使用同一个误差界？

观察：

$$
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
$$

当：

$$
\varepsilon\to0
$$

且：

$$
A>0
$$

时：

$$
\frac A{2\varepsilon}\to\infty
$$

因此，这个 SDE 的 Girsanov 误差界不能直接取：

$$
\varepsilon=0
$$

得到 ODE 的误差界。

原因在于，Girsanov 定理比较的是具有非退化扩散项的随机过程。

当：

$$
\varepsilon=0
$$

时，动力学退化为确定性 ODE：

$$
dY_t=b(t,Y_t)dt
$$

此时路径分布的数学性质发生变化。

因此，ODE 的分布误差需要其他分析方法。

这与第 2.5 节的理论结论一致：对于确定性生成模型，仅仅控制速度场的普通回归误差，通常不足以直接得到与 SDE 相同形式的 likelihood 保证，还需要额外的正则性或分布误差控制。


### 7.3.5 为什么实验中的最优 $\varepsilon$ 不一定等于理论公式？

虽然：

$$
\varepsilon^*=\sqrt{\frac AB}
$$

提供了重要解释，但实际实验还受到数值积分影响。

例如，SDE 的 Euler–Maruyama 更新为：

$$
Y_{k+1}
=
Y_k+
h[b_\theta+\varepsilon s_\theta]
+
\sqrt{2\varepsilon h}\xi_k
$$

当：

$$
\varepsilon
$$

增加时，随机增量的尺度为：

$$
\sqrt{2\varepsilon h}
$$

因此，如果固定：

$$
h
$$

较大的扩散强度可能增加离散化误差。

所以实际生成误差可以概念性地写成：

$$
\boxed{
\mathcal E_{\mathrm{total}}
=
\mathcal E_{\mathrm{model}}
+
\mathcal E_{\mathrm{solver}}
}
\tag{7.42}
$$

其中：

$$
\mathcal E_{\mathrm{model}}
$$

与：

$$
e_b,\quad e_s,\quad\varepsilon
$$

有关。

而：

$$
\mathcal E_{\mathrm{solver}}
$$

与：

$$
h,\quad\varepsilon,\quad\text{数值求解器}
$$

有关。

因此，理论最优扩散强度与实际有限步采样的最优值可能不同。

这也解释了为什么论文在高维实验中强调，较大的 $\varepsilon$ 应配合更小的积分步长。


### 7.3.6 实验是否证明 SDE 一定优于 ODE？

没有。

论文实验支持的是：

**对于所研究的模型、数据分布、训练设置和计算预算，适当的 SDE 采样通常能够改善生成分布的拟合。**

但不能进一步推出：

$$
\boxed{
\text{SDE 在所有情况下都优于 ODE}
}
$$

原因有三个。

首先，当速度场和 Score 都精确时，ODE 和 SDE 可以具有相同的边缘分布。

其次，不同神经网络架构、训练目标和数据分布，会产生不同的误差结构。

最后，ODE 可以使用高阶自适应求解器，而 SDE 的采样效率还受到随机积分误差影响。

因此，实际应用需要综合比较：

$$
\boxed{
\text{生成质量}
+
\text{数值误差}
+
\text{NFE}
+
\text{推理延迟}
}
$$

而不能只比较 ODE 与 SDE 的理论形式。


### 7.3.7 本节总结

本节最重要的理论关系为：

$$
\boxed{
D_{\mathrm{KL}}
\lesssim
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
$$

其中：

$$
A=\mathbb E\int_0^1\|e_b\|^2dt
$$

$$
B=\mathbb E\int_0^1\|e_s\|^2dt
$$

它说明：

**扩散强度并不是越大越好，也不是越小越好，而应结合速度场与 Score 的误差结构进行选择。**

论文实验观察到的非零最优扩散强度，与这一理论解释相一致。

---


# 第 7 章总结

本章通过二维分布、128 维 Gaussian Mixture 和图像生成实验，研究了随机插值框架在有限模型精度下的实际表现。

可以得到四个核心认识。

**第一，概率路径的设计会影响模型学习难度。**

即使不同插值函数连接相同的端点分布，它们对应的速度场和 Score 也可能不同。

因此：

$$
\gamma(t)
$$

不仅是一个数学上的自由参数，也可能直接影响生成效果。

**第二，ODE 与 SDE 在实际训练中不一定具有相同的生成质量。**

理论上：

$$
\rho_t^{\mathrm{ODE}}=\rho_t^{\mathrm{SDE}}
$$

但当：

$$
\widehat b\neq b,\qquad
\widehat s\neq s
$$

时：

$$
\widehat\rho_t^{\mathrm{ODE}}
\neq
\widehat\rho_t^{\mathrm{SDE}}
$$

完全可能发生。

**第三，适当的随机扩散可能改善生成分布的拟合。**

其理论解释来自：

$$
\boxed{
D_{\mathrm{KL}}
\lesssim
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
$$

但过大的扩散强度也可能增加 Score 误差的影响和数值积分成本。

**第四，随机插值框架具有较强的灵活性。**

它不仅可以实现：

$$
\mathcal N(0,I_d)\longrightarrow\rho_{\mathrm{data}}
$$

也可以通过 Mirror Interpolant 实现：

$$
\rho_{\mathrm{data}}\longrightarrow\rho_{\mathrm{data}}
$$

并利用随机动力学产生新的样本。

因此，本章最终验证了论文的核心思想：

$$
\boxed{
\text{概率路径设计}
\quad+\quad
\text{速度场与 Score 学习}
\quad+\quad
\text{ODE/SDE 采样}
}
$$

三者可以分别设计、相互组合，形成一类统一而灵活的生成模型。

**下一章将进入 Conclusion，对论文的理论贡献、适用条件与局限性进行整体总结。**


<a id="chapter-8"></a>

# 第 8 章：Conclusion

前面七章已经从数学理论、模型训练、数值采样和实验验证四个方面，系统讨论了随机插值（Stochastic Interpolants）框架。

本章首先总结论文的主要贡献，然后分析其理论适用条件与局限，最后讨论它对 Flow Matching、Diffusion，以及实际生成模型设计的启示。

本章的核心结论是：

**随机插值提供了一种先设计概率路径、再构造生成动力学的统一方法。Flow 与 Diffusion 可以被理解为同一概率路径上的不同动力学实现，而不必被视为两套完全独立的生成模型。**

---


## 8.1 论文主要贡献总结


### 8.1.1 贡献一：通过随机插值统一概率路径的构造

论文首先提出一般形式：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{8.1}
$$

其中：

- $X_0\sim\rho_0$：初始分布样本。
- $X_1\sim\rho_1$：目标分布样本。
- $(X_0,X_1)\sim\nu$：端点联合分布，即端点耦合。
- $Z\sim\mathcal N(0,I_d)$：独立的辅助高斯噪声。
- $I(t,X_0,X_1)$：端点之间的插值函数。
- $\gamma(t)$：控制额外插值噪声的时间函数。

满足：

$$
I(0,X_0,X_1)=X_0
$$

$$
I(1,X_0,X_1)=X_1
$$

以及：

$$
\gamma(0)=\gamma(1)=0
$$

因此：

$$
X_{t=0}=X_0,\qquad X_{t=1}=X_1
$$

从而保证：

$$
\boxed{
\rho_{t=0}=\rho_0,\qquad
\rho_{t=1}=\rho_1
}
\tag{8.2}
$$

这意味着，随机插值首先定义了一条概率分布路径：

$$
\boxed{
\rho_0\longrightarrow\rho_t\longrightarrow\rho_1
}
\tag{8.3}
$$

与直接从某个预先给定的 SDE 出发不同，这一框架允许我们主动设计：

$$
I,\qquad\gamma,\qquad\nu
$$

进而控制中间概率路径。

这为生成模型提供了更大的设计自由度。


### 8.1.2 贡献二：建立速度场、Score 与连续性方程的联系

随机插值的时间导数为：

$$
\dot X_t=
\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
$$

定义边缘速度场：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{8.4}
$$

定义 Score：

$$
\boxed{
s(t,x)=\nabla_x\log\rho_t(x)
}
\tag{8.5}
$$

利用高斯积分分部，可以得到：

$$
\boxed{
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}
}
\tag{8.6}
$$

其中该公式在 $\gamma(t)>0$ 的时间区间内成立。

另一方面，随机插值诱导的概率密度满足连续性方程：

$$
\boxed{
\partial_t\rho_t+
\nabla_x\cdot(\rho_tb)=0
}
\tag{8.7}
$$

它表明：

**随机插值虽然由随机变量构造，但其边缘概率密度的演化可以由确定性速度场描述。**

这也是从随机插值构造 Probability Flow ODE 的基础。


### 8.1.3 贡献三：从同一概率路径构造 ODE 与 SDE

根据连续性方程，可以构造确定性 ODE：

$$
\boxed{
dY_t=b(t,Y_t)dt
}
\tag{8.8}
$$

其边缘概率密度满足：

$$
Y_t\sim\rho_t
$$

在适当正则性与初始条件下成立。

进一步，利用：

$$
\rho_ts=\nabla\rho_t
$$

可以在 Fokker–Planck 方程中添加相互抵消的扩散项，得到：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{8.9}
$$

其中：

$$
\varepsilon(t)\ge0
$$

是采样扩散强度。

在满足适定性条件时，该 SDE 与 ODE 具有相同的单时刻边缘分布：

$$
\boxed{
\rho_t^{\mathrm{ODE}}=
\rho_t^{\mathrm{SDE}}=
\rho_t
}
\tag{8.10}
$$

但是，这并不意味着二者具有相同的完整路径分布。

ODE 的随机性主要来自初始样本。

SDE 除初始随机性之外，还受到布朗运动驱动。

因此，论文建立的统一关系是：

$$
\boxed{
\text{相同边缘概率路径}
\neq
\text{相同随机过程}
}
\tag{8.11}
$$


### 8.1.4 贡献四：将生成模型训练转化为监督回归

论文证明，速度场可以通过：

$$
\boxed{
\mathcal L_b(\theta)
=
\mathbb E\|b_\theta(t,X_t)-\dot X_t\|^2
}
\tag{8.12}
$$

学习。

因为：

$$
\boxed{
\arg\min_f
\mathbb E\|f(X)-Y\|^2
=
\mathbb E[Y\mid X]
}
\tag{8.13}
$$

所以最优预测器为：

$$
b^*(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

同样，Score 可以通过噪声预测学习。

例如，定义：

$$
\eta_\theta(t,x)\approx\mathbb E[Z\mid X_t=x]
$$

使用：

$$
\boxed{
\mathcal L_\eta(\theta)
=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
}
\tag{8.14}
$$

再恢复：

$$
\boxed{
s(t,x)=-\frac{\eta_\theta(t,x)}{\gamma(t)}
}
\tag{8.15}
$$

因此，训练阶段不需要模拟 ODE 或 SDE。

这一点使随机插值能够直接应用于高维生成模型。


### 8.1.5 贡献五：建立模型误差与生成分布误差的联系

论文不仅给出了理想生成动力学，还研究了学习误差对最终生成分布的影响。

对于采样 SDE：

$$
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
$$

假设学习得到：

$$
\widehat b,\qquad\widehat s
$$

定义：

$$
e_b=\widehat b-b,\qquad
e_s=\widehat s-s
$$

在 Girsanov 定理适用的条件下，可以建立路径空间 KL 误差关系：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\frac1{4\varepsilon}
\mathbb E_{\mathbb P}
\int_0^1
\|e_b+\varepsilon e_s\|^2dt
}
\tag{8.16}
$$

这里假设 $\varepsilon>0$ 为常数、初始分布相同，并满足相应可积性和绝对连续性条件。

进一步得到：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
\le
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
\tag{8.17}
$$

其中：

$$
A=\mathbb E\int_0^1\|e_b\|^2dt
$$

$$
B=\mathbb E\int_0^1\|e_s\|^2dt
$$

这解释了为什么非零扩散强度可能改善近似模型的生成效果。

但这里的结果是一个误差上界，不意味着所有任务都存在相同的最优扩散强度。


### 8.1.6 贡献六：连接多种生成模型

随机插值框架进一步建立了与以下方法的联系：

**Flow Matching**

通过：

$$
X_t=(1-t)X_0+tX_1
$$

学习：

$$
b(t,x)=\mathbb E[X_1-X_0\mid X_t=x]
$$

**Score-Based Diffusion**

通过高斯加噪过程：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

恢复：

$$
s(t,x)=\nabla_x\log\rho_t(x)
$$

并构造反向 SDE。

**Denoising**

通过：

$$
D(t,x)=\mathbb E[X_1\mid X_t=x]
$$

建立数据预测、噪声预测与 Score 之间的关系。

**Rectified Flow**

通过改变端点耦合，构造更平直的生成轨迹。

**Schrödinger Bridge**

通过相对于参考随机过程的路径空间 KL 最小化，研究满足端点分布约束的最优随机动力学。

这些方法并不是完全相同的算法，但可以在概率路径与随机动力学的层面建立统一联系。

---


## 8.2 理论局限与适用条件

随机插值框架虽然具有较强的理论一般性，但仍然存在一些需要注意的条件。


### 8.2.1 局限一：理论公式依赖正则性假设

例如：

$$
s(t,x)=\nabla_x\log\rho_t(x)
$$

要求：

$$
\rho_t(x)>0
$$

并具有适当的可微性。

连续性方程：

$$
\partial_t\rho_t+\nabla\cdot(\rho_tb)=0
$$

需要随机插值满足相应的可积性和可微性条件。

ODE：

$$
\dot Y_t=b(t,Y_t)
$$

要具有唯一解，通常还需要对速度场施加适当的正则性条件，例如局部 Lipschitz 性及防止有限时间爆炸的增长条件。

因此，随机插值能够形式上定义一条概率路径，并不意味着对应的所有 ODE/SDE 都自动满足存在唯一性。


### 8.2.2 局限二：时间端点可能出现奇异性

对于：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

有：

$$
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
$$

如果：

$$
\alpha(1)=0
$$

则：

$$
\frac1{\alpha(t)}
$$

在终点附近可能变得很大。

同样：

$$
b(t,x)=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z(t,x)
$$

在：

$$
\beta(0)=0
$$

时存在显式除零问题。

即使真实速度场存在有限极限，神经网络误差也可能在数值转换过程中被放大。

因此，实际采样通常需要：

- 采用数值稳定的预测参数化。
- 对端点使用解析极限或专门处理。
- 必要时在截断时间区间内积分。
- 选择合适的时间网格。


### 8.2.3 局限三：边缘分布一致不意味着生成路径一致

对于同一随机插值概率路径：

$$
\rho_t
$$

可以构造：

$$
dY_t=b\,dt
$$

以及：

$$
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
$$

它们具有相同的单时刻边缘分布。

但完整路径分布不同。

因此：

$$
\boxed{
\mathcal L(Y_t^{\mathrm{ODE}})
=
\mathcal L(Y_t^{\mathrm{SDE}})
}
$$

不意味着：

$$
\boxed{
\mathcal L((Y_t^{\mathrm{ODE}})_{t\in[0,1]})
=
\mathcal L((Y_t^{\mathrm{SDE}})_{t\in[0,1]})
}
$$

这一点对于轨迹生成尤其重要。

例如，自动驾驶规划中的：

$$
X_1\in\mathbb R^{T\times2}
$$

本身表示未来 $T$ 帧的轨迹。

而生成模型的：

$$
t\in[0,1]
$$

表示去噪或生成时间。

二者是不同的时间概念。

ODE/SDE 在生成时间上的路径不同，并不意味着它们最终输出的物理轨迹必须具有不同的统计分布。


### 8.2.4 局限四：理论最优不等于有限模型最优

不同预测目标在理论上可能可以相互转换。

例如：

$$
s=-\frac{\eta_z}{\alpha}
$$

$$
b=\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z
$$

但在实际训练中，存在：

- 有限模型容量。
- 有限训练样本。
- 优化误差。
- 不同时间的监督尺度差异。
- 预测误差在变量转换时被放大。

因此，不能仅根据代数等价性判断哪一种预测参数化最好。

论文第 7 章的实验也说明，直接学习速度场和通过其他预测器恢复速度场，可能具有不同的实际效果。


### 8.2.5 局限五：KL 误差保证并不直接覆盖所有采样器

Girsanov 误差分析主要针对具有相同非退化扩散系数的 SDE。

而对于 ODE：

$$
\varepsilon=0
$$

不能直接将 SDE 的：

$$
\frac1\varepsilon
$$

误差界取极限。

此外，即使连续时间 SDE 的 KL 误差得到控制，实际采样仍然需要数值离散化。

因此，理论误差界并不自动保证有限步 Euler–Maruyama 或其他求解器具有相同精度。


### 8.2.6 局限六：随机插值不自动提供最优概率路径

随机插值允许任意设计：

$$
I,\quad\gamma,\quad\nu
$$

但并不意味着任意选择都是最优的。

例如，采用：

$$
X_t=(1-t)X_0+tX_1
$$

只说明每组端点之间使用直线插值。

如果端点耦合：

$$
\nu
$$

不是最优传输耦合，那么该概率路径不一定是 Wasserstein 测地线。

同样，使用 Reflow 使生成轨迹更加平直，也不自动保证得到全局最优传输映射。

因此，概率路径设计本身仍然是一个重要研究问题。

---


## 8.3 对 Flow Matching 和 Diffusion 的启示


### 8.3.1 启示一：Flow Matching 与 Diffusion 的核心差异不只是 ODE/SDE

通常我们会将两类模型简单区分为：

$$
\text{Flow Matching}\longrightarrow\text{ODE}
$$

$$
\text{Diffusion}\longrightarrow\text{SDE}
$$

但随机插值框架说明，这种区分并不完整。

因为同一条概率路径：

$$
\rho_t
$$

既可以使用 ODE 实现，也可以使用 SDE 实现。

因此，更合理的分析方式是将生成模型拆成三个部分：

$$
\boxed{
\text{概率路径}
+
\text{预测目标}
+
\text{采样动力学}
}
\tag{8.18}
$$

例如：

**概率路径：**

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

**预测目标：**

$$
x_1,\quad Z,\quad v,\quad s
$$

**采样动力学：**

$$
\text{ODE},\quad\text{SDE}
$$

这三个部分可以在一定条件下独立设计。


### 8.3.2 启示二：Flow Matching 不一定只能使用直线路径

最常见的 Conditional Flow Matching 使用：

$$
X_t=(1-t)Z+tX_1
$$

但随机插值允许：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

例如：

$$
\alpha(t)=\cos\left(\frac{\pi t}{2}\right)
$$

$$
\beta(t)=\sin\left(\frac{\pi t}{2}\right)
$$

也可以进一步加入辅助噪声：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

因此，Flow Matching 的关键并不是“直线”，而是通过监督回归学习边缘速度场：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{8.19}
$$


### 8.3.3 启示三：Diffusion 的 Score 可以从去噪器恢复

对于：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

有：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
}
\tag{8.20}
$$

其中：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

因此，Diffusion 中的噪声预测并不是与 Score 无关的技巧，而是学习 Score 的一种参数化方式。

进一步：

$$
\boxed{
D(t,x)=
\frac{x+\alpha^2(t)s(t,x)}{\beta(t)}
}
\tag{8.21}
$$

说明数据预测、噪声预测和 Score 预测之间存在直接联系。


### 8.3.4 启示四：可以将 Flow Matching 与随机 SDE 采样结合

假设已经训练了 Flow Matching 速度场：

$$
b_\theta(t,x)
$$

如果还能够获得 Score：

$$
s_\phi(t,x)
$$

就可以构造：

$$
\boxed{
dY_t=
[b_\theta(t,Y_t)+\varepsilon s_\phi(t,Y_t)]dt+
\sqrt{2\varepsilon}dW_t
}
\tag{8.22}
$$

这意味着：

**Flow Matching 训练与 SDE 采样并不矛盾。**

对于轨迹规划，这提供了一种可能的设计方向：

训练阶段使用确定性的速度回归目标。

推理阶段根据任务需要，选择：

- ODE：低随机性、便于复现。
- SDE：引入额外随机探索。

但需要强调，SDE 的引入不保证一定提高轨迹质量，还需要评估安全性、舒适性、可行性和推理成本。


### 8.3.5 启示五：预测目标与求解器应该联合考虑

对于单侧线性插值：

$$
X_t=\alpha Z+\beta X_1
$$

可以选择：

$$
v_\theta,\quad\eta_\theta,\quad D_\theta
$$

作为预测目标。

但是，最终采样器需要的可能是：

$$
b_\theta
$$

或者：

$$
b_\theta+\varepsilon s_\theta
$$

因此，模型预测空间和采样器并不是完全无关的两个问题。

例如：

$$
b=
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z
$$

如果：

$$
\beta(t)\to0
$$

那么通过噪声预测器恢复速度场可能存在数值不稳定。

因此，应该综合考虑：

$$
\boxed{
\text{训练监督尺度}
+
\text{预测变量转换}
+
\text{采样器数值误差}
}
$$

而不是仅比较某一种损失函数在训练集上的 MSE。


### 8.3.6 第 8 章总结

论文的核心贡献可以概括为：

$$
\boxed{
\text{随机插值}
\Longrightarrow
\text{概率路径}
\Longrightarrow
\begin{cases}
\text{Probability Flow ODE}\\
\text{Forward SDE}\\
\text{Backward SDE}
\end{cases}
}
$$

并通过条件期望，将速度场和 Score 的学习转化为监督回归。

对生成模型研究而言，最重要的启示是：

**Flow Matching、Diffusion、Denoising 和 Rectified Flow 可以在同一概率路径框架下理解，但它们在路径设计、训练目标、采样动力学和误差性质上仍然存在实质区别。**

下面进入第 9 章。

这一章不再局限于论文原文，而是将前面讨论过的单侧线性随机插值、CondOT、VP-SDE、VE-SDE 和 Trigonometric Interpolant 统一到一套公式中，并逐一代入推导。

---


<a id="chapter-9"></a>

# 第 9 章：统一框架的完整推导（扩展）

> **本章属于在原论文理论基础上的扩展整理。** 目标不是逐段复述论文，而是建立一套统一的数学表示，使 CondOT / Flow Matching、OU / VP-SDE、VE-SDE 和 Trigonometric Interpolant 都能通过代入不同的时间系数得到。
>
> 本章特别区分三个容易混淆的概念：**随机插值的单时刻分布、生成 ODE 的概率路径，以及扩散 SDE 的完整随机过程。**


## 9.1 单侧线性随机插值的统一表示


### 9.1.1 为什么需要统一表示？

前面已经学习了几种看起来不同的生成过程。

Flow Matching 常用：

$$
X_t=(1-t)Z+tX_1
$$

VP Diffusion 常用：

$$
X_\tau=\sqrt{\bar\alpha(\tau)}X_1+
\sqrt{1-\bar\alpha(\tau)}Z
$$

VE Diffusion 常用：

$$
X_\tau=X_1+\sigma(\tau)Z
$$

Trigonometric Interpolant 使用：

$$
X_t=
\cos\left(\frac{\pi t}{2}\right)Z+
\sin\left(\frac{\pi t}{2}\right)X_1
$$

这些表达式虽然不同，但具有相同的数学结构：

$$
\boxed{
X_t=a(t)Z+c(t)X_1
}
\tag{9.1}
$$

因此，我们可以先研究一般的 $a(t),c(t)$，然后将具体模型代入。

这样就不需要对每一种模型重新推导 Score、速度场、ODE 和 SDE。


### 9.1.2 统一定义

设：

$$
X_1\sim p_{\mathrm{data}}
$$

表示真实数据。

设：

$$
Z\sim\mathcal N(0,I_d)
$$

表示独立标准高斯噪声。

定义：

$$
\boxed{
X_t=a(t)Z+c(t)X_1,\qquad t\in[0,1]
}
\tag{9.2}
$$

这里统一采用：

$$
\boxed{
t=0:\text{噪声端},\qquad
t=1:\text{数据端}
}
$$

对于标准噪声到数据生成，要求：

$$
a(0)=1,\qquad c(0)=0
$$

$$
a(1)=0,\qquad c(1)=1
$$

于是：

$$
X_0=Z
$$

$$
X_1=X_1
$$

式 (9.2) 中右侧的 $X_1$ 是真实数据随机变量，左侧 $X_t$ 是插值随机变量；两者在 $t=1$ 时相同。

后面为了避免混淆，我们有时将真实数据记为：

$$
D=X_1
$$

于是：

$$
X_t=a(t)Z+c(t)D
$$

对于 VP、VE 等扩散模型，原始文献通常采用相反的时间方向：

$$
\tau=0:\text{数据端},\qquad
\tau=T:\text{噪声端}
$$

因此，在代入这些模型时，我们将使用时间变换：

$$
\boxed{
\tau=1-t
}
\tag{9.3}
$$

或者更一般的：

$$
\tau=\tau(t),\qquad \tau'(t)<0
$$

这是后面统一推导中最需要注意的符号约定。


### 9.1.3 条件概率密度

对于：

$$
X_t=a(t)Z+c(t)D
$$

固定：

$$
D=x_1
$$

有：

$$
X_t\mid D=x_1
=
c(t)x_1+a(t)Z
$$

因此：

$$
\boxed{
p_t(x\mid x_1)
=
\mathcal N\left(
x;c(t)x_1,a^2(t)I_d
\right)
}
\tag{9.4}
$$

即：

$$
p_t(x\mid x_1)
=
\frac1{(2\pi a^2)^{d/2}}
\exp\left(
-\frac{\|x-cx_1\|^2}{2a^2}
\right)
\tag{9.5}
$$

这里假设：

$$
a(t)>0
$$

对真实数据分布进行边缘化：

$$
\boxed{
p_t(x)=
\int p_t(x\mid x_1)p_{\mathrm{data}}(x_1)\,dx_1
}
\tag{9.6}
$$

因此：

$$
p_t
$$

是经过缩放和高斯平滑的数据分布。

当：

$$
a(t)>0
$$

时，通常可以得到光滑的中间概率密度。

但当：

$$
a(1)=0
$$

时，高斯平滑消失，终点恢复真实数据分布。

如果真实数据分布不具有光滑密度，终点 Score 不一定存在。

这也是后续公式需要特别处理时间端点的原因。

---


## 9.2 从一个噪声预测器推导速度场与 Score


### 9.2.1 核心结论

对于：

$$
X_t=a(t)Z+c(t)D
$$

定义噪声预测器的理想输出：

$$
\boxed{
\eta(t,x)=\mathbb E[Z\mid X_t=x]
}
\tag{9.7}
$$

则：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
\tag{9.8}
$$

以及：

$$
\boxed{
b(t,x)=
\frac{\dot c(t)}{c(t)}x+
\left[
\dot a(t)-\frac{a(t)\dot c(t)}{c(t)}
\right]\eta(t,x)
}
\tag{9.9}
$$

因此：

$$
\boxed{
\text{一个噪声预测器}
\Longrightarrow
\begin{cases}
\text{Score}\\
\text{边缘速度场}
\end{cases}
}
$$

这两个公式是整个第 9 章最重要的结果。

下面完整推导。


### 9.2.2 第一步：从高斯条件密度推导 Score


对条件高斯密度应用 Fisher Identity，在 $a(t)>0$ 且允许交换积分与求导时：

完整推导见[附录 D.1.2～D.1.3：Fisher Identity 与噪声预测](#app-d-1-3)。

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
\tag{9.11}
$$


### 9.2.3 第二步：推导数据去噪器


定义 $m(t,x)=\mathbb E[D\mid X_t=x]$。由 $x=a\eta+cm$，在 $c(t)\ne0$ 时：

完整推导见[附录 D.2.2：线性高斯形式的 Tweedie 公式](#app-d-2)。

$$
\boxed{
m(t,x)=\frac{x-a\eta(t,x)}{c(t)}
}
\tag{9.13}
$$

再代入：

$$
\eta=-as
$$

得到：

$$
\boxed{
m(t,x)=
\frac{x+a^2(t)s(t,x)}{c(t)}
}
\tag{9.14}
$$

这就是统一形式下的 Tweedie 公式。

需要注意：

$$
c(t)\neq0
$$

才可以直接使用式 (9.13) 和式 (9.14)。

在噪声端：

$$
c(0)=0
$$

时，需要使用其他等价表达式或端点极限。


### 9.2.4 第三步：推导边缘速度场


对 $X_t=aZ+cD$，将 $b=\dot a\eta+\dot cm$ 与 $m=(x-a\eta)/c$ 结合，在 $c(t)\ne0$ 时得到：

完整推导见[附录 F.1.1：单侧线性插值的速度场](#app-f-1-1)。

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta(t,x)
}
\tag{9.17}
$$

这就是统一的边缘速度场公式。


### 9.2.5 第四步：使用 Score 表示速度场


将 $\eta=-as$ 代入噪声预测形式，得到：

完整推导见[附录 F.1.2：速度场的 Score 表示](#app-f-1-2)。

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\frac{a^2\dot c}{c}-a\dot a
\right)s(t,x)
}
\tag{9.18}
$$

为了简化符号，定义：

$$
\boxed{
A(t)=\frac{\dot c(t)}{c(t)}
}
\tag{9.19}
$$

以及：

$$
\boxed{
H(t)=
\frac{a^2(t)\dot c(t)}{c(t)}
-a(t)\dot a(t)
}
\tag{9.20}
$$

于是：

$$
\boxed{
b(t,x)=A(t)x+H(t)s(t,x)
}
\tag{9.21}
$$

这也是连接扩散模型的关键表达式。


### 9.2.6 三种预测目标之间的完整转换


这里 $a,c,\eta,m$ 分别对应附录中的 $\alpha,\beta,\eta_z,\eta_1$。定义 $\Delta(t)=a\dot c-c\dot a$。当 $\Delta(t)\ne0$ 时，观测和速度构成的线性方程组可逆，因此：

完整推导见[附录 F.1.4：预测目标转换的矩阵推导](#app-f-1-4)。

$$
\boxed{
\eta(t,x)=
\frac{\dot c(t)x-c(t)b(t,x)}
{\Delta(t)}
}
\tag{9.26}
$$

以及：

$$
\boxed{
m(t,x)=
\frac{a(t)b(t,x)-\dot a(t)x}
{\Delta(t)}
}
\tag{9.27}
$$

这意味着，在满足可逆条件时：

$$
\boxed{
\text{速度预测}
\Longleftrightarrow
\text{噪声预测}
\Longleftrightarrow
\text{数据预测}
}
$$

但这种等价是理想预测场之间的代数等价，不意味着它们具有完全相同的训练误差。

---


## 9.3 统一的 Probability Flow ODE、前向 SDE 与反向 SDE


### 9.3.1 从连续性方程开始

根据前面的推导，随机插值的边缘密度满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
\tag{9.28}
$$

其中：

$$
b(t,x)=A(t)x+H(t)s(t,x)
$$

因此：

$$
\boxed{
\partial_tp_t+
\nabla\cdot\left[
p_t(Ax+Hs)
\right]=0
}
\tag{9.29}
$$

这条方程描述了随机插值概率路径的演化。


### 9.3.2 Probability Flow ODE

定义：

$$
\boxed{
dY_t=b(t,Y_t)dt
}
\tag{9.30}
$$

代入式 (9.21)：

$$
\boxed{
\frac{dY_t}{dt}
=
A(t)Y_t+H(t)s(t,Y_t)
}
\tag{9.31}
$$

进一步使用噪声预测器：

$$
s=-\frac{\eta}{a}
$$

得到：

$$
\boxed{
\frac{dY_t}{dt}
=
A(t)Y_t-\frac{H(t)}{a(t)}\eta(t,Y_t)
}
\tag{9.32}
$$

这就是统一的 Probability Flow ODE。

它既可以用于 Flow Matching，也可以用于 Diffusion 对应的确定性概率流。


### 9.3.3 前向生成 SDE


对于状态无关的扩散系数和 $\varepsilon(t)\ge0$，以下 SDE 在相同初始分布与适定性条件下保持目标边缘密度：

完整推导见[附录 B.3.3：统一前向生成 SDE](#app-b-3-3)。

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{9.37}
$$

进一步代入：

$$
b=Ax+Hs
$$

得到：

$$
\boxed{
dY_t=
\left[
A(t)Y_t+
(H(t)+\varepsilon(t))s(t,Y_t)
\right]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{9.38}
$$

使用噪声预测器：

$$
\boxed{
dY_t=
\left[
A(t)Y_t-
\frac{H(t)+\varepsilon(t)}{a(t)}
\eta(t,Y_t)
\right]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{9.39}
$$

这就是统一的**前向生成 SDE**。

这里的“前向”特指生成时间 $t:0\to1$，不是扩散模型通常所说的数据加噪前向过程。


### 9.3.4 反向 SDE：为什么 Score 前面的符号改变？


定义 $u=1-t$、$R_u=Y_{1-u}$。由 $f=b+\varepsilon s$、$g^2=2\varepsilon$，时间反演后的漂移为 $-f+g^2s=-b+\varepsilon s$：

完整推导见[附录 C.3：递增反向时间与漂移修正](#app-c-3-2)。

$$
\boxed{
dR_u=
[-b(1-u,R_u)+\varepsilon(1-u)s(1-u,R_u)]du
+
\sqrt{2\varepsilon(1-u)}d\overline W_u
}
\tag{9.42}
$$

这才是使用递增反向时间 $u$ 的标准 SDE。

如果仍使用原时间 $t$，并明确采用反向 Itô 积分约定，则可简写为：

$$
\boxed{
dY_t=
[b(t,Y_t)-\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}d\overleftarrow W_t
}
\tag{9.43}
$$

其中：

$$
t:1\to0
$$

必须注意：

**式 (9.43) 中的 $dt$ 是沿递减原时间积分的，不能将它当作普通递增时间 SDE 直接使用。**


### 9.3.5 一个统一公式生成三种动力学

至此，已经得到：

**Probability Flow ODE：**

$$
\boxed{
dY_t=b\,dt
}
\tag{9.44}
$$

**前向生成 SDE，$t:0\to1$：**

$$
\boxed{
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
}
\tag{9.45}
$$

**反向 SDE，$t:1\to0$：**

$$
\boxed{
dY_t=(b-\varepsilon s)dt+
\sqrt{2\varepsilon}d\overleftarrow W_t
}
\tag{9.46}
$$

这里的反向 SDE 是式 (9.45) 所定义过程的时间反演。

三者使用相同的：

$$
b(t,x),\qquad s(t,x)
$$

而这两个量都可以从同一个噪声预测器恢复。

因此：

$$
\boxed{
\eta(t,x)
\Longrightarrow
\begin{cases}
b(t,x)\\
s(t,x)
\end{cases}
\Longrightarrow
\begin{cases}
\text{ODE}\\
\text{Forward SDE}\\
\text{Reverse SDE}
\end{cases}
}
\tag{9.47}
$$

这就是本章的统一数学框架。

---


## 9.4 CondOT / Flow Matching 的代入推导


### 9.4.1 CondOT 的时间系数

最常见的 Conditional Flow Matching 使用：

$$
\boxed{
X_t=(1-t)Z+tD
}
\tag{9.48}
$$

其中：

$$
Z\sim\mathcal N(0,I_d),\qquad D\sim p_{\mathrm{data}}
$$

对应：

$$
\boxed{
a(t)=1-t,\qquad c(t)=t
}
\tag{9.49}
$$

对时间求导：

$$
\dot a(t)=-1
$$

$$
\dot c(t)=1
$$

因此，样本级条件速度为：

$$
\boxed{
\dot X_t=D-Z
}
\tag{9.50}
$$

这就是 Flow Matching 最常见的监督目标。

训练损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E_{t,Z,D}
\left[
\|v_\theta(t,X_t)-(D-Z)\|^2
\right]
}
\tag{9.51}
$$

在条件轨迹规划中，只需将模型改为：

$$
v_\theta(t,X_t,c_{\mathrm{scene}})
$$

其中 $c_{\mathrm{scene}}$ 是场景条件。


### 9.4.2 从噪声预测器恢复速度场


将 $a=1-t,c=t$ 代入单侧线性通式，得到：

完整推导见[附录 F.4.1：CondOT 的系数代入](#app-f-4-1)。

$$
\boxed{
b(t,x)=\frac{x-\eta(t,x)}{t}
}
\tag{9.52}
$$

这是 CondOT 的噪声预测形式。

同时：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{1-t}
}
\tag{9.53}
$$

因此，一个噪声预测器可以同时恢复：

$$
b,\qquad s
$$

需要注意，式 (9.52) 在 $t=0$ 存在表面奇异性。

实际使用时，不能简单地将 $t=0$ 直接代入。


### 9.4.3 使用数据预测器恢复速度场


将 $\alpha=1-t,\beta=t$ 代入数据预测形式，在 $t\ne1$ 时有：

完整推导见[附录 F.1.3：数据预测形式的速度场](#app-f-1-3)。

$$
\boxed{
b(t,x)=\frac{m(t,x)-x}{1-t}
}
\tag{9.54}
$$

其中：

$$
m(t,x)=\mathbb E[D\mid X_t=x]
$$

因此，CondOT 速度场具有两种等价表示：

$$
\boxed{
b(t,x)=
\frac{x-\eta(t,x)}t
=
\frac{m(t,x)-x}{1-t}
}
\tag{9.55}
$$

这说明：

- 噪声预测适合表达从噪声端恢复速度的关系。
- 数据预测适合表达从数据端恢复速度的关系。

但两种形式分别在不同端点出现除零问题。


### 9.4.4 CondOT 的 Probability Flow ODE

直接代入：

$$
\dot Y_t=b(t,Y_t)
$$

得到：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac{Y_t-\eta(t,Y_t)}t
}
\tag{9.56}
$$

或者：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac{m(t,Y_t)-Y_t}{1-t}
}
\tag{9.57}
$$

如果直接训练速度预测器：

$$
v_\theta(t,x)\approx b(t,x)
$$

则不需要显式进行这些转换：

$$
\boxed{
\frac{dY_t}{dt}=v_\theta(t,Y_t)
}
\tag{9.58}
$$

这也是直接速度预测的一个实际优势。


### 9.4.5 CondOT 的前向生成 SDE


把 CondOT 的速度与 Score 代入 $b+\varepsilon s$，得到：

完整推导见[附录 B.3.3：同边缘分布的 SDE 构造](#app-b-3-3)。

$$
\boxed{
dY_t=
\left[
\frac{Y_t-\eta(t,Y_t)}t
-
\frac{\varepsilon(t)}{1-t}\eta(t,Y_t)
\right]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{9.59}
$$

这是 CondOT 对应的一族 SDE。

当：

$$
\varepsilon(t)=0
$$

时，恢复 Flow Matching ODE。

当：

$$
\varepsilon(t)>0
$$

时，在精确 Score 和适当正则性条件下，可以构造具有相同边缘概率路径的随机生成过程。

这里的 $\varepsilon(t)$ 是可选择的采样扩散强度，并不是 CondOT 插值自身规定的噪声方差。


### 9.4.6 CondOT 的反向 SDE


将 CondOT 的速度与 Score 代入原时间递减的 $b-\varepsilon s$ 形式，得到：

完整推导见[附录 C.3：时间反演约定](#app-c-3-2)。

$$
\boxed{
dY_t=
\left[
\frac{Y_t-\eta(t,Y_t)}t
+
\frac{\varepsilon(t)}{1-t}\eta(t,Y_t)
\right]dt
+
\sqrt{2\varepsilon(t)}d\overleftarrow W_t
}
\tag{9.60}
$$

其中：

$$
t:1\to0
$$

如果希望使用普通递增时间的数值求解器，应该改用：

$$
u=1-t
$$

并使用第 9.3.4 节的时间反演公式。


### 9.4.7 CondOT 与最优传输的关系

需要特别区分：

$$
X_t=(1-t)Z+tD
$$

与：

$$
(X_0,X_1)\sim\pi_{\mathrm{OT}}
$$

这两个概念。

前者只规定了每组端点之间的插值形式。

后者规定了端点之间的耦合方式。

如果：

$$
Z\perp D
$$

则端点是独立耦合。

这并不意味着：

$$
(Z,D)
$$

构成最优传输耦合。

因此：

$$
\boxed{
\text{直线条件插值}
\neq
\text{最优传输耦合}
}
$$

在一些 Flow Matching 文献中，CondOT 指采用线性条件概率路径的构造；它本身并不要求数据端点与噪声端点构成全局 OT 耦合。

---


## 9.5 OU / VP-SDE 的代入推导

这一节需要特别注意时间方向。

扩散模型通常从数据端开始加噪：

$$
\tau:0\to T
$$

而本章统一采用：

$$
t:0\to1
$$

表示从噪声端向数据端生成。

因此，我们先在扩散时间 $\tau$ 下推导，再转换到生成时间 $t$。


### 9.5.1 从 OU / VP 前向加噪 SDE 开始


考虑 $dX_\tau=-\tfrac12\beta(\tau)X_\tau d\tau+\sqrt{\beta(\tau)}dW_\tau$，$X_0=D$。定义 $B(\tau)=\int_0^\tau\beta(r)dr$、$\alpha(\tau)=e^{-B(\tau)/2}$，则：

完整推导见[附录 F.3.2：VP-SDE 的解析条件分布](#app-f-3-2)。

$$
\boxed{
X_\tau\mid D
\sim
\mathcal N\left(
\alpha(\tau)D,
[1-\alpha^2(\tau)]I_d
\right)
}
\tag{9.65}
$$

这就是 VP-SDE 的解析条件边缘分布。


### 9.5.2 将 VP-SDE 写成单侧线性插值

由于式 (9.65) 是高斯分布，因此对于每个固定时间 $\tau$，有：

$$
\boxed{
X_\tau\overset d=
\alpha(\tau)D+
\sqrt{1-\alpha^2(\tau)}Z
}
\tag{9.66}
$$

这里：

$$
\overset d=
$$

表示单时刻分布相同。

它不表示使用同一个固定 $Z$ 的随机插值，与原始 VP-SDE 具有相同的完整路径分布。

这是理解随机插值与 Diffusion 关系的关键。

现在令：

$$
\boxed{
\tau=1-t
}
\tag{9.67}
$$

为简化记号，假设扩散时间已经归一化到 $[0,1]$。

定义：

$$
\boxed{
c(t)=\alpha(1-t)
}
\tag{9.68}
$$

以及：

$$
\boxed{
a(t)=\sqrt{1-\alpha^2(1-t)}
}
\tag{9.69}
$$

于是：

$$
\boxed{
X_t=a(t)Z+c(t)D
}
\tag{9.70}
$$

这正是统一的单侧线性随机插值。

需要注意，有限扩散时间下：

$$
\alpha(1)>0
$$

通常成立。

因此：

$$
c(0)=\alpha(1)\neq0
$$

严格来说，初始边缘分布只是近似标准高斯，而不一定精确等于标准高斯。

只有当终止噪声水平足够大，使：

$$
\alpha(1)\approx0
$$

时，才可以近似认为：

$$
X_{t=0}\sim\mathcal N(0,I_d)
$$


### 9.5.3 计算 VP 的时间系数


由 $c(t)=\alpha(1-t)$ 与 $d\alpha/d\tau=-\beta\alpha/2$，链式法则给出：

完整推导见[附录 F.4.2：VP 的时间系数](#app-f-4-2)。

$$
\boxed{
\dot c(t)=
\frac12\beta(1-t)c(t)
}
\tag{9.71}
$$

为简洁起见，定义：

$$
\widetilde\beta(t)=\beta(1-t)
$$

于是：

$$
\boxed{
\frac{\dot c}{c}=\frac12\widetilde\beta(t)
}
\tag{9.72}
$$


另一方面，由 $a^2=1-c^2$ 求导可得（详见[附录 F.4.2](#app-f-4-2)）：

$$
\boxed{
a\dot a=
-\frac12\widetilde\beta(t)c^2(t)
}
\tag{9.73}
$$


### 9.5.4 代入统一速度场公式


利用 $a^2+c^2=1$、$\dot c/c=\widetilde\beta/2$，代入统一速度场公式可得：

完整推导见[附录 F.4.3：VP 的统一速度场](#app-f-4-3)。

$$
\boxed{
b_{\mathrm{VP}}(t,x)=
\frac12\widetilde\beta(t)x+
\frac12\widetilde\beta(t)s(t,x)
}
\tag{9.75}
$$

这就是 VP-SDE 在噪声到数据时间方向上的 Probability Flow ODE 速度场。


### 9.5.5 VP 的 Probability Flow ODE

代入：

$$
\dot Y_t=b_{\mathrm{VP}}(t,Y_t)
$$

得到：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac12\widetilde\beta(t)
\left[
Y_t+s(t,Y_t)
\right]
}
\tag{9.76}
$$

再使用：

$$
s=-\frac{\eta}{a}
$$

得到：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac12\widetilde\beta(t)
\left[
Y_t-\frac{\eta(t,Y_t)}{a(t)}
\right]
}
\tag{9.77}
$$

这是 VP 概率流 ODE 的噪声预测形式。


### 9.5.6 VP 的反向去噪 SDE

现在回到扩散模型的原始时间 $\tau$。

前向加噪 SDE 为：

$$
dX_\tau=
-\frac12\beta(\tau)X_\tau d\tau+
\sqrt{\beta(\tau)}dW_\tau
$$

其反向时间 SDE 在递减的 $\tau$ 下写为：

$$
\boxed{
dX_\tau=
\left[
-\frac12\beta(\tau)X_\tau-
\beta(\tau)s_\tau(X_\tau)
\right]d\tau
+
\sqrt{\beta(\tau)}d\overleftarrow W_\tau
}
\tag{9.78}
$$

其中：

$$
\tau:1\to0
$$

注意：

$$
d\tau<0
$$

因此，式 (9.78) 中的漂移符号不能直接按照递增时间解释。

如果转换成递增生成时间：

$$
t=1-\tau
$$

则：

$$
\boxed{
dY_t=
\left[
\frac12\widetilde\beta(t)Y_t+
\widetilde\beta(t)s(t,Y_t)
\right]dt
+
\sqrt{\widetilde\beta(t)}dW_t
}
\tag{9.79}
$$

这就是 VP 的标准反向去噪 SDE，以噪声到数据的时间方向表示。


### 9.5.7 验证它属于统一 SDE 族

统一的前向生成 SDE 为：

$$
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
$$

我们已经得到：

$$
b_{\mathrm{VP}}=
\frac12\widetilde\beta x+
\frac12\widetilde\beta s
$$

选择：

$$
\boxed{
\varepsilon(t)=\frac12\widetilde\beta(t)
}
\tag{9.80}
$$

则：

$$
\begin{aligned}
b+\varepsilon s
&=
\frac12\widetilde\beta x+
\frac12\widetilde\beta s+
\frac12\widetilde\beta s\\
&=
\frac12\widetilde\beta x+
\widetilde\beta s
\end{aligned}
$$

同时：

$$
\sqrt{2\varepsilon}=\sqrt{\widetilde\beta}
$$

因此：

$$
\boxed{
dY_t=
\left[
\frac12\widetilde\beta Y_t+
\widetilde\beta s(t,Y_t)
\right]dt+
\sqrt{\widetilde\beta}dW_t
}
\tag{9.81}
$$

与式 (9.79) 完全一致。

这证明：

$$
\boxed{
\text{VP 反向去噪 SDE}
=
\text{统一前向生成 SDE 的一个特例}
}
$$

这里的“前向生成”指 $t:0\to1$，而 VP 文献中的“反向去噪”指 $\tau:1\to0$。两者是同一个物理采样方向，只是时间变量不同。


### 9.5.8 OU 与 VP 的关系

当：

$$
\beta(\tau)=\beta_0
$$

为常数时：

$$
dX_\tau=
-\frac12\beta_0X_\tau d\tau+
\sqrt{\beta_0}dW_\tau
$$

就是 Ornstein–Uhlenbeck（OU）过程的一种标准参数化。

此时：

$$
\alpha(\tau)=e^{-\beta_0\tau/2}
$$

条件边缘分布：

$$
\boxed{
X_\tau\mid D
\sim
\mathcal N\left(
e^{-\beta_0\tau/2}D,
(1-e^{-\beta_0\tau})I_d
\right)
}
\tag{9.82}
$$

当：

$$
\tau\to\infty
$$

时：

$$
e^{-\beta_0\tau/2}\to0
$$

因此：

$$
X_\tau\overset d\longrightarrow\mathcal N(0,I_d)
$$

所以：

$$
\boxed{
\text{OU 是常系数 VP-SDE 的特例}
}
$$

而一般 VP-SDE 可以理解为时间变化系数的 OU 型扩散过程。

---


## 9.6 VE-SDE 的代入推导与端点问题


### 9.6.1 VE 与 VP 的核心区别

VP-SDE 的前向加噪过程为：

$$
dX_\tau=
-\frac12\beta(\tau)X_\tau d\tau+
\sqrt{\beta(\tau)}dW_\tau
$$

它同时具有：

- 将数据向零收缩的漂移。
- 增加随机噪声的扩散项。

因此，其条件边缘分布为：

$$
X_\tau\mid D
\sim
\mathcal N\left(
\alpha(\tau)D,
[1-\alpha^2(\tau)]I_d
\right)
$$

而 Variance Exploding SDE（VE-SDE）通常没有收缩漂移：

$$
\boxed{
dX_\tau=g(\tau)dW_\tau
}
\tag{9.83}
$$

因此，数据均值保持不变，噪声方差随时间增加。

这是 VE 与 VP 的根本区别。


### 9.6.2 VE-SDE 的解析条件分布


对 $dX_\tau=g(\tau)dW_\tau$、$X_0=D$，令 $\sigma^2(\tau)=\int_0^\tau g^2(r)dr$，有：

完整推导见[附录 F.3.3：VE 的解析条件分布](#app-f-3-3)。

$$
\boxed{
X_\tau\mid D
\sim
\mathcal N(D,\sigma^2(\tau)I_d)
}
\tag{9.85}
$$

或者：

$$
\boxed{
X_\tau\overset d=D+\sigma(\tau)Z
}
\tag{9.86}
$$

其中：

$$
Z\sim\mathcal N(0,I_d)
$$

同样，这里的等式是单时刻分布等价，不表示固定 $Z$ 的随机插值与原始 VE-SDE 具有相同路径分布。

由式 (9.84)：

$$
\boxed{
g^2(\tau)=\frac{d\sigma^2(\tau)}{d\tau}
}
\tag{9.87}
$$

这是 VE-SDE 中噪声日程与扩散系数之间的基本关系。


### 9.6.3 转换到统一生成时间

令：

$$
\tau=1-t
$$

则：

$$
X_t=D+\sigma(1-t)Z
$$

因此，对应统一形式：

$$
X_t=a(t)Z+c(t)D
$$

有：

$$
\boxed{
a(t)=\sigma(1-t)
}
\tag{9.88}
$$

$$
\boxed{
c(t)=1
}
\tag{9.89}
$$

因此：

$$
\dot c(t)=0
$$

$$
\dot a(t)=-\sigma'(1-t)
$$

这与 CondOT 和 VP 的时间系数明显不同。


### 9.6.4 从统一公式推导 VE 速度场


由 $c=1$，有 $b=\dot a\eta=-a\dot a\,s$。记 $\widetilde g(t)=g(1-t)$，利用 $a(t)=\sigma(1-t)$ 和 $g^2=d\sigma^2/d\tau$，可得：

完整推导见[附录 F.4.4：VE 的生成速度场](#app-f-4-4)。

$$
\boxed{
b_{\mathrm{VE}}(t,x)=
\frac12\widetilde g^2(t)s(t,x)
}
\tag{9.92}
$$

这是 VE 在噪声到数据时间方向上的 Probability Flow ODE 速度场。


### 9.6.5 VE 的 Probability Flow ODE

直接得到：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac12\widetilde g^2(t)s(t,Y_t)
}
\tag{9.93}
$$

使用噪声预测器：

$$
s=-\frac{\eta}{a}
$$

得到：

$$
\boxed{
\frac{dY_t}{dt}
=
-\frac{\widetilde g^2(t)}{2a(t)}
\eta(t,Y_t)
}
\tag{9.94}
$$

这就是 VE 的统一噪声预测形式。

与 VP 相比，VE 的 ODE 没有显式的：

$$
A(t)Y_t
$$

项。

原因是：

$$
c(t)=1
$$

因此：

$$
\frac{\dot c}{c}=0
$$


### 9.6.6 VE 的反向去噪 SDE

在原始扩散时间 $\tau$ 下：

$$
dX_\tau=g(\tau)dW_\tau
$$

对应的反向去噪 SDE 为：

$$
\boxed{
dX_\tau=
-g^2(\tau)s_\tau(X_\tau)d\tau+
g(\tau)d\overleftarrow W_\tau
}
\tag{9.95}
$$

其中：

$$
\tau:1\to0
$$

转换到递增生成时间：

$$
t=1-\tau
$$

得到：

$$
\boxed{
dY_t=
\widetilde g^2(t)s(t,Y_t)dt+
\widetilde g(t)dW_t
}
\tag{9.96}
$$

现在使用统一的前向生成 SDE：

$$
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
$$

已经得到：

$$
b_{\mathrm{VE}}=\frac12\widetilde g^2s
$$

选择：

$$
\boxed{
\varepsilon(t)=\frac12\widetilde g^2(t)
}
\tag{9.97}
$$


代入后 $b+\varepsilon s=\widetilde g^2s$、$\sqrt{2\varepsilon}=\widetilde g$。

最终恢复：

$$
\boxed{
dY_t=
\widetilde g^2(t)s(t,Y_t)dt+
\widetilde g(t)dW_t
}
\tag{9.98}
$$

与 VE 的反向去噪 SDE 完全一致。

因此：

$$
\boxed{
\text{VE 反向去噪 SDE}
=
\text{统一前向生成 SDE 的一个特例}
}
$$


### 9.6.7 VE 的第一个端点问题：数据端噪声不一定为零

在许多实际 VE 模型中，噪声日程设置为：

$$
\sigma_{\min}>0
$$

到：

$$
\sigma_{\max}
$$

因此，在最小噪声时间：

$$
\tau=0
$$

可能仍然有：

$$
\sigma(0)=\sigma_{\min}>0
$$

于是：

$$
X_{\tau=0}=D+\sigma_{\min}Z
$$

而不是严格的：

$$
X_{\tau=0}=D
$$

因此：

$$
\boxed{
p_{\tau=0}
=
p_{\mathrm{data}}*
\mathcal N(0,\sigma_{\min}^2I_d)
}
\tag{9.99}
$$

这里：

$$
*
$$

表示概率密度卷积。

这说明：

**当 $\sigma_{\min}>0$ 时，VE 的最小噪声分布仍然是经过高斯平滑的数据分布。**

如果希望严格连接真实数据分布，需要：

$$
\sigma(0)=0
$$

或者额外处理最终去噪步骤。


### 9.6.8 VE 的第二个端点问题：最大噪声分布不一定是标准高斯

对于：

$$
X_{\tau=T}=D+\sigma_{\max}Z
$$

有：

$$
p_T=
p_{\mathrm{data}}*
\mathcal N(0,\sigma_{\max}^2I_d)
$$

当：

$$
\sigma_{\max}
$$

很大时，数据项相对于噪声项的影响较小。

因此，可以近似使用：

$$
\boxed{
X_T\sim\mathcal N(0,\sigma_{\max}^2I_d)
}
\tag{9.100}
$$

作为生成初始分布。

但是，这通常只是近似，而不是严格相等。

更重要的是：

$$
\mathcal N(0,\sigma_{\max}^2I_d)
$$

也不是标准高斯：

$$
\mathcal N(0,I_d)
$$

除非进一步进行归一化。

因此，VE 不能直接被认为满足第 9.1 节的标准端点条件：

$$
a(0)=1,\qquad c(0)=0
$$

因为 VE 实际上具有：

$$
a(0)=\sigma_{\max}
$$

$$
c(0)=1
$$

这说明：

**VE 可以使用统一的线性高斯表示，但不严格属于标准高斯到数据端点条件下的插值，除非进行额外构造或近似。**


### 9.6.9 VE 与随机插值的真正联系

经过以上推导，我们可以得到：

$$
\boxed{
X_t=D+a(t)Z
}
$$

对应：

$$
\boxed{
b(t,x)=-a(t)\dot a(t)s(t,x)
}
$$

进一步选择：

$$
\boxed{
\varepsilon(t)=-a(t)\dot a(t)
}
$$

在 $a(t)$ 随生成时间递减、从而 $\varepsilon(t)\ge0$ 时，有：

$$
b+\varepsilon s
=
-2a\dot a\,s
$$

由于：

$$
\widetilde g^2=-2a\dot a
$$

因此：

$$
\boxed{
dY_t=
\widetilde g^2(t)s(t,Y_t)dt+
\widetilde g(t)dW_t
}
$$

这就恢复了 VE 的标准反向生成过程。

---


## 9.7 Trigonometric Interpolant 的代入推导


### 9.7.1 定义

考虑：

$$
\boxed{
X_t=
\cos\left(\frac{\pi t}{2}\right)Z+
\sin\left(\frac{\pi t}{2}\right)D
}
\tag{9.101}
$$

定义：

$$
\theta(t)=\frac{\pi t}{2}
$$

于是：

$$
\boxed{
a(t)=\cos\theta(t)
}
\tag{9.102}
$$

$$
\boxed{
c(t)=\sin\theta(t)
}
\tag{9.103}
$$

显然：

$$
a(0)=1,\qquad c(0)=0
$$

$$
a(1)=0,\qquad c(1)=1
$$

因此，它严格满足标准高斯到数据的端点条件。

另外：

$$
\boxed{
a^2(t)+c^2(t)=1
}
\tag{9.104}
$$

这与 VP 的方差保持结构相同。

但是，两者的时间日程不一定相同。


### 9.7.2 计算时间导数

由于：

$$
\theta(t)=\frac{\pi t}{2}
$$

有：

$$
\dot\theta=\frac\pi2
$$

因此：

$$
\boxed{
\dot a(t)=
-\frac\pi2\sin\theta
}
\tag{9.105}
$$

$$
\boxed{
\dot c(t)=
\frac\pi2\cos\theta
}
\tag{9.106}
$$

样本级速度：

$$
\dot X_t=\dot aZ+\dot cD
$$

因此：

$$
\boxed{
\dot X_t=
-\frac\pi2\sin\theta\,Z+
\frac\pi2\cos\theta\,D
}
\tag{9.107}
$$

对应的 Flow Matching 训练损失：

$$
\boxed{
\mathcal L_{\mathrm{Trig}}
=
\mathbb E
\left[
\left\|
v_\theta(t,X_t)
-
\frac\pi2(-\sin\theta Z+\cos\theta D)
\right\|^2
\right]
}
\tag{9.108}
$$


### 9.7.3 推导 Score

根据统一公式：

$$
s=-\frac{\eta}{a}
$$

直接得到：

$$
\boxed{
s(t,x)=
-\frac{\eta(t,x)}{\cos\theta}
}
\tag{9.109}
$$

这里要求：

$$
\cos\theta>0
$$

即：

$$
t<1
$$


### 9.7.4 推导速度场

使用：

$$
b=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta
$$

首先：

$$
\frac{\dot c}{c}
=
\frac{\frac\pi2\cos\theta}{\sin\theta}
$$

因此：

$$
\boxed{
\frac{\dot c}{c}
=
\frac\pi2\cot\theta
}
\tag{9.110}
$$

然后：

$$
\begin{aligned}
\dot a-\frac{a\dot c}{c}
&=
-\frac\pi2\sin\theta
-
\frac\pi2\frac{\cos^2\theta}{\sin\theta}\\
&=
-\frac\pi2
\frac{\sin^2\theta+\cos^2\theta}{\sin\theta}\\
&=
-\frac\pi{2\sin\theta}
\end{aligned}
$$

因此：

$$
\boxed{
b_{\mathrm{Trig}}(t,x)=
\frac\pi2\cot\theta\,x
-
\frac\pi{2\sin\theta}\eta(t,x)
}
\tag{9.111}
$$

进一步使用：

$$
\eta=-\cos\theta\,s
$$

得到：

$$
\begin{aligned}
b_{\mathrm{Trig}}
&=
\frac\pi2\cot\theta\,x
+
\frac\pi2\frac{\cos\theta}{\sin\theta}s
\end{aligned}
$$

因此：

$$
\boxed{
b_{\mathrm{Trig}}(t,x)=
\frac\pi2\cot\theta
[x+s(t,x)]
}
\tag{9.112}
$$

其中：

$$
\theta=\frac{\pi t}{2}
$$

这是 Trigonometric Interpolant 的统一速度场。


### 9.7.5 Probability Flow ODE

因此：

$$
\boxed{
\frac{dY_t}{dt}
=
\frac\pi2\cot\left(\frac{\pi t}{2}\right)
\left[
Y_t+s(t,Y_t)
\right]
}
\tag{9.113}
$$

与 VP 的概率流 ODE：

$$
\frac{dY_t}{dt}
=
\frac12\widetilde\beta(t)
[Y_t+s(t,Y_t)]
$$

进行比较。

可以发现，两者形式完全一致，只需要令：

$$
\boxed{
\widetilde\beta(t)=
\pi\cot\left(\frac{\pi t}{2}\right)
}
\tag{9.114}
$$

因此：

$$
\boxed{
\text{Trigonometric Interpolant}
\Longleftrightarrow
\text{一种特殊时间日程的 VP 概率路径}
}
$$

这里的等价指对应的单时刻条件边缘分布与概率流动力学。


### 9.7.6 为什么这个 VP 日程在端点存在奇异性？

观察：

$$
\widetilde\beta(t)=
\pi\cot\left(\frac{\pi t}{2}\right)
$$

当：

$$
t\to0
$$

时：

$$
\cot\left(\frac{\pi t}{2}\right)
\sim\frac2{\pi t}
$$

因此：

$$
\boxed{
\widetilde\beta(t)\sim\frac2t
}
\tag{9.115}
$$

所以：

$$
\widetilde\beta(t)\to\infty
$$

这说明 Trigonometric Interpolant 虽然满足精确的标准高斯到数据端点条件，但将它表示为 VP 型 SDE 时，所对应的扩散日程在纯噪声端具有奇异性。

这并不是偶然现象。

对于 VP：

$$
c(t)=\alpha(1-t)
$$

如果要求：

$$
c(0)=0
$$

那么在有限时间内，要使原始 VP 的信号系数严格衰减到零，通常需要累积噪声强度发散。

这与普通有限时间、有限积分强度的 VP-SDE 不同。

因此：

**Trigonometric Interpolant 可以对应一种端点奇异的 VP 型日程，但不能不加说明地等同于常规有限噪声日程的 VP-SDE。**


### 9.7.7 Trigonometric Interpolant 的生成 SDE

统一公式：

$$
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
$$

代入：

$$
b=
\frac\pi2\cot\theta(x+s)
$$

得到：

$$
\boxed{
dY_t=
\left[
\frac\pi2\cot\theta\,Y_t+
\left(
\frac\pi2\cot\theta+\varepsilon(t)
\right)s(t,Y_t)
\right]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{9.116}
$$

如果希望恢复与其对应的 VP 型反向去噪 SDE，选择：

$$
\boxed{
\varepsilon(t)=\frac\pi2\cot\theta
}
\tag{9.117}
$$

则：

$$
\boxed{
dY_t=
\left[
\frac\pi2\cot\theta\,Y_t+
\pi\cot\theta\,s(t,Y_t)
\right]dt+
\sqrt{\pi\cot\theta}\,dW_t
}
\tag{9.118}
$$

这里：

$$
0<t<1
$$

且纯噪声端需要考虑扩散系数的奇异性。


### 9.7.8 与 CondOT 的比较

CondOT：

$$
X_t=(1-t)Z+tD
$$

满足：

$$
a(t)+c(t)=1
$$

Trigonometric Interpolant：

$$
X_t=\cos\theta Z+\sin\theta D
$$

满足：

$$
a^2(t)+c^2(t)=1
$$

二者分别对应：

$$
\boxed{
\text{CondOT：线性系数约束}
}
$$

$$
\boxed{
\text{Trigonometric：平方系数约束}
}
$$

这意味着：

CondOT 的条件路径在样本空间中是直线。

Trigonometric Interpolant 的条件路径一般不是直线，而是在由 $Z,D$ 张成的空间中沿三角函数系数演化。

但需要再次强调：

**条件路径的几何形状，不等于边缘 Probability Flow ODE 的轨迹形状。**

即使 CondOT 使用直线条件路径，其边缘 ODE 轨迹也可能弯曲。

---


## 9.8 不同生成模型的统一公式与对照总结


### 9.8.1 首先建立统一公式

所有模型从：

$$
\boxed{
X_t=a(t)Z+c(t)D
}
\tag{9.119}
$$

开始。

定义：

$$
\eta(t,x)=\mathbb E[Z\mid X_t=x]
$$

则：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
\tag{9.120}
$$

定义：

$$
A(t)=\frac{\dot c(t)}{c(t)}
$$

$$
H(t)=
\frac{a^2(t)\dot c(t)}{c(t)}
-a(t)\dot a(t)
$$

则：

$$
\boxed{
b(t,x)=A(t)x+H(t)s(t,x)
}
\tag{9.121}
$$

对应：

$$
\boxed{
\text{ODE}:\qquad
dY_t=b(t,Y_t)dt
}
\tag{9.122}
$$

$$
\boxed{
\text{生成 SDE}:\qquad
dY_t=
[b+\varepsilon s]dt+
\sqrt{2\varepsilon}dW_t
}
\tag{9.123}
$$

反向 SDE 则由该生成 SDE 的时间反演得到。

这套公式就是整个统一框架的核心。


### 9.8.2 不同模型的时间系数对照

以下统一使用：

$$
t:0\to1
$$

表示噪声到数据的生成方向。

| 模型 | $a(t)$：噪声系数 | $c(t)$：数据系数 |
|---|---|---|
| CondOT / Linear FM | $1-t$ | $t$ |
| VP-SDE | $\sqrt{1-\alpha^2(1-t)}$ | $\alpha(1-t)$ |
| OU | $\sqrt{1-e^{-\beta_0(1-t)}}$ | $e^{-\beta_0(1-t)/2}$ |
| VE-SDE | $\sigma(1-t)$ | $1$ |
| Trigonometric | $\cos(\pi t/2)$ | $\sin(\pi t/2)$ |

这里 OU 和 VP 的有限时间端点通常不严格等于标准高斯；VE 还存在初始分布和数据端点的近似问题，不能忽略。


### 9.8.3 不同模型的 Score 对照

对于所有满足：

$$
a(t)>0
$$

的线性高斯模型：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
$$

因此：

| 模型 | Score |
|---|---|
| CondOT | $-\eta/(1-t)$ |
| VP | $-\eta/\sqrt{1-\alpha^2(1-t)}$ |
| OU | $-\eta/\sqrt{1-e^{-\beta_0(1-t)}}$ |
| VE | $-\eta/\sigma(1-t)$ |
| Trigonometric | $-\eta/\cos(\pi t/2)$ |

这说明：

**噪声预测与 Score 的关系不是 VP 或 DDPM 特有的，而是高斯线性插值的普遍性质。**


### 9.8.4 不同模型的边缘速度场对照

令：

$$
\widetilde\beta(t)=\beta(1-t)
$$

$$
\widetilde g(t)=g(1-t)
$$

则：

| 模型 | 边缘速度场 $b(t,x)$ |
|---|---|
| CondOT | $\displaystyle\frac{x-\eta}{t}$ |
| VP | $\displaystyle\frac12\widetilde\beta(x+s)$ |
| OU | $\displaystyle\frac12\beta_0(x+s)$ |
| VE | $\displaystyle\frac12\widetilde g^2s$ |
| Trigonometric | $\displaystyle\frac\pi2\cot(\frac{\pi t}{2})(x+s)$ |

其中：

$$
s=-\frac{\eta}{a}
$$

因此，每个模型的速度场都可以只使用一个噪声预测器计算。


### 9.8.5 不同模型的 Probability Flow ODE

**CondOT：**

$$
\boxed{
\dot Y_t=\frac{Y_t-\eta(t,Y_t)}t
}
\tag{9.124}
$$

**VP：**

$$
\boxed{
\dot Y_t=
\frac12\widetilde\beta(t)
[Y_t+s(t,Y_t)]
}
\tag{9.125}
$$

**OU：**

$$
\boxed{
\dot Y_t=
\frac12\beta_0[Y_t+s(t,Y_t)]
}
\tag{9.126}
$$

**VE：**

$$
\boxed{
\dot Y_t=
\frac12\widetilde g^2(t)s(t,Y_t)
}
\tag{9.127}
$$

**Trigonometric：**

$$
\boxed{
\dot Y_t=
\frac\pi2\cot\left(\frac{\pi t}{2}\right)
[Y_t+s(t,Y_t)]
}
\tag{9.128}
$$

这里所有 ODE 都使用相同的生成时间方向。


### 9.8.6 不同模型的标准随机生成 SDE

对于 CondOT，没有唯一指定的标准 SDE 扩散强度。

可以任意选择满足适定性要求的：

$$
\varepsilon(t)\ge0
$$

并构造：

$$
\boxed{
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
}
$$

对于 VP，选择：

$$
\varepsilon(t)=\frac12\widetilde\beta(t)
$$

得到：

$$
\boxed{
dY_t=
\left[
\frac12\widetilde\beta Y_t+
\widetilde\beta s
\right]dt+
\sqrt{\widetilde\beta}dW_t
}
\tag{9.129}
$$

对于 OU：

$$
\boxed{
dY_t=
\left[
\frac12\beta_0Y_t+\beta_0s
\right]dt+
\sqrt{\beta_0}dW_t
}
\tag{9.130}
$$

对于 VE：

$$
\boxed{
dY_t=
\widetilde g^2s\,dt+
\widetilde g\,dW_t
}
\tag{9.131}
$$

对于 Trigonometric：

$$
\boxed{
dY_t=
\left[
\frac\pi2\cot\theta\,Y_t+
\pi\cot\theta\,s
\right]dt+
\sqrt{\pi\cot\theta}dW_t
}
\tag{9.132}
$$

其中：

$$
\theta=\frac{\pi t}{2}
$$

这些 SDE 的标准扩散强度是从各自的扩散模型结构中恢复的，而不是由线性插值形式唯一决定的。


### 9.8.7 统一框架中最容易混淆的三个问题

**问题一：相同的单时刻边缘分布，是否意味着相同的随机过程？**

不是。

例如：

$$
X_\tau\overset d=
\alpha(\tau)D+\sigma(\tau)Z
$$

只说明对于每个固定 $\tau$，右侧具有与扩散 SDE 相同的概率分布。

但固定一个 $Z$ 构造的随机插值，与由布朗运动驱动的扩散 SDE，通常具有不同的跨时间联合分布。

因此：

$$
\boxed{
\text{边缘分布相同}
\not\Rightarrow
\text{路径分布相同}
}
$$

**问题二：一个噪声预测器是否足以恢复 ODE 和 SDE？**

对于本章的高斯线性插值，在：

$$
a(t)>0
$$

以及所需时间系数非退化的条件下，答案是肯定的。

因为：

$$
\eta\Longrightarrow s
$$

并且：

$$
\eta\Longrightarrow b
$$

因此：

$$
\boxed{
\eta\Longrightarrow\text{ODE/SDE}
}
$$

但这只是数学上的表达能力结论。

实际训练中，噪声预测误差可能经过：

$$
\frac1{a(t)}
$$

等系数放大。

因此，直接速度预测和噪声预测可能具有不同的数值性能。

**问题三：Flow Matching 和 Diffusion 是否完全等价？**

不能这样说。

它们可以使用相同的线性高斯概率路径表示，也可以通过 Score 与速度场转换建立联系。

但仍然可能存在不同的：

- 时间日程。
- 端点耦合。
- 训练目标。
- 采样动力学。
- 数值积分方法。
- 近似误差结构。

因此，更准确的结论是：

$$
\boxed{
\text{Flow Matching 与 Diffusion 可以在随机插值框架下统一描述}
}
$$

而不是：

$$
\text{Flow Matching}=\text{Diffusion}
$$


### 9.8.8 对自动驾驶轨迹规划的进一步启示

考虑条件轨迹生成：

$$
D\in\mathbb R^{T\times d_{\mathrm{traj}}}
$$

其中：

- $T$：未来规划时域内的轨迹点数量。
- $d_{\mathrm{traj}}$：每个轨迹点的表示维度。
- $c_{\mathrm{scene}}$：场景条件编码。

定义：

$$
\boxed{
X_t=a(t)Z+c(t)D
}
\tag{9.133}
$$

并训练：

$$
\eta_\theta(t,X_t,c_{\mathrm{scene}})
$$

则：

$$
\boxed{
s_\theta(t,x,c_{\mathrm{scene}})
=
-\frac{\eta_\theta(t,x,c_{\mathrm{scene}})}{a(t)}
}
\tag{9.134}
$$

以及：

$$
\boxed{
b_\theta(t,x,c_{\mathrm{scene}})
=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)
\eta_\theta(t,x,c_{\mathrm{scene}})
}
\tag{9.135}
$$

因此，可以使用同一个模型实现：

**确定性生成：**

$$
\boxed{
dY_t=b_\theta(t,Y_t,c_{\mathrm{scene}})dt
}
\tag{9.136}
$$

**随机性生成：**

$$
\boxed{
\begin{aligned}
dY_t={}&
[b_\theta(t,Y_t,c_{\mathrm{scene}})
+\varepsilon(t)s_\theta(t,Y_t,c_{\mathrm{scene}})]dt\\
&+\sqrt{2\varepsilon(t)}dW_t
\end{aligned}
}
\tag{9.137}
$$

这意味着，在同一套模型参数下，可以研究：

$$
\varepsilon=0
$$

和：

$$
\varepsilon>0
$$

两种采样策略。

例如：

- 使用 ODE 生成可复现的候选轨迹。
- 使用 SDE 引入额外随机探索。
- 比较不同 $\varepsilon$ 对轨迹多模态性、碰撞率和舒适性的影响。
- 研究训练时的预测目标与推理时的求解器是否匹配。

但对于实际自动驾驶规划，还必须考虑：

$$
\text{动力学可行性}
$$

$$
\text{碰撞约束}
$$

$$
\text{交通规则}
$$

$$
\text{闭环稳定性}
$$

随机插值理论本身并不自动保证这些性质。

---


# 第 9 章总结

本章从统一的单侧线性高斯插值出发：

$$
\boxed{
X_t=a(t)Z+c(t)D
}
$$

首先推导：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
$$

然后推导：

$$
\boxed{
b(t,x)=
\frac{\dot c(t)}{c(t)}x+
\left[
\dot a(t)-\frac{a(t)\dot c(t)}{c(t)}
\right]\eta(t,x)
}
$$

进一步构造：

$$
\boxed{
\begin{aligned}
\text{ODE:}\quad&
dY_t=b\,dt\\[2mm]
\text{Forward SDE:}\quad&
dY_t=(b+\varepsilon s)dt+
\sqrt{2\varepsilon}dW_t\\[2mm]
\text{Reverse SDE:}\quad&
dY_t=(b-\varepsilon s)dt+
\sqrt{2\varepsilon}d\overleftarrow W_t
\end{aligned}
}
\tag{9.138}
$$

其中前向与反向分别指统一生成时间 $t$ 的递增与递减方向，反向式采用反向 Itô 约定。

通过代入不同的：

$$
a(t),\qquad c(t)
$$

我们依次恢复了：

$$
\boxed{
\begin{gathered}
\text{CondOT / Flow Matching}\\
\text{OU / VP-SDE}\\
\text{VE-SDE}\\
\text{Trigonometric Interpolant}
\end{gathered}
}
$$

整个推导说明：

**Flow Matching 与 Diffusion 之间最深层的联系，不在于它们是否使用相同的神经网络或相同的数值采样器，而在于它们都可以通过概率路径、条件期望、速度场与 Score 来描述。**

不过，这种统一需要严格区分：

$$
\boxed{
\text{单时刻边缘分布}
\neq
\text{完整随机过程}
}
$$

$$
\boxed{
\text{数学预测目标等价}
\neq
\text{有限模型训练效果相同}
}
$$

$$
\boxed{
\text{连续时间理论等价}
\neq
\text{有限步数值采样等价}
}
$$

这三点也是将随机插值理论应用于实际 Flow Matching、Diffusion 和轨迹规划模型时，最需要保留的数学认识。


# 附录：数学基础与公式推导

前面第 0～9 章介绍了随机插值、Flow Matching 和 Diffusion 的统一数学框架。附录 A～D 介绍相关数学基础，附录 E 给出符号表与公式索引，附录 F 展开线性插值与生成动力学的详细推导。

全文继续使用以下基本记号：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

其中 $X_0,X_1$ 是端点随机变量，$Z\sim\mathcal N(0,I_d)$ 是独立标准高斯噪声。

在单侧线性插值中，使用：

$$
X_t=a(t)Z+c(t)D
$$

其中 $D\sim p_{\mathrm{data}}$。

除特别说明外，统一采用生成时间：

$$
t=0:\text{噪声端},\qquad t=1:\text{数据端}
$$

附录公式独立编号为：

$$
(A.1),(B.1),(C.1),(D.1),\ldots
$$

---


# 附录 A：概率论与条件期望


## A.1 联合分布、条件分布与 Bayes 公式


### A.1.1 为什么生成模型需要联合分布？

在生成模型中，我们经常同时考虑两个随机变量：

$$
X_0\sim p_0,\qquad X_1\sim p_1
$$

但仅知道它们各自的边缘分布，并不能确定它们之间的关系。

例如：

$$
X_0\sim\mathcal N(0,1)
$$

$$
X_1\sim\mathcal N(0,1)
$$

可以有：

$$
X_1=X_0
$$

也可以有：

$$
X_1=-X_0
$$

还可以令二者相互独立。

这三种情况下，边缘分布完全相同，但联合分布不同。

定义联合分布：

$$
\boxed{
p(x_0,x_1)
}
\tag{A.1}
$$

则边缘分布通过积分得到：

$$
\boxed{
p_0(x_0)=\int p(x_0,x_1)\,dx_1
}
\tag{A.2}
$$

$$
\boxed{
p_1(x_1)=\int p(x_0,x_1)\,dx_0
}
\tag{A.3}
$$

满足指定边缘分布的所有联合分布，称为两个分布之间的**耦合（Coupling）**。

记为：

$$
\boxed{
\Pi(p_0,p_1)
}
\tag{A.4}
$$

因此，随机插值：

$$
X_t=(1-t)X_0+tX_1
$$

不仅取决于两个端点分布，还取决于端点耦合。

这是普通 Flow Matching、OT-CFM 和 Rectified Flow 之间的重要区别之一。


### A.1.2 条件概率分布

根据联合分布，可以定义：

$$
\boxed{
p(x_1\mid x_0)
=
\frac{p(x_0,x_1)}{p_0(x_0)}
}
\tag{A.5}
$$

其中要求：

$$
p_0(x_0)>0
$$

类似地：

$$
p(x_0\mid x_1)
=
\frac{p(x_0,x_1)}{p_1(x_1)}
$$

由联合分布分解：

$$
\boxed{
p(x_0,x_1)
=
p(x_1\mid x_0)p_0(x_0)
}
\tag{A.6}
$$

也可以写成：

$$
p(x_0,x_1)
=
p(x_0\mid x_1)p_1(x_1)
$$

这两种分解方式对应不同的条件生成方向。


### A.1.3 Bayes 公式

由：

$$
p(x_0,x_1)
=
p(x_1\mid x_0)p_0(x_0)
$$

以及：

$$
p(x_0,x_1)
=
p(x_0\mid x_1)p_1(x_1)
$$

得到：

$$
\boxed{
p(x_0\mid x_1)
=
\frac{p(x_1\mid x_0)p_0(x_0)}
{p_1(x_1)}
}
\tag{A.7}
$$

在 Diffusion 中，这一公式用于建立：

$$
p(x_0\mid x_t)
$$

与：

$$
p(x_t\mid x_0)
$$

之间的联系。

例如，高斯加噪：

$$
X_t=aZ+cD
$$

具有已知条件分布：

$$
p_t(x\mid D)
=
\mathcal N(x;cD,a^2I)
$$

因此：

$$
\boxed{
p(D\mid X_t=x)
=
\frac{p_t(x\mid D)p_{\mathrm{data}}(D)}
{p_t(x)}
}
\tag{A.8}
$$

这就是数据去噪器所对应的后验分布。

---


## A.2 条件期望及其 MSE 最优性


### A.2.1 条件期望的定义

给定随机变量 $X,Y$，条件期望：

$$
\boxed{
m(x)=\mathbb E[Y\mid X=x]
}
\tag{A.9}
$$

表示在观察到 $X=x$ 后，对 $Y$ 的条件平均。

如果条件概率密度存在：

$$
m(x)=\int y\,p(y\mid x)\,dy
$$

注意，条件期望一般不是某一个真实样本。

例如：

$$
Y\mid X=x
$$

可能具有两个模态：

$$
Y=-1,\qquad Y=1
$$

且概率相等。

那么：

$$
\mathbb E[Y\mid X=x]=0
$$

但：

$$
0
$$

可能不是该条件分布的高概率区域。

这也是直接 MSE 轨迹回归容易出现均值化现象的数学原因。


<a id="app-a-2-2"></a>

### A.2.2 为什么条件期望是 MSE 的最优预测器？

考虑：

$$
\boxed{
\mathcal L(f)=
\mathbb E[\|f(X)-Y\|^2]
}
\tag{A.10}
$$

希望求：

$$
f^*=\arg\min_f\mathcal L(f)
$$

定义：

$$
m(X)=\mathbb E[Y\mid X]
$$

将误差分解为：

$$
f(X)-Y=
[f(X)-m(X)]+[m(X)-Y]
$$

平方展开：

$$
\begin{aligned}
\|f(X)-Y\|^2
={}&
\|f(X)-m(X)\|^2\\
&+\|m(X)-Y\|^2\\
&+2[f(X)-m(X)]^\top[m(X)-Y]
\end{aligned}
$$

对两边取期望。

关键是交叉项：

$$
\mathbb E\left[
[f(X)-m(X)]^\top[m(X)-Y]
\right]
$$

使用全期望公式：

$$
\mathbb E[U]
=
\mathbb E[\mathbb E[U\mid X]]
$$

得到：

$$
\begin{aligned}
&\mathbb E\left[
[f(X)-m(X)]^\top[m(X)-Y]
\right]\\
&=
\mathbb E\left[
[f(X)-m(X)]^\top
\mathbb E[m(X)-Y\mid X]
\right]
\end{aligned}
$$

因为：

$$
\mathbb E[m(X)-Y\mid X]
=
m(X)-\mathbb E[Y\mid X]
=0
$$

所以交叉项为零。

最终：

$$
\boxed{
\mathcal L(f)
=
\mathbb E\|f(X)-m(X)\|^2
+
\mathbb E\|m(X)-Y\|^2
}
\tag{A.11}
$$

第二项与 $f$ 无关。

因此：

$$
\boxed{
f^*(x)=\mathbb E[Y\mid X=x]
}
\tag{A.12}
$$

这是 Flow Matching、噪声预测和去噪器训练的共同数学基础。


### A.2.3 应用于 Flow Matching

随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

定义：

$$
V_t=\dot X_t
$$

训练：

$$
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|v_\theta(t,X_t)-V_t\|^2
$$

因此：

$$
\boxed{
v^*(t,x)=
\mathbb E[\dot X_t\mid X_t=x]
}
\tag{A.13}
$$

这正是随机插值的边缘速度场。

需要注意，模型学到的不是每条条件插值路径的真实速度，而是所有经过当前状态 $x$ 的条件速度的平均。

因此，即使：

$$
\dot X_t=X_1-X_0
$$

对每条条件直线而言是常数，也不能推出：

$$
v^*(t,x)
$$

沿生成 ODE 轨迹保持常数。


### A.2.4 应用于噪声预测

考虑：

$$
X_t=a(t)Z+c(t)D
$$

训练：

$$
\mathcal L_\eta=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
$$

最优预测器：

$$
\boxed{
\eta^*(t,x)=\mathbb E[Z\mid X_t=x]
}
\tag{A.14}
$$

然后：

$$
\boxed{
s(t,x)=-\frac{\eta^*(t,x)}{a(t)}
}
\tag{A.15}
$$

因此，噪声预测的最优性直接建立在条件期望的 MSE 最优性之上。

---


## A.3 高斯分布与高斯随机变量合并


### A.3.1 多元高斯分布

设：

$$
X\sim\mathcal N(\mu,\Sigma)
$$

其中：

$$
X\in\mathbb R^d
$$

$$
\mu\in\mathbb R^d
$$

$$
\Sigma\in\mathbb R^{d\times d}
$$

且 $\Sigma$ 对称正定。

概率密度为：

$$
\boxed{
p(x)=
\frac1{(2\pi)^{d/2}|\Sigma|^{1/2}}
\exp\left[
-\frac12(x-\mu)^\top\Sigma^{-1}(x-\mu)
\right]
}
\tag{A.16}
$$

对于标准高斯：

$$
Z\sim\mathcal N(0,I_d)
$$

则：

$$
p(z)=
\frac1{(2\pi)^{d/2}}
\exp\left(-\frac12\|z\|^2\right)
$$


### A.3.2 高斯随机变量的线性变换

如果：

$$
X\sim\mathcal N(\mu,\Sigma)
$$

定义：

$$
Y=AX+b
$$

则：

$$
\boxed{
Y\sim\mathcal N(A\mu+b,A\Sigma A^\top)
}
\tag{A.17}
$$

这是因为：

$$
\mathbb E[Y]=A\mu+b
$$

以及：

$$
\begin{aligned}
\operatorname{Cov}(Y)
&=
\mathbb E[(Y-\mathbb E[Y])(Y-\mathbb E[Y])^\top]\\
&=
A\Sigma A^\top
\end{aligned}
$$


<a id="app-a-3-3"></a>

### A.3.3 独立高斯随机变量相加

设：

$$
X\sim\mathcal N(\mu_X,\Sigma_X)
$$

$$
Y\sim\mathcal N(\mu_Y,\Sigma_Y)
$$

且：

$$
X\perp Y
$$

则：

$$
\boxed{
aX+bY
\sim
\mathcal N(
a\mu_X+b\mu_Y,
a^2\Sigma_X+b^2\Sigma_Y
)
}
\tag{A.18}
$$

例如：

$$
Z_1,Z_2\overset{\mathrm{i.i.d.}}{\sim}
\mathcal N(0,I_d)
$$

则：

$$
\boxed{
aZ_1+bZ_2
\sim
\mathcal N(0,(a^2+b^2)I_d)
}
\tag{A.19}
$$

因此：

$$
aZ_1+bZ_2
\overset d=
\sqrt{a^2+b^2}Z
$$

其中：

$$
Z\sim\mathcal N(0,I_d)
$$

这是推导随机插值中高斯噪声合并的基本公式。

需要强调：这里要求 $Z_1,Z_2$ 独立。如果不独立，协方差还会包含交叉项。


### A.3.4 高斯条件分布

设联合随机变量：

$$
\begin{bmatrix}
X\\Y
\end{bmatrix}
\sim
\mathcal N
\left(
\begin{bmatrix}
\mu_X\\\mu_Y
\end{bmatrix},
\begin{bmatrix}
\Sigma_{XX}&\Sigma_{XY}\\
\Sigma_{YX}&\Sigma_{YY}
\end{bmatrix}
\right)
$$

则：

$$
\boxed{
\mathbb E[X\mid Y=y]
=
\mu_X+
\Sigma_{XY}\Sigma_{YY}^{-1}(y-\mu_Y)
}
\tag{A.20}
$$

条件协方差为：

$$
\boxed{
\operatorname{Cov}(X\mid Y)
=
\Sigma_{XX}-
\Sigma_{XY}\Sigma_{YY}^{-1}\Sigma_{YX}
}
\tag{A.21}
$$

例如，设：

$$
D\sim\mathcal N(0,\sigma_D^2I)
$$

$$
Z\sim\mathcal N(0,I)
$$

且二者独立。

令：

$$
X_t=aZ+cD
$$

则：

$$
\operatorname{Cov}(D,X_t)=c\sigma_D^2I
$$

$$
\operatorname{Cov}(X_t)=
(a^2+c^2\sigma_D^2)I
$$

所以：

$$
\boxed{
\mathbb E[D\mid X_t=x]
=
\frac{c\sigma_D^2}
{a^2+c^2\sigma_D^2}x
}
\tag{A.22}
$$

这个例子说明，当数据本身也是高斯时，最优去噪器具有线性形式。

对于真实图像或轨迹数据，后验通常不再是高斯，因此需要神经网络学习。

---


## A.4 KL 散度与数据处理不等式


### A.4.1 KL 散度的定义

设两个概率密度：

$$
p(x),\qquad q(x)
$$

且：

$$
p\ll q
$$

即 $p$ 对 $q$ 绝对连续。

KL 散度定义为：

$$
\boxed{
D_{\mathrm{KL}}(p\|q)
=
\int p(x)\log\frac{p(x)}{q(x)}dx
}
\tag{A.23}
$$

也可以写成：

$$
D_{\mathrm{KL}}(p\|q)
=
\mathbb E_{X\sim p}
\left[
\log\frac{p(X)}{q(X)}
\right]
$$

KL 散度具有：

$$
\boxed{
D_{\mathrm{KL}}(p\|q)\ge0
}
\tag{A.24}
$$

并且：

$$
D_{\mathrm{KL}}(p\|q)=0
$$

当且仅当：

$$
p=q
$$

几乎处处成立。

KL 散度不是对称的：

$$
D_{\mathrm{KL}}(p\|q)
\neq
D_{\mathrm{KL}}(q\|p)
$$

因此它不是严格意义上的距离。


### A.4.2 KL 非负性的推导

使用不等式：

$$
\log u\le u-1
$$

令：

$$
u=\frac{q(x)}{p(x)}
$$

则：

$$
-\log\frac{q(x)}{p(x)}
\ge
1-\frac{q(x)}{p(x)}
$$

两边乘以 $p(x)$ 并积分：

$$
\begin{aligned}
D_{\mathrm{KL}}(p\|q)
&=
-\int p\log\frac qp\,dx\\
&\ge
\int\left(p-q\right)dx\\
&=1-1\\
&=0
\end{aligned}
$$

因此：

$$
D_{\mathrm{KL}}(p\|q)\ge0
$$


### A.4.3 数据处理不等式

设：

$$
X\sim P,\qquad X\sim Q
$$

表示两种不同的输入分布。

通过同一个随机映射：

$$
K(y\mid x)
$$

得到输出分布：

$$
P_Y(y)=\int K(y\mid x)P(x)dx
$$

$$
Q_Y(y)=\int K(y\mid x)Q(x)dx
$$

则：

$$
\boxed{
D_{\mathrm{KL}}(P_Y\|Q_Y)
\le
D_{\mathrm{KL}}(P\|Q)
}
\tag{A.25}
$$

这就是数据处理不等式。

它的直观含义是：

**对两个概率分布进行相同的数据处理，不会增加它们的 KL 可区分程度。**


### A.4.4 数据处理不等式的推导

定义两个联合分布：

$$
P_{XY}(x,y)=P(x)K(y\mid x)
$$

$$
Q_{XY}(x,y)=Q(x)K(y\mid x)
$$

计算联合 KL：

$$
\begin{aligned}
D_{\mathrm{KL}}(P_{XY}\|Q_{XY})
&=
\iint P(x)K(y\mid x)
\log
\frac{P(x)K(y\mid x)}
{Q(x)K(y\mid x)}
\,dxdy\\
&=
\iint P(x)K(y\mid x)
\log\frac{P(x)}{Q(x)}
\,dxdy\\
&=
D_{\mathrm{KL}}(P\|Q)
\end{aligned}
$$

另一方面，利用 KL 的链式分解：

$$
\begin{aligned}
D_{\mathrm{KL}}(P_{XY}\|Q_{XY})
={}&
D_{\mathrm{KL}}(P_Y\|Q_Y)\\
&+
\mathbb E_{Y\sim P_Y}
D_{\mathrm{KL}}(P_{X\mid Y}\|Q_{X\mid Y})
\end{aligned}
$$

由于条件 KL 非负：

$$
D_{\mathrm{KL}}(P_{XY}\|Q_{XY})
\ge
D_{\mathrm{KL}}(P_Y\|Q_Y)
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(P_Y\|Q_Y)
\le
D_{\mathrm{KL}}(P\|Q)
}
$$


<a id="app-a-4-5"></a>

### A.4.5 为什么它对 SDE 生成模型重要？

设：

$$
\mathbb P
$$

表示真实 SDE 的路径空间分布。

设：

$$
\widehat{\mathbb P}
$$

表示学习得到的 SDE 的路径空间分布。

从完整路径中提取终点：

$$
F((X_t)_{t\in[0,1]})=X_1
$$

这是一个确定性映射。

因此：

$$
\boxed{
D_{\mathrm{KL}}(p_1\|\widehat p_1)
\le
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
}
\tag{A.26}
$$

这意味着：

**如果能够控制整个随机过程的路径空间 KL，就能够控制最终生成分布的 KL。**

这正是 Girsanov 定理在随机生成模型误差分析中的重要作用。

---


# 附录 B：微积分与偏微分方程


## B.1 梯度、散度、拉普拉斯与分部积分


### B.1.1 梯度

设标量函数：

$$
f:\mathbb R^d\to\mathbb R
$$

梯度定义为：

$$
\boxed{
\nabla f(x)=
\begin{bmatrix}
\frac{\partial f}{\partial x_1}\\
\vdots\\
\frac{\partial f}{\partial x_d}
\end{bmatrix}
}
\tag{B.1}
$$

梯度表示函数在空间中的局部变化方向。

例如：

$$
f(x)=\frac12\|x\|^2
$$

则：

$$
\nabla f(x)=x
$$

对于概率密度：

$$
p(x)
$$

Score 定义为：

$$
\boxed{
s(x)=\nabla\log p(x)
}
\tag{B.2}
$$

因此：

$$
s(x)=\frac{\nabla p(x)}{p(x)}
$$

也就是：

$$
\boxed{
p(x)s(x)=\nabla p(x)
}
\tag{B.3}
$$

这个恒等式是统一 ODE/SDE 推导的核心。


### B.1.2 散度

设向量场：

$$
v:\mathbb R^d\to\mathbb R^d
$$

其中：

$$
v(x)=
\begin{bmatrix}
v_1(x)\\
\vdots\\
v_d(x)
\end{bmatrix}
$$

散度定义为：

$$
\boxed{
\nabla\cdot v(x)
=
\sum_{i=1}^d
\frac{\partial v_i}{\partial x_i}
}
\tag{B.4}
$$

散度描述向量场在局部的体积扩张或收缩趋势。

例如：

$$
v(x)=x
$$

则：

$$
\nabla\cdot v=d
$$

如果：

$$
v(x)=-x
$$

则：

$$
\nabla\cdot v=-d
$$

前者对应局部扩张，后者对应局部收缩。


### B.1.3 拉普拉斯算子

拉普拉斯算子定义为：

$$
\boxed{
\Delta f(x)=
\nabla\cdot\nabla f(x)
=
\sum_{i=1}^d
\frac{\partial^2f}{\partial x_i^2}
}
\tag{B.5}
$$

例如：

$$
f(x)=\frac12\|x\|^2
$$

则：

$$
\Delta f=d
$$

在概率论中，拉普拉斯算子与布朗运动引起的扩散密切相关。


### B.1.4 乘积求导公式

对于标量函数：

$$
p(x)
$$

和向量场：

$$
v(x)
$$

有：

$$
\boxed{
\nabla\cdot(pv)
=
v\cdot\nabla p+
p\nabla\cdot v
}
\tag{B.6}
$$

证明只需逐坐标展开：

$$
\begin{aligned}
\nabla\cdot(pv)
&=
\sum_i\frac{\partial(pv_i)}{\partial x_i}\\
&=
\sum_i
\left[
v_i\frac{\partial p}{\partial x_i}
+
p\frac{\partial v_i}{\partial x_i}
\right]\\
&=
v\cdot\nabla p+p\nabla\cdot v
\end{aligned}
$$


### B.1.5 多维分部积分

对于足够光滑的函数 $f$ 和向量场 $v$，在边界项消失的条件下：

$$
\boxed{
\int_{\mathbb R^d}
f(x)\nabla\cdot v(x)\,dx
=
-\int_{\mathbb R^d}
\nabla f(x)\cdot v(x)\,dx
}
\tag{B.7}
$$

例如：

$$
\int f\nabla\cdot(pv)\,dx
=
-\int p\,v\cdot\nabla f\,dx
$$

这是从弱形式推导连续性方程的重要工具。

---


## B.2 连续性方程与 Liouville 方程


### B.2.1 从概率质量守恒出发

考虑一个随时间变化的概率密度：

$$
p_t(x)
$$

以及速度场：

$$
b(t,x)
$$

假设概率质量沿速度场流动。

对于空间区域：

$$
\Omega\subset\mathbb R^d
$$

其中的概率质量为：

$$
M_\Omega(t)=\int_\Omega p_t(x)dx
$$

质量变化率等于穿过边界的净流出量的相反数：

$$
\frac{d}{dt}M_\Omega(t)
=
-\int_{\partial\Omega}
p_tb\cdot n\,dS
$$

其中：

$$
n
$$

表示区域边界的外法向量。

使用散度定理：

$$
\int_{\partial\Omega}
p_tb\cdot n\,dS
=
\int_\Omega\nabla\cdot(p_tb)dx
$$

得到：

$$
\int_\Omega
\left[
\partial_tp_t+\nabla\cdot(p_tb)
\right]dx=0
$$

由于区域 $\Omega$ 可以任意选择，因此：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
\tag{B.8}
$$

这就是连续性方程。


<a id="app-b-2-2"></a>

### B.2.2 从随机插值推导连续性方程

设：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

定义：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{B.9}
$$

考虑任意光滑测试函数：

$$
\varphi:\mathbb R^d\to\mathbb R
$$

根据链式法则：

$$
\frac{d}{dt}\varphi(X_t)
=
\nabla\varphi(X_t)\cdot\dot X_t
$$

取期望：

$$
\frac{d}{dt}\mathbb E[\varphi(X_t)]
=
\mathbb E[
\nabla\varphi(X_t)\cdot\dot X_t
]
$$

使用条件期望：

$$
\begin{aligned}
\mathbb E[
\nabla\varphi(X_t)\cdot\dot X_t
]
&=
\mathbb E[
\nabla\varphi(X_t)\cdot
\mathbb E[\dot X_t\mid X_t]
]\\
&=
\mathbb E[
\nabla\varphi(X_t)\cdot b(t,X_t)
]
\end{aligned}
$$

因此：

$$
\frac{d}{dt}
\int\varphi(x)p_t(x)dx
=
\int\nabla\varphi(x)\cdot b(t,x)p_t(x)dx
$$

左侧：

$$
\int\varphi(x)\partial_tp_t(x)dx
$$

右侧通过分部积分：

$$
-\int\varphi(x)\nabla\cdot(p_tb)dx
$$

所以：

$$
\int\varphi(x)
\left[
\partial_tp_t+\nabla\cdot(p_tb)
\right]dx=0
$$

由于 $\varphi$ 任意：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
\tag{B.10}
$$

这就证明了随机插值诱导的边缘概率密度满足连续性方程。


上述测试函数可取光滑紧支撑函数，以消去边界项；交换时间求导与期望需要足够的可积性和正则性。得到相同连续性方程后，还需相同初始条件及解的唯一性，才能推出 ODE 与随机插值的边缘密度一致。


### B.2.3 Liouville 方程

考虑确定性 ODE：

$$
\boxed{
\frac{dY_t}{dt}=b(t,Y_t)
}
\tag{B.11}
$$

如果：

$$
Y_0\sim p_0
$$

则其概率密度满足：

$$
\boxed{
\partial_tp_t+
\nabla\cdot(p_tb)=0
}
\tag{B.12}
$$

在确定性动力系统中，这通常称为 Liouville 方程。

因此：

$$
\boxed{
\text{连续性方程}
=
\text{确定性 ODE 的概率密度演化方程}
}
$$

这就是 Probability Flow ODE 的数学基础。


<a id="app-b-2-4"></a>

### B.2.4 沿 ODE 轨迹的密度变化

从：

$$
\partial_tp+\nabla\cdot(pb)=0
$$

利用乘积法则：

$$
\partial_tp+b\cdot\nabla p+p\nabla\cdot b=0
$$

因此：

$$
\partial_tp+b\cdot\nabla p
=
-p\nabla\cdot b
$$

沿着 ODE：

$$
\dot Y_t=b(t,Y_t)
$$

使用链式法则：

$$
\frac{d}{dt}p_t(Y_t)
=
\partial_tp_t(Y_t)+
\nabla p_t(Y_t)\cdot\dot Y_t
$$

代入：

$$
\dot Y_t=b(t,Y_t)
$$

得到：

$$
\frac{d}{dt}p_t(Y_t)
=
-p_t(Y_t)\nabla\cdot b(t,Y_t)
$$

两边除以：

$$
p_t(Y_t)
$$

得到：

$$
\boxed{
\frac{d}{dt}\log p_t(Y_t)
=
-\nabla\cdot b(t,Y_t)
}
\tag{B.13}
$$

积分：

$$
\boxed{
\log p_1(Y_1)
=
\log p_0(Y_0)
-
\int_0^1
\nabla\cdot b(t,Y_t)dt
}
\tag{B.14}
$$

这就是 Continuous Normalizing Flow 的 likelihood 公式。

---


## B.3 Fokker–Planck 方程


### B.3.1 一般 Itô SDE

考虑：

$$
\boxed{
dX_t=f(t,X_t)dt+G(t,X_t)dW_t
}
\tag{B.15}
$$

其中：

$$
X_t\in\mathbb R^d
$$

$$
f(t,x)\in\mathbb R^d
$$

$$
G(t,x)\in\mathbb R^{d\times m}
$$

$$
W_t\in\mathbb R^m
$$

定义扩散协方差矩阵：

$$
\boxed{
Q(t,x)=G(t,x)G^\top(t,x)
}
\tag{B.16}
$$

则概率密度满足：

$$
\boxed{
\partial_tp_t
=
-\sum_i\partial_{x_i}(f_ip_t)
+
\frac12\sum_{i,j}
\partial_{x_i}\partial_{x_j}(Q_{ij}p_t)
}
\tag{B.17}
$$

这就是一般形式的 Fokker–Planck 方程。


### B.3.2 各向同性扩散

如果：

$$
G(t,x)=g(t)I_d
$$

则：

$$
Q(t,x)=g^2(t)I_d
$$

因此：

$$
\boxed{
\partial_tp_t=
-\nabla\cdot(p_tf)+
\frac12g^2(t)\Delta p_t
}
\tag{B.18}
$$

如果进一步定义：

$$
g(t)=\sqrt{2\varepsilon(t)}
$$

则：

$$
\boxed{
\partial_tp_t=
-\nabla\cdot(p_tf)+
\varepsilon(t)\Delta p_t
}
\tag{B.19}
$$

这是正文使用最多的形式。


<a id="app-b-3-3"></a>

### B.3.3 为什么可以添加 Score 而不改变概率路径？

设目标概率路径满足：

$$
\partial_tp_t=-\nabla\cdot(p_tb)
$$

希望构造：

$$
dY_t=f\,dt+\sqrt{2\varepsilon}dW_t
$$

对应：

$$
\partial_tp_t=
-\nabla\cdot(p_tf)+\varepsilon\Delta p_t
$$

令：

$$
\boxed{
f=b+\varepsilon s
}
\tag{B.20}
$$

其中：

$$
s=\nabla\log p_t
$$

则：

$$
p_ts=\nabla p_t
$$

因此：

$$
\begin{aligned}
-\nabla\cdot(p_tf)+\varepsilon\Delta p_t
&=
-\nabla\cdot[p_t(b+\varepsilon s)]
+\varepsilon\Delta p_t\\
&=
-\nabla\cdot(p_tb)
-\varepsilon\nabla\cdot(p_ts)
+\varepsilon\Delta p_t\\
&=
-\nabla\cdot(p_tb)
\end{aligned}
$$

最终：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{B.21}
$$

与：

$$
dY_t=b(t,Y_t)dt
$$

具有相同的边缘概率路径。

这里的结论需要相应的初始条件与方程适定性保证。

---


## B.4 Hamilton–Jacobi 方程


### B.4.1 为什么最优传输会出现 Hamilton–Jacobi 方程？

在最优传输中，我们希望寻找从：

$$
p_0
$$

到：

$$
p_1
$$

的最优概率流动。

Benamou–Brenier 动态最优传输问题为：

$$
\boxed{
\min_{p_t,v_t}
\int_0^1\int
\frac12\|v_t(x)\|^2p_t(x)\,dxdt
}
\tag{B.22}
$$

约束：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tv_t)=0
}
\tag{B.23}
$$

以及：

$$
p_{t=0}=p_0,\qquad p_{t=1}=p_1
$$

这里：

$$
v_t(x)
$$

表示概率质量的速度场。


### B.4.2 引入拉格朗日乘子

定义：

$$
\phi(t,x)
$$

作为连续性方程的拉格朗日乘子。

构造：

$$
\begin{aligned}
\mathcal J(p,v,\phi)
=
\int_0^1\int
\bigg[
\frac12p\|v\|^2+
\phi(\partial_tp+\nabla\cdot(pv))
\bigg]dxdt
\end{aligned}
\tag{B.24}
$$

对时间和空间进行分部积分，忽略固定端点项：

$$
\mathcal J
=
\int_0^1\int
p\left[
\frac12\|v\|^2-
\partial_t\phi-
v\cdot\nabla\phi
\right]dxdt
+\text{端点项}
$$

现在对速度场 $v$ 求变分。

被积函数关于 $v$ 的部分为：

$$
\frac12\|v\|^2-v\cdot\nabla\phi
$$

求导：

$$
v-\nabla\phi=0
$$

因此：

$$
\boxed{
v=\nabla\phi
}
\tag{B.25}
$$

再代入：

$$
\frac12\|\nabla\phi\|^2-
\nabla\phi\cdot\nabla\phi
=
-\frac12\|\nabla\phi\|^2
$$

得到关于 $p$ 的变分条件：

$$
-\partial_t\phi-
\frac12\|\nabla\phi\|^2=0
$$

因此：

$$
\boxed{
\partial_t\phi+
\frac12\|\nabla\phi\|^2=0
}
\tag{B.26}
$$

这就是 Hamilton–Jacobi 方程。


### B.4.3 物理与生成模型中的含义

Hamilton–Jacobi 方程说明，在二次动能最优传输中，最优速度场具有梯度形式：

$$
v=\nabla\phi
$$

并且势函数满足：

$$
\partial_t\phi+\frac12\|\nabla\phi\|^2=0
$$

因此，最优传输不仅要求概率质量守恒，还要求速度场满足额外的最优性条件。

而普通 Flow Matching 只通过监督学习：

$$
v(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

并不自动保证：

$$
v=\nabla\phi
$$

也不自动保证满足 Hamilton–Jacobi 方程。

所以：

$$
\boxed{
\text{Flow Matching 的概率路径}
\neq
\text{必然是最优传输路径}
}
$$

---


# 附录 C：随机过程与随机微积分


## C.1 Brownian Motion 与 Brownian Bridge


### C.1.1 Brownian Motion 的定义

标准布朗运动：

$$
\boxed{
W_t,\qquad t\ge0
}
$$

满足：

1. $W_0=0$。
2. 具有连续样本路径。
3. 增量独立。
4. 对 $0\le s<t$：

$$
\boxed{
W_t-W_s\sim\mathcal N(0,(t-s)I_d)
}
\tag{C.1}
$$

因此：

$$
\mathbb E[W_t]=0
$$

$$
\boxed{
\operatorname{Cov}(W_s,W_t)=\min(s,t)I_d
}
\tag{C.2}
$$

特别地：

$$
W_t\sim\mathcal N(0,tI_d)
$$


### C.1.2 为什么布朗运动不可微？

考虑差商：

$$
\frac{W_{t+h}-W_t}{h}
$$

由于：

$$
W_{t+h}-W_t\sim\mathcal N(0,hI_d)
$$

所以：

$$
\frac{W_{t+h}-W_t}{h}
\sim
\mathcal N\left(0,\frac1hI_d\right)
$$

当：

$$
h\to0
$$

时，差商的方差：

$$
\frac1h\to\infty
$$

因此，布朗运动不能像普通光滑函数那样进行时间求导。

这也是为什么 SDE 需要 Itô 积分，而不能简单地将：

$$
dW_t
$$

理解为普通微分。


### C.1.3 Brownian Bridge 的定义

Brownian Bridge 是被约束在指定终点的布朗运动。

最简单的零端点 Brownian Bridge 为：

$$
\boxed{
B_t=W_t-tW_1,\qquad t\in[0,1]
}
\tag{C.3}
$$

显然：

$$
B_0=0
$$

$$
B_1=W_1-W_1=0
$$

因此，它从零出发，并在时间 1 回到零。


<a id="app-c-1-4"></a>

### C.1.4 Brownian Bridge 的协方差

由：

$$
B_t=W_t-tW_1
$$

计算：

$$
\begin{aligned}
\operatorname{Cov}(B_s,B_t)
={}&
\operatorname{Cov}(W_s,W_t)\\
&-t\operatorname{Cov}(W_s,W_1)\\
&-s\operatorname{Cov}(W_1,W_t)\\
&+st\operatorname{Cov}(W_1,W_1)
\end{aligned}
$$

由于：

$$
\operatorname{Cov}(W_s,W_t)=\min(s,t)I
$$

$$
\operatorname{Cov}(W_s,W_1)=sI
$$

$$
\operatorname{Cov}(W_1,W_t)=tI
$$

因此：

$$
\begin{aligned}
\operatorname{Cov}(B_s,B_t)
&=
[\min(s,t)-st-st+st]I\\
&=
[\min(s,t)-st]I
\end{aligned}
$$

最终：

$$
\boxed{
\operatorname{Cov}(B_s,B_t)
=
[\min(s,t)-st]I_d
}
\tag{C.4}
$$

当：

$$
s=t
$$

时：

$$
\boxed{
\operatorname{Var}(B_t)=t(1-t)I_d
}
\tag{C.5}
$$

因此：

$$
\boxed{
B_t\sim\mathcal N(0,t(1-t)I_d)
}
\tag{C.6}
$$


### C.1.5 任意端点的 Brownian Bridge

给定两个固定端点：

$$
x_0,\qquad x_1
$$

定义：

$$
\boxed{
X_t=(1-t)x_0+tx_1+B_t
}
\tag{C.7}
$$

则：

$$
X_0=x_0
$$

$$
X_1=x_1
$$

并且：

$$
\boxed{
X_t\sim
\mathcal N(
(1-t)x_0+tx_1,
t(1-t)I_d
)
}
\tag{C.8}
$$

因此，在单时刻边缘分布层面，可以写成：

$$
\boxed{
X_t\overset d=
(1-t)x_0+tx_1+
\sqrt{t(1-t)}Z
}
\tag{C.9}
$$

其中：

$$
Z\sim\mathcal N(0,I_d)
$$

但是需要注意：

$$
B_t
$$

是一个具有时间相关结构的高斯过程。

而：

$$
\sqrt{t(1-t)}Z
$$

如果所有时间使用同一个 $Z$，则其跨时间协方差为：

$$
\sqrt{s(1-s)t(1-t)}I_d
$$

这与 Brownian Bridge 的：

$$
[\min(s,t)-st]I_d
$$

不同。

因此：

$$
\boxed{
\text{固定高斯噪声插值}
\neq
\text{Brownian Bridge 随机过程}
}
$$

二者只具有相同的单时刻条件边缘分布。

---


## C.2 Itô 积分、Itô 公式与 Itô 等距


### C.2.1 Itô 积分

考虑：

$$
\int_0^T H_t\,dW_t
$$

其中：

$$
H_t
$$

是适应于布朗运动过滤族的随机过程，并满足相应平方可积条件。

Itô 积分可以通过左端点和式理解：

$$
\boxed{
\int_0^T H_t\,dW_t
=
L^2\!-\!\lim_{\|\Pi\|\to0}
\sum_{k=0}^{N-1}
H_{t_k}(W_{t_{k+1}}-W_{t_k})
}
\tag{C.10}
$$

这里：

$$
\Pi=\{0=t_0<\cdots<t_N=T\}
$$

表示时间划分。

左端点选择非常重要，因为：

$$
H_{t_k}
$$

不能依赖未来布朗增量：

$$
W_{t_{k+1}}-W_{t_k}
$$

这使 Itô 积分具有鞅性质。


### C.2.2 Itô 等距

对于满足条件的 $H_t$：

$$
\boxed{
\mathbb E
\left[
\left(
\int_0^T H_t\,dW_t
\right)^2
\right]
=
\mathbb E
\int_0^T H_t^2dt
}
\tag{C.11}
$$

这是 Itô 等距。

它可以用于计算线性 SDE 的协方差。

例如：

$$
X_T=\int_0^Tg(t)dW_t
$$

其中 $g(t)$ 为确定性函数。

则：

$$
\mathbb E[X_T]=0
$$

并且：

$$
\boxed{
\operatorname{Var}(X_T)=
\int_0^Tg^2(t)dt
}
\tag{C.12}
$$

这就是 VP-SDE 和 VE-SDE 解析高斯条件分布的数学基础。


### C.2.3 Itô 公式

考虑一维 SDE：

$$
dX_t=f(t,X_t)dt+g(t,X_t)dW_t
$$

设：

$$
\varphi(t,x)
$$

具有足够的光滑性。

Itô 公式为：

$$
\boxed{
\begin{aligned}
d\varphi(t,X_t)
={}&
\left[
\partial_t\varphi+
f\partial_x\varphi+
\frac12g^2\partial_{xx}\varphi
\right]dt\\
&+
g\partial_x\varphi\,dW_t
\end{aligned}
}
\tag{C.13}
$$

与普通链式法则相比，多出了：

$$
\boxed{
\frac12g^2\partial_{xx}\varphi\,dt
}
$$

这一项来自布朗运动的二次变差。


### C.2.4 为什么会出现二阶导数项？

对：

$$
\varphi(X_t)
$$

做二阶 Taylor 展开：

$$
d\varphi
\approx
\varphi'(X_t)dX_t+
\frac12\varphi''(X_t)(dX_t)^2
$$

代入：

$$
dX_t=fdt+gdW_t
$$

得到：

$$
(dX_t)^2
=
f^2(dt)^2+
2fg\,dt\,dW_t+
g^2(dW_t)^2
$$

在 Itô 微积分中：

$$
(dt)^2=0
$$

$$
dt\,dW_t=0
$$

$$
(dW_t)^2=dt
$$

因此：

$$
(dX_t)^2=g^2dt
$$

最终：

$$
d\varphi
=
\varphi'dX_t+
\frac12g^2\varphi''dt
$$

这就解释了 Itô 公式中的二阶导数项。


### C.2.5 多维 Itô 公式

对于：

$$
dX_t=f(t,X_t)dt+G(t,X_t)dW_t
$$

定义：

$$
Q=GG^\top
$$

则：

$$
\boxed{
\begin{aligned}
d\varphi(t,X_t)
={}&
\left[
\partial_t\varphi+
f\cdot\nabla\varphi+
\frac12\operatorname{Tr}(Q\nabla^2\varphi)
\right]dt\\
&+
\nabla\varphi^\top G\,dW_t
\end{aligned}
}
\tag{C.14}
$$

其中：

$$
\nabla^2\varphi
$$

是 Hessian 矩阵。


### C.2.6 从 Itô 公式推导 Fokker–Planck 方程

对式 (C.14) 取期望。

由于 Itô 积分的期望为零：

$$
\mathbb E\left[
\int_0^t\nabla\varphi^\top G\,dW
\right]=0
$$

因此：

$$
\frac{d}{dt}\mathbb E[\varphi(X_t)]
=
\mathbb E
\left[
f\cdot\nabla\varphi+
\frac12\operatorname{Tr}(Q\nabla^2\varphi)
\right]
$$

写成积分：

$$
\frac{d}{dt}\int\varphi(x)p_t(x)dx
=
\int
\left[
f\cdot\nabla\varphi+
\frac12\sum_{i,j}Q_{ij}\partial_{ij}\varphi
\right]p_tdx
$$

对空间变量分部积分：

$$
\int f\cdot\nabla\varphi\,p_tdx
=
-\int\varphi\nabla\cdot(fp_t)dx
$$

二阶项进行两次分部积分：

$$
\int Q_{ij}p_t\partial_{ij}\varphi\,dx
=
\int\varphi\partial_{ij}(Q_{ij}p_t)dx
$$

因此：

$$
\boxed{
\partial_tp_t=
-\nabla\cdot(fp_t)+
\frac12\sum_{i,j}
\partial_{ij}(Q_{ij}p_t)
}
\tag{C.15}
$$

这就是 Fokker–Planck 方程。

---


## C.3 SDE 时间反转


### C.3.1 为什么反向 SDE 不是简单改变 $dt$ 的符号？

考虑：

$$
dX_t=f(t,X_t)dt+g(t)dW_t
$$

如果希望从：

$$
X_T
$$

反向生成：

$$
X_0
$$

不能简单写成：

$$
dX_t=-fdt+gdW_t
$$

因为反向过程的漂移不仅取决于原始漂移，还取决于当前概率密度。

直观来说，反向生成需要知道：

**哪些状态在当前时刻具有更高的概率，以及这些状态可能来自哪里。**

这一信息由 Score 提供：

$$
s(t,x)=\nabla\log p_t(x)
$$


<a id="app-c-3-2"></a>

### C.3.2 时间反转公式

考虑：

$$
\boxed{
dX_t=f(t,X_t)dt+g(t)dW_t
}
\tag{C.16}
$$

其中：

$$
g(t)>0
$$

不依赖状态。

定义递增的反向时间：

$$
u=T-t
$$

以及：

$$
R_u=X_{T-u}
$$

在适当正则性条件下，反向过程满足：

$$
\boxed{
dR_u=
\left[
-f(T-u,R_u)+
g^2(T-u)s(T-u,R_u)
\right]du
+
g(T-u)d\overline W_u
}
\tag{C.17}
$$

其中：

$$
s(t,x)=\nabla_x\log p_t(x)
$$

这是 Score-Based Diffusion 反向 SDE 的基础公式。


若使用原时间递减的记法，同一时间反演公式写为：

$$
dX_t=[f(t,X_t)-g^2(t)s(t,X_t)]dt+g(t)d\overleftarrow W_t,
\qquad t:T\to0.
$$

特别地，取 $f=b+\varepsilon s$、$g^2=2\varepsilon$，递增反向时间下的漂移为 $-b+\varepsilon s$，原时间递减记法下为 $b-\varepsilon s$。后向 Itô 积分不能按普通递增时间的布朗增量解释。


### C.3.3 为什么出现 $g^2s$？


由[附录 D.3.1](#app-d-3)，前向 SDE 的概率流速度为 $b=f-\tfrac12g^2s$。

时间反转后，概率流速度改变符号：

$$
b_{\mathrm{reverse}}=-b
$$

同时，反向 SDE 的扩散系数仍为 $g$。

因此：

$$
b_{\mathrm{reverse}}
=
f_{\mathrm{reverse}}-\frac12g^2s_{\mathrm{reverse}}
$$

这里：

$$
s_{\mathrm{reverse}}(u,x)=s(T-u,x)
$$

所以：

$$
-b=
f_{\mathrm{reverse}}-\frac12g^2s
$$

代入：

$$
b=f-\frac12g^2s
$$

得到：

$$
-f+\frac12g^2s
=
f_{\mathrm{reverse}}-\frac12g^2s
$$

因此：

$$
\boxed{
f_{\mathrm{reverse}}=-f+g^2s
}
\tag{C.19}
$$

这解释了反向 SDE 中 Score 项的系数为什么是：

$$
g^2
$$

而不是：

$$
\frac12g^2
$$


### C.3.4 一般状态相关扩散的补充

如果：

$$
Q(t,x)=G(t,x)G^\top(t,x)
$$

依赖状态，那么时间反演公式还需要包含扩散矩阵的散度项。

在适当条件下：

$$
\boxed{
f_{\mathrm{reverse},i}
=
-f_i+
\frac1p
\sum_j\partial_{x_j}(Q_{ij}p)
}
\tag{C.20}
$$

展开：

$$
\boxed{
f_{\mathrm{reverse},i}
=
-f_i+
\sum_j\partial_{x_j}Q_{ij}
+
\sum_jQ_{ij}\partial_{x_j}\log p
}
\tag{C.21}
$$

因此，只有当 $Q$ 不依赖状态时，才可以直接简化为：

$$
f_{\mathrm{reverse}}=-f+Qs
$$

这也是正文主要采用时间相关、状态无关扩散系数的原因。

---


<a id="app-c-4"></a>

## C.4 Girsanov 定理


### C.4.1 为什么需要 Girsanov 定理？

假设两个 SDE：

$$
dX_t=f(t,X_t)dt+g(t)dW_t
$$

$$
dY_t=\widehat f(t,Y_t)dt+g(t)dW_t
$$

具有相同扩散系数，但漂移不同。

我们希望比较它们的路径空间概率分布：

$$
\mathbb P,\qquad\widehat{\mathbb P}
$$

Girsanov 定理提供了通过漂移变化描述概率测度变化的方法。


### C.4.2 漂移差异的归一化

设：

$$
\boxed{
u(t,x)=g^{-1}(t)[f(t,x)-\widehat f(t,x)]
}
\tag{C.22}
$$

这里为简化推导，假设 $g(t)>0$ 为标量。

在适当绝对连续性与可积性条件下，可以构造路径测度之间的 Radon–Nikodym 导数。

形式上：

$$
\boxed{
\log\frac{d\mathbb P}{d\widehat{\mathbb P}}
=
\int_0^T u(t,X_t)\cdot dW_t^{\mathbb P}
+
\frac12\int_0^T\|u(t,X_t)\|^2dt
}
\tag{C.23}
$$

这里的布朗运动和符号约定取决于所选择的参考测度；式 (C.23) 使用真实过程测度 $\mathbb P$ 下的表示。


### C.4.3 路径空间 KL 的推导

根据定义：

$$
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\mathbb E_{\mathbb P}
\left[
\log\frac{d\mathbb P}{d\widehat{\mathbb P}}
\right]
$$

代入：

$$
\begin{aligned}
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
={}&
\mathbb E_{\mathbb P}
\int_0^T u\cdot dW_t^{\mathbb P}\\
&+
\frac12
\mathbb E_{\mathbb P}
\int_0^T\|u\|^2dt
\end{aligned}
$$

由于 Itô 积分期望为零：

$$
\mathbb E_{\mathbb P}
\int_0^T u\cdot dW_t^{\mathbb P}=0
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\frac12
\mathbb E_{\mathbb P}
\int_0^T
\left\|
g^{-1}(f-\widehat f)
\right\|^2dt
}
\tag{C.24}
$$

这里假设两个过程初始分布相同；若初始分布不同，还需要加入相应的初始 KL 项。


### C.4.4 应用于随机插值 SDE

考虑：

$$
dX_t=
[b+\varepsilon s]dt+
\sqrt{2\varepsilon}dW_t
$$

以及：

$$
dY_t=
[\widehat b+\varepsilon\widehat s]dt+
\sqrt{2\varepsilon}dW_t
$$

定义：

$$
e_b=\widehat b-b
$$

$$
e_s=\widehat s-s
$$

则：

$$
f-\widehat f=-(e_b+\varepsilon e_s)
$$

同时：

$$
g=\sqrt{2\varepsilon}
$$

因此：

$$
g^{-1}(f-\widehat f)
=
-\frac{e_b+\varepsilon e_s}{\sqrt{2\varepsilon}}
$$

代入：

$$
\begin{aligned}
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
&=
\frac12
\mathbb E_{\mathbb P}
\int_0^1
\frac{\|e_b+\varepsilon e_s\|^2}{2\varepsilon}dt
\end{aligned}
$$

最终：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\frac1{4\varepsilon}
\mathbb E_{\mathbb P}
\int_0^1
\|e_b+\varepsilon e_s\|^2dt
}
\tag{C.25}
$$

其中假设：

$$
\varepsilon>0
$$

为常数，并满足 Girsanov 定理的适用条件。

再利用数据处理不等式：

$$
\boxed{
D_{\mathrm{KL}}(p_1\|\widehat p_1)
\le
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
}
\tag{C.26}
$$

因此，速度场和 Score 的预测误差可以用于控制最终生成分布的 KL。

---


## C.5 Feynman–Kac 公式


### C.5.1 Feynman–Kac 解决什么问题？

Feynman–Kac 公式建立了：

$$
\boxed{
\text{偏微分方程的解}
\Longleftrightarrow
\text{随机过程上的期望}
}
$$

之间的联系。

它可以用于：

- 求解扩散型 PDE。
- 表示条件期望。
- 建立随机控制与 PDE 的关系。
- 研究扩散模型的概率密度和 likelihood。


### C.5.2 一般形式

考虑 SDE：

$$
dX_t=f(t,X_t)dt+G(t,X_t)dW_t
$$

其生成元为：

$$
\boxed{
\mathcal L_t\varphi
=
f\cdot\nabla\varphi+
\frac12\operatorname{Tr}(GG^\top\nabla^2\varphi)
}
\tag{C.27}
$$

考虑 PDE：

$$
\boxed{
\partial_tu+
\mathcal L_tu-
V(t,x)u=0
}
\tag{C.28}
$$

终端条件：

$$
u(T,x)=h(x)
$$

则在适当条件下：

$$
\boxed{
u(t,x)=
\mathbb E\left[
h(X_T)
\exp\left(
-\int_t^T V(r,X_r)dr
\right)
\middle|X_t=x
\right]
}
\tag{C.29}
$$


### C.5.3 推导思路

定义：

$$
M_r=
\exp\left(
-\int_t^rV(q,X_q)dq
\right)u(r,X_r)
$$

对 $M_r$ 使用 Itô 乘积公式。

首先：

$$
d\exp\left(-\int_t^rVdq\right)
=
-V(r,X_r)
\exp\left(-\int_t^rVdq\right)dr
$$

另一方面：

$$
du(r,X_r)
=
[\partial_ru+\mathcal L_ru]dr+
\nabla u^\top GdW_r
$$

因此：

$$
\begin{aligned}
dM_r
={}&
e^{-\int_t^rVdq}
[\partial_ru+\mathcal L_ru-Vu]dr\\
&+
e^{-\int_t^rVdq}\nabla u^\top GdW_r
\end{aligned}
$$

由于 PDE 满足：

$$
\partial_ru+\mathcal L_ru-Vu=0
$$

所以漂移项消失：

$$
dM_r=
e^{-\int_t^rVdq}\nabla u^\top GdW_r
$$

在适当可积性条件下，$M_r$ 是鞅。

因此：

$$
\mathbb E[M_T\mid X_t=x]=M_t=u(t,x)
$$

代入：

$$
u(T,X_T)=h(X_T)
$$

得到：

$$
\boxed{
u(t,x)=
\mathbb E\left[
h(X_T)e^{-\int_t^TV(r,X_r)dr}
\middle|X_t=x
\right]
}
$$

这就完成了 Feynman–Kac 公式的推导。


### C.5.4 与生成模型 likelihood 的联系

对于 ODE，密度变化可以直接通过散度积分计算：

$$
\log p_1(Y_1)
=
\log p_0(Y_0)
-
\int_0^1\nabla\cdot b(t,Y_t)dt
$$

对于 SDE，概率密度满足二阶 Fokker–Planck 方程。

因此，不能直接沿单条随机轨迹使用完全相同的 ODE likelihood 公式。

Feynman–Kac 公式提供了一种将相应线性 PDE 的解表示为随机过程期望的方法。

但具体 likelihood 表达式需要明确所使用的 PDE、时间方向和边界条件，不能直接将 ODE 的散度公式照搬到 SDE 上。

---


# 附录 D：生成模型相关数学知识


## D.1 Score Function 与 Fisher Identity


### D.1.1 Score Function

给定概率密度：

$$
p(x)
$$

定义：

$$
\boxed{
s(x)=\nabla_x\log p(x)
}
\tag{D.1}
$$

例如：

$$
p(x)=\mathcal N(x;\mu,\sigma^2I)
$$

则：

$$
\log p(x)
=
-\frac d2\log(2\pi\sigma^2)
-\frac{\|x-\mu\|^2}{2\sigma^2}
$$

求梯度：

$$
\boxed{
s(x)=-\frac{x-\mu}{\sigma^2}
}
\tag{D.2}
$$

因此，Score 指向高斯分布的均值方向。


### D.1.2 Fisher Identity

考虑隐变量模型：

$$
p(x,z)=p(x\mid z)p(z)
$$

边缘分布：

$$
p(x)=\int p(x,z)dz
$$

计算：

$$
\nabla_x\log p(x)
=
\frac{\nabla_xp(x)}{p(x)}
$$

代入：

$$
p(x)=\int p(x,z)dz
$$

得到：

$$
\nabla_xp(x)
=
\int\nabla_xp(x,z)dz
$$

利用：

$$
\nabla_xp(x,z)
=
p(x,z)\nabla_x\log p(x,z)
$$

因此：

$$
\begin{aligned}
\nabla_x\log p(x)
&=
\int
\frac{p(x,z)}{p(x)}
\nabla_x\log p(x,z)dz\\
&=
\int
p(z\mid x)\nabla_x\log p(x,z)dz
\end{aligned}
$$

由于：

$$
p(x,z)=p(x\mid z)p(z)
$$

且先验 $p(z)$ 不依赖 $x$：

$$
\nabla_x\log p(x,z)
=
\nabla_x\log p(x\mid z)
$$

因此：

$$
\boxed{
\nabla_x\log p(x)
=
\mathbb E[
\nabla_x\log p(x\mid Z)
\mid X=x
]
}
\tag{D.3}
$$

这就是本附录使用的 Fisher Identity。


<a id="app-d-1-3"></a>

### D.1.3 应用于随机插值

考虑：

$$
X_t=aZ+cD
$$

条件分布：

$$
p_t(x\mid D)
=
\mathcal N(x;cD,a^2I)
$$

其条件 Score：

$$
\nabla_x\log p_t(x\mid D)
=
-\frac{x-cD}{a^2}
$$

由于：

$$
x-cD=aZ
$$

所以：

$$
\nabla_x\log p_t(x\mid D)
=
-\frac Za
$$

使用 Fisher Identity：

$$
s(t,x)
=
\mathbb E\left[
-\frac Za
\middle|X_t=x
\right]
$$

最终：

$$
\boxed{
s(t,x)=
-\frac{\mathbb E[Z\mid X_t=x]}{a(t)}
}
\tag{D.4}
$$

这就是噪声预测恢复 Score 的数学依据。

---


一般随机插值也使用同一证明：将潜变量换为端点对 $(X_0,X_1)$，把条件均值 $cD$ 换为 $I(t,X_0,X_1)$，把标准差 $a$ 换为 $\gamma(t)>0$。于是：

$$
\nabla_x\log p_t(x\mid X_0,X_1)
=-\frac{x-I(t,X_0,X_1)}{\gamma^2(t)}
=-\frac{Z}{\gamma(t)},\qquad
s(t,x)=-\frac{\mathbb E[Z\mid X_t=x]}{\gamma(t)}.
$$

这里的条件 Score 与边缘 Score 必须区分；后者来自前者的后验平均。交换求导与积分需要相应正则性条件，噪声系数为零的端点不能直接使用上述除法。


<a id="app-d-2"></a>

## D.2 Tweedie Formula 与 Denoising Score Matching


### D.2.1 Tweedie Formula

考虑最基本的高斯加噪模型：

$$
\boxed{
X=D+\sigma Z
}
\tag{D.5}
$$

其中：

$$
D\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I_d)
$$

定义去噪器：

$$
m(x)=\mathbb E[D\mid X=x]
$$

根据：

$$
X=D+\sigma Z
$$

有：

$$
x=m(x)+\sigma\mathbb E[Z\mid X=x]
$$

因此：

$$
m(x)=x-\sigma\eta(x)
$$

而：

$$
s(x)=-\frac{\eta(x)}{\sigma}
$$

所以：

$$
\eta(x)=-\sigma s(x)
$$

代入：

$$
m(x)=x+\sigma^2s(x)
$$

最终：

$$
\boxed{
\mathbb E[D\mid X=x]
=
x+\sigma^2\nabla_x\log p_X(x)
}
\tag{D.6}
$$

这就是 Tweedie Formula。


### D.2.2 一般线性高斯形式

考虑：

$$
X_t=a(t)Z+c(t)D
$$

则：

$$
x=a\eta+cm
$$

以及：

$$
\eta=-as
$$

因此：

$$
x=-a^2s+cm
$$

解得：

$$
\boxed{
m(t,x)=
\frac{x+a^2(t)s(t,x)}{c(t)}
}
\tag{D.7}
$$

这就是正文使用的广义 Tweedie 公式。


### D.2.3 Denoising Score Matching

考虑：

$$
X_t=D+\sigma Z
$$

条件 Score 为：

$$
\nabla_x\log p(x\mid D)
=
-\frac{x-D}{\sigma^2}
$$

定义 Score 网络：

$$
s_\theta(x)
$$

使用损失：

$$
\boxed{
\mathcal L_{\mathrm{DSM}}
=
\mathbb E_{D,X\mid D}
\left[
\left\|
s_\theta(X)+\frac{X-D}{\sigma^2}
\right\|^2
\right]
}
\tag{D.8}
$$

由于：

$$
X-D=\sigma Z
$$

所以：

$$
\mathcal L_{\mathrm{DSM}}
=
\mathbb E
\left[
\left\|
s_\theta(X)+\frac Z\sigma
\right\|^2
\right]
$$

根据条件期望的 MSE 最优性：

$$
s_\theta^*(x)
=
\mathbb E\left[
-\frac Z\sigma
\middle|X=x
\right]
$$

因此：

$$
\boxed{
s_\theta^*(x)=\nabla_x\log p_X(x)
}
\tag{D.9}
$$

这说明 Denoising Score Matching 可以通过已知的条件高斯噪声，学习未知边缘分布的 Score。

---


<a id="app-d-3"></a>

## D.3 Probability Flow ODE


### D.3.1 从一般扩散 SDE 出发

考虑：

$$
\boxed{
dX_t=f(t,X_t)dt+g(t)dW_t
}
\tag{D.10}
$$

其 Fokker–Planck 方程为：

$$
\partial_tp=
-\nabla\cdot(fp)+
\frac12g^2\Delta p
$$

使用：

$$
\Delta p=\nabla\cdot(ps)
$$

得到：

$$
\partial_tp=
-\nabla\cdot(fp)+
\frac12g^2\nabla\cdot(ps)
$$

合并：

$$
\partial_tp=
-\nabla\cdot
\left[
p\left(f-\frac12g^2s\right)
\right]
$$

因此，定义：

$$
\boxed{
b(t,x)=
f(t,x)-\frac12g^2(t)s(t,x)
}
\tag{D.11}
$$

就有：

$$
\partial_tp+\nabla\cdot(pb)=0
$$

所以：

$$
\boxed{
dY_t=
\left[
f(t,Y_t)-\frac12g^2(t)s(t,Y_t)
\right]dt
}
\tag{D.12}
$$

与原 SDE 具有相同的边缘概率密度。

这就是 Probability Flow ODE。


### D.3.2 与随机插值统一公式的关系

随机插值从速度场 $b$ 出发，构造：

$$
f=b+\varepsilon s
$$

并选择：

$$
g^2=2\varepsilon
$$

代入 Probability Flow ODE：

$$
f-\frac12g^2s
=
b+\varepsilon s-\varepsilon s
$$

因此：

$$
\boxed{
f-\frac12g^2s=b
}
\tag{D.13}
$$

这说明：

随机插值构造 SDE，与从 SDE 恢复 Probability Flow ODE，是同一个数学关系的两个方向。

---


## D.4 Fisher Divergence


### D.4.1 定义

给定两个概率密度：

$$
p(x),\qquad q(x)
$$

其 Score 分别为：

$$
s_p(x)=\nabla\log p(x)
$$

$$
s_q(x)=\nabla\log q(x)
$$

Fisher Divergence 定义为：

$$
\boxed{
D_F(p\|q)
=
\frac12
\mathbb E_{X\sim p}
\left[
\|s_p(X)-s_q(X)\|^2
\right]
}
\tag{D.14}
$$

不同文献可能省略前面的：

$$
\frac12
$$

但本质相同。

它衡量的是两个概率分布的 Score 差异。


### D.4.2 与 KL 散度的区别

KL 散度：

$$
D_{\mathrm{KL}}(p\|q)
=
\mathbb E_p
\left[
\log\frac pq
\right]
$$

比较概率密度本身。

Fisher Divergence：

$$
D_F(p\|q)
=
\frac12\mathbb E_p
\|\nabla\log p-\nabla\log q\|^2
$$

比较对数概率密度的梯度。

因此：

$$
\boxed{
\text{KL 比较密度值}
}
$$

$$
\boxed{
\text{Fisher Divergence 比较 Score}
}
$$


### D.4.3 为什么 Score Matching 可以不需要归一化常数？

假设模型：

$$
q_\theta(x)=\frac{\widetilde q_\theta(x)}{Z_\theta}
$$

其中：

$$
Z_\theta=\int\widetilde q_\theta(x)dx
$$

可能难以计算。

但：

$$
\log q_\theta(x)
=
\log\widetilde q_\theta(x)-\log Z_\theta
$$

对 $x$ 求梯度：

$$
\nabla_x\log q_\theta(x)
=
\nabla_x\log\widetilde q_\theta(x)
$$

因为：

$$
\nabla_x\log Z_\theta=0
$$

所以：

$$
\boxed{
s_\theta(x)=
\nabla_x\log\widetilde q_\theta(x)
}
\tag{D.15}
$$

不需要知道归一化常数。

这也是 Score-Based Generative Models 的一个重要数学优势。


### D.4.4 Score Matching 的分部积分形式

考虑：

$$
\mathcal L(\theta)
=
\frac12
\mathbb E_p
\|s_\theta-s_p\|^2
$$

展开：

$$
\begin{aligned}
\mathcal L(\theta)
={}&
\frac12\mathbb E_p\|s_\theta\|^2\\
&-\mathbb E_p[s_\theta\cdot s_p]\\
&+\frac12\mathbb E_p\|s_p\|^2
\end{aligned}
$$

最后一项与 $\theta$ 无关。

对于交叉项：

$$
\mathbb E_p[s_\theta\cdot s_p]
=
\int p\,s_\theta\cdot\nabla\log p\,dx
$$

由于：

$$
p\nabla\log p=\nabla p
$$

因此：

$$
\mathbb E_p[s_\theta\cdot s_p]
=
\int s_\theta\cdot\nabla p\,dx
$$

通过分部积分：

$$
\int s_\theta\cdot\nabla p\,dx
=
-\int p\nabla\cdot s_\theta\,dx
$$

所以：

$$
\boxed{
\mathcal L(\theta)
=
\mathbb E_p
\left[
\frac12\|s_\theta(x)\|^2+
\nabla\cdot s_\theta(x)
\right]
+\text{常数}
}
\tag{D.16}
$$

这就是经典 Score Matching 的目标形式。

---


## D.5 最优传输与 Schrödinger Bridge


### D.5.1 静态最优传输

给定两个分布：

$$
p_0,\qquad p_1
$$

定义所有满足指定边缘分布的耦合集合：

$$
\Pi(p_0,p_1)
$$

最优传输问题为：

$$
\boxed{
\min_{\pi\in\Pi(p_0,p_1)}
\int c(x_0,x_1)\,\pi(dx_0,dx_1)
}
\tag{D.17}
$$

其中：

$$
c(x_0,x_1)
$$

是传输代价。

对于二次代价：

$$
c(x_0,x_1)=\|x_1-x_0\|^2
$$

得到 Wasserstein-2 距离：

$$
\boxed{
W_2^2(p_0,p_1)
=
\inf_{\pi\in\Pi(p_0,p_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{D.18}
$$


### D.5.2 动态最优传输

Benamou–Brenier 形式为：

$$
\boxed{
\min_{p_t,v_t}
\int_0^1\int
\|v_t(x)\|^2p_t(x)\,dxdt
}
\tag{D.19}
$$

约束：

$$
\partial_tp_t+\nabla\cdot(p_tv_t)=0
$$

以及：

$$
p_{t=0}=p_0,\qquad p_{t=1}=p_1
$$

在标准归一化下，最优值为：

$$
W_2^2(p_0,p_1)
$$

如果动能中使用：

$$
\frac12\|v\|^2
$$

则最优值相应为：

$$
\frac12W_2^2(p_0,p_1)
$$

因此，最优传输可以理解为：

**在所有满足端点分布约束的概率流动中，寻找总动能最小的路径。**


### D.5.3 Schrödinger Bridge 的定义

最优传输通常考虑确定性传输代价。

Schrödinger Bridge 则引入参考随机过程。

设：

$$
\mathbb R
$$

表示参考路径空间概率测度。

希望寻找：

$$
\mathbb P
$$

使其满足：

$$
P_0=p_0,\qquad P_1=p_1
$$

并最小化：

$$
\boxed{
\min_{\mathbb P:\,P_0=p_0,\ P_1=p_1}
D_{\mathrm{KL}}(\mathbb P\|\mathbb R)
}
\tag{D.20}
$$

这就是 Schrödinger Bridge 问题。

其含义是：

**在所有能够连接指定端点分布的随机过程中，寻找与参考随机过程最接近的过程。**


### D.5.4 为什么它与随机控制有关？

假设参考过程为：

$$
\boxed{
dX_t=f_0(t,X_t)dt+\sqrt{2\varepsilon}dW_t
}
\tag{D.21}
$$

受控过程为：

$$
\boxed{
dX_t=
[f_0(t,X_t)+u(t,X_t)]dt+
\sqrt{2\varepsilon}dW_t
}
\tag{D.22}
$$


在相同初始分布和 Girsanov 定理适用条件下，直接应用[附录 C.4.3](#app-c-4)（漂移差取为控制场），得到：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P^u\|\mathbb R)
=
\frac1{4\varepsilon}
\mathbb E_{\mathbb P^u}
\int_0^1\|u(t,X_t)\|^2dt
}
\tag{D.23}
$$

因此，Schrödinger Bridge 可以写成：

$$
\boxed{
\min_u
\mathbb E_{\mathbb P^u}
\int_0^1\frac{\|u(t,X_t)\|^2}{4\varepsilon}dt
}
\tag{D.24}
$$

约束：

$$
X_0\sim p_0,\qquad X_1\sim p_1
$$

这就是一个随机最优控制问题。


### D.5.5 与最优传输的联系

最优传输：

$$
\min\int_0^1\int
\frac12\|v\|^2p\,dxdt
$$

约束：

$$
\partial_tp+\nabla\cdot(pv)=0
$$

Schrödinger Bridge：

$$
\min\int_0^1\int
\frac12\|u\|^2p\,dxdt
$$

在适当归一化后，约束为带扩散项的 Fokker–Planck 方程：

$$
\partial_tp=
-\nabla\cdot(pu)+\varepsilon\Delta p
$$

因此：

$$
\boxed{
\text{最优传输：确定性概率流}
}
$$

$$
\boxed{
\text{Schrödinger Bridge：随机概率流}
}
$$

在适当条件下，当参考扩散强度：

$$
\varepsilon\to0
$$

时，Schrödinger Bridge 与最优传输之间存在小噪声极限联系。

但需要注意，这并不意味着任意随机插值都是 Schrödinger Bridge。

随机插值只要求构造满足端点分布的概率路径，而 Schrödinger Bridge 还要求满足路径空间 KL 最优性。

---


## D.6 变分法与 Euler–Lagrange 方程


### D.6.1 为什么需要变分法？

普通优化问题寻找：

$$
x^*=\arg\min_xf(x)
$$

变分法则寻找一个函数：

$$
x^*(t)
$$

使某个泛函最小。

例如：

$$
\boxed{
J[x]=
\int_0^1
L(t,x(t),\dot x(t))dt
}
\tag{D.25}
$$

其中：

$$
L
$$

称为 Lagrangian。

我们希望寻找：

$$
x^*(t)=\arg\min_xJ[x]
$$

并满足指定端点条件。


### D.6.2 引入函数扰动

考虑：

$$
x_\delta(t)=x(t)+\delta h(t)
$$

其中：

$$
h(0)=h(1)=0
$$

表示端点固定。

定义：

$$
F(\delta)=J[x_\delta]
$$

最优条件为：

$$
F'(0)=0
$$

展开：

$$
F(\delta)
=
\int_0^1
L(t,x+\delta h,\dot x+\delta\dot h)dt
$$

对 $\delta$ 求导：

$$
\begin{aligned}
F'(0)
=
\int_0^1
\left[
\frac{\partial L}{\partial x}\cdot h+
\frac{\partial L}{\partial\dot x}\cdot\dot h
\right]dt
\end{aligned}
\tag{D.26}
$$


### D.6.3 分部积分

对第二项：

$$
\int_0^1
\frac{\partial L}{\partial\dot x}\cdot\dot h\,dt
$$

进行分部积分：

$$
\begin{aligned}
&\int_0^1
\frac{\partial L}{\partial\dot x}\cdot\dot h\,dt\\
&=
\left[
\frac{\partial L}{\partial\dot x}\cdot h
\right]_0^1
-
\int_0^1
\frac{d}{dt}
\left(
\frac{\partial L}{\partial\dot x}
\right)\cdot h\,dt
\end{aligned}
$$

由于：

$$
h(0)=h(1)=0
$$

边界项为零。

因此：

$$
F'(0)
=
\int_0^1
\left[
\frac{\partial L}{\partial x}
-
\frac{d}{dt}
\left(
\frac{\partial L}{\partial\dot x}
\right)
\right]\cdot h\,dt
$$

由于：

$$
h(t)
$$

可以任意选择，因此：

$$
\boxed{
\frac{\partial L}{\partial x}
-
\frac{d}{dt}
\left(
\frac{\partial L}{\partial\dot x}
\right)=0
}
\tag{D.27}
$$

这就是 Euler–Lagrange 方程。


### D.6.4 例子：为什么最小动能路径是直线？

考虑：

$$
\boxed{
J[x]=
\int_0^1
\frac12\|\dot x(t)\|^2dt
}
\tag{D.28}
$$

其中：

$$
L=\frac12\|\dot x\|^2
$$

计算：

$$
\frac{\partial L}{\partial x}=0
$$

$$
\frac{\partial L}{\partial\dot x}=\dot x
$$

代入 Euler–Lagrange 方程：

$$
0-\frac{d}{dt}\dot x=0
$$

因此：

$$
\boxed{
\ddot x(t)=0
}
\tag{D.29}
$$

积分：

$$
\dot x(t)=v
$$

再次积分：

$$
x(t)=x_0+vt
$$

利用：

$$
x(1)=x_1
$$

得到：

$$
v=x_1-x_0
$$

因此：

$$
\boxed{
x(t)=(1-t)x_0+tx_1
}
\tag{D.30}
$$

这说明：

**对于固定端点、固定时间区间和二次动能代价，最优路径是匀速直线。**

但这只是单条路径的最优性。

对于概率分布之间的最优传输，还需要优化端点耦合。

因此，单条条件插值为直线，并不意味着整个生成过程自动达到最优传输。

---


# 附录 E：符号表与公式索引


## E.1 全文符号定义


### E.1.1 概率分布与随机变量

| 符号 | 含义 |
|---|---|
| $X_0$ | 初始端点随机变量 |
| $X_1$ | 目标端点随机变量 |
| $D$ | 真实数据随机变量，通常等于 $X_1$ |
| $Z$ | 标准高斯随机变量 |
| $X_t$ | 随机插值变量 |
| $Y_t$ | 生成 ODE/SDE 的状态 |
| $p_t(x)$ | 时间 $t$ 的边缘概率密度 |
| $p(x\mid y)$ | 条件概率密度 |
| $\nu$ | 端点耦合分布 |
| $\Pi(p_0,p_1)$ | 所有满足端点边缘分布的耦合集合 |
| $\mathbb P$ | 路径空间概率测度 |
| $\mathbb R$ | 参考随机过程的路径空间概率测度 |


### E.1.2 随机插值与生成模型

| 符号 | 含义 |
|---|---|
| $I(t,X_0,X_1)$ | 插值函数 |
| $\gamma(t)$ | 辅助高斯插值噪声系数 |
| $a(t)$ | 单侧线性插值中的噪声系数 |
| $c(t)$ | 单侧线性插值中的数据系数 |
| $\eta(t,x)$ | 条件噪声均值 $\mathbb E[Z\mid X_t=x]$ |
| $m(t,x)$ | 条件数据均值 $\mathbb E[D\mid X_t=x]$ |
| $b(t,x)$ | 边缘概率流速度场 |
| $s(t,x)$ | Score，$\nabla_x\log p_t(x)$ |
| $\varepsilon(t)$ | 生成 SDE 的采样扩散强度 |
| $W_t$ | 标准布朗运动 |
| $\overleftarrow W_t$ | 反向时间 Itô 记法中的布朗噪声 |
| $g(t)$ | SDE 扩散系数 |
| $\beta(t)$ | VP-SDE 噪声日程 |
| $\sigma(t)$ | VE-SDE 噪声标准差 |


### E.1.3 数学算子

| 符号 | 含义 |
|---|---|
| $\mathbb E[\cdot]$ | 数学期望 |
| $\operatorname{Var}$ | 方差 |
| $\operatorname{Cov}$ | 协方差 |
| $\nabla$ | 梯度 |
| $\nabla\cdot$ | 散度 |
| $\Delta$ | 拉普拉斯算子 |
| $\nabla^2$ | Hessian |
| $\operatorname{Tr}$ | 矩阵迹 |
| $D_{\mathrm{KL}}$ | KL 散度 |
| $D_F$ | Fisher Divergence |
| $W_2$ | 二阶 Wasserstein 距离 |
| $\overset d=$ | 概率分布相同 |
| $\perp$ | 随机变量独立 |
| $\dot a(t)$ | 对时间求导 |

---


## E.2 主要公式及适用条件


### E.2.1 随机插值

一般形式：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{E.1}
$$

适用条件：

- 端点耦合已定义。
- 插值函数满足相应端点条件。
- 辅助高斯噪声独立于端点变量。
- 所需时间导数与期望存在。


### E.2.2 边缘速度场

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{E.2}
$$

对应连续性方程：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
\tag{E.3}
$$


### E.2.3 Score

$$
\boxed{
s(t,x)=\nabla_x\log p_t(x)
}
\tag{E.4}
$$

对于：

$$
X_t=a(t)Z+c(t)D
$$

有：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
\tag{E.5}
$$

适用条件：

$$
a(t)>0
$$

并满足相应的密度可微性条件。


### E.2.4 单侧线性插值的速度场

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta(t,x)
}
\tag{E.6}
$$

适用条件：

$$
c(t)\neq0
$$

并且所需条件期望存在。


### E.2.5 Probability Flow ODE

$$
\boxed{
dY_t=b(t,Y_t)dt
}
\tag{E.7}
$$

在初始分布相同、动力学适定且对应连续性方程解唯一的条件下，具有目标边缘概率路径。


### E.2.6 前向生成 SDE

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{E.8}
$$

适用条件：

- $\varepsilon(t)\ge0$。
- 扩散系数不依赖状态。
- Score 与速度场足够正则。
- SDE 和对应 Fokker–Planck 方程适定。


### E.2.7 SDE 时间反演

对于：

$$
dX_t=f(t,X_t)dt+g(t)dW_t
$$

定义：

$$
u=T-t
$$

则：

$$
\boxed{
dR_u=
[-f(T-u,R_u)+g^2(T-u)s(T-u,R_u)]du+
g(T-u)d\overline W_u
}
\tag{E.9}
$$

适用条件：

- 扩散系数不依赖状态。
- 存在足够正则的正密度。
- 满足时间反演定理所需条件。


### E.2.8 Girsanov KL

对于相同初始分布、相同非退化扩散系数的 SDE：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\widehat{\mathbb P})
=
\frac12
\mathbb E_{\mathbb P}
\int_0^T
\|G^{-1}(f-\widehat f)\|^2dt
}
\tag{E.10}
$$

这里假设扩散矩阵可逆，并满足相应绝对连续性与可积性条件。


### E.2.9 ODE likelihood

$$
\boxed{
\log p_1(Y_1)
=
\log p_0(Y_0)
-
\int_0^1\nabla\cdot b(t,Y_t)dt
}
\tag{E.11}
$$

适用于具有适当可微性与可逆流映射的确定性 ODE。

---


## E.3 正文与附录公式的对应关系

| 正文主题 | 关键公式 | 数学基础 |
|---|---|---|
| 随机插值定义 | $X_t=I+\gamma Z$ | A.1、A.3 |
| Flow Matching 训练 | $\mathbb E\|v_\theta-\dot X_t\|^2$ | A.2 |
| 噪声预测 | $\mathbb E\|\eta_\theta-Z\|^2$ | A.2、D.1 |
| Score 恢复 | $s=-\eta/a$ | D.1、D.2 |
| 连续性方程 | $\partial_tp+\nabla\cdot(pb)=0$ | B.2 |
| Probability Flow ODE | $dY=b\,dt$ | B.2、D.3 |
| 生成 SDE | $dY=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW$ | B.3 |
| 反向 SDE | 时间反演漂移公式 | C.3 |
| VP-SDE | 线性 SDE 解析解 | A.3、C.2 |
| VE-SDE | $g^2=d\sigma^2/dt$ | C.2 |
| Brownian Bridge | $B_t=W_t-tW_1$ | C.1 |
| KL 误差分析 | 路径空间 KL | A.4、C.4 |
| ODE likelihood | 散度积分 | B.2 |
| SDE likelihood | Feynman–Kac 表示 | C.5 |
| OT-CFM | 最优端点耦合 | D.5 |
| Schrödinger Bridge | 路径空间 KL 最小化 | C.4、D.5 |
| 最优传输速度场 | Hamilton–Jacobi 方程 | B.4、D.6 |
| Rectified Flow | 直线插值与动能 | B.4、D.6 |

---


**公式推导索引：** 单侧线性速度与预测转换见[F.1](#app-f-1)；KL 误差拆分及最优扩散强度见[F.2](#app-f-2)；OU/VP/VE 的解析条件分布见[F.3](#app-f-3)；CondOT 与 VP 的具体代入见[F.4](#app-f-4)；Mirror 的速度公式见[F.5](#app-f-5)。通用证明的具体位置见文首“数学推导导航”。

## E.4 原论文结论与扩展推导的区分

为了避免将论文原始贡献与后续统一整理混淆，需要明确区分两部分。


### E.4.1 原论文的主要理论内容

《Stochastic Interpolants: A Unifying Framework for Flows and Diffusions》的核心研究内容包括：

1. 定义一般随机插值。
2. 推导随机插值的边缘速度场与 Score。
3. 建立对应的连续性方程。
4. 构造共享边缘概率路径的 ODE 和 SDE。
5. 通过监督回归学习速度场与 Score。
6. 分析生成动力学的误差与概率密度估计。
7. 讨论不同随机插值构造及其与已有生成模型的联系。
8. 通过数值实验研究插值方式、训练目标和采样扩散强度的影响。


### E.4.2 本学习笔记的扩展内容

第 9 章和本附录还进行了进一步的统一整理，包括：

- 使用 $a(t),c(t)$ 统一单侧线性高斯插值。
- 通过一个噪声预测器同时恢复 Score 和速度场。
- 将 CondOT、VP、OU、VE 和 Trigonometric Interpolant 逐一代入统一公式。
- 显式区分扩散时间与生成时间。
- 对比固定高斯噪声插值与真实扩散 SDE 的路径分布。
- 分析 VP、VE 和 Trigonometric Interpolant 的端点问题。
- 将 Girsanov、Feynman–Kac、Hamilton–Jacobi 和 Euler–Lagrange 等数学工具整理为独立附录。
- 讨论统一框架在条件轨迹规划中的潜在应用。

这些扩展用于帮助理解论文及其与其他方法的数学联系，**不应全部归属于原论文的独立贡献**。

---


<a id="appendix-f"></a>

# 附录 F：线性插值与生成动力学的公式推导

本附录详细推导单侧线性插值的参数转换、SDE 误差上界，以及 OU、VP、VE、CondOT 和 Mirror 的相关公式。通用数学结论见附录 A～D。不同小节的记号对应如下：单侧插值中的 $(a,c,\eta,m)$ 对应 $(\alpha,\beta,\eta_z,\eta_1)$；F.2 的 $A,B$ 是积分误差，F.1 的 $A,H$ 是时间系数，二者用途不同。

<a id="app-f-1"></a>

## F.1 单侧线性插值：条件均值、速度与预测目标转换

先使用[附录 D.1.3](#app-d-1-3) 的 Score 恒等式与[附录 D.2.2](#app-d-2) 的数据去噪器公式，再进行以下代数运算。


<a id="app-f-1-1"></a>

### F.1.1 从噪声预测器恢复边缘速度场

对应正文第 9.2.4 节。

随机插值为：

$$
X_t=a(t)Z+c(t)D
$$

对时间求导：

$$
\boxed{
\dot X_t=\dot a(t)Z+\dot c(t)D
}
\tag{9.15}
$$

定义边缘速度场：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

代入：

$$
b(t,x)=
\dot a\,\mathbb E[Z\mid X_t=x]
+
\dot c\,\mathbb E[D\mid X_t=x]
$$

即：

$$
b=\dot a\eta+\dot cm
\tag{9.16}
$$

利用：

$$
m=\frac{x-a\eta}{c}
$$

得到：

$$
\begin{aligned}
b
&=
\dot a\eta+
\dot c\frac{x-a\eta}{c}\\
&=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta
\end{aligned}
$$

最终：

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta(t,x)
}

$$

这就是统一的边缘速度场公式。


<a id="app-f-1-2"></a>

### F.1.2 使用 Score 表示速度场与 A、H 系数

对应正文第 9.2.5 节。

由于：

$$
\eta=-as
$$

代入式 (9.17)：

$$
b=
\frac{\dot c}{c}x-
a\left(
\dot a-\frac{a\dot c}{c}
\right)s
$$

展开：

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\frac{a^2\dot c}{c}-a\dot a
\right)s(t,x)
}

$$


<a id="app-f-1-3"></a>

### F.1.3 使用数据预测器恢复 Score 与速度场

对应正文第 4.4.4 节。

可以。

前面使用：

$$
\eta_z=\mathbb E[Z\mid X_t=x]
$$

作为唯一预测器。

但也可以使用：

$$
\eta_1=\mathbb E[X_1\mid X_t=x]
$$

作为唯一预测器。

根据：

$$
x=\alpha\eta_z+\beta\eta_1
$$

当 $\alpha\neq0$ 时：

$$
\eta_z=\frac{x-\beta\eta_1}{\alpha}
\tag{4.43}
$$

代入 Score：

$$
s=-\frac{\eta_z}{\alpha}
$$

得到：

$$
\boxed{
s(t,x)=
-\frac{x-\beta(t)\eta_1(t,x)}{\alpha^2(t)}
}
\tag{4.44}
$$

再代入速度场：

$$
b=\dot\alpha\eta_z+\dot\beta\eta_1
$$

得到：

$$
b=
\dot\alpha\frac{x-\beta\eta_1}{\alpha}
+
\dot\beta\eta_1
$$

整理：

$$
\boxed{
b(t,x)=
\frac{\dot\alpha(t)}{\alpha(t)}x+
\left(
\dot\beta(t)
-\frac{\beta(t)\dot\alpha(t)}{\alpha(t)}
\right)\eta_1(t,x)
}
\tag{4.45}
$$


<a id="app-f-1-4"></a>

### F.1.4 矩阵求逆与速度、噪声、数据预测之间的转换

对应正文第 4.4.5 节。

可以，但需要满足一个非退化条件。

根据：

$$
b=\dot\alpha\eta_z+\dot\beta\eta_1
$$

以及：

$$
x=\alpha\eta_z+\beta\eta_1
$$

我们可以写成矩阵形式：

$$
\begin{pmatrix}
x\\
b
\end{pmatrix}
=
\begin{pmatrix}
\alpha&\beta\\
\dot\alpha&\dot\beta
\end{pmatrix}
\begin{pmatrix}
\eta_z\\
\eta_1
\end{pmatrix}
\tag{4.47}
$$

这里的矩阵系数作用于每个向量分量。

定义行列式：

$$
\boxed{
D(t)=\alpha(t)\dot\beta(t)-\beta(t)\dot\alpha(t)
}
\tag{4.48}
$$

如果：

$$
D(t)\neq0
$$

那么这个矩阵可逆。

根据二阶矩阵求逆公式：

$$
\begin{pmatrix}
\eta_z\\
\eta_1
\end{pmatrix}
=
\frac1D
\begin{pmatrix}
\dot\beta&-\beta\\
-\dot\alpha&\alpha
\end{pmatrix}
\begin{pmatrix}
x\\
b
\end{pmatrix}
$$

因此：

$$
\boxed{
\eta_z(t,x)=
\frac{\dot\beta(t)x-\beta(t)b(t,x)}{D(t)}
}

$$

以及：

$$
\boxed{
\eta_1(t,x)=
\frac{\alpha(t)b(t,x)-\dot\alpha(t)x}{D(t)}
}

$$

再根据：

$$
s=-\frac{\eta_z}{\alpha}
$$

得到：

$$
\boxed{
s(t,x)=
-\frac{\dot\beta(t)x-\beta(t)b(t,x)}
{\alpha(t)D(t)}
}

$$


<a id="app-f-2"></a>

## F.2 SDE 漂移误差的拆分与扩散强度权衡

对应正文第 2.5.5 节。

前面已经证明，随机插值对应的前向 SDE 漂移为：

$$
f=b+\varepsilon s
\tag{2.102}
$$

实际模型使用：

$$
\widehat f=\widehat b+\varepsilon\widehat s
\tag{2.103}
$$

因此：

$$
f-\widehat f
=
(b-\widehat b)
+
\varepsilon(s-\widehat s)
\tag{2.104}
$$

定义：

$$
\delta_b=b-\widehat b
$$

$$
\delta_s=s-\widehat s
$$

则：

$$
f-\widehat f=\delta_b+\varepsilon\delta_s
$$

利用不等式：

$$
\|a+c\|^2\le2\|a\|^2+2\|c\|^2
$$

得到：

$$
\|f-\widehat f\|^2
\le
2\|\delta_b\|^2+
2\varepsilon^2\|\delta_s\|^2
\tag{2.105}
$$

代入式 (2.101)：

$$
\begin{aligned}
D_{\mathrm{KL}}(\rho_1\|q_1)
\le{}&
\frac{1}{2\varepsilon}
\int_0^1\mathbb E_{\rho_t}\|\delta_b\|^2dt\\
&+
\frac{\varepsilon}{2}
\int_0^1\mathbb E_{\rho_t}\|\delta_s\|^2dt
\end{aligned}
\tag{2.106}
$$

定义两个积分误差：

$$
A=
\int_0^1
\mathbb E_{\rho_t}\|b-\widehat b\|^2dt
\tag{2.107}
$$

$$
B=
\int_0^1
\mathbb E_{\rho_t}\|s-\widehat s\|^2dt
\tag{2.108}
$$

得到：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|q_1)
\le
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}

$$

当 $A,B>0$ 时，对右侧求导：

$$
-\frac{A}{2\varepsilon^2}+\frac B2=0
$$

解得：

$$
\boxed{
\varepsilon^*=\sqrt{\frac AB}
}

$$


<a id="app-f-3"></a>

## F.3 OU、VP 与 VE：解析条件边缘分布


<a id="app-f-3-1"></a>

### F.3.1 OU：积分因子的逐步计算

参考过程为 $dY_\tau=-\lambda Y_\tau d\tau+\sqrt{2\lambda}dW_\tau$，$\lambda>0$。该解析解对应正文式 (5.8)。

原方程：

$$
dY_\tau=-\lambda Y_\tau d\tau+\sqrt{2\lambda}\,dW_\tau
$$

两边乘以：

$$
e^{\lambda\tau}
$$

利用 Itô 乘积法则：

$$
d(e^{\lambda\tau}Y_\tau)
=
\lambda e^{\lambda\tau}Y_\tau d\tau
+
e^{\lambda\tau}dY_\tau
$$

代入 SDE：

$$
\begin{aligned}
d(e^{\lambda\tau}Y_\tau)
={}&
\lambda e^{\lambda\tau}Y_\tau d\tau\\
&+
e^{\lambda\tau}
\left(
-\lambda Y_\tau d\tau+\sqrt{2\lambda}\,dW_\tau
\right)
\end{aligned}
$$

漂移项抵消：

$$
d(e^{\lambda\tau}Y_\tau)
=
\sqrt{2\lambda}e^{\lambda\tau}dW_\tau
$$

从 $0$ 积分到 $\tau$：

$$
e^{\lambda\tau}Y_\tau-Y_0
=
\sqrt{2\lambda}\int_0^\tau e^{\lambda r}dW_r
$$

因此：

$$
\boxed{
Y_\tau=
e^{-\lambda\tau}Y_0+
\sqrt{2\lambda}\int_0^\tau e^{-\lambda(\tau-r)}dW_r
}

$$


<a id="app-f-3-2"></a>

### F.3.2 VP：变系数解析解与噪声方差

以下对应第 5.1.3、9.5.1 节。将 F.3.1 的积分因子换成 $m^{-1}(\tau)$，即可处理随时间变化的系数。

令：

$$
B(\tau)=\int_0^\tau\beta_{\mathrm{VP}}(r)dr
\tag{5.13}
$$

定义：

$$
m(\tau)=e^{-\frac12B(\tau)}
\tag{5.14}
$$

则：

$$
\dot m(\tau)
=
-\frac12\beta_{\mathrm{VP}}(\tau)m(\tau)
$$

因此，SDE 的解析解可以写成：

$$
Y_\tau=
m(\tau)Y_0+
m(\tau)
\int_0^\tau
\frac{\sqrt{\beta_{\mathrm{VP}}(r)}}{m(r)}dW_r

$$

这个结果可以使用与 OU 过程相同的积分因子法得到。

定义随机积分项：

$$
N_\tau=
m(\tau)
\int_0^\tau
\frac{\sqrt{\beta_{\mathrm{VP}}(r)}}{m(r)}dW_r
$$

根据 Itô 等距：

$$
\operatorname{Cov}(N_\tau)
=
m^2(\tau)
\int_0^\tau
\frac{\beta_{\mathrm{VP}}(r)}{m^2(r)}dr\,I_d
\tag{5.16}
$$

由于：

$$
m^{-2}(r)=e^{B(r)}
$$

所以：

$$
\frac{d}{dr}e^{B(r)}
=
\beta_{\mathrm{VP}}(r)e^{B(r)}
$$

因此：

$$
\begin{aligned}
\int_0^\tau
\frac{\beta_{\mathrm{VP}}(r)}{m^2(r)}dr
&=
\int_0^\tau
\beta_{\mathrm{VP}}(r)e^{B(r)}dr\\
&=e^{B(\tau)}-1
\end{aligned}
$$

代入：

$$
\begin{aligned}
\operatorname{Cov}(N_\tau)
&=
e^{-B(\tau)}
[e^{B(\tau)}-1]I_d\\
&=
[1-e^{-B(\tau)}]I_d
\end{aligned}
$$

因此：

$$
\boxed{
Y_\tau\mid Y_0=x_1
\sim
\mathcal N\left(
e^{-\frac12B(\tau)}x_1,
[1-e^{-B(\tau)}]I_d
\right)
}

$$
取 $\beta_{\mathrm{VP}}=2\lambda$，即得 OU 的噪声方差 $1-e^{-2\lambda\tau}$。第 9 章的 $\alpha(\tau)$ 对应这里的 $m(\tau)$。


<a id="app-f-3-3"></a>

### F.3.3 VE：随机积分与累计噪声方差

对应正文第 9.6.2 节。

考虑：

$$
dX_\tau=g(\tau)dW_\tau
$$

积分得到：

$$
X_\tau=
D+\int_0^\tau g(r)dW_r
$$

根据 Itô 积分的性质：

$$
\int_0^\tau g(r)dW_r
\sim
\mathcal N\left(
0,
\left[\int_0^\tau g^2(r)dr\right]I_d
\right)
$$

定义：

$$
\boxed{
\sigma^2(\tau)=\int_0^\tau g^2(r)dr
}
\tag{9.84}
$$

因此：

$$
\boxed{
X_\tau\mid D
\sim
\mathcal N(D,\sigma^2(\tau)I_d)
}

$$
由定义还直接得到 $g^2(\tau)=d\sigma^2(\tau)/d\tau$。对于非零最小噪声，初始方差与新增方差的区别见正文第 5.1.6、9.6.7 节。


<a id="app-f-4"></a>

## F.4 具体时间系数的代入计算


<a id="app-f-4-1"></a>

### F.4.1 CondOT：代入时间系数计算速度场

对应正文第 9.4.2 节。

使用统一公式：

$$
b=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta
$$

代入：

$$
a=1-t,\qquad c=t
$$

得到：

$$
\frac{\dot c}{c}=\frac1t
$$

以及：

$$
\begin{aligned}
\dot a-\frac{a\dot c}{c}
&=-1-\frac{1-t}{t}\\
&=-\frac1t
\end{aligned}
$$

因此：

$$
\boxed{
b(t,x)=\frac{x-\eta(t,x)}{t}
}

$$


<a id="app-f-4-2"></a>

### F.4.2 VP：生成时间系数的链式求导

对应正文第 9.5.3 节。

由：

$$
c(t)=\alpha(1-t)
$$

以及：

$$
\frac{d\alpha(\tau)}{d\tau}
=
-\frac12\beta(\tau)\alpha(\tau)
$$

使用链式法则：

$$
\dot c(t)
=
\frac{d\alpha}{d\tau}
\frac{d\tau}{dt}
$$

由于：

$$
\frac{d\tau}{dt}=-1
$$

因此：

$$
\boxed{
\dot c(t)=
\frac12\beta(1-t)c(t)
}

$$

为简洁起见，定义：

$$
\widetilde\beta(t)=\beta(1-t)
$$

于是：

$$
\boxed{
\frac{\dot c}{c}=\frac12\widetilde\beta(t)
}

$$

另一方面：

$$
a^2(t)=1-c^2(t)
$$

对时间求导：

$$
2a\dot a=-2c\dot c
$$

因此：

$$
a\dot a=-c\dot c
$$

代入：

$$
\dot c=\frac12\widetilde\beta c
$$

得到：

$$
\boxed{
a\dot a=
-\frac12\widetilde\beta(t)c^2(t)
}

$$


<a id="app-f-4-3"></a>

### F.4.3 VP：从系数恢复概率流速度场

对应正文第 9.5.4 节。

回顾：

$$
b(t,x)=
\frac{\dot c}{c}x+
\left(
\frac{a^2\dot c}{c}-a\dot a
\right)s(t,x)
$$

第一项：

$$
\frac{\dot c}{c}=\frac12\widetilde\beta
$$

第二项：

$$
\begin{aligned}
\frac{a^2\dot c}{c}-a\dot a
&=
\frac12\widetilde\beta a^2+
\frac12\widetilde\beta c^2\\
&=
\frac12\widetilde\beta(a^2+c^2)
\end{aligned}
$$

由于 VP 满足：

$$
a^2+c^2=1
$$

因此：

$$
\boxed{
\frac{a^2\dot c}{c}-a\dot a
=
\frac12\widetilde\beta
}
\tag{9.74}
$$

最终：

$$
\boxed{
b_{\mathrm{VP}}(t,x)=
\frac12\widetilde\beta(t)x+
\frac12\widetilde\beta(t)s(t,x)
}

$$

这就是 VP-SDE 在噪声到数据时间方向上的 Probability Flow ODE 速度场。


<a id="app-f-4-4"></a>

### F.4.4 VE：由生成时间下的噪声方差恢复速度

对应正文第 9.6.4 节。

回顾：

$$
b(t,x)=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta(t,x)
$$

由于：

$$
c=1,\qquad\dot c=0
$$

因此：

$$
\boxed{
b_{\mathrm{VE}}(t,x)=\dot a(t)\eta(t,x)
}
\tag{9.90}
$$

又因为：

$$
s=-\frac{\eta}{a}
$$

所以：

$$
\eta=-as
$$

得到：

$$
\boxed{
b_{\mathrm{VE}}(t,x)=
-a(t)\dot a(t)s(t,x)
}
\tag{9.91}
$$

注意：

$$
-a\dot a=-\frac12\frac{d[a^2(t)]}{dt}
$$

而：

$$
a^2(t)=\sigma^2(1-t)
$$

因此：

$$
\frac{d[a^2(t)]}{dt}
=
-\frac{d\sigma^2(\tau)}{d\tau}
\bigg|_{\tau=1-t}
$$

所以：

$$
-a\dot a=
\frac12g^2(1-t)
$$

定义：

$$
\widetilde g(t)=g(1-t)
$$

最终：

$$
\boxed{
b_{\mathrm{VE}}(t,x)=
\frac12\widetilde g^2(t)s(t,x)
}

$$

这是 VE 在噪声到数据时间方向上的 Probability Flow ODE 速度场。


<a id="app-f-5"></a>

## F.5 Mirror：只添加高斯噪声时的速度场

对应正文第 3.3.5 节。

如果：

$$
K(t,x)=x
$$

则：

$$
\partial_tK=0
$$

因此：

$$
v(t,x)=0
$$

于是：

$$
\boxed{
b(t,x)=-\gamma(t)\dot\gamma(t)s(t,x)
}

$$

这说明在这种特殊情况下，**边缘速度场完全由 Score 决定**。

例如：

$$
\gamma(t)=\sqrt{a\,t(1-t)}
$$

则：

$$
\gamma^2(t)=a\,t(1-t)
$$

对时间求导：

$$
2\gamma\dot\gamma=a(1-2t)
$$

因此：

$$
\gamma\dot\gamma=\frac a2(1-2t)
$$

代入：

$$
\boxed{
b(t,x)=
-\frac a2(1-2t)s(t,x)
}

$$


---


# 全文最终总结

通过正文九章和数学附录，我们可以将随机插值框架压缩为以下数学关系。

首先，定义概率路径：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
$$

然后，通过条件期望得到边缘速度场：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

通过概率密度得到 Score：

$$
\boxed{
s(t,x)=\nabla_x\log p_t(x)
}
$$

二者分别描述：

- **速度场 $b$**：概率质量如何流动。
- **Score $s$**：当前概率密度在空间中如何变化。

概率路径满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
$$

由此构造：

$$
\boxed{
\text{Probability Flow ODE}:
\quad dY_t=b\,dt
}
$$

以及：

$$
\boxed{
\text{Generative SDE}:
\quad dY_t=(b+\varepsilon s)dt+
\sqrt{2\varepsilon}dW_t
}
$$

对于单侧线性高斯插值：

$$
\boxed{
X_t=a(t)Z+c(t)D
}
$$

进一步有：

$$
\boxed{
s(t,x)=-\frac{\eta(t,x)}{a(t)}
}
$$

$$
\boxed{
b(t,x)=
\frac{\dot c}{c}x+
\left(
\dot a-\frac{a\dot c}{c}
\right)\eta(t,x)
}
$$

其中：

$$
\eta(t,x)=\mathbb E[Z\mid X_t=x]
$$

因此：

$$
\boxed{
\text{一个噪声预测器}
\Longrightarrow
\text{Score 与速度场}
\Longrightarrow
\text{ODE 与 SDE}
}
$$

最后，通过不同的时间系数：

$$
a(t),\qquad c(t)
$$

可以建立与：

$$
\boxed{
\begin{gathered}
\text{Flow Matching}\\
\text{VP / OU Diffusion}\\
\text{VE Diffusion}\\
\text{Trigonometric Interpolant}
\end{gathered}
}
$$

之间的数学联系。

**整套理论最重要的认识是：生成模型可以从概率路径、预测目标和采样动力学三个层面分别理解与设计。它们之间存在严格的数学联系，但这种联系不意味着不同模型的训练效果、数值误差或完整随机路径必然相同。**
